An agent in this competition can lose a fifth of its opening cash on the first night and never find out.

Buy a cow on day 0, place it, feed it, and on day 1 send the hand back with a FEED.
Nothing happens: the hand's inventory was emptied into the shed at midnight, so the FEED found no wheat, the engine accepted the action and changed nothing, and at the end of day 2 the cow is gone.
The next observation shows a pasture.
It looked exactly like that before the cow arrived.

That is the shape of a whole class of losses in this game.
**The observation is a snapshot of state, not a log of events.** Nothing in it says an animal left, a crop dried out, a harvest rotted on the tile, or an action was ignored.
An agent that keeps no memory of yesterday cannot tell an empty pasture from an emptied one, and most agents in the field are tapes that keep no memory at all.

**Why the bank does not reveal it either.** A lost cow costs 400 coins of capital and forfeits a season of milk and fertilizer worth several times that.
Two or three of them a game is thousands of coins, on banks near a hundred thousand.
But the shop draw alone moves the final bank by a standard deviation of about 26,000 coins from one seed to the next, so a leak of that size sits inside the noise of a single game and inside the spread of a dozen.
It is a defect that looks like an ordinary bad day, every day.

So I moved the counting to the one place that sees every event: the engine.
My C++ port of the environment now keeps **four books** at the exact sites where the events happen: the losses it inflicts, the actions it accepted and ignored, the efficiency the farm left on the table, and what the season stranded in the shed at the end.
This page points all four at every agent I hold: **154 agents, mine and the community's, 4,928 games**, every one played in the same worlds against the same rival.

**82 of the 154 lose animals.** The median agent leaves 18 units of grown produce a game to rot where it stands, lets 11 units of fertilizer go uncollected, holds 14 units of milk and fruit at the tile's cap because nobody came, and pays hired hands for 441 turns of standing still.
And the warning before an animal is lost arrives a **median of 20 turns early**, which is the part worth having: it is time enough to act.

The engine is public, the agents are published notebooks kept in a public dataset, and the code for every number is in the cells, so all of it can be re-run by anyone.

This page is the third of a family.
[X-ray your agent](https://www.kaggle.com/code/destbreso/x-ray-your-agent) reads what one agent plays from its replays, and [A macroeconomic x-ray of the meta](https://www.kaggle.com/code/destbreso/a-macroeconomic-x-ray-of-the-kaggriculture-meta) reads what the whole field's economy does.
This one takes both readings from inside the engine instead of from the outside, and asks a question neither could: **not what an agent does, but what it loses and leaves while doing it**, one agent at a time and then across the field.


In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import numpy as np

plt.rcParams.update({'figure.dpi': 130, 'font.size': 9, 'axes.grid': True,
                     'grid.alpha': .25, 'axes.spines.top': False, 'axes.spines.right': False})
INK, WARN, GOOD, MINE = '#22303f', '#c0392b', '#1e8449', '#b7791f'

CENSUS = json.loads(r'''{"rival":"yhay81-shop-router-0913","worlds":16,"rows":[{"agent":"v7.76_learned (mine, live 56220106)","mine":true,"games":32,"bank_median":68683.0,"games_losing_an_animal":32,"escape_first_day_min":1,"escapes_by_day":{"1":64,"8":12},"per_game":{"animals_escaped":2.375,"escape_capital":968.75,"escape_yield_units":0.0,"animal_unfed_days":25.375,"feed_no_wheat":4.625,"feed_no_animal":130.0,"feed_redundant":0.0,"plants_dry":2.062,"dry_seed_cost":118.125,"plants_decayed":14.625,"decay_units":14.625,"plant_unwatered_days":351.062,"shed_discarded_units":1.375,"dead_actions":612.375,"water_dead":48.812,"harvest_dead":71.562,"plant_dead":4.75,"sell_zero_fill":189.25,"buy_zero_fill":10.438,"sell_dead_units":90301.156,"sold_units":1474.719,"silent_loss_coins":1086.875,"silent_loss_units":16.875,"refused_buy_product":232.625,"refused_buy_seed":4.688,"refused_buy_animal":3.562,"tile_cap_units":15.75,"fertilizer_forgone":12.688,"care_unfed":19.562,"animal_shed_days":0.875,"idle_tile_days":143.375,"weed_tile_days":4.531,"hand_pass_turns":397.031,"phantom_hand_actions":0.0,"sell_floor_units":96.438,"hire_premium":4086.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":2.062,"seeds_unplanted_cost_now":88.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":11.875,"drop_dead":0.0,"place_dead":20.219,"build_dead":0.0,"dig_dead":4.25,"fertilize_dead":18.594,"collect_dead":141.188,"care_dead":156.5,"farmer_pass_turns":62.25}},{"agent":"v7.77_learned (mine, repaired)","mine":true,"games":32,"bank_median":97153.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":149.812,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"ahmedberatozer-v27","mine":false,"games":32,"bank_median":80469.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.562,"shed_discarded_units":8.562,"dead_actions":44.469,"water_dead":12.062,"harvest_dead":6.25,"plant_dead":0.0,"sell_zero_fill":102.688,"buy_zero_fill":0.0,"sell_dead_units":88068.812,"sold_units":1528.188,"silent_loss_coins":109.375,"silent_loss_units":27.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":65.0,"weed_tile_days":2.562,"hand_pass_turns":399.594,"phantom_hand_actions":0.0,"sell_floor_units":196.875,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.469,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"ahmedberatozer-v31-production-and-sale-priority","mine":false,"games":32,"bank_median":80222.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.562,"shed_discarded_units":2.125,"dead_actions":44.344,"water_dead":12.062,"harvest_dead":6.188,"plant_dead":0.0,"sell_zero_fill":102.281,"buy_zero_fill":0.0,"sell_dead_units":88067.281,"sold_units":1547.094,"silent_loss_coins":109.375,"silent_loss_units":21.0,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":80.469,"weed_tile_days":2.594,"hand_pass_turns":393.719,"phantom_hand_actions":0.0,"sell_floor_units":206.375,"hire_premium":3953.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.406,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"ahmedberatozer-v34-observed-market-timing","mine":false,"games":32,"bank_median":80194.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.562,"shed_discarded_units":1.312,"dead_actions":44.438,"water_dead":12.062,"harvest_dead":6.281,"plant_dead":0.0,"sell_zero_fill":107.562,"buy_zero_fill":0.0,"sell_dead_units":88069.156,"sold_units":1547.812,"silent_loss_coins":109.375,"silent_loss_units":20.188,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":14.125,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":80.469,"weed_tile_days":2.594,"hand_pass_turns":393.719,"phantom_hand_actions":0.0,"sell_floor_units":196.594,"hire_premium":3953.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.406,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"ahmedberatozer-v35-reactive-sales-sheep-expansion","mine":false,"games":32,"bank_median":96842.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":1.312,"dead_actions":67.0,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":100.844,"buy_zero_fill":0.0,"sell_dead_units":86052.156,"sold_units":1581.906,"silent_loss_coins":109.375,"silent_loss_units":20.219,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":15.188,"fertilizer_forgone":14.812,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":408.188,"phantom_hand_actions":0.0,"sell_floor_units":165.844,"hire_premium":4715.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":60.75}},{"agent":"ahmedberatozer-v36-guarded-four-turn-sales","mine":false,"games":32,"bank_median":97025.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":1.438,"dead_actions":67.0,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":100.656,"buy_zero_fill":0.0,"sell_dead_units":86051.656,"sold_units":1581.781,"silent_loss_coins":109.375,"silent_loss_units":20.344,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":15.188,"fertilizer_forgone":14.812,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":408.188,"phantom_hand_actions":0.0,"sell_floor_units":165.0,"hire_premium":4715.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":60.75}},{"agent":"ahmedberatozer-v38-smarter-feed-stronger-margins","mine":false,"games":32,"bank_median":97349.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":64.344,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":23.0,"dead_actions":67.312,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":114.812,"buy_zero_fill":0.0,"sell_dead_units":86121.969,"sold_units":1623.281,"silent_loss_coins":109.375,"silent_loss_units":41.906,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":28.094,"fertilizer_forgone":14.812,"care_unfed":59.562,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":446.125,"phantom_hand_actions":0.0,"sell_floor_units":90.562,"hire_premium":5410.406,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.344,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":65.562}},{"agent":"ahmedberatozer-v39-ready-before-the-rush","mine":false,"games":32,"bank_median":97891.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":73.25,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":21.625,"dead_actions":67.031,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":115.375,"buy_zero_fill":0.0,"sell_dead_units":86129.219,"sold_units":1619.312,"silent_loss_coins":109.375,"silent_loss_units":40.531,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":28.156,"fertilizer_forgone":14.812,"care_unfed":68.969,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":454.844,"phantom_hand_actions":0.0,"sell_floor_units":89.688,"hire_premium":5410.406,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.25,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":65.875}},{"agent":"ahmedberatozer-v40-plans-that-fit-the-shops","mine":false,"games":32,"bank_median":101068.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":69.156,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":382.469,"shed_discarded_units":15.188,"dead_actions":62.375,"water_dead":28.625,"harvest_dead":6.188,"plant_dead":2.0,"sell_zero_fill":133.0,"buy_zero_fill":0.062,"sell_dead_units":86198.688,"sold_units":1628.438,"silent_loss_coins":109.375,"silent_loss_units":34.094,"refused_buy_product":0.062,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":27.938,"fertilizer_forgone":14.812,"care_unfed":64.938,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":458.344,"phantom_hand_actions":0.0,"sell_floor_units":75.094,"hire_premium":5436.125,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":2.375,"build_dead":0.0,"dig_dead":2.406,"fertilize_dead":0.719,"collect_dead":7.938,"care_dead":12.125,"farmer_pass_turns":66.5}},{"agent":"ahmedberatozer-v41-review-candidate","mine":false,"games":32,"bank_median":86205.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":83.375,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":100.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":377.625,"shed_discarded_units":20.969,"dead_actions":45.375,"water_dead":12.062,"harvest_dead":6.344,"plant_dead":0.0,"sell_zero_fill":123.625,"buy_zero_fill":1.0,"sell_dead_units":88213.406,"sold_units":1586.438,"silent_loss_coins":100.0,"silent_loss_units":38.906,"refused_buy_product":1.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":25.25,"fertilizer_forgone":14.688,"care_unfed":78.875,"animal_shed_days":0.938,"idle_tile_days":100.656,"weed_tile_days":2.781,"hand_pass_turns":464.75,"phantom_hand_actions":0.0,"sell_floor_units":114.844,"hire_premium":5120.906,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.0,"seeds_unplanted_cost_now":15.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.469,"build_dead":0.0,"dig_dead":1.875,"fertilize_dead":0.188,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":67.656}},{"agent":"ahmedberatozer-v42-production-that-fits-the-market","mine":false,"games":32,"bank_median":89705.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":81.344,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":100.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":384.25,"shed_discarded_units":13.281,"dead_actions":41.438,"water_dead":10.625,"harvest_dead":5.469,"plant_dead":0.0,"sell_zero_fill":141.344,"buy_zero_fill":1.0,"sell_dead_units":88283.188,"sold_units":1797.625,"silent_loss_coins":100.0,"silent_loss_units":31.219,"refused_buy_product":5.125,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":22.781,"fertilizer_forgone":14.875,"care_unfed":76.906,"animal_shed_days":0.938,"idle_tile_days":100.656,"weed_tile_days":2.781,"hand_pass_turns":466.875,"phantom_hand_actions":0.0,"sell_floor_units":99.656,"hire_premium":5111.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.0,"seeds_unplanted_cost_now":15.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":1.344,"build_dead":0.0,"dig_dead":3.312,"fertilize_dead":0.812,"collect_dead":7.688,"care_dead":12.188,"farmer_pass_turns":68.5}},{"agent":"aurax7-shop-router-reactive-v2","mine":false,"games":32,"bank_median":80149.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":2.125,"dead_actions":42.719,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":104.125,"buy_zero_fill":0.0,"sell_dead_units":88072.875,"sold_units":1534.812,"silent_loss_coins":110.0,"silent_loss_units":21.125,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":64.062,"weed_tile_days":2.719,"hand_pass_turns":400.406,"phantom_hand_actions":0.0,"sell_floor_units":199.438,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"aurax7-shop-router-reactive-v4","mine":false,"games":32,"bank_median":97349.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":64.344,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":23.0,"dead_actions":67.312,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":114.812,"buy_zero_fill":0.0,"sell_dead_units":86121.969,"sold_units":1623.281,"silent_loss_coins":109.375,"silent_loss_units":41.906,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":28.094,"fertilizer_forgone":14.812,"care_unfed":59.562,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":446.125,"phantom_hand_actions":0.0,"sell_floor_units":90.562,"hire_premium":5410.406,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.344,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":65.562}},{"agent":"avioon-apex-v7","mine":false,"games":32,"bank_median":70544.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":35.188,"feed_no_wheat":0.812,"feed_no_animal":14.625,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":25.0,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":387.031,"shed_discarded_units":3.0,"dead_actions":87.406,"water_dead":11.969,"harvest_dead":9.875,"plant_dead":2.0,"sell_zero_fill":41.562,"buy_zero_fill":5.938,"sell_dead_units":345.844,"sold_units":1489.844,"silent_loss_coins":350.0,"silent_loss_units":29.812,"refused_buy_product":10.688,"refused_buy_seed":1.0,"refused_buy_animal":1.0,"tile_cap_units":2.0,"fertilizer_forgone":10.0,"care_unfed":18.188,"animal_shed_days":1.0,"idle_tile_days":56.938,"weed_tile_days":3.469,"hand_pass_turns":512.125,"phantom_hand_actions":0.0,"sell_floor_units":217.031,"hire_premium":5687.25,"shed_units_now":2.0,"shed_value_now":90.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":4.0,"seeds_unplanted_cost_now":50.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.812,"drop_dead":0.0,"place_dead":3.062,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.938,"collect_dead":17.062,"care_dead":25.25,"farmer_pass_turns":62.75}},{"agent":"boatlee-v16-rc5-r5a-recovery","mine":false,"games":32,"bank_median":78838.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":31.0,"feed_no_wheat":14.0,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":3.031,"dry_seed_cost":123.125,"plants_decayed":10.0,"decay_units":20.0,"plant_unwatered_days":464.594,"shed_discarded_units":0.0,"dead_actions":748.625,"water_dead":22.531,"harvest_dead":4.25,"plant_dead":4.0,"sell_zero_fill":23.156,"buy_zero_fill":5.0,"sell_dead_units":304.375,"sold_units":1374.625,"silent_loss_coins":123.125,"silent_loss_units":22.0,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":5.0,"care_unfed":25.0,"animal_shed_days":1.0,"idle_tile_days":168.0,"weed_tile_days":11.062,"hand_pass_turns":238.875,"phantom_hand_actions":0.0,"sell_floor_units":157.938,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.031,"seeds_unplanted_cost_now":3.125,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.0,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.125,"collect_dead":9.0,"care_dead":645.688,"farmer_pass_turns":61.0}},{"agent":"boatlee-v20-adaptive-r1-multi-route","mine":false,"games":32,"bank_median":56396.0,"games_losing_an_animal":6,"escape_first_day_min":18,"escapes_by_day":{"18":2,"28":12},"per_game":{"animals_escaped":0.438,"escape_capital":212.5,"escape_yield_units":0.0,"animal_unfed_days":10.062,"feed_no_wheat":1.062,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.594,"dry_seed_cost":507.5,"plants_decayed":25.094,"decay_units":27.312,"plant_unwatered_days":411.344,"shed_discarded_units":11.75,"dead_actions":22.438,"water_dead":4.844,"harvest_dead":1.969,"plant_dead":0.375,"sell_zero_fill":241.0,"buy_zero_fill":0.875,"sell_dead_units":3023.719,"sold_units":1594.375,"silent_loss_coins":720.0,"silent_loss_units":40.875,"refused_buy_product":1.875,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":10.5,"care_unfed":1.0,"animal_shed_days":1.75,"idle_tile_days":237.656,"weed_tile_days":34.875,"hand_pass_turns":511.781,"phantom_hand_actions":0.0,"sell_floor_units":237.969,"hire_premium":5691.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":15.875,"seeds_unplanted_cost_now":237.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":10.938,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.5,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":47.375}},{"agent":"boatlee-v29-market-hysteresis","mine":false,"games":32,"bank_median":78690.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":11.0,"feed_no_wheat":1.0,"feed_no_animal":26.0,"feed_redundant":2.0,"plants_dry":3.0,"dry_seed_cost":30.0,"plants_decayed":33.688,"decay_units":33.688,"plant_unwatered_days":387.438,"shed_discarded_units":6.188,"dead_actions":148.188,"water_dead":30.0,"harvest_dead":22.312,"plant_dead":3.375,"sell_zero_fill":74.312,"buy_zero_fill":1.875,"sell_dead_units":426.25,"sold_units":1435.625,"silent_loss_coins":30.0,"silent_loss_units":42.875,"refused_buy_product":44.25,"refused_buy_seed":3.312,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":0.0,"care_unfed":10.0,"animal_shed_days":27.0,"idle_tile_days":130.5,"weed_tile_days":10.188,"hand_pass_turns":764.906,"phantom_hand_actions":0.0,"sell_floor_units":156.0,"hire_premium":6006.0,"shed_units_now":5.938,"shed_value_now":172.375,"animals_in_shed_now":1.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":4.75,"build_dead":0.0,"dig_dead":2.125,"fertilize_dead":4.625,"collect_dead":27.0,"care_dead":25.0,"farmer_pass_turns":70.0}},{"agent":"bruceqdu-route1","mine":false,"games":32,"bank_median":60943.0,"games_losing_an_animal":32,"escape_first_day_min":28,"escapes_by_day":{"28":32},"per_game":{"animals_escaped":1.0,"escape_capital":500.0,"escape_yield_units":0.0,"animal_unfed_days":7.375,"feed_no_wheat":0.375,"feed_no_animal":1.312,"feed_redundant":0.0,"plants_dry":9.562,"dry_seed_cost":804.062,"plants_decayed":28.938,"decay_units":30.875,"plant_unwatered_days":302.625,"shed_discarded_units":0.0,"dead_actions":67.688,"water_dead":29.719,"harvest_dead":7.375,"plant_dead":1.5,"sell_zero_fill":161.469,"buy_zero_fill":4.562,"sell_dead_units":1661.438,"sold_units":1821.562,"silent_loss_coins":1304.062,"silent_loss_units":33.344,"refused_buy_product":36.312,"refused_buy_seed":1.062,"refused_buy_animal":0.0,"tile_cap_units":15.812,"fertilizer_forgone":6.438,"care_unfed":1.375,"animal_shed_days":4.312,"idle_tile_days":353.75,"weed_tile_days":49.281,"hand_pass_turns":324.812,"phantom_hand_actions":42.25,"sell_floor_units":166.781,"hire_premium":7484.375,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.062,"seeds_unplanted_now":0.875,"seeds_unplanted_cost_now":50.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":11.281,"drop_dead":0.0,"place_dead":0.125,"build_dead":0.062,"dig_dead":4.25,"fertilize_dead":7.25,"collect_dead":1.188,"care_dead":3.25,"farmer_pass_turns":45.0}},{"agent":"denizeryilmaz-v16-rc5","mine":false,"games":32,"bank_median":78838.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":31.0,"feed_no_wheat":14.0,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":3.031,"dry_seed_cost":123.125,"plants_decayed":10.0,"decay_units":20.0,"plant_unwatered_days":464.594,"shed_discarded_units":0.0,"dead_actions":748.625,"water_dead":22.531,"harvest_dead":4.25,"plant_dead":4.0,"sell_zero_fill":23.156,"buy_zero_fill":5.0,"sell_dead_units":304.375,"sold_units":1374.625,"silent_loss_coins":123.125,"silent_loss_units":22.0,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":5.0,"care_unfed":25.0,"animal_shed_days":1.0,"idle_tile_days":168.0,"weed_tile_days":11.062,"hand_pass_turns":238.875,"phantom_hand_actions":0.0,"sell_floor_units":157.938,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.031,"seeds_unplanted_cost_now":3.125,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.0,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.125,"collect_dead":9.0,"care_dead":645.688,"farmer_pass_turns":61.0}},{"agent":"dmitriigluzdov-7-turn-rescue","mine":false,"games":32,"bank_median":80474.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":8.562,"dead_actions":42.406,"water_dead":11.469,"harvest_dead":5.875,"plant_dead":0.0,"sell_zero_fill":100.688,"buy_zero_fill":0.0,"sell_dead_units":88063.188,"sold_units":1533.938,"silent_loss_coins":110.0,"silent_loss_units":27.562,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":64.062,"weed_tile_days":2.719,"hand_pass_turns":383.156,"phantom_hand_actions":0.0,"sell_floor_units":199.875,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.312,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"dmitriigluzdov-a-smaller-market-shock","mine":false,"games":32,"bank_median":80432.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":4.438,"dead_actions":42.656,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":106.625,"buy_zero_fill":0.0,"sell_dead_units":88061.938,"sold_units":1544.125,"silent_loss_coins":110.0,"silent_loss_units":23.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":74.375,"weed_tile_days":2.75,"hand_pass_turns":388.219,"phantom_hand_actions":0.0,"sell_floor_units":197.438,"hire_premium":3838.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"dmitriigluzdov-herd-safe-sale-window-lb-2700","mine":false,"games":32,"bank_median":79967.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":4.438,"dead_actions":42.656,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":105.812,"buy_zero_fill":0.0,"sell_dead_units":88061.938,"sold_units":1544.125,"silent_loss_coins":110.0,"silent_loss_units":23.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":74.375,"weed_tile_days":2.75,"hand_pass_turns":388.219,"phantom_hand_actions":0.0,"sell_floor_units":196.531,"hire_premium":3838.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"mine, earlier build 24","mine":true,"games":32,"bank_median":1270.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":0.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":17.531,"dry_seed_cost":1403.125,"plants_decayed":1.906,"decay_units":1.906,"plant_unwatered_days":22.688,"shed_discarded_units":0.0,"dead_actions":332.969,"water_dead":0.0,"harvest_dead":332.969,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":12.969,"silent_loss_coins":1403.125,"silent_loss_units":18.844,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":0.0,"care_unfed":0.0,"animal_shed_days":0.0,"idle_tile_days":639.156,"weed_tile_days":379.406,"hand_pass_turns":0.0,"phantom_hand_actions":0.0,"sell_floor_units":0.0,"hire_premium":0.0,"shed_units_now":4.0,"shed_value_now":955.031,"animals_in_shed_now":0.0,"seeds_unplanted_now":2.094,"seeds_unplanted_cost_now":154.688,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":69.812}},{"agent":"flexonafft-fieldbook-closeout","mine":false,"games":32,"bank_median":77511.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":12.688,"feed_no_wheat":0.812,"feed_no_animal":18.75,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":32.5,"plants_decayed":31.062,"decay_units":31.062,"plant_unwatered_days":390.75,"shed_discarded_units":5.219,"dead_actions":115.938,"water_dead":29.438,"harvest_dead":9.25,"plant_dead":0.438,"sell_zero_fill":16.344,"buy_zero_fill":3.938,"sell_dead_units":81.031,"sold_units":1430.844,"silent_loss_coins":357.5,"silent_loss_units":37.031,"refused_buy_product":7.375,"refused_buy_seed":0.375,"refused_buy_animal":0.188,"tile_cap_units":4.125,"fertilizer_forgone":3.562,"care_unfed":10.875,"animal_shed_days":0.25,"idle_tile_days":60.938,"weed_tile_days":2.406,"hand_pass_turns":427.75,"phantom_hand_actions":0.0,"sell_floor_units":160.969,"hire_premium":5686.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.812,"seeds_unplanted_cost_now":311.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.125,"drop_dead":0.0,"place_dead":0.562,"build_dead":0.0,"dig_dead":0.625,"fertilize_dead":1.0,"collect_dead":19.812,"care_dead":35.125,"farmer_pass_turns":57.938}},{"agent":"flexonafft-fieldbook-day9","mine":false,"games":32,"bank_median":77511.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":12.688,"feed_no_wheat":0.812,"feed_no_animal":18.75,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":32.5,"plants_decayed":31.062,"decay_units":31.062,"plant_unwatered_days":390.75,"shed_discarded_units":5.219,"dead_actions":115.938,"water_dead":29.438,"harvest_dead":9.25,"plant_dead":0.438,"sell_zero_fill":17.281,"buy_zero_fill":3.938,"sell_dead_units":174.781,"sold_units":1430.844,"silent_loss_coins":357.5,"silent_loss_units":37.031,"refused_buy_product":7.375,"refused_buy_seed":0.375,"refused_buy_animal":0.188,"tile_cap_units":4.125,"fertilizer_forgone":3.562,"care_unfed":10.875,"animal_shed_days":0.25,"idle_tile_days":60.938,"weed_tile_days":2.406,"hand_pass_turns":427.75,"phantom_hand_actions":0.0,"sell_floor_units":160.969,"hire_premium":5686.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.812,"seeds_unplanted_cost_now":311.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.125,"drop_dead":0.0,"place_dead":0.562,"build_dead":0.0,"dig_dead":0.625,"fertilize_dead":1.0,"collect_dead":19.812,"care_dead":35.125,"farmer_pass_turns":57.938}},{"agent":"flexonafft-multiroute","mine":false,"games":32,"bank_median":56396.0,"games_losing_an_animal":6,"escape_first_day_min":18,"escapes_by_day":{"18":2,"28":12},"per_game":{"animals_escaped":0.438,"escape_capital":212.5,"escape_yield_units":0.0,"animal_unfed_days":10.062,"feed_no_wheat":1.062,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.594,"dry_seed_cost":507.5,"plants_decayed":25.094,"decay_units":27.312,"plant_unwatered_days":411.344,"shed_discarded_units":11.75,"dead_actions":22.438,"water_dead":4.844,"harvest_dead":1.969,"plant_dead":0.375,"sell_zero_fill":241.0,"buy_zero_fill":0.875,"sell_dead_units":3023.719,"sold_units":1594.375,"silent_loss_coins":720.0,"silent_loss_units":40.875,"refused_buy_product":1.875,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":10.5,"care_unfed":1.0,"animal_shed_days":1.75,"idle_tile_days":237.656,"weed_tile_days":34.875,"hand_pass_turns":511.781,"phantom_hand_actions":0.0,"sell_floor_units":237.969,"hire_premium":5691.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":15.875,"seeds_unplanted_cost_now":237.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":10.938,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.5,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":47.375}},{"agent":"hgh1024-herd-safe-v2-sale-horizon-2","mine":false,"games":32,"bank_median":80139.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":4.438,"dead_actions":42.656,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":105.75,"buy_zero_fill":0.0,"sell_dead_units":88061.938,"sold_units":1544.125,"silent_loss_coins":110.0,"silent_loss_units":23.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":74.375,"weed_tile_days":2.75,"hand_pass_turns":388.219,"phantom_hand_actions":0.0,"sell_floor_units":196.406,"hire_premium":3838.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"indarkarhana-e776-latent-pasture","mine":false,"games":32,"bank_median":72830.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":32},"per_game":{"animals_escaped":1.0,"escape_capital":412.5,"escape_yield_units":0.0,"animal_unfed_days":13.312,"feed_no_wheat":1.312,"feed_no_animal":4.0,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":30.0,"plants_decayed":35.688,"decay_units":35.688,"plant_unwatered_days":403.469,"shed_discarded_units":10.375,"dead_actions":30.031,"water_dead":3.062,"harvest_dead":8.469,"plant_dead":0.375,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":1428.562,"silent_loss_coins":442.5,"silent_loss_units":49.062,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":6.938,"fertilizer_forgone":0.0,"care_unfed":10.188,"animal_shed_days":0.0,"idle_tile_days":58.5,"weed_tile_days":3.344,"hand_pass_turns":829.969,"phantom_hand_actions":0.0,"sell_floor_units":162.438,"hire_premium":6096.0,"shed_units_now":0.312,"shed_value_now":0.312,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.031,"seeds_unplanted_cost_now":0.312,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":2.812,"build_dead":0.0,"dig_dead":0.312,"fertilize_dead":0.625,"collect_dead":5.0,"care_dead":4.0,"farmer_pass_turns":70.0}},{"agent":"kaitofukami-v25-meta-reset","mine":false,"games":32,"bank_median":71533.0,"games_losing_an_animal":32,"escape_first_day_min":22,"escapes_by_day":{"22":32},"per_game":{"animals_escaped":1.0,"escape_capital":400.0,"escape_yield_units":3.0,"animal_unfed_days":23.062,"feed_no_wheat":1.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":3.469,"dry_seed_cost":166.875,"plants_decayed":8.0,"decay_units":22.0,"plant_unwatered_days":436.344,"shed_discarded_units":3.719,"dead_actions":67.969,"water_dead":39.688,"harvest_dead":13.156,"plant_dead":5.125,"sell_zero_fill":0.625,"buy_zero_fill":2.0,"sell_dead_units":37.656,"sold_units":1416.344,"silent_loss_coins":566.875,"silent_loss_units":30.719,"refused_buy_product":1.5,"refused_buy_seed":5.125,"refused_buy_animal":0.0,"tile_cap_units":17.0,"fertilizer_forgone":5.0,"care_unfed":13.062,"animal_shed_days":3.0,"idle_tile_days":197.125,"weed_tile_days":11.375,"hand_pass_turns":946.594,"phantom_hand_actions":0.0,"sell_floor_units":178.75,"hire_premium":9546.0,"shed_units_now":5.688,"shed_value_now":5.812,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":10.312,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.031,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":3.219,"fertilize_dead":5.688,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":198.0}},{"agent":"leoprovorov-two-coins-at-high-noon-small-improvement","mine":false,"games":32,"bank_median":80111.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":4.438,"dead_actions":42.656,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":105.75,"buy_zero_fill":0.0,"sell_dead_units":88061.938,"sold_units":1544.125,"silent_loss_coins":110.0,"silent_loss_units":23.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":74.375,"weed_tile_days":2.75,"hand_pass_turns":388.219,"phantom_hand_actions":0.0,"sell_floor_units":196.406,"hire_premium":3838.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"llccqq624-adaptive-counterbook","mine":false,"games":32,"bank_median":56396.0,"games_losing_an_animal":6,"escape_first_day_min":18,"escapes_by_day":{"18":2,"28":12},"per_game":{"animals_escaped":0.438,"escape_capital":212.5,"escape_yield_units":0.0,"animal_unfed_days":10.062,"feed_no_wheat":1.062,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.594,"dry_seed_cost":507.5,"plants_decayed":25.094,"decay_units":27.312,"plant_unwatered_days":411.344,"shed_discarded_units":11.75,"dead_actions":22.438,"water_dead":4.844,"harvest_dead":1.969,"plant_dead":0.375,"sell_zero_fill":241.0,"buy_zero_fill":0.875,"sell_dead_units":3023.719,"sold_units":1594.375,"silent_loss_coins":720.0,"silent_loss_units":40.875,"refused_buy_product":1.875,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":10.5,"care_unfed":1.0,"animal_shed_days":1.75,"idle_tile_days":237.656,"weed_tile_days":34.875,"hand_pass_turns":511.781,"phantom_hand_actions":0.0,"sell_floor_units":237.969,"hire_premium":5691.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":15.875,"seeds_unplanted_cost_now":237.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":10.938,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.5,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":47.375}},{"agent":"llccqq624-adaptive-shop-guard","mine":false,"games":32,"bank_median":78668.0,"games_losing_an_animal":2,"escape_first_day_min":10,"escapes_by_day":{"10":6},"per_game":{"animals_escaped":0.188,"escape_capital":75.0,"escape_yield_units":0.0,"animal_unfed_days":11.438,"feed_no_wheat":5.438,"feed_no_animal":6.062,"feed_redundant":0.0,"plants_dry":4.375,"dry_seed_cost":141.25,"plants_decayed":18.438,"decay_units":23.625,"plant_unwatered_days":465.969,"shed_discarded_units":6.906,"dead_actions":76.844,"water_dead":22.938,"harvest_dead":12.156,"plant_dead":3.906,"sell_zero_fill":99.688,"buy_zero_fill":1.812,"sell_dead_units":340.375,"sold_units":1255.406,"silent_loss_coins":216.25,"silent_loss_units":33.844,"refused_buy_product":1.125,"refused_buy_seed":2.0,"refused_buy_animal":0.062,"tile_cap_units":12.281,"fertilizer_forgone":12.562,"care_unfed":10.438,"animal_shed_days":10.312,"idle_tile_days":154.938,"weed_tile_days":27.625,"hand_pass_turns":353.281,"phantom_hand_actions":0.0,"sell_floor_units":202.875,"hire_premium":6826.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.125,"seeds_unplanted_now":5.156,"seeds_unplanted_cost_now":55.938,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":7.812,"drop_dead":0.0,"place_dead":0.25,"build_dead":0.062,"dig_dead":1.469,"fertilize_dead":4.375,"collect_dead":6.188,"care_dead":6.188,"farmer_pass_turns":6.375}},{"agent":"llccqq624-premium-queue-split","mine":false,"games":32,"bank_median":78850.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":31.0,"feed_no_wheat":14.0,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":3.031,"dry_seed_cost":123.125,"plants_decayed":10.0,"decay_units":20.0,"plant_unwatered_days":464.594,"shed_discarded_units":0.0,"dead_actions":748.625,"water_dead":22.531,"harvest_dead":4.25,"plant_dead":4.0,"sell_zero_fill":34.094,"buy_zero_fill":5.0,"sell_dead_units":304.375,"sold_units":1374.625,"silent_loss_coins":123.125,"silent_loss_units":22.0,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":5.0,"care_unfed":25.0,"animal_shed_days":1.0,"idle_tile_days":168.0,"weed_tile_days":11.062,"hand_pass_turns":238.875,"phantom_hand_actions":0.0,"sell_floor_units":157.938,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.031,"seeds_unplanted_cost_now":3.125,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.0,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.125,"collect_dead":9.0,"care_dead":645.688,"farmer_pass_turns":61.0}},{"agent":"llccqq624-shops-remember-the-route","mine":false,"games":32,"bank_median":78668.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":11.375,"feed_no_wheat":5.625,"feed_no_animal":1.438,"feed_redundant":0.0,"plants_dry":4.188,"dry_seed_cost":133.75,"plants_decayed":18.0,"decay_units":22.75,"plant_unwatered_days":470.469,"shed_discarded_units":3.219,"dead_actions":51.844,"water_dead":18.438,"harvest_dead":7.906,"plant_dead":3.406,"sell_zero_fill":113.062,"buy_zero_fill":1.812,"sell_dead_units":368.5,"sold_units":1270.312,"silent_loss_coins":133.75,"silent_loss_units":29.156,"refused_buy_product":1.25,"refused_buy_seed":1.938,"refused_buy_animal":0.062,"tile_cap_units":12.781,"fertilizer_forgone":13.062,"care_unfed":10.75,"animal_shed_days":7.562,"idle_tile_days":145.188,"weed_tile_days":19.312,"hand_pass_turns":353.906,"phantom_hand_actions":0.0,"sell_floor_units":208.5,"hire_premium":6947.438,"shed_units_now":1.031,"shed_value_now":21.375,"animals_in_shed_now":0.0,"seeds_unplanted_now":5.344,"seeds_unplanted_cost_now":57.812,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":8.062,"drop_dead":0.0,"place_dead":0.062,"build_dead":0.0,"dig_dead":1.406,"fertilize_dead":2.75,"collect_dead":1.375,"care_dead":1.375,"farmer_pass_turns":5.688}},{"agent":"lucifer19-harvest-nocturne-v2-a-lighter-start","mine":false,"games":32,"bank_median":80452.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":4.438,"dead_actions":42.656,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":105.812,"buy_zero_fill":0.0,"sell_dead_units":88061.938,"sold_units":1544.125,"silent_loss_coins":110.0,"silent_loss_units":23.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":74.375,"weed_tile_days":2.75,"hand_pass_turns":388.219,"phantom_hand_actions":0.0,"sell_floor_units":193.656,"hire_premium":3838.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"lucifer19_harvest_nocturne_v2_a_lighter_","mine":false,"games":32,"bank_median":80452.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":4.438,"dead_actions":42.656,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":105.812,"buy_zero_fill":0.0,"sell_dead_units":88061.938,"sold_units":1544.125,"silent_loss_coins":110.0,"silent_loss_units":23.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":74.375,"weed_tile_days":2.75,"hand_pass_turns":388.219,"phantom_hand_actions":0.0,"sell_floor_units":193.656,"hire_premium":3838.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"lynnsakurai-farming-score-v4-a-better-shop","mine":false,"games":32,"bank_median":97458.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":6.812,"dead_actions":67.0,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":104.438,"buy_zero_fill":0.0,"sell_dead_units":86063.594,"sold_units":1635.531,"silent_loss_coins":109.375,"silent_loss_units":25.719,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":24.938,"fertilizer_forgone":14.812,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":145.469,"weed_tile_days":5.781,"hand_pass_turns":419.656,"phantom_hand_actions":0.0,"sell_floor_units":145.0,"hire_premium":5221.156,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":60.75}},{"agent":"moon_v189c","mine":false,"games":32,"bank_median":79407.0,"games_losing_an_animal":30,"escape_first_day_min":28,"escapes_by_day":{"28":30},"per_game":{"animals_escaped":0.938,"escape_capital":375.0,"escape_yield_units":0.0,"animal_unfed_days":25.625,"feed_no_wheat":0.938,"feed_no_animal":1.312,"feed_redundant":0.0,"plants_dry":1.062,"dry_seed_cost":13.438,"plants_decayed":32.969,"decay_units":33.969,"plant_unwatered_days":397.906,"shed_discarded_units":1.062,"dead_actions":7.562,"water_dead":0.219,"harvest_dead":1.219,"plant_dead":0.0,"sell_zero_fill":13.688,"buy_zero_fill":1.375,"sell_dead_units":632.062,"sold_units":1814.938,"silent_loss_coins":388.438,"silent_loss_units":36.062,"refused_buy_product":61.25,"refused_buy_seed":0.0,"refused_buy_animal":0.062,"tile_cap_units":3.812,"fertilizer_forgone":0.0,"care_unfed":17.812,"animal_shed_days":0.0,"idle_tile_days":63.469,"weed_tile_days":1.906,"hand_pass_turns":831.969,"phantom_hand_actions":0.0,"sell_floor_units":109.906,"hire_premium":5063.0,"shed_units_now":3.125,"shed_value_now":13.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.0,"seeds_unplanted_cost_now":400.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.125,"drop_dead":0.0,"place_dead":0.562,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.188,"collect_dead":1.562,"care_dead":1.438,"farmer_pass_turns":93.0}},{"agent":"municef1-lb-2448-single-file-agent","mine":false,"games":32,"bank_median":94667.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":29.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":370.562,"shed_discarded_units":0.0,"dead_actions":39.812,"water_dead":20.438,"harvest_dead":8.25,"plant_dead":2.0,"sell_zero_fill":0.0,"buy_zero_fill":2.0,"sell_dead_units":0.0,"sold_units":1524.688,"silent_loss_coins":110.0,"silent_loss_units":18.938,"refused_buy_product":1.0,"refused_buy_seed":2.0,"refused_buy_animal":0.0,"tile_cap_units":17.0,"fertilizer_forgone":13.0,"care_unfed":22.0,"animal_shed_days":1.0,"idle_tile_days":87.062,"weed_tile_days":3.812,"hand_pass_turns":404.969,"phantom_hand_actions":0.0,"sell_floor_units":184.438,"hire_premium":3464.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.062,"seeds_unplanted_cost_now":36.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":2.0,"build_dead":0.0,"dig_dead":1.062,"fertilize_dead":0.062,"collect_dead":3.0,"care_dead":3.0,"farmer_pass_turns":66.0}},{"agent":"pilkwang-precomputed-schedule-policy","mine":false,"games":32,"bank_median":53103.0,"games_losing_an_animal":7,"escape_first_day_min":12,"escapes_by_day":{"12":4,"17":2,"23":9,"24":4},"per_game":{"animals_escaped":0.594,"escape_capital":237.5,"escape_yield_units":0.0,"animal_unfed_days":17.969,"feed_no_wheat":10.125,"feed_no_animal":11.625,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":186.25,"plants_decayed":22.5,"decay_units":24.812,"plant_unwatered_days":378.281,"shed_discarded_units":0.562,"dead_actions":107.688,"water_dead":25.938,"harvest_dead":14.531,"plant_dead":3.906,"sell_zero_fill":265.594,"buy_zero_fill":0.906,"sell_dead_units":2332.594,"sold_units":1408.406,"silent_loss_coins":423.75,"silent_loss_units":26.938,"refused_buy_product":3.688,"refused_buy_seed":0.438,"refused_buy_animal":0.0,"tile_cap_units":11.5,"fertilizer_forgone":9.219,"care_unfed":9.531,"animal_shed_days":7.25,"idle_tile_days":224.438,"weed_tile_days":47.938,"hand_pass_turns":590.438,"phantom_hand_actions":21.75,"sell_floor_units":250.75,"hire_premium":5644.062,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.344,"seeds_unplanted_now":4.656,"seeds_unplanted_cost_now":133.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":13.812,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.094,"dig_dead":0.25,"fertilize_dead":3.562,"collect_dead":10.969,"care_dead":11.562,"farmer_pass_turns":72.0}},{"agent":"pilkwang_structured_economic_policy","mine":false,"games":32,"bank_median":97025.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.719,"shed_discarded_units":1.312,"dead_actions":67.0,"water_dead":30.0,"harvest_dead":7.25,"plant_dead":2.0,"sell_zero_fill":100.969,"buy_zero_fill":0.0,"sell_dead_units":86051.281,"sold_units":1556.406,"silent_loss_coins":109.375,"silent_loss_units":20.219,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":15.188,"fertilizer_forgone":14.812,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":105.094,"weed_tile_days":3.0,"hand_pass_turns":387.25,"phantom_hand_actions":0.0,"sell_floor_units":159.938,"hire_premium":3985.125,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.031,"seeds_unplanted_cost_now":83.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":21.688,"farmer_pass_turns":60.75}},{"agent":"prvsiyan-wheat-q45","mine":false,"games":32,"bank_median":88385.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":22.0,"feed_no_wheat":0.0,"feed_no_animal":1.25,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":17.594,"decay_units":17.594,"plant_unwatered_days":372.625,"shed_discarded_units":0.688,"dead_actions":43.844,"water_dead":29.844,"harvest_dead":4.875,"plant_dead":3.344,"sell_zero_fill":1.344,"buy_zero_fill":2.562,"sell_dead_units":25.75,"sold_units":1514.25,"silent_loss_coins":110.0,"silent_loss_units":19.281,"refused_buy_product":4.75,"refused_buy_seed":1.375,"refused_buy_animal":0.062,"tile_cap_units":1.0,"fertilizer_forgone":13.875,"care_unfed":19.0,"animal_shed_days":1.0,"idle_tile_days":94.281,"weed_tile_days":9.188,"hand_pass_turns":560.0,"phantom_hand_actions":0.0,"sell_floor_units":204.562,"hire_premium":4809.0,"shed_units_now":1.312,"shed_value_now":24.375,"animals_in_shed_now":0.0,"seeds_unplanted_now":13.969,"seeds_unplanted_cost_now":233.75,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.062,"build_dead":0.0,"dig_dead":0.281,"fertilize_dead":0.812,"collect_dead":1.062,"care_dead":1.25,"farmer_pass_turns":71.0}},{"agent":"rayk-topmeta-consensus-route","mine":false,"games":32,"bank_median":56185.0,"games_losing_an_animal":6,"escape_first_day_min":18,"escapes_by_day":{"18":2,"28":12},"per_game":{"animals_escaped":0.438,"escape_capital":212.5,"escape_yield_units":0.0,"animal_unfed_days":9.188,"feed_no_wheat":0.188,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.594,"dry_seed_cost":507.5,"plants_decayed":25.094,"decay_units":27.312,"plant_unwatered_days":411.344,"shed_discarded_units":11.75,"dead_actions":20.688,"water_dead":4.844,"harvest_dead":1.969,"plant_dead":0.375,"sell_zero_fill":240.219,"buy_zero_fill":0.0,"sell_dead_units":3020.219,"sold_units":1597.875,"silent_loss_coins":720.0,"silent_loss_units":40.875,"refused_buy_product":1.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":10.5,"care_unfed":0.125,"animal_shed_days":1.75,"idle_tile_days":237.656,"weed_tile_days":34.875,"hand_pass_turns":511.781,"phantom_hand_actions":0.0,"sell_floor_units":240.469,"hire_premium":5691.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":10.062,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.5,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":47.375}},{"agent":"salemali7-3000-score","mine":false,"games":32,"bank_median":78838.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":31.0,"feed_no_wheat":14.0,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":3.031,"dry_seed_cost":123.125,"plants_decayed":10.0,"decay_units":20.0,"plant_unwatered_days":464.594,"shed_discarded_units":0.0,"dead_actions":748.625,"water_dead":22.531,"harvest_dead":4.25,"plant_dead":4.0,"sell_zero_fill":23.156,"buy_zero_fill":5.0,"sell_dead_units":304.375,"sold_units":1374.625,"silent_loss_coins":123.125,"silent_loss_units":22.0,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":5.0,"care_unfed":25.0,"animal_shed_days":1.0,"idle_tile_days":168.0,"weed_tile_days":11.062,"hand_pass_turns":238.875,"phantom_hand_actions":0.0,"sell_floor_units":157.938,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.031,"seeds_unplanted_cost_now":3.125,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.0,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.125,"collect_dead":9.0,"care_dead":645.688,"farmer_pass_turns":61.0}},{"agent":"tetsutani-market-smart-farming","mine":false,"games":32,"bank_median":80476.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":8.562,"dead_actions":42.281,"water_dead":11.469,"harvest_dead":5.75,"plant_dead":0.0,"sell_zero_fill":101.5,"buy_zero_fill":0.0,"sell_dead_units":88066.875,"sold_units":1536.0,"silent_loss_coins":110.0,"silent_loss_units":27.562,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":64.062,"weed_tile_days":2.719,"hand_pass_turns":376.469,"phantom_hand_actions":0.0,"sell_floor_units":201.938,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.312,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"tschinkel-router-v5","mine":false,"games":32,"bank_median":81999.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":64},"per_game":{"animals_escaped":2.562,"escape_capital":1225.0,"escape_yield_units":0.0,"animal_unfed_days":32.625,"feed_no_wheat":0.875,"feed_no_animal":11.875,"feed_redundant":0.0,"plants_dry":1.875,"dry_seed_cost":18.75,"plants_decayed":23.75,"decay_units":25.375,"plant_unwatered_days":395.969,"shed_discarded_units":21.5,"dead_actions":93.094,"water_dead":14.344,"harvest_dead":14.125,"plant_dead":1.125,"sell_zero_fill":81.688,"buy_zero_fill":3.875,"sell_dead_units":9625479.094,"sold_units":1684.219,"silent_loss_coins":1243.75,"silent_loss_units":48.75,"refused_buy_product":6.625,"refused_buy_seed":1.062,"refused_buy_animal":0.875,"tile_cap_units":20.5,"fertilizer_forgone":20.375,"care_unfed":17.312,"animal_shed_days":4.75,"idle_tile_days":53.0,"weed_tile_days":7.281,"hand_pass_turns":461.188,"phantom_hand_actions":0.0,"sell_floor_units":172.281,"hire_premium":5120.0,"shed_units_now":0.812,"shed_value_now":13.312,"animals_in_shed_now":0.188,"seeds_unplanted_now":10.438,"seeds_unplanted_cost_now":110.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":2.188,"drop_dead":0.0,"place_dead":7.375,"build_dead":0.0,"dig_dead":0.062,"fertilize_dead":1.562,"collect_dead":17.75,"care_dead":21.812,"farmer_pass_turns":56.625}},{"agent":"tschinkel-router-v55","mine":false,"games":32,"bank_median":77834.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":22},"per_game":{"animals_escaped":1.25,"escape_capital":568.75,"escape_yield_units":0.0,"animal_unfed_days":29.875,"feed_no_wheat":0.875,"feed_no_animal":12.5,"feed_redundant":0.0,"plants_dry":0.938,"dry_seed_cost":26.25,"plants_decayed":25.188,"decay_units":28.438,"plant_unwatered_days":379.469,"shed_discarded_units":1.312,"dead_actions":111.031,"water_dead":25.219,"harvest_dead":15.0,"plant_dead":1.062,"sell_zero_fill":32.0,"buy_zero_fill":3.562,"sell_dead_units":1751309.781,"sold_units":1700.469,"silent_loss_coins":595.0,"silent_loss_units":30.5,"refused_buy_product":6.812,"refused_buy_seed":1.0,"refused_buy_animal":0.375,"tile_cap_units":13.25,"fertilizer_forgone":17.75,"care_unfed":16.5,"animal_shed_days":1.0,"idle_tile_days":72.688,"weed_tile_days":7.188,"hand_pass_turns":441.062,"phantom_hand_actions":0.0,"sell_floor_units":174.562,"hire_premium":5115.312,"shed_units_now":4.688,"shed_value_now":227.5,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.25,"seeds_unplanted_cost_now":126.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":4.312,"drop_dead":0.0,"place_dead":6.938,"build_dead":0.0,"dig_dead":0.562,"fertilize_dead":4.25,"collect_dead":16.5,"care_dead":23.812,"farmer_pass_turns":56.125}},{"agent":"tschinkel-state-router","mine":false,"games":32,"bank_median":77570.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26,"25":120,"27":75},"per_game":{"animals_escaped":6.906,"escape_capital":2996.875,"escape_yield_units":0.0,"animal_unfed_days":41.281,"feed_no_wheat":0.812,"feed_no_animal":12.75,"feed_redundant":0.0,"plants_dry":1.25,"dry_seed_cost":12.5,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":376.656,"shed_discarded_units":16.469,"dead_actions":113.031,"water_dead":21.469,"harvest_dead":15.5,"plant_dead":1.0,"sell_zero_fill":1.0,"buy_zero_fill":4.75,"sell_dead_units":20.562,"sold_units":1484.594,"silent_loss_coins":3009.375,"silent_loss_units":42.656,"refused_buy_product":10.719,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":16.25,"fertilizer_forgone":8.125,"care_unfed":23.812,"animal_shed_days":1.0,"idle_tile_days":65.312,"weed_tile_days":1.781,"hand_pass_turns":519.562,"phantom_hand_actions":0.938,"sell_floor_units":159.5,"hire_premium":5317.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":18.062,"seeds_unplanted_cost_now":186.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.875,"drop_dead":0.0,"place_dead":8.25,"build_dead":0.0,"dig_dead":0.062,"fertilize_dead":1.344,"collect_dead":26.469,"care_dead":23.5,"farmer_pass_turns":60.156}},{"agent":"v21-r1-public-state-route-portfolio","mine":false,"games":32,"bank_median":63367.0,"games_losing_an_animal":2,"escape_first_day_min":18,"escapes_by_day":{"18":2},"per_game":{"animals_escaped":0.062,"escape_capital":25.0,"escape_yield_units":0.0,"animal_unfed_days":9.062,"feed_no_wheat":1.062,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.281,"dry_seed_cost":532.5,"plants_decayed":23.969,"decay_units":24.938,"plant_unwatered_days":405.719,"shed_discarded_units":12.25,"dead_actions":19.188,"water_dead":2.656,"harvest_dead":3.156,"plant_dead":0.0,"sell_zero_fill":244.375,"buy_zero_fill":0.875,"sell_dead_units":3111.719,"sold_units":1526.125,"silent_loss_coins":557.5,"silent_loss_units":38.25,"refused_buy_product":1.875,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":9.125,"care_unfed":1.0,"animal_shed_days":1.0,"idle_tile_days":199.719,"weed_tile_days":29.562,"hand_pass_turns":508.656,"phantom_hand_actions":0.0,"sell_floor_units":225.969,"hire_premium":5673.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":16.0,"seeds_unplanted_cost_now":240.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":8.688,"drop_dead":0.0,"place_dead":0.062,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.812,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":45.0}},{"agent":"mine, earlier build 51","mine":true,"games":32,"bank_median":42487.0,"games_losing_an_animal":32,"escape_first_day_min":16,"escapes_by_day":{"16":80,"17":16,"18":6,"19":12,"20":54,"27":4,"28":126},"per_game":{"animals_escaped":9.312,"escape_capital":4050.0,"escape_yield_units":0.0,"animal_unfed_days":22.875,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":20.375,"dry_seed_cost":654.375,"plants_decayed":18.438,"decay_units":24.688,"plant_unwatered_days":94.156,"shed_discarded_units":39.25,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":5.5,"buy_zero_fill":0.062,"sell_dead_units":73.719,"sold_units":962.312,"silent_loss_coins":4704.375,"silent_loss_units":83.062,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":31.688,"fertilizer_forgone":2.875,"care_unfed":0.0,"animal_shed_days":0.562,"idle_tile_days":390.0,"weed_tile_days":45.531,"hand_pass_turns":903.438,"phantom_hand_actions":0.0,"sell_floor_units":25.438,"hire_premium":7655.594,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":12.438,"seeds_unplanted_cost_now":628.75,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":85.5}},{"agent":"mine, earlier build 52","mine":true,"games":32,"bank_median":42812.0,"games_losing_an_animal":32,"escape_first_day_min":16,"escapes_by_day":{"16":80,"17":16,"18":6,"19":12,"20":54,"27":4,"28":126},"per_game":{"animals_escaped":9.312,"escape_capital":4050.0,"escape_yield_units":0.0,"animal_unfed_days":23.062,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":25.062,"dry_seed_cost":710.625,"plants_decayed":19.062,"decay_units":25.25,"plant_unwatered_days":105.125,"shed_discarded_units":39.875,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":4.188,"buy_zero_fill":0.125,"sell_dead_units":69.0,"sold_units":979.875,"silent_loss_coins":4760.625,"silent_loss_units":88.875,"refused_buy_product":0.312,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":31.562,"fertilizer_forgone":3.188,"care_unfed":0.0,"animal_shed_days":0.562,"idle_tile_days":363.188,"weed_tile_days":61.688,"hand_pass_turns":800.906,"phantom_hand_actions":0.0,"sell_floor_units":24.938,"hire_premium":7911.875,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":19.312,"seeds_unplanted_cost_now":764.375,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":75.812}},{"agent":"mine, earlier build 53","mine":true,"games":32,"bank_median":31742.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":2,"16":18,"17":32,"18":62,"19":90,"20":54,"28":42},"per_game":{"animals_escaped":9.375,"escape_capital":4018.75,"escape_yield_units":0.0,"animal_unfed_days":24.312,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":191.25,"plants_decayed":4.25,"decay_units":5.188,"plant_unwatered_days":5.875,"shed_discarded_units":8.875,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":2.375,"buy_zero_fill":0.0,"sell_dead_units":42.188,"sold_units":867.5,"silent_loss_coins":4210.0,"silent_loss_units":14.312,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":59.75,"fertilizer_forgone":2.062,"care_unfed":0.0,"animal_shed_days":3.812,"idle_tile_days":401.25,"weed_tile_days":3.969,"hand_pass_turns":1628.875,"phantom_hand_actions":0.0,"sell_floor_units":31.938,"hire_premium":3936.688,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":13.312,"seeds_unplanted_cost_now":1323.75,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":228.906}},{"agent":"mine, earlier build 54","mine":true,"games":32,"bank_median":27890.0,"games_losing_an_animal":32,"escape_first_day_min":17,"escapes_by_day":{"17":12,"18":48,"19":8,"20":72,"28":52},"per_game":{"animals_escaped":6.0,"escape_capital":2618.75,"escape_yield_units":0.0,"animal_unfed_days":15.438,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":15.062,"dry_seed_cost":263.75,"plants_decayed":5.5,"decay_units":8.188,"plant_unwatered_days":40.438,"shed_discarded_units":1.875,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":1.188,"buy_zero_fill":0.0,"sell_dead_units":57.25,"sold_units":783.0,"silent_loss_coins":2882.5,"silent_loss_units":24.562,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":27.688,"fertilizer_forgone":0.375,"care_unfed":0.0,"animal_shed_days":4.062,"idle_tile_days":68.062,"weed_tile_days":40.781,"hand_pass_turns":1437.0,"phantom_hand_actions":0.0,"sell_floor_units":17.125,"hire_premium":1450.125,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":42.5,"seeds_unplanted_cost_now":1965.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":212.75}},{"agent":"mine, earlier build 55","mine":true,"games":32,"bank_median":36060.0,"games_losing_an_animal":32,"escape_first_day_min":17,"escapes_by_day":{"17":60,"18":54,"20":36,"21":18,"28":108},"per_game":{"animals_escaped":8.625,"escape_capital":3731.25,"escape_yield_units":0.0,"animal_unfed_days":19.625,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":18.625,"dry_seed_cost":394.375,"plants_decayed":22.875,"decay_units":28.938,"plant_unwatered_days":62.188,"shed_discarded_units":24.25,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":2.812,"buy_zero_fill":0.0,"sell_dead_units":52.5,"sold_units":890.438,"silent_loss_coins":4125.625,"silent_loss_units":68.938,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":40.125,"fertilizer_forgone":0.375,"care_unfed":0.0,"animal_shed_days":6.25,"idle_tile_days":212.125,"weed_tile_days":46.844,"hand_pass_turns":1086.812,"phantom_hand_actions":0.0,"sell_floor_units":24.688,"hire_premium":4428.688,"shed_units_now":0.062,"shed_value_now":2.625,"animals_in_shed_now":0.0,"seeds_unplanted_now":15.25,"seeds_unplanted_cost_now":702.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":163.406}},{"agent":"mine, earlier build 56","mine":true,"games":32,"bank_median":42568.0,"games_losing_an_animal":32,"escape_first_day_min":16,"escapes_by_day":{"16":34,"17":44,"20":54,"28":150},"per_game":{"animals_escaped":8.812,"escape_capital":3837.5,"escape_yield_units":0.0,"animal_unfed_days":21.5,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":17.188,"dry_seed_cost":558.75,"plants_decayed":14.562,"decay_units":15.25,"plant_unwatered_days":119.562,"shed_discarded_units":12.125,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":4.75,"buy_zero_fill":0.125,"sell_dead_units":67.812,"sold_units":940.938,"silent_loss_coins":4396.25,"silent_loss_units":45.125,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":54.562,"fertilizer_forgone":2.688,"care_unfed":0.0,"animal_shed_days":1.562,"idle_tile_days":404.875,"weed_tile_days":36.594,"hand_pass_turns":834.031,"phantom_hand_actions":0.0,"sell_floor_units":27.438,"hire_premium":7656.281,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":15.25,"seeds_unplanted_cost_now":718.75,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":68.969}},{"agent":"mine, earlier build 57","mine":true,"games":32,"bank_median":41315.0,"games_losing_an_animal":32,"escape_first_day_min":16,"escapes_by_day":{"16":52,"17":36,"20":54,"21":18,"28":106},"per_game":{"animals_escaped":8.312,"escape_capital":3618.75,"escape_yield_units":0.0,"animal_unfed_days":18.125,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":18.188,"dry_seed_cost":612.5,"plants_decayed":15.375,"decay_units":18.0,"plant_unwatered_days":81.75,"shed_discarded_units":17.5,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":4.0,"buy_zero_fill":0.0,"sell_dead_units":60.125,"sold_units":916.812,"silent_loss_coins":4231.25,"silent_loss_units":52.875,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":33.125,"fertilizer_forgone":1.188,"care_unfed":0.0,"animal_shed_days":1.938,"idle_tile_days":366.438,"weed_tile_days":36.875,"hand_pass_turns":985.219,"phantom_hand_actions":0.0,"sell_floor_units":20.375,"hire_premium":7186.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":14.188,"seeds_unplanted_cost_now":742.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":95.875}},{"agent":"mine, earlier build 58","mine":true,"games":32,"bank_median":44651.0,"games_losing_an_animal":32,"escape_first_day_min":16,"escapes_by_day":{"16":56,"17":10,"18":24,"20":30,"21":36,"27":4,"28":122},"per_game":{"animals_escaped":8.812,"escape_capital":3781.25,"escape_yield_units":0.0,"animal_unfed_days":19.938,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":9.812,"dry_seed_cost":363.125,"plants_decayed":19.562,"decay_units":23.5,"plant_unwatered_days":66.469,"shed_discarded_units":24.062,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":3.688,"buy_zero_fill":0.062,"sell_dead_units":58.375,"sold_units":945.125,"silent_loss_coins":4144.375,"silent_loss_units":56.75,"refused_buy_product":0.062,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":34.0,"fertilizer_forgone":1.438,"care_unfed":0.0,"animal_shed_days":0.812,"idle_tile_days":403.688,"weed_tile_days":31.156,"hand_pass_turns":973.812,"phantom_hand_actions":0.0,"sell_floor_units":20.75,"hire_premium":7452.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":11.625,"seeds_unplanted_cost_now":560.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":96.969}},{"agent":"mine, earlier build 59","mine":true,"games":32,"bank_median":71731.0,"games_losing_an_animal":2,"escape_first_day_min":16,"escapes_by_day":{"16":2},"per_game":{"animals_escaped":0.062,"escape_capital":25.0,"escape_yield_units":0.0,"animal_unfed_days":19.094,"feed_no_wheat":1.156,"feed_no_animal":0.688,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.781,"decay_units":19.594,"plant_unwatered_days":463.406,"shed_discarded_units":0.062,"dead_actions":31.25,"water_dead":15.375,"harvest_dead":5.562,"plant_dead":2.188,"sell_zero_fill":4.438,"buy_zero_fill":0.062,"sell_dead_units":58.531,"sold_units":1404.469,"silent_loss_coins":145.0,"silent_loss_units":21.656,"refused_buy_product":0.0,"refused_buy_seed":1.125,"refused_buy_animal":0.0,"tile_cap_units":15.406,"fertilizer_forgone":4.938,"care_unfed":10.031,"animal_shed_days":7.0,"idle_tile_days":163.969,"weed_tile_days":18.562,"hand_pass_turns":814.0,"phantom_hand_actions":0.0,"sell_floor_units":178.656,"hire_premium":6886.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.062,"seeds_unplanted_cost_now":84.688,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.719,"fertilize_dead":2.188,"collect_dead":1.688,"care_dead":0.688,"farmer_pass_turns":180.0}},{"agent":"mine, earlier build 60","mine":true,"games":32,"bank_median":68823.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":33.188,"feed_no_wheat":16.188,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":4.0,"dry_seed_cost":130.0,"plants_decayed":8.906,"decay_units":16.719,"plant_unwatered_days":465.562,"shed_discarded_units":0.0,"dead_actions":757.531,"water_dead":30.094,"harvest_dead":6.562,"plant_dead":4.094,"sell_zero_fill":42.875,"buy_zero_fill":5.0,"sell_dead_units":328.0,"sold_units":1351.0,"silent_loss_coins":130.0,"silent_loss_units":20.719,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":11.0,"care_unfed":26.188,"animal_shed_days":1.0,"idle_tile_days":175.719,"weed_tile_days":19.0,"hand_pass_turns":204.0,"phantom_hand_actions":0.0,"sell_floor_units":158.844,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.094,"seeds_unplanted_cost_now":27.812,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.281,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":2.938,"fertilize_dead":0.375,"collect_dead":9.0,"care_dead":640.0,"farmer_pass_turns":54.0}},{"agent":"mine, earlier build 61","mine":true,"games":32,"bank_median":68823.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":33.188,"feed_no_wheat":16.188,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":4.0,"dry_seed_cost":130.0,"plants_decayed":8.906,"decay_units":16.719,"plant_unwatered_days":465.562,"shed_discarded_units":0.0,"dead_actions":757.531,"water_dead":30.094,"harvest_dead":6.562,"plant_dead":4.094,"sell_zero_fill":42.875,"buy_zero_fill":5.0,"sell_dead_units":328.0,"sold_units":1351.0,"silent_loss_coins":130.0,"silent_loss_units":20.719,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":11.0,"care_unfed":26.188,"animal_shed_days":1.0,"idle_tile_days":175.719,"weed_tile_days":19.0,"hand_pass_turns":204.0,"phantom_hand_actions":0.0,"sell_floor_units":158.844,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.094,"seeds_unplanted_cost_now":27.812,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.281,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":2.938,"fertilize_dead":0.375,"collect_dead":9.0,"care_dead":640.0,"farmer_pass_turns":54.0}},{"agent":"mine, earlier build 62","mine":true,"games":32,"bank_median":69468.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":17.188,"feed_no_wheat":2.125,"feed_no_animal":26.562,"feed_redundant":0.0,"plants_dry":4.0,"dry_seed_cost":130.0,"plants_decayed":8.906,"decay_units":16.719,"plant_unwatered_days":465.562,"shed_discarded_units":0.0,"dead_actions":801.625,"water_dead":30.094,"harvest_dead":12.5,"plant_dead":4.094,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":1327.25,"silent_loss_coins":130.0,"silent_loss_units":20.719,"refused_buy_product":0.0,"refused_buy_seed":6.0,"refused_buy_animal":0.938,"tile_cap_units":12.0,"fertilizer_forgone":8.0,"care_unfed":11.188,"animal_shed_days":0.938,"idle_tile_days":177.156,"weed_tile_days":19.0,"hand_pass_turns":204.0,"phantom_hand_actions":0.375,"sell_floor_units":169.188,"hire_premium":6887.875,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.094,"seeds_unplanted_cost_now":27.812,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":0.938,"drop_dead":0.0,"place_dead":1.938,"build_dead":0.0,"dig_dead":2.938,"fertilize_dead":0.375,"collect_dead":26.5,"care_dead":658.562,"farmer_pass_turns":54.0}},{"agent":"mine, earlier build 63","mine":true,"games":32,"bank_median":39592.0,"games_losing_an_animal":15,"escape_first_day_min":7,"escapes_by_day":{"7":2,"17":2,"18":7,"21":2,"22":2,"23":2},"per_game":{"animals_escaped":0.469,"escape_capital":175.0,"escape_yield_units":0.0,"animal_unfed_days":17.969,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":22.406,"plants_dry":73.969,"dry_seed_cost":1622.5,"plants_decayed":12.094,"decay_units":26.125,"plant_unwatered_days":195.5,"shed_discarded_units":3.219,"dead_actions":37.219,"water_dead":9.875,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":44.688,"buy_zero_fill":0.75,"sell_dead_units":22932.719,"sold_units":822.281,"silent_loss_coins":1797.5,"silent_loss_units":106.531,"refused_buy_product":7.188,"refused_buy_seed":0.188,"refused_buy_animal":2.0,"tile_cap_units":4.906,"fertilizer_forgone":3.438,"care_unfed":16.594,"animal_shed_days":8.875,"idle_tile_days":1096.594,"weed_tile_days":251.094,"hand_pass_turns":219.094,"phantom_hand_actions":0.0,"sell_floor_units":40.0,"hire_premium":4698.938,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":16.438,"seeds_unplanted_cost_now":1338.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.531,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":3.406,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":3.875}},{"agent":"mine, earlier build 64","mine":true,"games":32,"bank_median":53368.0,"games_losing_an_animal":8,"escape_first_day_min":18,"escapes_by_day":{"18":8,"23":4},"per_game":{"animals_escaped":0.375,"escape_capital":150.0,"escape_yield_units":0.0,"animal_unfed_days":11.688,"feed_no_wheat":3.812,"feed_no_animal":7.812,"feed_redundant":0.0,"plants_dry":1.562,"dry_seed_cost":148.75,"plants_decayed":22.938,"decay_units":25.25,"plant_unwatered_days":383.219,"shed_discarded_units":0.0,"dead_actions":74.062,"water_dead":16.812,"harvest_dead":9.062,"plant_dead":3.094,"sell_zero_fill":241.406,"buy_zero_fill":0.125,"sell_dead_units":2275.219,"sold_units":1596.781,"silent_loss_coins":298.75,"silent_loss_units":26.125,"refused_buy_product":3.0,"refused_buy_seed":0.125,"refused_buy_animal":0.0,"tile_cap_units":14.625,"fertilizer_forgone":9.062,"care_unfed":3.438,"animal_shed_days":4.875,"idle_tile_days":208.188,"weed_tile_days":40.5,"hand_pass_turns":592.75,"phantom_hand_actions":10.938,"sell_floor_units":258.75,"hire_premium":5676.125,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.219,"seeds_unplanted_now":19.344,"seeds_unplanted_cost_now":315.312,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":14.75,"drop_dead":0.0,"place_dead":0.75,"build_dead":0.094,"dig_dead":0.125,"fertilize_dead":2.531,"collect_dead":7.469,"care_dead":7.75,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 65","mine":true,"games":32,"bank_median":39976.0,"games_losing_an_animal":32,"escape_first_day_min":2,"escapes_by_day":{"2":32},"per_game":{"animals_escaped":1.0,"escape_capital":500.0,"escape_yield_units":0.0,"animal_unfed_days":11.0,"feed_no_wheat":5.0,"feed_no_animal":133.5,"feed_redundant":0.0,"plants_dry":0.0,"dry_seed_cost":0.0,"plants_decayed":4.031,"decay_units":4.156,"plant_unwatered_days":168.156,"shed_discarded_units":1.125,"dead_actions":1297.375,"water_dead":113.188,"harvest_dead":90.75,"plant_dead":2.031,"sell_zero_fill":0.0,"buy_zero_fill":9.062,"sell_dead_units":0.0,"sold_units":1139.031,"silent_loss_coins":500.0,"silent_loss_units":5.281,"refused_buy_product":10.062,"refused_buy_seed":8.0,"refused_buy_animal":2.0,"tile_cap_units":1.875,"fertilizer_forgone":6.062,"care_unfed":4.0,"animal_shed_days":121.5,"idle_tile_days":360.406,"weed_tile_days":21.562,"hand_pass_turns":594.5,"phantom_hand_actions":2.5,"sell_floor_units":23.0,"hire_premium":5691.125,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":6.062,"seeds_unplanted_now":65.094,"seeds_unplanted_cost_now":2993.75,"move_dead":0.0,"locked_dead":628.0,"pickup_dead":4.062,"drop_dead":0.0,"place_dead":27.156,"build_dead":0.062,"dig_dead":6.688,"fertilize_dead":22.0,"collect_dead":132.438,"care_dead":132.5,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 66","mine":true,"games":32,"bank_median":53004.0,"games_losing_an_animal":8,"escape_first_day_min":18,"escapes_by_day":{"18":8,"23":4},"per_game":{"animals_escaped":0.375,"escape_capital":150.0,"escape_yield_units":0.0,"animal_unfed_days":11.688,"feed_no_wheat":3.812,"feed_no_animal":7.812,"feed_redundant":0.0,"plants_dry":1.562,"dry_seed_cost":148.75,"plants_decayed":22.938,"decay_units":25.25,"plant_unwatered_days":383.219,"shed_discarded_units":0.0,"dead_actions":74.062,"water_dead":16.812,"harvest_dead":9.062,"plant_dead":3.094,"sell_zero_fill":241.812,"buy_zero_fill":0.125,"sell_dead_units":2251.219,"sold_units":1596.781,"silent_loss_coins":298.75,"silent_loss_units":26.125,"refused_buy_product":3.0,"refused_buy_seed":0.125,"refused_buy_animal":0.0,"tile_cap_units":14.625,"fertilizer_forgone":9.062,"care_unfed":3.438,"animal_shed_days":4.875,"idle_tile_days":208.188,"weed_tile_days":40.5,"hand_pass_turns":592.75,"phantom_hand_actions":10.938,"sell_floor_units":257.969,"hire_premium":5676.125,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.219,"seeds_unplanted_now":19.344,"seeds_unplanted_cost_now":315.312,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":14.75,"drop_dead":0.0,"place_dead":0.75,"build_dead":0.094,"dig_dead":0.125,"fertilize_dead":2.531,"collect_dead":7.469,"care_dead":7.75,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 67","mine":true,"games":32,"bank_median":53368.0,"games_losing_an_animal":8,"escape_first_day_min":18,"escapes_by_day":{"18":8,"23":4},"per_game":{"animals_escaped":0.375,"escape_capital":150.0,"escape_yield_units":0.0,"animal_unfed_days":11.688,"feed_no_wheat":3.812,"feed_no_animal":7.812,"feed_redundant":0.0,"plants_dry":1.562,"dry_seed_cost":148.75,"plants_decayed":22.938,"decay_units":25.25,"plant_unwatered_days":383.219,"shed_discarded_units":0.0,"dead_actions":74.062,"water_dead":16.812,"harvest_dead":9.062,"plant_dead":3.094,"sell_zero_fill":241.656,"buy_zero_fill":0.125,"sell_dead_units":2275.219,"sold_units":1596.781,"silent_loss_coins":298.75,"silent_loss_units":26.125,"refused_buy_product":3.0,"refused_buy_seed":0.125,"refused_buy_animal":0.0,"tile_cap_units":14.625,"fertilizer_forgone":9.062,"care_unfed":3.438,"animal_shed_days":4.875,"idle_tile_days":208.188,"weed_tile_days":40.5,"hand_pass_turns":592.75,"phantom_hand_actions":10.938,"sell_floor_units":258.75,"hire_premium":5676.125,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.219,"seeds_unplanted_now":19.344,"seeds_unplanted_cost_now":315.312,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":14.75,"drop_dead":0.0,"place_dead":0.75,"build_dead":0.094,"dig_dead":0.125,"fertilize_dead":2.531,"collect_dead":7.469,"care_dead":7.75,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 68","mine":true,"games":32,"bank_median":68823.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":33.188,"feed_no_wheat":16.188,"feed_no_animal":7.0,"feed_redundant":0.0,"plants_dry":4.0,"dry_seed_cost":130.0,"plants_decayed":8.906,"decay_units":16.719,"plant_unwatered_days":465.562,"shed_discarded_units":0.0,"dead_actions":757.531,"water_dead":30.094,"harvest_dead":6.562,"plant_dead":4.094,"sell_zero_fill":42.812,"buy_zero_fill":5.0,"sell_dead_units":328.0,"sold_units":1351.0,"silent_loss_coins":130.0,"silent_loss_units":20.719,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":11.0,"care_unfed":26.188,"animal_shed_days":1.0,"idle_tile_days":175.719,"weed_tile_days":19.0,"hand_pass_turns":204.0,"phantom_hand_actions":0.0,"sell_floor_units":158.844,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.094,"seeds_unplanted_cost_now":27.812,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.281,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":2.938,"fertilize_dead":0.375,"collect_dead":9.0,"care_dead":640.0,"farmer_pass_turns":54.0}},{"agent":"mine, earlier build 69","mine":true,"games":32,"bank_median":56185.0,"games_losing_an_animal":6,"escape_first_day_min":18,"escapes_by_day":{"18":2,"28":12},"per_game":{"animals_escaped":0.438,"escape_capital":212.5,"escape_yield_units":0.0,"animal_unfed_days":9.188,"feed_no_wheat":0.188,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.594,"dry_seed_cost":507.5,"plants_decayed":25.094,"decay_units":27.312,"plant_unwatered_days":411.344,"shed_discarded_units":11.75,"dead_actions":20.688,"water_dead":4.844,"harvest_dead":1.969,"plant_dead":0.375,"sell_zero_fill":240.219,"buy_zero_fill":0.0,"sell_dead_units":3020.219,"sold_units":1597.875,"silent_loss_coins":720.0,"silent_loss_units":40.875,"refused_buy_product":1.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":10.5,"care_unfed":0.125,"animal_shed_days":1.75,"idle_tile_days":237.656,"weed_tile_days":34.875,"hand_pass_turns":511.781,"phantom_hand_actions":0.0,"sell_floor_units":240.469,"hire_premium":5691.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":10.062,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.5,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":47.375}},{"agent":"mine, earlier build 70","mine":true,"games":32,"bank_median":56185.0,"games_losing_an_animal":6,"escape_first_day_min":18,"escapes_by_day":{"18":2,"28":12},"per_game":{"animals_escaped":0.438,"escape_capital":212.5,"escape_yield_units":0.0,"animal_unfed_days":9.188,"feed_no_wheat":0.188,"feed_no_animal":0.625,"feed_redundant":0.0,"plants_dry":6.594,"dry_seed_cost":507.5,"plants_decayed":25.094,"decay_units":27.312,"plant_unwatered_days":411.344,"shed_discarded_units":11.75,"dead_actions":20.688,"water_dead":4.844,"harvest_dead":1.969,"plant_dead":0.375,"sell_zero_fill":240.219,"buy_zero_fill":0.0,"sell_dead_units":3020.219,"sold_units":1597.875,"silent_loss_coins":720.0,"silent_loss_units":40.875,"refused_buy_product":1.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.969,"fertilizer_forgone":10.5,"care_unfed":0.125,"animal_shed_days":1.75,"idle_tile_days":237.656,"weed_tile_days":34.875,"hand_pass_turns":511.781,"phantom_hand_actions":0.0,"sell_floor_units":240.469,"hire_premium":5691.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":10.062,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.5,"collect_dead":0.562,"care_dead":0.562,"farmer_pass_turns":47.375}},{"agent":"mine, earlier build 71","mine":true,"games":32,"bank_median":78773.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":4},"per_game":{"animals_escaped":0.125,"escape_capital":56.25,"escape_yield_units":0.0,"animal_unfed_days":31.812,"feed_no_wheat":14.938,"feed_no_animal":9.375,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.844,"decay_units":19.656,"plant_unwatered_days":462.375,"shed_discarded_units":0.312,"dead_actions":766.938,"water_dead":27.875,"harvest_dead":6.875,"plant_dead":5.094,"sell_zero_fill":17.188,"buy_zero_fill":5.0,"sell_dead_units":314.312,"sold_units":1364.688,"silent_loss_coins":176.25,"silent_loss_units":21.969,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.125,"fertilizer_forgone":4.938,"care_unfed":25.75,"animal_shed_days":1.0,"idle_tile_days":174.438,"weed_tile_days":19.031,"hand_pass_turns":239.0,"phantom_hand_actions":23.0,"sell_floor_units":162.781,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.094,"seeds_unplanted_cost_now":30.625,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.219,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.75,"collect_dead":11.5,"care_dead":648.312,"farmer_pass_turns":61.0}},{"agent":"mine, earlier build 72","mine":true,"games":32,"bank_median":82737.0,"games_losing_an_animal":2,"escape_first_day_min":16,"escapes_by_day":{"16":2},"per_game":{"animals_escaped":0.062,"escape_capital":25.0,"escape_yield_units":0.0,"animal_unfed_days":19.094,"feed_no_wheat":1.156,"feed_no_animal":0.688,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.594,"decay_units":19.406,"plant_unwatered_days":461.531,"shed_discarded_units":0.062,"dead_actions":37.5,"water_dead":18.062,"harvest_dead":8.25,"plant_dead":2.5,"sell_zero_fill":5.375,"buy_zero_fill":0.688,"sell_dead_units":108.594,"sold_units":1396.406,"silent_loss_coins":145.0,"silent_loss_units":21.469,"refused_buy_product":1.375,"refused_buy_seed":1.5,"refused_buy_animal":0.0,"tile_cap_units":11.719,"fertilizer_forgone":4.938,"care_unfed":10.031,"animal_shed_days":7.0,"idle_tile_days":168.406,"weed_tile_days":18.562,"hand_pass_turns":814.0,"phantom_hand_actions":0.0,"sell_floor_units":172.812,"hire_premium":6886.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.0,"seeds_unplanted_cost_now":78.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":1.906,"fertilize_dead":2.562,"collect_dead":1.688,"care_dead":0.688,"farmer_pass_turns":180.0}},{"agent":"mine, earlier build 73","mine":true,"games":32,"bank_median":53142.0,"games_losing_an_animal":4,"escape_first_day_min":12,"escapes_by_day":{"12":4,"20":4,"21":2},"per_game":{"animals_escaped":0.312,"escape_capital":125.0,"escape_yield_units":0.0,"animal_unfed_days":16.688,"feed_no_wheat":8.688,"feed_no_animal":10.625,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":186.25,"plants_decayed":22.5,"decay_units":24.812,"plant_unwatered_days":378.281,"shed_discarded_units":0.0,"dead_actions":97.438,"water_dead":25.938,"harvest_dead":13.75,"plant_dead":3.906,"sell_zero_fill":264.562,"buy_zero_fill":0.906,"sell_dead_units":2328.844,"sold_units":1412.156,"silent_loss_coins":311.25,"silent_loss_units":26.375,"refused_buy_product":3.688,"refused_buy_seed":0.438,"refused_buy_animal":0.0,"tile_cap_units":11.5,"fertilizer_forgone":9.312,"care_unfed":8.375,"animal_shed_days":7.25,"idle_tile_days":224.438,"weed_tile_days":47.938,"hand_pass_turns":590.438,"phantom_hand_actions":21.75,"sell_floor_units":247.094,"hire_premium":5644.062,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.344,"seeds_unplanted_now":4.656,"seeds_unplanted_cost_now":133.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":9.125,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.094,"dig_dead":0.25,"fertilize_dead":3.344,"collect_dead":9.906,"care_dead":10.5,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 74","mine":true,"games":32,"bank_median":53142.0,"games_losing_an_animal":4,"escape_first_day_min":12,"escapes_by_day":{"12":4,"20":4,"21":2},"per_game":{"animals_escaped":0.312,"escape_capital":125.0,"escape_yield_units":0.0,"animal_unfed_days":16.688,"feed_no_wheat":8.688,"feed_no_animal":10.625,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":186.25,"plants_decayed":22.5,"decay_units":24.812,"plant_unwatered_days":378.281,"shed_discarded_units":0.0,"dead_actions":97.438,"water_dead":25.938,"harvest_dead":13.75,"plant_dead":3.906,"sell_zero_fill":264.562,"buy_zero_fill":0.906,"sell_dead_units":2328.844,"sold_units":1412.156,"silent_loss_coins":311.25,"silent_loss_units":26.375,"refused_buy_product":3.688,"refused_buy_seed":0.438,"refused_buy_animal":0.0,"tile_cap_units":11.5,"fertilizer_forgone":9.312,"care_unfed":8.375,"animal_shed_days":7.25,"idle_tile_days":224.438,"weed_tile_days":47.938,"hand_pass_turns":590.438,"phantom_hand_actions":21.75,"sell_floor_units":247.094,"hire_premium":5644.062,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.344,"seeds_unplanted_now":4.656,"seeds_unplanted_cost_now":133.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":9.125,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.094,"dig_dead":0.25,"fertilize_dead":3.344,"collect_dead":9.906,"care_dead":10.5,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 75","mine":true,"games":32,"bank_median":53142.0,"games_losing_an_animal":4,"escape_first_day_min":12,"escapes_by_day":{"12":4,"20":4,"21":2},"per_game":{"animals_escaped":0.312,"escape_capital":125.0,"escape_yield_units":0.0,"animal_unfed_days":16.688,"feed_no_wheat":8.688,"feed_no_animal":10.625,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":186.25,"plants_decayed":22.5,"decay_units":24.812,"plant_unwatered_days":378.281,"shed_discarded_units":0.0,"dead_actions":97.438,"water_dead":25.938,"harvest_dead":13.75,"plant_dead":3.906,"sell_zero_fill":264.562,"buy_zero_fill":0.906,"sell_dead_units":2328.844,"sold_units":1412.156,"silent_loss_coins":311.25,"silent_loss_units":26.375,"refused_buy_product":3.688,"refused_buy_seed":0.438,"refused_buy_animal":0.0,"tile_cap_units":11.5,"fertilizer_forgone":9.312,"care_unfed":8.375,"animal_shed_days":7.25,"idle_tile_days":224.438,"weed_tile_days":47.938,"hand_pass_turns":590.438,"phantom_hand_actions":21.75,"sell_floor_units":247.094,"hire_premium":5644.062,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.344,"seeds_unplanted_now":4.656,"seeds_unplanted_cost_now":133.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":9.125,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.094,"dig_dead":0.25,"fertilize_dead":3.344,"collect_dead":9.906,"care_dead":10.5,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 76","mine":true,"games":32,"bank_median":53142.0,"games_losing_an_animal":4,"escape_first_day_min":12,"escapes_by_day":{"12":4,"20":4,"21":2},"per_game":{"animals_escaped":0.312,"escape_capital":125.0,"escape_yield_units":0.0,"animal_unfed_days":16.688,"feed_no_wheat":8.688,"feed_no_animal":10.625,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":186.25,"plants_decayed":22.5,"decay_units":24.812,"plant_unwatered_days":378.281,"shed_discarded_units":0.0,"dead_actions":97.438,"water_dead":25.938,"harvest_dead":13.75,"plant_dead":3.906,"sell_zero_fill":264.562,"buy_zero_fill":0.906,"sell_dead_units":2328.844,"sold_units":1412.156,"silent_loss_coins":311.25,"silent_loss_units":26.375,"refused_buy_product":3.688,"refused_buy_seed":0.438,"refused_buy_animal":0.0,"tile_cap_units":11.5,"fertilizer_forgone":9.312,"care_unfed":8.375,"animal_shed_days":7.25,"idle_tile_days":224.438,"weed_tile_days":47.938,"hand_pass_turns":590.438,"phantom_hand_actions":21.75,"sell_floor_units":247.094,"hire_premium":5644.062,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.344,"seeds_unplanted_now":4.656,"seeds_unplanted_cost_now":133.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":9.125,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.094,"dig_dead":0.25,"fertilize_dead":3.344,"collect_dead":9.906,"care_dead":10.5,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 77","mine":true,"games":32,"bank_median":53391.0,"games_losing_an_animal":2,"escape_first_day_min":12,"escapes_by_day":{"12":4,"20":4},"per_game":{"animals_escaped":0.25,"escape_capital":100.0,"escape_yield_units":0.0,"animal_unfed_days":17.25,"feed_no_wheat":9.25,"feed_no_animal":10.188,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":186.25,"plants_decayed":22.5,"decay_units":24.812,"plant_unwatered_days":378.281,"shed_discarded_units":0.594,"dead_actions":97.219,"water_dead":25.938,"harvest_dead":13.5,"plant_dead":3.906,"sell_zero_fill":266.438,"buy_zero_fill":0.906,"sell_dead_units":2329.875,"sold_units":1411.125,"silent_loss_coins":286.25,"silent_loss_units":26.969,"refused_buy_product":3.688,"refused_buy_seed":0.438,"refused_buy_animal":0.0,"tile_cap_units":11.5,"fertilizer_forgone":9.375,"care_unfed":9.0,"animal_shed_days":7.25,"idle_tile_days":224.438,"weed_tile_days":47.938,"hand_pass_turns":590.438,"phantom_hand_actions":21.75,"sell_floor_units":247.562,"hire_premium":5644.062,"shed_units_now":0.125,"shed_value_now":2.562,"animals_in_shed_now":0.344,"seeds_unplanted_now":4.656,"seeds_unplanted_cost_now":133.438,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":9.969,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.094,"dig_dead":0.25,"fertilize_dead":3.156,"collect_dead":9.531,"care_dead":10.125,"farmer_pass_turns":72.0}},{"agent":"mine, earlier build 78","mine":true,"games":32,"bank_median":70286.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{"19":1},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":11.5,"feed_no_wheat":6.5,"feed_no_animal":1.312,"feed_redundant":0.0,"plants_dry":4.969,"dry_seed_cost":139.688,"plants_decayed":14.938,"decay_units":19.938,"plant_unwatered_days":481.844,"shed_discarded_units":0.688,"dead_actions":69.0,"water_dead":23.094,"harvest_dead":9.531,"plant_dead":4.25,"sell_zero_fill":158.188,"buy_zero_fill":2.0,"sell_dead_units":511.656,"sold_units":1271.344,"silent_loss_coins":139.688,"silent_loss_units":24.594,"refused_buy_product":0.812,"refused_buy_seed":2.625,"refused_buy_animal":0.0,"tile_cap_units":15.844,"fertilizer_forgone":14.938,"care_unfed":11.5,"animal_shed_days":8.312,"idle_tile_days":130.062,"weed_tile_days":18.438,"hand_pass_turns":397.0,"phantom_hand_actions":1.312,"sell_floor_units":187.562,"hire_premium":8018.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.062,"seeds_unplanted_now":7.625,"seeds_unplanted_cost_now":81.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":12.125,"drop_dead":1.0,"place_dead":0.062,"build_dead":0.062,"dig_dead":3.25,"fertilize_dead":3.438,"collect_dead":2.188,"care_dead":2.188,"farmer_pass_turns":3.0}},{"agent":"mine, earlier build 79","mine":true,"games":32,"bank_median":3000.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":0.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":0.0,"dry_seed_cost":0.0,"plants_decayed":0.0,"decay_units":0.0,"plant_unwatered_days":0.0,"shed_discarded_units":0.0,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":0.0,"silent_loss_coins":0.0,"silent_loss_units":0.0,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":0.0,"care_unfed":0.0,"animal_shed_days":0.0,"idle_tile_days":725.0,"weed_tile_days":51.562,"hand_pass_turns":0.0,"phantom_hand_actions":0.0,"sell_floor_units":0.0,"hire_premium":0.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":719.0}},{"agent":"mine, earlier build 80","mine":true,"games":32,"bank_median":77522.0,"games_losing_an_animal":22,"escape_first_day_min":8,"escapes_by_day":{"8":22},"per_game":{"animals_escaped":0.688,"escape_capital":275.0,"escape_yield_units":0.0,"animal_unfed_days":47.375,"feed_no_wheat":0.0,"feed_no_animal":24.875,"feed_redundant":0.0,"plants_dry":0.562,"dry_seed_cost":33.75,"plants_decayed":18.844,"decay_units":19.906,"plant_unwatered_days":389.781,"shed_discarded_units":40.781,"dead_actions":106.781,"water_dead":9.938,"harvest_dead":14.969,"plant_dead":0.938,"sell_zero_fill":13.031,"buy_zero_fill":1.312,"sell_dead_units":954.062,"sold_units":1347.312,"silent_loss_coins":308.75,"silent_loss_units":60.938,"refused_buy_product":2.688,"refused_buy_seed":0.25,"refused_buy_animal":0.562,"tile_cap_units":3.625,"fertilizer_forgone":5.5,"care_unfed":24.25,"animal_shed_days":0.25,"idle_tile_days":85.906,"weed_tile_days":18.812,"hand_pass_turns":718.688,"phantom_hand_actions":0.0,"sell_floor_units":96.594,"hire_premium":4428.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":8.438,"seeds_unplanted_cost_now":832.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.375,"drop_dead":0.0,"place_dead":2.812,"build_dead":0.0,"dig_dead":1.562,"fertilize_dead":1.438,"collect_dead":25.875,"care_dead":24.0,"farmer_pass_turns":83.625}},{"agent":"mine, earlier build 81","mine":true,"games":32,"bank_median":46459.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":12.0,"feed_no_wheat":0.0,"feed_no_animal":108.0,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":10.0,"plants_decayed":3.938,"decay_units":11.812,"plant_unwatered_days":196.312,"shed_discarded_units":71.938,"dead_actions":1206.656,"water_dead":112.062,"harvest_dead":101.188,"plant_dead":1.0,"sell_zero_fill":0.0,"buy_zero_fill":0.438,"sell_dead_units":0.0,"sold_units":833.5,"silent_loss_coins":10.0,"silent_loss_units":84.75,"refused_buy_product":0.062,"refused_buy_seed":0.75,"refused_buy_animal":0.938,"tile_cap_units":4.0,"fertilizer_forgone":1.0,"care_unfed":7.0,"animal_shed_days":116.438,"idle_tile_days":391.062,"weed_tile_days":20.031,"hand_pass_turns":814.0,"phantom_hand_actions":0.0,"sell_floor_units":49.125,"hire_premium":6886.0,"shed_units_now":12.0,"shed_value_now":471.0,"animals_in_shed_now":5.062,"seeds_unplanted_now":80.125,"seeds_unplanted_cost_now":4080.625,"move_dead":0.0,"locked_dead":621.875,"pickup_dead":3.0,"drop_dead":0.0,"place_dead":5.0,"build_dead":0.0,"dig_dead":13.281,"fertilize_dead":26.25,"collect_dead":112.0,"care_dead":103.0,"farmer_pass_turns":180.0}},{"agent":"mine, earlier build 82","mine":true,"games":32,"bank_median":77511.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":12.688,"feed_no_wheat":0.812,"feed_no_animal":18.75,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":32.5,"plants_decayed":31.062,"decay_units":31.062,"plant_unwatered_days":390.75,"shed_discarded_units":5.219,"dead_actions":115.938,"water_dead":29.438,"harvest_dead":9.25,"plant_dead":0.438,"sell_zero_fill":16.281,"buy_zero_fill":3.938,"sell_dead_units":74.781,"sold_units":1430.844,"silent_loss_coins":357.5,"silent_loss_units":37.031,"refused_buy_product":7.375,"refused_buy_seed":0.375,"refused_buy_animal":0.188,"tile_cap_units":4.125,"fertilizer_forgone":3.562,"care_unfed":10.875,"animal_shed_days":0.25,"idle_tile_days":60.938,"weed_tile_days":2.406,"hand_pass_turns":427.75,"phantom_hand_actions":0.0,"sell_floor_units":160.969,"hire_premium":5686.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.812,"seeds_unplanted_cost_now":311.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.125,"drop_dead":0.0,"place_dead":0.562,"build_dead":0.0,"dig_dead":0.625,"fertilize_dead":1.0,"collect_dead":19.812,"care_dead":35.125,"farmer_pass_turns":57.938}},{"agent":"mine, earlier build 83","mine":true,"games":32,"bank_median":78690.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":11.0,"feed_no_wheat":1.0,"feed_no_animal":26.0,"feed_redundant":2.0,"plants_dry":3.0,"dry_seed_cost":30.0,"plants_decayed":33.688,"decay_units":33.688,"plant_unwatered_days":387.438,"shed_discarded_units":6.188,"dead_actions":148.188,"water_dead":30.0,"harvest_dead":22.312,"plant_dead":3.375,"sell_zero_fill":74.312,"buy_zero_fill":1.875,"sell_dead_units":426.25,"sold_units":1435.625,"silent_loss_coins":30.0,"silent_loss_units":42.875,"refused_buy_product":44.25,"refused_buy_seed":3.312,"refused_buy_animal":0.0,"tile_cap_units":5.0,"fertilizer_forgone":0.0,"care_unfed":10.0,"animal_shed_days":27.0,"idle_tile_days":130.5,"weed_tile_days":10.188,"hand_pass_turns":764.906,"phantom_hand_actions":0.0,"sell_floor_units":156.0,"hire_premium":6006.0,"shed_units_now":5.938,"shed_value_now":172.375,"animals_in_shed_now":1.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":4.75,"build_dead":0.0,"dig_dead":2.125,"fertilize_dead":4.625,"collect_dead":27.0,"care_dead":25.0,"farmer_pass_turns":70.0}},{"agent":"mine, earlier build 84","mine":true,"games":32,"bank_median":70569.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":35.188,"feed_no_wheat":0.812,"feed_no_animal":14.625,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":383.656,"shed_discarded_units":10.188,"dead_actions":99.969,"water_dead":18.219,"harvest_dead":13.0,"plant_dead":5.062,"sell_zero_fill":33.938,"buy_zero_fill":6.188,"sell_dead_units":283.406,"sold_units":1476.219,"silent_loss_coins":343.125,"silent_loss_units":36.938,"refused_buy_product":10.938,"refused_buy_seed":1.0,"refused_buy_animal":1.0,"tile_cap_units":2.0,"fertilizer_forgone":10.0,"care_unfed":18.188,"animal_shed_days":1.0,"idle_tile_days":62.875,"weed_tile_days":3.469,"hand_pass_turns":509.688,"phantom_hand_actions":0.0,"sell_floor_units":216.75,"hire_premium":5669.375,"shed_units_now":2.0,"shed_value_now":66.188,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.062,"seeds_unplanted_cost_now":36.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.812,"drop_dead":0.0,"place_dead":3.188,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.938,"collect_dead":17.062,"care_dead":25.25,"farmer_pass_turns":62.625}},{"agent":"mine, earlier build 85","mine":true,"games":32,"bank_median":83160.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":34.5,"feed_no_wheat":0.0,"feed_no_animal":4.5,"feed_redundant":1.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":21.969,"decay_units":22.969,"plant_unwatered_days":376.406,"shed_discarded_units":2.812,"dead_actions":81.688,"water_dead":38.688,"harvest_dead":12.875,"plant_dead":4.719,"sell_zero_fill":13.906,"buy_zero_fill":2.75,"sell_dead_units":184.438,"sold_units":1521.562,"silent_loss_coins":18.125,"silent_loss_units":27.594,"refused_buy_product":3.75,"refused_buy_seed":2.0,"refused_buy_animal":0.25,"tile_cap_units":1.0,"fertilizer_forgone":9.5,"care_unfed":15.75,"animal_shed_days":2.0,"idle_tile_days":108.594,"weed_tile_days":7.625,"hand_pass_turns":635.0,"phantom_hand_actions":0.0,"sell_floor_units":232.969,"hire_premium":5696.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":5.719,"seeds_unplanted_cost_now":240.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.75,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":3.844,"fertilize_dead":4.312,"collect_dead":5.25,"care_dead":4.5,"farmer_pass_turns":64.0}},{"agent":"mine, earlier build 86","mine":true,"games":32,"bank_median":83160.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":34.5,"feed_no_wheat":0.0,"feed_no_animal":4.5,"feed_redundant":1.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":21.969,"decay_units":22.969,"plant_unwatered_days":376.406,"shed_discarded_units":2.812,"dead_actions":81.688,"water_dead":38.688,"harvest_dead":12.875,"plant_dead":4.719,"sell_zero_fill":13.969,"buy_zero_fill":2.75,"sell_dead_units":184.5,"sold_units":1521.562,"silent_loss_coins":18.125,"silent_loss_units":27.594,"refused_buy_product":3.75,"refused_buy_seed":2.0,"refused_buy_animal":0.25,"tile_cap_units":1.0,"fertilizer_forgone":9.5,"care_unfed":15.75,"animal_shed_days":2.0,"idle_tile_days":108.594,"weed_tile_days":7.625,"hand_pass_turns":635.0,"phantom_hand_actions":0.0,"sell_floor_units":232.969,"hire_premium":5696.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":5.719,"seeds_unplanted_cost_now":240.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.75,"drop_dead":0.0,"place_dead":1.25,"build_dead":0.0,"dig_dead":3.844,"fertilize_dead":4.312,"collect_dead":5.25,"care_dead":4.5,"farmer_pass_turns":64.0}},{"agent":"mine, earlier build 87","mine":true,"games":32,"bank_median":70569.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":35.188,"feed_no_wheat":0.812,"feed_no_animal":14.625,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":383.656,"shed_discarded_units":10.188,"dead_actions":99.969,"water_dead":18.219,"harvest_dead":13.0,"plant_dead":5.062,"sell_zero_fill":33.938,"buy_zero_fill":6.188,"sell_dead_units":283.469,"sold_units":1476.219,"silent_loss_coins":343.125,"silent_loss_units":36.938,"refused_buy_product":10.938,"refused_buy_seed":1.0,"refused_buy_animal":1.0,"tile_cap_units":2.0,"fertilizer_forgone":10.0,"care_unfed":18.188,"animal_shed_days":1.0,"idle_tile_days":62.875,"weed_tile_days":3.469,"hand_pass_turns":509.688,"phantom_hand_actions":0.0,"sell_floor_units":216.75,"hire_premium":5669.375,"shed_units_now":2.0,"shed_value_now":66.188,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.062,"seeds_unplanted_cost_now":36.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.812,"drop_dead":0.0,"place_dead":3.188,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.938,"collect_dead":17.062,"care_dead":25.25,"farmer_pass_turns":62.625}},{"agent":"mine, earlier build 88","mine":true,"games":32,"bank_median":70705.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":35.188,"feed_no_wheat":0.812,"feed_no_animal":14.625,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":383.656,"shed_discarded_units":7.5,"dead_actions":101.781,"water_dead":18.219,"harvest_dead":13.0,"plant_dead":5.062,"sell_zero_fill":38.812,"buy_zero_fill":6.188,"sell_dead_units":353.406,"sold_units":1479.281,"silent_loss_coins":343.125,"silent_loss_units":34.25,"refused_buy_product":10.938,"refused_buy_seed":1.0,"refused_buy_animal":1.0,"tile_cap_units":2.0,"fertilizer_forgone":10.0,"care_unfed":18.188,"animal_shed_days":1.0,"idle_tile_days":62.875,"weed_tile_days":3.469,"hand_pass_turns":509.688,"phantom_hand_actions":0.0,"sell_floor_units":218.0,"hire_premium":5669.375,"shed_units_now":1.625,"shed_value_now":61.75,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.062,"seeds_unplanted_cost_now":36.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":3.625,"drop_dead":0.0,"place_dead":3.188,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.938,"collect_dead":17.062,"care_dead":25.25,"farmer_pass_turns":62.625}},{"agent":"mine, earlier build 89","mine":true,"games":32,"bank_median":70705.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":35.188,"feed_no_wheat":0.812,"feed_no_animal":14.625,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":383.656,"shed_discarded_units":7.5,"dead_actions":101.781,"water_dead":18.219,"harvest_dead":13.0,"plant_dead":5.062,"sell_zero_fill":38.812,"buy_zero_fill":6.188,"sell_dead_units":353.406,"sold_units":1479.281,"silent_loss_coins":343.125,"silent_loss_units":34.25,"refused_buy_product":10.938,"refused_buy_seed":1.0,"refused_buy_animal":1.0,"tile_cap_units":2.0,"fertilizer_forgone":10.0,"care_unfed":18.188,"animal_shed_days":1.0,"idle_tile_days":62.875,"weed_tile_days":3.469,"hand_pass_turns":509.688,"phantom_hand_actions":0.0,"sell_floor_units":218.0,"hire_premium":5669.375,"shed_units_now":1.625,"shed_value_now":61.75,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.062,"seeds_unplanted_cost_now":36.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":3.625,"drop_dead":0.0,"place_dead":3.188,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.938,"collect_dead":17.062,"care_dead":25.25,"farmer_pass_turns":62.625}},{"agent":"mine, earlier build 90","mine":true,"games":32,"bank_median":70705.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":26},"per_game":{"animals_escaped":0.812,"escape_capital":325.0,"escape_yield_units":0.0,"animal_unfed_days":35.188,"feed_no_wheat":0.812,"feed_no_animal":14.625,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":18.125,"plants_decayed":23.938,"decay_units":24.938,"plant_unwatered_days":383.656,"shed_discarded_units":7.5,"dead_actions":101.781,"water_dead":18.219,"harvest_dead":13.0,"plant_dead":5.062,"sell_zero_fill":38.812,"buy_zero_fill":6.188,"sell_dead_units":353.406,"sold_units":1479.281,"silent_loss_coins":343.125,"silent_loss_units":34.25,"refused_buy_product":10.938,"refused_buy_seed":1.0,"refused_buy_animal":1.0,"tile_cap_units":2.0,"fertilizer_forgone":10.0,"care_unfed":18.188,"animal_shed_days":1.0,"idle_tile_days":62.875,"weed_tile_days":3.469,"hand_pass_turns":509.688,"phantom_hand_actions":0.0,"sell_floor_units":218.0,"hire_premium":5669.375,"shed_units_now":1.625,"shed_value_now":61.75,"animals_in_shed_now":0.0,"seeds_unplanted_now":3.062,"seeds_unplanted_cost_now":36.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":3.625,"drop_dead":0.0,"place_dead":3.188,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.938,"collect_dead":17.062,"care_dead":25.25,"farmer_pass_turns":62.625}},{"agent":"mine, earlier build 91","mine":true,"games":32,"bank_median":62740.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":102,"9":2},"per_game":{"animals_escaped":3.25,"escape_capital":1368.75,"escape_yield_units":0.0,"animal_unfed_days":26.0,"feed_no_wheat":13.125,"feed_no_animal":80.812,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.562,"decay_units":19.5,"plant_unwatered_days":445.688,"shed_discarded_units":63.156,"dead_actions":378.625,"water_dead":41.875,"harvest_dead":44.469,"plant_dead":5.875,"sell_zero_fill":17.312,"buy_zero_fill":11.438,"sell_dead_units":301.75,"sold_units":1170.25,"silent_loss_coins":1488.75,"silent_loss_units":84.656,"refused_buy_product":16.875,"refused_buy_seed":9.719,"refused_buy_animal":1.0,"tile_cap_units":6.688,"fertilizer_forgone":0.75,"care_unfed":16.5,"animal_shed_days":7.0,"idle_tile_days":208.0,"weed_tile_days":21.219,"hand_pass_turns":781.0,"phantom_hand_actions":0.0,"sell_floor_units":112.938,"hire_premium":6886.0,"shed_units_now":17.344,"shed_value_now":634.969,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.156,"seeds_unplanted_cost_now":65.938,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":5.688,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.062,"dig_dead":3.938,"fertilize_dead":20.906,"collect_dead":82.875,"care_dead":78.0,"farmer_pass_turns":180.0}},{"agent":"mine, earlier build 92","mine":true,"games":32,"bank_median":67599.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":4,"14":6,"15":31,"16":30,"17":36,"18":25,"19":29,"20":14,"21":14,"22":8,"23":3,"24":8,"25":5,"26":5,"27":1},"per_game":{"animals_escaped":6.062,"escape_capital":2734.375,"escape_yield_units":1.625,"animal_unfed_days":39.625,"feed_no_wheat":7.625,"feed_no_animal":45.688,"feed_redundant":3.0,"plants_dry":20.719,"dry_seed_cost":884.062,"plants_decayed":29.344,"decay_units":42.844,"plant_unwatered_days":434.0,"shed_discarded_units":14.125,"dead_actions":460.875,"water_dead":104.5,"harvest_dead":10.0,"plant_dead":5.969,"sell_zero_fill":5.844,"buy_zero_fill":0.625,"sell_dead_units":41.281,"sold_units":1500.969,"silent_loss_coins":3618.438,"silent_loss_units":73.938,"refused_buy_product":0.5,"refused_buy_seed":1.562,"refused_buy_animal":0.0,"tile_cap_units":7.812,"fertilizer_forgone":20.625,"care_unfed":22.062,"animal_shed_days":3.781,"idle_tile_days":134.812,"weed_tile_days":29.812,"hand_pass_turns":224.312,"phantom_hand_actions":0.0,"sell_floor_units":108.75,"hire_premium":7382.0,"shed_units_now":1.156,"shed_value_now":32.375,"animals_in_shed_now":0.0,"seeds_unplanted_now":11.062,"seeds_unplanted_cost_now":239.062,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":2.469,"drop_dead":0.0,"place_dead":2.781,"build_dead":174.375,"dig_dead":0.5,"fertilize_dead":3.938,"collect_dead":48.844,"care_dead":51.188,"farmer_pass_turns":9.625}},{"agent":"mine, earlier build 93","mine":true,"games":32,"bank_median":69037.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":2,"12":1,"15":18,"16":41,"17":35,"18":22,"19":28,"20":13,"21":15,"22":12,"23":4,"24":8,"25":10,"26":6,"27":1,"28":3},"per_game":{"animals_escaped":6.219,"escape_capital":2890.625,"escape_yield_units":1.312,"animal_unfed_days":41.0,"feed_no_wheat":7.75,"feed_no_animal":46.812,"feed_redundant":3.0,"plants_dry":18.688,"dry_seed_cost":751.875,"plants_decayed":29.656,"decay_units":42.0,"plant_unwatered_days":437.5,"shed_discarded_units":17.219,"dead_actions":469.812,"water_dead":106.688,"harvest_dead":11.625,"plant_dead":6.344,"sell_zero_fill":6.25,"buy_zero_fill":0.625,"sell_dead_units":43.438,"sold_units":1512.406,"silent_loss_coins":3642.5,"silent_loss_units":73.844,"refused_buy_product":0.312,"refused_buy_seed":1.562,"refused_buy_animal":0.0,"tile_cap_units":7.875,"fertilizer_forgone":16.5,"care_unfed":23.156,"animal_shed_days":3.781,"idle_tile_days":125.844,"weed_tile_days":28.438,"hand_pass_turns":225.719,"phantom_hand_actions":0.0,"sell_floor_units":97.656,"hire_premium":7382.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":10.531,"seeds_unplanted_cost_now":195.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":2.812,"drop_dead":0.0,"place_dead":3.062,"build_dead":174.312,"dig_dead":0.312,"fertilize_dead":4.906,"collect_dead":50.062,"care_dead":52.125,"farmer_pass_turns":9.875}},{"agent":"mine, earlier build 94","mine":true,"games":32,"bank_median":69037.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":2,"12":1,"15":18,"16":41,"17":35,"18":22,"19":28,"20":13,"21":15,"22":12,"23":4,"24":8,"25":10,"26":6,"27":1,"28":3},"per_game":{"animals_escaped":6.219,"escape_capital":2890.625,"escape_yield_units":1.312,"animal_unfed_days":41.0,"feed_no_wheat":7.75,"feed_no_animal":46.812,"feed_redundant":3.0,"plants_dry":18.688,"dry_seed_cost":751.875,"plants_decayed":29.656,"decay_units":42.0,"plant_unwatered_days":437.5,"shed_discarded_units":17.219,"dead_actions":469.812,"water_dead":106.688,"harvest_dead":11.625,"plant_dead":6.344,"sell_zero_fill":6.25,"buy_zero_fill":0.625,"sell_dead_units":43.438,"sold_units":1512.406,"silent_loss_coins":3642.5,"silent_loss_units":73.844,"refused_buy_product":0.312,"refused_buy_seed":1.562,"refused_buy_animal":0.0,"tile_cap_units":7.875,"fertilizer_forgone":16.5,"care_unfed":23.156,"animal_shed_days":3.781,"idle_tile_days":125.844,"weed_tile_days":28.438,"hand_pass_turns":225.719,"phantom_hand_actions":0.0,"sell_floor_units":97.656,"hire_premium":7382.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":10.531,"seeds_unplanted_cost_now":195.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":2.812,"drop_dead":0.0,"place_dead":3.062,"build_dead":174.312,"dig_dead":0.312,"fertilize_dead":4.906,"collect_dead":50.062,"care_dead":52.125,"farmer_pass_turns":9.875}},{"agent":"mine, earlier build 95","mine":true,"games":32,"bank_median":87379.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":22.0,"feed_no_wheat":0.0,"feed_no_animal":1.25,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":17.594,"decay_units":17.594,"plant_unwatered_days":372.812,"shed_discarded_units":0.688,"dead_actions":37.906,"water_dead":25.719,"harvest_dead":3.938,"plant_dead":2.406,"sell_zero_fill":1.156,"buy_zero_fill":2.562,"sell_dead_units":22.375,"sold_units":1518.938,"silent_loss_coins":110.0,"silent_loss_units":19.281,"refused_buy_product":4.75,"refused_buy_seed":1.375,"refused_buy_animal":0.062,"tile_cap_units":1.0,"fertilizer_forgone":13.875,"care_unfed":19.0,"animal_shed_days":1.0,"idle_tile_days":90.906,"weed_tile_days":3.656,"hand_pass_turns":559.781,"phantom_hand_actions":0.0,"sell_floor_units":210.781,"hire_premium":4809.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":13.031,"seeds_unplanted_cost_now":223.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.062,"build_dead":0.0,"dig_dead":0.344,"fertilize_dead":0.812,"collect_dead":1.062,"care_dead":1.25,"farmer_pass_turns":70.938}},{"agent":"mine, earlier build 96","mine":true,"games":32,"bank_median":88385.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":2,"15":4,"16":4,"18":2},"per_game":{"animals_escaped":0.312,"escape_capital":150.0,"escape_yield_units":0.125,"animal_unfed_days":22.375,"feed_no_wheat":0.125,"feed_no_animal":3.438,"feed_redundant":0.0,"plants_dry":2.312,"dry_seed_cost":118.75,"plants_decayed":19.031,"decay_units":19.344,"plant_unwatered_days":379.938,"shed_discarded_units":0.688,"dead_actions":54.531,"water_dead":28.344,"harvest_dead":4.562,"plant_dead":2.469,"sell_zero_fill":1.906,"buy_zero_fill":2.562,"sell_dead_units":25.312,"sold_units":1513.688,"silent_loss_coins":268.75,"silent_loss_units":21.406,"refused_buy_product":4.688,"refused_buy_seed":1.625,"refused_buy_animal":0.062,"tile_cap_units":1.812,"fertilizer_forgone":15.812,"care_unfed":18.562,"animal_shed_days":0.938,"idle_tile_days":91.781,"weed_tile_days":5.219,"hand_pass_turns":549.281,"phantom_hand_actions":0.0,"sell_floor_units":201.406,"hire_premium":4969.688,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":13.219,"seeds_unplanted_cost_now":225.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.188,"drop_dead":0.0,"place_dead":1.562,"build_dead":5.438,"dig_dead":0.344,"fertilize_dead":1.0,"collect_dead":3.312,"care_dead":3.75,"farmer_pass_turns":70.438}},{"agent":"mine, earlier build 97","mine":true,"games":32,"bank_median":88385.0,"games_losing_an_animal":14,"escape_first_day_min":8,"escapes_by_day":{"8":14,"15":4,"16":4,"18":2,"25":16,"27":10},"per_game":{"animals_escaped":1.5,"escape_capital":656.25,"escape_yield_units":0.125,"animal_unfed_days":28.062,"feed_no_wheat":0.5,"feed_no_animal":8.688,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":81.25,"plants_decayed":21.344,"decay_units":22.031,"plant_unwatered_days":379.25,"shed_discarded_units":3.938,"dead_actions":91.469,"water_dead":35.531,"harvest_dead":9.75,"plant_dead":3.094,"sell_zero_fill":3.406,"buy_zero_fill":2.312,"sell_dead_units":47.875,"sold_units":1511.312,"silent_loss_coins":737.5,"silent_loss_units":27.344,"refused_buy_product":4.75,"refused_buy_seed":1.562,"refused_buy_animal":0.0,"tile_cap_units":9.312,"fertilizer_forgone":14.188,"care_unfed":18.188,"animal_shed_days":0.938,"idle_tile_days":92.531,"weed_tile_days":4.812,"hand_pass_turns":534.969,"phantom_hand_actions":0.0,"sell_floor_units":176.281,"hire_premium":5145.688,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":17.719,"seeds_unplanted_cost_now":263.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.5,"drop_dead":0.0,"place_dead":5.0,"build_dead":5.438,"dig_dead":0.344,"fertilize_dead":0.812,"collect_dead":11.938,"care_dead":9.875,"farmer_pass_turns":69.5}},{"agent":"mine, earlier build 98","mine":true,"games":32,"bank_median":70931.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":4},"per_game":{"animals_escaped":0.125,"escape_capital":62.5,"escape_yield_units":0.0,"animal_unfed_days":19.312,"feed_no_wheat":0.188,"feed_no_animal":2.312,"feed_redundant":0.25,"plants_dry":1.312,"dry_seed_cost":18.75,"plants_decayed":31.062,"decay_units":31.375,"plant_unwatered_days":395.75,"shed_discarded_units":13.656,"dead_actions":46.094,"water_dead":19.75,"harvest_dead":2.438,"plant_dead":0.062,"sell_zero_fill":8.344,"buy_zero_fill":1.625,"sell_dead_units":8000.781,"sold_units":1444.031,"silent_loss_coins":81.25,"silent_loss_units":46.281,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":1.625,"tile_cap_units":8.688,"fertilizer_forgone":4.125,"care_unfed":13.312,"animal_shed_days":0.188,"idle_tile_days":53.125,"weed_tile_days":2.656,"hand_pass_turns":485.406,"phantom_hand_actions":0.0,"sell_floor_units":194.156,"hire_premium":5828.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.312,"drop_dead":0.0,"place_dead":1.562,"build_dead":0.0,"dig_dead":0.531,"fertilize_dead":0.562,"collect_dead":2.812,"care_dead":15.312,"farmer_pass_turns":63.0}},{"agent":"mine, earlier build 99","mine":true,"games":32,"bank_median":88385.0,"games_losing_an_animal":14,"escape_first_day_min":8,"escapes_by_day":{"8":14,"15":4,"16":4,"18":2,"25":16,"27":10},"per_game":{"animals_escaped":1.5,"escape_capital":656.25,"escape_yield_units":0.125,"animal_unfed_days":28.062,"feed_no_wheat":0.5,"feed_no_animal":8.688,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":81.25,"plants_decayed":21.344,"decay_units":22.031,"plant_unwatered_days":379.25,"shed_discarded_units":3.938,"dead_actions":91.469,"water_dead":35.531,"harvest_dead":9.75,"plant_dead":3.094,"sell_zero_fill":10.594,"buy_zero_fill":2.312,"sell_dead_units":7228.188,"sold_units":1511.312,"silent_loss_coins":737.5,"silent_loss_units":27.344,"refused_buy_product":4.75,"refused_buy_seed":1.562,"refused_buy_animal":0.0,"tile_cap_units":9.312,"fertilizer_forgone":14.188,"care_unfed":18.188,"animal_shed_days":0.938,"idle_tile_days":92.531,"weed_tile_days":4.812,"hand_pass_turns":534.969,"phantom_hand_actions":0.0,"sell_floor_units":176.281,"hire_premium":5145.688,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":17.719,"seeds_unplanted_cost_now":263.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.5,"drop_dead":0.0,"place_dead":5.0,"build_dead":5.438,"dig_dead":0.344,"fertilize_dead":0.812,"collect_dead":11.938,"care_dead":9.875,"farmer_pass_turns":69.5}},{"agent":"mine, earlier build 100","mine":true,"games":32,"bank_median":81999.0,"games_losing_an_animal":32,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":64},"per_game":{"animals_escaped":2.562,"escape_capital":1225.0,"escape_yield_units":0.0,"animal_unfed_days":32.625,"feed_no_wheat":0.875,"feed_no_animal":11.875,"feed_redundant":0.0,"plants_dry":1.875,"dry_seed_cost":18.75,"plants_decayed":23.75,"decay_units":25.375,"plant_unwatered_days":395.969,"shed_discarded_units":21.5,"dead_actions":93.094,"water_dead":14.344,"harvest_dead":14.125,"plant_dead":1.125,"sell_zero_fill":81.688,"buy_zero_fill":3.875,"sell_dead_units":9625479.094,"sold_units":1684.219,"silent_loss_coins":1243.75,"silent_loss_units":48.75,"refused_buy_product":6.625,"refused_buy_seed":1.062,"refused_buy_animal":0.875,"tile_cap_units":20.5,"fertilizer_forgone":20.375,"care_unfed":17.312,"animal_shed_days":4.75,"idle_tile_days":53.0,"weed_tile_days":7.281,"hand_pass_turns":461.188,"phantom_hand_actions":0.0,"sell_floor_units":172.281,"hire_premium":5120.0,"shed_units_now":0.812,"shed_value_now":13.312,"animals_in_shed_now":0.188,"seeds_unplanted_now":10.438,"seeds_unplanted_cost_now":110.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":2.188,"drop_dead":0.0,"place_dead":7.375,"build_dead":0.0,"dig_dead":0.062,"fertilize_dead":1.562,"collect_dead":17.75,"care_dead":21.812,"farmer_pass_turns":56.625}},{"agent":"mine, earlier build 101","mine":true,"games":32,"bank_median":77834.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":22},"per_game":{"animals_escaped":1.25,"escape_capital":568.75,"escape_yield_units":0.0,"animal_unfed_days":29.875,"feed_no_wheat":0.875,"feed_no_animal":12.5,"feed_redundant":0.0,"plants_dry":0.938,"dry_seed_cost":26.25,"plants_decayed":25.188,"decay_units":28.438,"plant_unwatered_days":379.469,"shed_discarded_units":1.312,"dead_actions":111.031,"water_dead":25.219,"harvest_dead":15.0,"plant_dead":1.062,"sell_zero_fill":32.0,"buy_zero_fill":3.562,"sell_dead_units":1751309.781,"sold_units":1700.469,"silent_loss_coins":595.0,"silent_loss_units":30.5,"refused_buy_product":6.812,"refused_buy_seed":1.0,"refused_buy_animal":0.375,"tile_cap_units":13.25,"fertilizer_forgone":17.75,"care_unfed":16.5,"animal_shed_days":1.0,"idle_tile_days":72.688,"weed_tile_days":7.188,"hand_pass_turns":441.062,"phantom_hand_actions":0.0,"sell_floor_units":174.562,"hire_premium":5115.312,"shed_units_now":4.688,"shed_value_now":227.5,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.25,"seeds_unplanted_cost_now":126.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":4.312,"drop_dead":0.0,"place_dead":6.938,"build_dead":0.0,"dig_dead":0.562,"fertilize_dead":4.25,"collect_dead":16.5,"care_dead":23.812,"farmer_pass_turns":56.125}},{"agent":"mine, earlier build 102","mine":true,"games":32,"bank_median":78352.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":4},"per_game":{"animals_escaped":0.125,"escape_capital":56.25,"escape_yield_units":0.0,"animal_unfed_days":31.438,"feed_no_wheat":14.562,"feed_no_animal":9.375,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.938,"decay_units":19.938,"plant_unwatered_days":462.656,"shed_discarded_units":0.312,"dead_actions":759.969,"water_dead":27.031,"harvest_dead":6.688,"plant_dead":4.906,"sell_zero_fill":20.25,"buy_zero_fill":5.0,"sell_dead_units":344.438,"sold_units":1367.438,"silent_loss_coins":176.25,"silent_loss_units":22.25,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.125,"fertilizer_forgone":4.938,"care_unfed":25.375,"animal_shed_days":1.0,"idle_tile_days":173.5,"weed_tile_days":17.812,"hand_pass_turns":242.0,"phantom_hand_actions":0.0,"sell_floor_units":164.469,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.906,"seeds_unplanted_cost_now":28.75,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.125,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":1.906,"fertilize_dead":0.75,"collect_dead":14.312,"care_dead":639.312,"farmer_pass_turns":60.812}},{"agent":"mine, earlier build 103","mine":true,"games":32,"bank_median":77834.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":22},"per_game":{"animals_escaped":1.25,"escape_capital":568.75,"escape_yield_units":0.0,"animal_unfed_days":29.875,"feed_no_wheat":0.875,"feed_no_animal":12.5,"feed_redundant":0.0,"plants_dry":0.938,"dry_seed_cost":26.25,"plants_decayed":25.188,"decay_units":28.438,"plant_unwatered_days":379.469,"shed_discarded_units":1.312,"dead_actions":111.031,"water_dead":25.219,"harvest_dead":15.0,"plant_dead":1.062,"sell_zero_fill":36.406,"buy_zero_fill":3.562,"sell_dead_units":1755742.781,"sold_units":1700.531,"silent_loss_coins":595.0,"silent_loss_units":30.5,"refused_buy_product":6.812,"refused_buy_seed":1.0,"refused_buy_animal":0.375,"tile_cap_units":13.25,"fertilizer_forgone":17.75,"care_unfed":16.5,"animal_shed_days":1.0,"idle_tile_days":72.688,"weed_tile_days":7.188,"hand_pass_turns":441.062,"phantom_hand_actions":0.0,"sell_floor_units":174.625,"hire_premium":5115.312,"shed_units_now":4.625,"shed_value_now":227.438,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.25,"seeds_unplanted_cost_now":126.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":4.312,"drop_dead":0.0,"place_dead":6.938,"build_dead":0.0,"dig_dead":0.562,"fertilize_dead":4.25,"collect_dead":16.5,"care_dead":23.812,"farmer_pass_turns":56.125}},{"agent":"mine, earlier build 104","mine":true,"games":32,"bank_median":77834.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":22},"per_game":{"animals_escaped":1.25,"escape_capital":568.75,"escape_yield_units":0.0,"animal_unfed_days":29.875,"feed_no_wheat":0.875,"feed_no_animal":12.5,"feed_redundant":0.0,"plants_dry":0.938,"dry_seed_cost":26.25,"plants_decayed":25.188,"decay_units":28.438,"plant_unwatered_days":379.469,"shed_discarded_units":1.312,"dead_actions":111.031,"water_dead":25.219,"harvest_dead":15.0,"plant_dead":1.062,"sell_zero_fill":36.406,"buy_zero_fill":3.562,"sell_dead_units":1755742.781,"sold_units":1700.531,"silent_loss_coins":595.0,"silent_loss_units":30.5,"refused_buy_product":6.812,"refused_buy_seed":1.0,"refused_buy_animal":0.375,"tile_cap_units":13.25,"fertilizer_forgone":17.75,"care_unfed":16.5,"animal_shed_days":1.0,"idle_tile_days":72.688,"weed_tile_days":7.188,"hand_pass_turns":441.062,"phantom_hand_actions":0.0,"sell_floor_units":174.625,"hire_premium":5115.312,"shed_units_now":4.625,"shed_value_now":227.438,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.25,"seeds_unplanted_cost_now":126.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":4.312,"drop_dead":0.0,"place_dead":6.938,"build_dead":0.0,"dig_dead":0.562,"fertilize_dead":4.25,"collect_dead":16.5,"care_dead":23.812,"farmer_pass_turns":56.125}},{"agent":"mine, earlier build 105","mine":true,"games":32,"bank_median":77811.0,"games_losing_an_animal":26,"escape_first_day_min":8,"escapes_by_day":{"8":18,"28":22},"per_game":{"animals_escaped":1.25,"escape_capital":568.75,"escape_yield_units":0.0,"animal_unfed_days":29.875,"feed_no_wheat":0.875,"feed_no_animal":12.5,"feed_redundant":0.0,"plants_dry":0.938,"dry_seed_cost":26.25,"plants_decayed":25.188,"decay_units":28.438,"plant_unwatered_days":379.469,"shed_discarded_units":0.812,"dead_actions":111.031,"water_dead":25.219,"harvest_dead":15.0,"plant_dead":1.062,"sell_zero_fill":37.969,"buy_zero_fill":3.562,"sell_dead_units":1755764.406,"sold_units":1701.031,"silent_loss_coins":595.0,"silent_loss_units":30.0,"refused_buy_product":6.812,"refused_buy_seed":1.0,"refused_buy_animal":0.375,"tile_cap_units":13.25,"fertilizer_forgone":17.75,"care_unfed":16.5,"animal_shed_days":1.0,"idle_tile_days":72.688,"weed_tile_days":7.188,"hand_pass_turns":441.062,"phantom_hand_actions":0.0,"sell_floor_units":174.875,"hire_premium":5115.312,"shed_units_now":4.625,"shed_value_now":227.438,"animals_in_shed_now":0.0,"seeds_unplanted_now":6.25,"seeds_unplanted_cost_now":126.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":4.312,"drop_dead":0.0,"place_dead":6.938,"build_dead":0.0,"dig_dead":0.562,"fertilize_dead":4.25,"collect_dead":16.5,"care_dead":23.812,"farmer_pass_turns":56.125}},{"agent":"mine, earlier build 106","mine":true,"games":32,"bank_median":86551.0,"games_losing_an_animal":32,"escape_first_day_min":1,"escapes_by_day":{"1":32,"8":4},"per_game":{"animals_escaped":1.125,"escape_capital":450.0,"escape_yield_units":0.0,"animal_unfed_days":20.0,"feed_no_wheat":2.125,"feed_no_animal":43.375,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":108.125,"plants_decayed":15.094,"decay_units":15.094,"plant_unwatered_days":355.5,"shed_discarded_units":47.031,"dead_actions":261.344,"water_dead":46.281,"harvest_dead":39.562,"plant_dead":5.781,"sell_zero_fill":89.594,"buy_zero_fill":8.438,"sell_dead_units":9309.906,"sold_units":1382.656,"silent_loss_coins":558.125,"silent_loss_units":62.938,"refused_buy_product":11.875,"refused_buy_seed":5.688,"refused_buy_animal":0.75,"tile_cap_units":22.562,"fertilizer_forgone":11.812,"care_unfed":15.875,"animal_shed_days":0.812,"idle_tile_days":124.844,"weed_tile_days":5.375,"hand_pass_turns":544.844,"phantom_hand_actions":0.0,"sell_floor_units":136.875,"hire_premium":4782.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.094,"seeds_unplanted_cost_now":9.375,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":3.375,"drop_dead":0.0,"place_dead":7.812,"build_dead":0.0,"dig_dead":3.219,"fertilize_dead":7.5,"collect_dead":55.062,"care_dead":47.25,"farmer_pass_turns":70.562}},{"agent":"mine, earlier build 107","mine":true,"games":32,"bank_median":86551.0,"games_losing_an_animal":32,"escape_first_day_min":1,"escapes_by_day":{"1":32,"8":4},"per_game":{"animals_escaped":1.125,"escape_capital":450.0,"escape_yield_units":0.0,"animal_unfed_days":20.0,"feed_no_wheat":2.125,"feed_no_animal":43.375,"feed_redundant":0.0,"plants_dry":1.812,"dry_seed_cost":108.125,"plants_decayed":15.094,"decay_units":15.094,"plant_unwatered_days":355.5,"shed_discarded_units":47.031,"dead_actions":260.844,"water_dead":46.281,"harvest_dead":39.062,"plant_dead":5.781,"sell_zero_fill":88.844,"buy_zero_fill":8.312,"sell_dead_units":9236.281,"sold_units":1383.531,"silent_loss_coins":558.125,"silent_loss_units":62.938,"refused_buy_product":11.375,"refused_buy_seed":5.688,"refused_buy_animal":0.75,"tile_cap_units":23.938,"fertilizer_forgone":11.812,"care_unfed":15.875,"animal_shed_days":0.812,"idle_tile_days":124.844,"weed_tile_days":5.375,"hand_pass_turns":544.844,"phantom_hand_actions":0.0,"sell_floor_units":132.938,"hire_premium":4782.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.094,"seeds_unplanted_cost_now":9.375,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":3.375,"drop_dead":0.0,"place_dead":7.812,"build_dead":0.0,"dig_dead":3.219,"fertilize_dead":7.5,"collect_dead":55.062,"care_dead":47.25,"farmer_pass_turns":70.562}},{"agent":"mine, earlier build 108","mine":true,"games":32,"bank_median":79910.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":31.562,"feed_no_wheat":2.75,"feed_no_animal":3.375,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":379.562,"shed_discarded_units":5.062,"dead_actions":27.344,"water_dead":1.375,"harvest_dead":7.25,"plant_dead":0.0,"sell_zero_fill":102.656,"buy_zero_fill":0.0,"sell_dead_units":93553.5,"sold_units":1515.625,"silent_loss_coins":110.0,"silent_loss_units":24.0,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":17.25,"fertilizer_forgone":12.625,"care_unfed":24.75,"animal_shed_days":0.812,"idle_tile_days":65.062,"weed_tile_days":2.625,"hand_pass_turns":429.406,"phantom_hand_actions":0.0,"sell_floor_units":195.594,"hire_premium":3618.812,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.375,"drop_dead":0.0,"place_dead":0.594,"build_dead":0.0,"dig_dead":0.875,"fertilize_dead":0.25,"collect_dead":7.125,"care_dead":3.375,"farmer_pass_turns":66.938}},{"agent":"mine, earlier build 109","mine":true,"games":32,"bank_median":80469.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.562,"shed_discarded_units":8.562,"dead_actions":44.469,"water_dead":12.062,"harvest_dead":6.25,"plant_dead":0.0,"sell_zero_fill":102.688,"buy_zero_fill":0.0,"sell_dead_units":88068.812,"sold_units":1528.188,"silent_loss_coins":109.375,"silent_loss_units":27.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":65.0,"weed_tile_days":2.562,"hand_pass_turns":399.594,"phantom_hand_actions":0.0,"sell_floor_units":196.875,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.469,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"mine, earlier build 110","mine":true,"games":32,"bank_median":80469.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.562,"shed_discarded_units":8.562,"dead_actions":44.469,"water_dead":12.062,"harvest_dead":6.25,"plant_dead":0.0,"sell_zero_fill":102.688,"buy_zero_fill":0.0,"sell_dead_units":88068.812,"sold_units":1528.188,"silent_loss_coins":109.375,"silent_loss_units":27.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":65.0,"weed_tile_days":2.562,"hand_pass_turns":399.594,"phantom_hand_actions":0.0,"sell_floor_units":196.875,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.469,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"mine, earlier build 111","mine":true,"games":32,"bank_median":80469.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.562,"shed_discarded_units":8.562,"dead_actions":44.469,"water_dead":12.062,"harvest_dead":6.25,"plant_dead":0.0,"sell_zero_fill":102.688,"buy_zero_fill":0.0,"sell_dead_units":88068.812,"sold_units":1528.188,"silent_loss_coins":109.375,"silent_loss_units":27.438,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":65.0,"weed_tile_days":2.562,"hand_pass_turns":399.594,"phantom_hand_actions":0.0,"sell_floor_units":196.875,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.469,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"mine, earlier build 112","mine":true,"games":32,"bank_median":92932.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":32.562,"feed_no_wheat":4.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.781,"shed_discarded_units":140.125,"dead_actions":71.25,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":110.875,"buy_zero_fill":0.0,"sell_dead_units":87894.094,"sold_units":1517.094,"silent_loss_coins":109.375,"silent_loss_units":159.031,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":15.25,"fertilizer_forgone":14.688,"care_unfed":27.75,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":369.938,"phantom_hand_actions":0.0,"sell_floor_units":120.438,"hire_premium":3519.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.844,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 113","mine":true,"games":32,"bank_median":97087.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":375.594,"shed_discarded_units":16.75,"dead_actions":64.75,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":115.25,"buy_zero_fill":0.0,"sell_dead_units":87957.594,"sold_units":1673.781,"silent_loss_coins":109.375,"silent_loss_units":35.656,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":19.312,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":379.688,"phantom_hand_actions":0.0,"sell_floor_units":146.75,"hire_premium":3805.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 114","mine":true,"games":32,"bank_median":96814.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":375.594,"shed_discarded_units":0.0,"dead_actions":64.75,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":117.062,"buy_zero_fill":0.0,"sell_dead_units":87977.469,"sold_units":1690.531,"silent_loss_coins":109.375,"silent_loss_units":18.906,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":19.312,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":379.688,"phantom_hand_actions":0.0,"sell_floor_units":157.812,"hire_premium":3805.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 115","mine":true,"games":32,"bank_median":96814.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":374.844,"shed_discarded_units":0.0,"dead_actions":64.75,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":100.75,"buy_zero_fill":0.0,"sell_dead_units":87810.25,"sold_units":1661.281,"silent_loss_coins":109.375,"silent_loss_units":18.906,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":15.562,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":370.688,"phantom_hand_actions":0.0,"sell_floor_units":160.594,"hire_premium":3541.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 116","mine":true,"games":32,"bank_median":96271.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":375.594,"shed_discarded_units":0.0,"dead_actions":64.75,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":130.875,"buy_zero_fill":0.0,"sell_dead_units":87985.281,"sold_units":1690.531,"silent_loss_coins":109.375,"silent_loss_units":18.906,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":19.312,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":379.688,"phantom_hand_actions":0.0,"sell_floor_units":157.031,"hire_premium":3805.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 117","mine":true,"games":32,"bank_median":80284.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":378.625,"shed_discarded_units":0.0,"dead_actions":42.156,"water_dead":11.375,"harvest_dead":6.0,"plant_dead":0.0,"sell_zero_fill":116.688,"buy_zero_fill":0.0,"sell_dead_units":89858.75,"sold_units":1565.938,"silent_loss_coins":109.375,"silent_loss_units":18.875,"refused_buy_product":216.25,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":14.312,"fertilizer_forgone":14.562,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":65.0,"weed_tile_days":2.562,"hand_pass_turns":379.531,"phantom_hand_actions":0.0,"sell_floor_units":200.0,"hire_premium":3602.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.562,"seeds_unplanted_cost_now":11.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.344,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 118","mine":true,"games":32,"bank_median":96814.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":375.594,"shed_discarded_units":0.0,"dead_actions":64.75,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":128.812,"buy_zero_fill":0.0,"sell_dead_units":87977.469,"sold_units":1690.531,"silent_loss_coins":109.375,"silent_loss_units":18.906,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":19.312,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":379.688,"phantom_hand_actions":0.0,"sell_floor_units":158.062,"hire_premium":3805.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 119","mine":true,"games":32,"bank_median":96814.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.188,"feed_no_wheat":0.062,"feed_no_animal":5.625,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":375.219,"shed_discarded_units":6.344,"dead_actions":85.625,"water_dead":30.062,"harvest_dead":9.938,"plant_dead":2.375,"sell_zero_fill":112.375,"buy_zero_fill":0.562,"sell_dead_units":87558.375,"sold_units":1674.0,"silent_loss_coins":109.375,"silent_loss_units":25.25,"refused_buy_product":110.375,"refused_buy_seed":1.25,"refused_buy_animal":0.312,"tile_cap_units":16.812,"fertilizer_forgone":13.75,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":85.219,"weed_tile_days":2.75,"hand_pass_turns":379.688,"phantom_hand_actions":0.0,"sell_floor_units":155.375,"hire_premium":3805.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.406,"seeds_unplanted_cost_now":86.875,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.375,"drop_dead":0.0,"place_dead":1.5,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.344,"collect_dead":8.25,"care_dead":26.062,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 120","mine":true,"games":32,"bank_median":96814.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.969,"decay_units":17.969,"plant_unwatered_days":375.594,"shed_discarded_units":0.0,"dead_actions":64.75,"water_dead":29.312,"harvest_dead":7.062,"plant_dead":2.0,"sell_zero_fill":117.938,"buy_zero_fill":0.0,"sell_dead_units":87977.469,"sold_units":1690.531,"silent_loss_coins":109.375,"silent_loss_units":18.906,"refused_buy_product":108.125,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":19.312,"fertilizer_forgone":14.688,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":84.469,"weed_tile_days":2.75,"hand_pass_turns":379.688,"phantom_hand_actions":0.0,"sell_floor_units":156.938,"hire_premium":3805.25,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.281,"seeds_unplanted_cost_now":85.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.188,"build_dead":0.0,"dig_dead":1.031,"fertilize_dead":0.031,"collect_dead":3.562,"care_dead":20.438,"farmer_pass_turns":61.5}},{"agent":"mine, earlier build 121","mine":true,"games":32,"bank_median":96814.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.5,"feed_no_wheat":0.062,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":375.375,"shed_discarded_units":0.0,"dead_actions":64.938,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":122.906,"buy_zero_fill":0.0,"sell_dead_units":89990.062,"sold_units":1697.375,"silent_loss_coins":109.375,"silent_loss_units":18.875,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":24.5,"animal_shed_days":0.938,"idle_tile_days":85.0,"weed_tile_days":2.75,"hand_pass_turns":388.531,"phantom_hand_actions":0.0,"sell_floor_units":165.906,"hire_premium":3866.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 122","mine":true,"games":32,"bank_median":96743.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":375.375,"shed_discarded_units":2.344,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.094,"buy_zero_fill":0.0,"sell_dead_units":90010.844,"sold_units":1695.031,"silent_loss_coins":109.375,"silent_loss_units":21.219,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":85.0,"weed_tile_days":2.75,"hand_pass_turns":388.531,"phantom_hand_actions":0.0,"sell_floor_units":160.0,"hire_premium":3866.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 123","mine":true,"games":32,"bank_median":96743.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.938,"dry_seed_cost":109.375,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":375.375,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.906,"buy_zero_fill":0.0,"sell_dead_units":90013.094,"sold_units":1692.781,"silent_loss_coins":109.375,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":85.0,"weed_tile_days":2.75,"hand_pass_turns":388.531,"phantom_hand_actions":0.0,"sell_floor_units":156.938,"hire_premium":3866.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 124","mine":true,"games":32,"bank_median":79102.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":2},"per_game":{"animals_escaped":0.062,"escape_capital":25.0,"escape_yield_units":0.0,"animal_unfed_days":26.062,"feed_no_wheat":10.375,"feed_no_animal":20.688,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.844,"decay_units":19.656,"plant_unwatered_days":462.438,"shed_discarded_units":5.906,"dead_actions":796.938,"water_dead":27.5,"harvest_dead":10.062,"plant_dead":5.031,"sell_zero_fill":25.188,"buy_zero_fill":5.0,"sell_dead_units":327.719,"sold_units":1343.281,"silent_loss_coins":145.0,"silent_loss_units":27.562,"refused_buy_product":6.0,"refused_buy_seed":6.062,"refused_buy_animal":0.625,"tile_cap_units":6.438,"fertilizer_forgone":3.688,"care_unfed":19.938,"animal_shed_days":0.375,"idle_tile_days":174.125,"weed_tile_days":18.531,"hand_pass_turns":243.0,"phantom_hand_actions":0.0,"sell_floor_units":151.125,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.969,"seeds_unplanted_cost_now":29.375,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":1.969,"drop_dead":0.0,"place_dead":1.625,"build_dead":0.0,"dig_dead":1.938,"fertilize_dead":2.312,"collect_dead":22.125,"care_dead":658.312,"farmer_pass_turns":61.0}},{"agent":"mine, earlier build 125","mine":true,"games":32,"bank_median":96982.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":156.344,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 126","mine":true,"games":32,"bank_median":96982.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":156.344,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 127","mine":true,"games":32,"bank_median":97153.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":149.812,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 128","mine":true,"games":32,"bank_median":97153.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":149.812,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 129","mine":true,"games":32,"bank_median":68683.0,"games_losing_an_animal":32,"escape_first_day_min":1,"escapes_by_day":{"1":64,"8":12},"per_game":{"animals_escaped":2.375,"escape_capital":968.75,"escape_yield_units":0.0,"animal_unfed_days":25.375,"feed_no_wheat":4.625,"feed_no_animal":130.0,"feed_redundant":0.0,"plants_dry":2.062,"dry_seed_cost":118.125,"plants_decayed":14.625,"decay_units":14.625,"plant_unwatered_days":351.062,"shed_discarded_units":1.375,"dead_actions":612.375,"water_dead":48.812,"harvest_dead":71.562,"plant_dead":4.75,"sell_zero_fill":189.25,"buy_zero_fill":10.438,"sell_dead_units":90301.156,"sold_units":1474.719,"silent_loss_coins":1086.875,"silent_loss_units":16.875,"refused_buy_product":232.625,"refused_buy_seed":4.688,"refused_buy_animal":3.562,"tile_cap_units":15.75,"fertilizer_forgone":12.688,"care_unfed":19.562,"animal_shed_days":0.875,"idle_tile_days":143.375,"weed_tile_days":4.531,"hand_pass_turns":397.031,"phantom_hand_actions":0.0,"sell_floor_units":96.438,"hire_premium":4086.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":2.062,"seeds_unplanted_cost_now":88.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":11.875,"drop_dead":0.0,"place_dead":20.219,"build_dead":0.0,"dig_dead":4.25,"fertilize_dead":18.594,"collect_dead":141.188,"care_dead":156.5,"farmer_pass_turns":62.25}},{"agent":"mine, earlier build 130","mine":true,"games":32,"bank_median":96982.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":156.344,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 131","mine":true,"games":32,"bank_median":68683.0,"games_losing_an_animal":32,"escape_first_day_min":1,"escapes_by_day":{"1":64,"8":12},"per_game":{"animals_escaped":2.375,"escape_capital":968.75,"escape_yield_units":0.0,"animal_unfed_days":25.375,"feed_no_wheat":4.625,"feed_no_animal":130.0,"feed_redundant":0.0,"plants_dry":2.062,"dry_seed_cost":118.125,"plants_decayed":14.625,"decay_units":14.625,"plant_unwatered_days":351.062,"shed_discarded_units":1.375,"dead_actions":612.375,"water_dead":48.812,"harvest_dead":71.562,"plant_dead":4.75,"sell_zero_fill":189.25,"buy_zero_fill":10.438,"sell_dead_units":90301.156,"sold_units":1474.719,"silent_loss_coins":1086.875,"silent_loss_units":16.875,"refused_buy_product":232.625,"refused_buy_seed":4.688,"refused_buy_animal":3.562,"tile_cap_units":15.75,"fertilizer_forgone":12.688,"care_unfed":19.562,"animal_shed_days":0.875,"idle_tile_days":143.375,"weed_tile_days":4.531,"hand_pass_turns":397.031,"phantom_hand_actions":0.0,"sell_floor_units":96.438,"hire_premium":4086.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":2.062,"seeds_unplanted_cost_now":88.125,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":11.875,"drop_dead":0.0,"place_dead":20.219,"build_dead":0.0,"dig_dead":4.25,"fertilize_dead":18.594,"collect_dead":141.188,"care_dead":156.5,"farmer_pass_turns":62.25}},{"agent":"mine, earlier build 132","mine":true,"games":32,"bank_median":97153.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":30.125,"feed_no_wheat":1.688,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.219,"dry_seed_cost":123.438,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":380.094,"shed_discarded_units":4.594,"dead_actions":68.188,"water_dead":29.188,"harvest_dead":8.0,"plant_dead":2.0,"sell_zero_fill":126.719,"buy_zero_fill":0.0,"sell_dead_units":90010.469,"sold_units":1696.156,"silent_loss_coins":123.438,"silent_loss_units":23.469,"refused_buy_product":216.312,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":18.062,"fertilizer_forgone":14.562,"care_unfed":26.125,"animal_shed_days":0.938,"idle_tile_days":101.094,"weed_tile_days":5.625,"hand_pass_turns":392.25,"phantom_hand_actions":0.0,"sell_floor_units":149.812,"hire_premium":4102.5,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.562,"seeds_unplanted_cost_now":91.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.688,"drop_dead":0.0,"place_dead":1.312,"build_dead":0.0,"dig_dead":1.0,"fertilize_dead":0.062,"collect_dead":4.0,"care_dead":19.25,"farmer_pass_turns":62.562}},{"agent":"mine, earlier build 133","mine":true,"games":32,"bank_median":89705.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":81.344,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":1.0,"dry_seed_cost":100.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":384.25,"shed_discarded_units":13.281,"dead_actions":41.438,"water_dead":10.625,"harvest_dead":5.469,"plant_dead":0.0,"sell_zero_fill":141.344,"buy_zero_fill":1.0,"sell_dead_units":88283.188,"sold_units":1797.625,"silent_loss_coins":100.0,"silent_loss_units":31.219,"refused_buy_product":5.125,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":22.781,"fertilizer_forgone":14.875,"care_unfed":76.906,"animal_shed_days":0.938,"idle_tile_days":100.656,"weed_tile_days":2.781,"hand_pass_turns":466.875,"phantom_hand_actions":0.0,"sell_floor_units":99.656,"hire_premium":5111.312,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.0,"seeds_unplanted_cost_now":15.625,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":1.344,"build_dead":0.0,"dig_dead":3.312,"fertilize_dead":0.812,"collect_dead":7.688,"care_dead":12.188,"farmer_pass_turns":68.5}},{"agent":"mine, earlier build 134","mine":true,"games":32,"bank_median":78810.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":4},"per_game":{"animals_escaped":0.125,"escape_capital":56.25,"escape_yield_units":0.0,"animal_unfed_days":31.812,"feed_no_wheat":14.938,"feed_no_animal":9.375,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.844,"decay_units":19.656,"plant_unwatered_days":462.375,"shed_discarded_units":0.281,"dead_actions":766.938,"water_dead":27.875,"harvest_dead":6.875,"plant_dead":5.094,"sell_zero_fill":42.625,"buy_zero_fill":5.0,"sell_dead_units":609.281,"sold_units":1364.719,"silent_loss_coins":176.25,"silent_loss_units":21.938,"refused_buy_product":6.0,"refused_buy_seed":6.0,"refused_buy_animal":0.0,"tile_cap_units":5.125,"fertilizer_forgone":4.938,"care_unfed":25.75,"animal_shed_days":1.0,"idle_tile_days":174.438,"weed_tile_days":19.031,"hand_pass_turns":239.0,"phantom_hand_actions":0.0,"sell_floor_units":161.562,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.094,"seeds_unplanted_cost_now":30.625,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":5.219,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":2.0,"fertilize_dead":0.75,"collect_dead":11.5,"care_dead":647.312,"farmer_pass_turns":61.0}},{"agent":"mine, earlier build 135","mine":true,"games":32,"bank_median":64993.0,"games_losing_an_animal":17,"escape_first_day_min":8,"escapes_by_day":{"8":2,"11":42},"per_game":{"animals_escaped":1.375,"escape_capital":550.0,"escape_yield_units":0.0,"animal_unfed_days":32.562,"feed_no_wheat":17.281,"feed_no_animal":34.594,"feed_redundant":0.0,"plants_dry":3.625,"dry_seed_cost":154.688,"plants_decayed":8.688,"decay_units":17.875,"plant_unwatered_days":421.656,"shed_discarded_units":24.906,"dead_actions":981.781,"water_dead":54.75,"harvest_dead":27.406,"plant_dead":8.906,"sell_zero_fill":33.281,"buy_zero_fill":8.031,"sell_dead_units":458.969,"sold_units":1212.031,"silent_loss_coins":704.688,"silent_loss_units":47.031,"refused_buy_product":17.5,"refused_buy_seed":9.188,"refused_buy_animal":1.688,"tile_cap_units":4.312,"fertilizer_forgone":2.844,"care_unfed":25.812,"animal_shed_days":106.312,"idle_tile_days":210.312,"weed_tile_days":33.531,"hand_pass_turns":241.812,"phantom_hand_actions":13.25,"sell_floor_units":127.0,"hire_premium":6700.094,"shed_units_now":0.938,"shed_value_now":71.25,"animals_in_shed_now":4.5,"seeds_unplanted_now":10.719,"seeds_unplanted_cost_now":414.375,"move_dead":0.0,"locked_dead":134.281,"pickup_dead":4.719,"drop_dead":0.0,"place_dead":0.625,"build_dead":0.0,"dig_dead":3.156,"fertilize_dead":9.781,"collect_dead":35.812,"care_dead":650.469,"farmer_pass_turns":61.0}},{"agent":"mine, earlier build 136","mine":true,"games":32,"bank_median":76595.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":2},"per_game":{"animals_escaped":0.062,"escape_capital":25.0,"escape_yield_units":0.0,"animal_unfed_days":29.312,"feed_no_wheat":13.625,"feed_no_animal":20.688,"feed_redundant":0.0,"plants_dry":3.0,"dry_seed_cost":120.0,"plants_decayed":9.844,"decay_units":19.656,"plant_unwatered_days":462.438,"shed_discarded_units":81.031,"dead_actions":803.125,"water_dead":27.5,"harvest_dead":10.062,"plant_dead":5.031,"sell_zero_fill":24.188,"buy_zero_fill":5.0,"sell_dead_units":349.219,"sold_units":1264.906,"silent_loss_coins":145.0,"silent_loss_units":102.688,"refused_buy_product":6.0,"refused_buy_seed":6.062,"refused_buy_animal":0.625,"tile_cap_units":6.438,"fertilizer_forgone":3.688,"care_unfed":19.938,"animal_shed_days":0.375,"idle_tile_days":174.125,"weed_tile_days":18.531,"hand_pass_turns":243.0,"phantom_hand_actions":0.0,"sell_floor_units":149.5,"hire_premium":6888.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.969,"seeds_unplanted_cost_now":29.375,"move_dead":0.0,"locked_dead":35.0,"pickup_dead":4.906,"drop_dead":0.0,"place_dead":1.625,"build_dead":0.0,"dig_dead":1.938,"fertilize_dead":2.312,"collect_dead":22.125,"care_dead":658.312,"farmer_pass_turns":61.0}},{"agent":"mine, earlier build 137","mine":true,"games":32,"bank_median":7699.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":0.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":0.0,"dry_seed_cost":0.0,"plants_decayed":0.0,"decay_units":0.0,"plant_unwatered_days":0.0,"shed_discarded_units":0.0,"dead_actions":1256.0,"water_dead":0.0,"harvest_dead":1256.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":196.0,"silent_loss_coins":0.0,"silent_loss_units":0.0,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":0.0,"care_unfed":0.0,"animal_shed_days":0.0,"idle_tile_days":522.0,"weed_tile_days":34.938,"hand_pass_turns":0.0,"phantom_hand_actions":0.0,"sell_floor_units":0.0,"hire_premium":420.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":12.0,"seeds_unplanted_cost_now":120.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":1.0}},{"agent":"mine, earlier build 138","mine":true,"games":32,"bank_median":14570.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":0.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":0.0,"dry_seed_cost":0.0,"plants_decayed":0.0,"decay_units":0.0,"plant_unwatered_days":7.0,"shed_discarded_units":0.0,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":124.0,"silent_loss_coins":0.0,"silent_loss_units":0.0,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":0.0,"care_unfed":0.0,"animal_shed_days":0.0,"idle_tile_days":469.0,"weed_tile_days":21.344,"hand_pass_turns":1785.0,"phantom_hand_actions":0.0,"sell_floor_units":26.0,"hire_premium":420.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":405.0}},{"agent":"mine, earlier build 139","mine":true,"games":32,"bank_median":70931.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":4},"per_game":{"animals_escaped":0.125,"escape_capital":62.5,"escape_yield_units":0.0,"animal_unfed_days":19.312,"feed_no_wheat":0.188,"feed_no_animal":2.312,"feed_redundant":0.25,"plants_dry":1.312,"dry_seed_cost":18.75,"plants_decayed":31.062,"decay_units":31.375,"plant_unwatered_days":395.75,"shed_discarded_units":13.656,"dead_actions":46.094,"water_dead":19.75,"harvest_dead":2.438,"plant_dead":0.062,"sell_zero_fill":8.344,"buy_zero_fill":1.625,"sell_dead_units":8000.781,"sold_units":1444.031,"silent_loss_coins":81.25,"silent_loss_units":46.281,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":1.625,"tile_cap_units":8.688,"fertilizer_forgone":4.125,"care_unfed":13.312,"animal_shed_days":0.188,"idle_tile_days":53.125,"weed_tile_days":2.656,"hand_pass_turns":485.406,"phantom_hand_actions":0.0,"sell_floor_units":194.156,"hire_premium":5828.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.312,"drop_dead":0.0,"place_dead":1.562,"build_dead":0.0,"dig_dead":0.531,"fertilize_dead":0.562,"collect_dead":2.812,"care_dead":15.312,"farmer_pass_turns":63.0}},{"agent":"mine, earlier build 140","mine":true,"games":32,"bank_median":76298.0,"games_losing_an_animal":8,"escape_first_day_min":8,"escapes_by_day":{"8":4,"19":12,"20":4,"22":4},"per_game":{"animals_escaped":0.75,"escape_capital":337.5,"escape_yield_units":0.5,"animal_unfed_days":21.688,"feed_no_wheat":2.75,"feed_no_animal":8.812,"feed_redundant":1.375,"plants_dry":4.75,"dry_seed_cost":151.25,"plants_decayed":32.875,"decay_units":37.375,"plant_unwatered_days":397.0,"shed_discarded_units":15.188,"dead_actions":132.094,"water_dead":51.75,"harvest_dead":21.5,"plant_dead":0.5,"sell_zero_fill":72.125,"buy_zero_fill":1.562,"sell_dead_units":9104.688,"sold_units":1396.125,"silent_loss_coins":488.75,"silent_loss_units":57.688,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":1.562,"tile_cap_units":10.562,"fertilizer_forgone":4.625,"care_unfed":14.75,"animal_shed_days":0.125,"idle_tile_days":114.0,"weed_tile_days":19.469,"hand_pass_turns":496.094,"phantom_hand_actions":0.0,"sell_floor_units":165.219,"hire_premium":6214.125,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.875,"seeds_unplanted_cost_now":18.125,"move_dead":1.938,"locked_dead":0.875,"pickup_dead":0.938,"drop_dead":1.375,"place_dead":2.312,"build_dead":0.0,"dig_dead":1.469,"fertilize_dead":1.75,"collect_dead":11.75,"care_dead":23.0,"farmer_pass_turns":62.812}},{"agent":"mine, earlier build 141","mine":true,"games":32,"bank_median":16636.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":0.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":44.375,"dry_seed_cost":583.75,"plants_decayed":2.062,"decay_units":6.156,"plant_unwatered_days":84.906,"shed_discarded_units":0.0,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":504.219,"silent_loss_coins":583.75,"silent_loss_units":58.906,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":0.0,"care_unfed":0.0,"animal_shed_days":0.0,"idle_tile_days":712.812,"weed_tile_days":261.094,"hand_pass_turns":1320.969,"phantom_hand_actions":0.0,"sell_floor_units":17.0,"hire_premium":5029.562,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":9.938,"seeds_unplanted_cost_now":99.375,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":260.188}},{"agent":"mine, earlier build 142","mine":true,"games":32,"bank_median":76298.0,"games_losing_an_animal":8,"escape_first_day_min":8,"escapes_by_day":{"8":4,"19":12,"20":4,"22":4},"per_game":{"animals_escaped":0.75,"escape_capital":337.5,"escape_yield_units":0.5,"animal_unfed_days":21.688,"feed_no_wheat":2.75,"feed_no_animal":8.812,"feed_redundant":1.375,"plants_dry":4.75,"dry_seed_cost":151.25,"plants_decayed":32.875,"decay_units":37.375,"plant_unwatered_days":397.0,"shed_discarded_units":15.188,"dead_actions":132.094,"water_dead":51.75,"harvest_dead":21.5,"plant_dead":0.5,"sell_zero_fill":72.125,"buy_zero_fill":1.562,"sell_dead_units":9104.688,"sold_units":1396.125,"silent_loss_coins":488.75,"silent_loss_units":57.688,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":1.562,"tile_cap_units":10.562,"fertilizer_forgone":4.625,"care_unfed":14.75,"animal_shed_days":0.125,"idle_tile_days":114.0,"weed_tile_days":19.469,"hand_pass_turns":496.094,"phantom_hand_actions":0.0,"sell_floor_units":165.219,"hire_premium":6214.125,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.875,"seeds_unplanted_cost_now":18.125,"move_dead":1.938,"locked_dead":0.875,"pickup_dead":0.938,"drop_dead":1.375,"place_dead":2.312,"build_dead":0.0,"dig_dead":1.469,"fertilize_dead":1.75,"collect_dead":11.75,"care_dead":23.0,"farmer_pass_turns":62.812}},{"agent":"mine, earlier build 143","mine":true,"games":32,"bank_median":70931.0,"games_losing_an_animal":2,"escape_first_day_min":8,"escapes_by_day":{"8":4},"per_game":{"animals_escaped":0.125,"escape_capital":62.5,"escape_yield_units":0.0,"animal_unfed_days":19.312,"feed_no_wheat":0.188,"feed_no_animal":2.312,"feed_redundant":0.25,"plants_dry":1.312,"dry_seed_cost":18.75,"plants_decayed":31.062,"decay_units":31.375,"plant_unwatered_days":395.75,"shed_discarded_units":13.656,"dead_actions":46.094,"water_dead":19.75,"harvest_dead":2.438,"plant_dead":0.062,"sell_zero_fill":8.344,"buy_zero_fill":1.625,"sell_dead_units":8000.781,"sold_units":1444.031,"silent_loss_coins":81.25,"silent_loss_units":46.281,"refused_buy_product":0.125,"refused_buy_seed":0.0,"refused_buy_animal":1.625,"tile_cap_units":8.688,"fertilizer_forgone":4.125,"care_unfed":13.312,"animal_shed_days":0.188,"idle_tile_days":53.125,"weed_tile_days":2.656,"hand_pass_turns":485.406,"phantom_hand_actions":0.0,"sell_floor_units":194.156,"hire_premium":5828.188,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.312,"drop_dead":0.0,"place_dead":1.562,"build_dead":0.0,"dig_dead":0.531,"fertilize_dead":0.562,"collect_dead":2.812,"care_dead":15.312,"farmer_pass_turns":63.0}},{"agent":"mine, earlier build 144","mine":true,"games":32,"bank_median":17120.0,"games_losing_an_animal":32,"escape_first_day_min":13,"escapes_by_day":{"13":32},"per_game":{"animals_escaped":1.0,"escape_capital":300.0,"escape_yield_units":0.0,"animal_unfed_days":3.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":42.594,"dry_seed_cost":425.938,"plants_decayed":12.438,"decay_units":36.938,"plant_unwatered_days":123.812,"shed_discarded_units":17.438,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":722.875,"silent_loss_coins":725.938,"silent_loss_units":101.531,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":1.0,"care_unfed":0.0,"animal_shed_days":146.0,"idle_tile_days":598.719,"weed_tile_days":210.938,"hand_pass_turns":1315.875,"phantom_hand_actions":0.0,"sell_floor_units":12.0,"hire_premium":3503.188,"shed_units_now":40.0,"shed_value_now":1575.0,"animals_in_shed_now":6.0,"seeds_unplanted_now":2.438,"seeds_unplanted_cost_now":24.375,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":375.0}},{"agent":"mine, earlier build 145","mine":true,"games":32,"bank_median":15314.0,"games_losing_an_animal":32,"escape_first_day_min":12,"escapes_by_day":{"12":32,"13":32,"14":32,"15":32,"16":16},"per_game":{"animals_escaped":4.5,"escape_capital":1350.0,"escape_yield_units":0.0,"animal_unfed_days":17.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":26.938,"dry_seed_cost":1280.0,"plants_decayed":8.062,"decay_units":11.75,"plant_unwatered_days":66.344,"shed_discarded_units":20.0,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":635.188,"silent_loss_coins":2630.0,"silent_loss_units":49.188,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":1.0,"care_unfed":0.0,"animal_shed_days":113.5,"idle_tile_days":606.188,"weed_tile_days":155.781,"hand_pass_turns":1398.469,"phantom_hand_actions":0.0,"sell_floor_units":7.0,"hire_premium":2860.375,"shed_units_now":42.062,"shed_value_now":2137.875,"animals_in_shed_now":4.0,"seeds_unplanted_now":2.875,"seeds_unplanted_cost_now":50.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":187.5}},{"agent":"mine, earlier build 146","mine":true,"games":32,"bank_median":15979.0,"games_losing_an_animal":32,"escape_first_day_min":13,"escapes_by_day":{"13":32},"per_game":{"animals_escaped":1.0,"escape_capital":300.0,"escape_yield_units":0.0,"animal_unfed_days":3.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":30.312,"dry_seed_cost":303.125,"plants_decayed":5.688,"decay_units":17.062,"plant_unwatered_days":44.125,"shed_discarded_units":94.875,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":683.562,"silent_loss_coins":603.125,"silent_loss_units":142.25,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":1.0,"care_unfed":0.0,"animal_shed_days":146.0,"idle_tile_days":692.062,"weed_tile_days":349.875,"hand_pass_turns":1467.812,"phantom_hand_actions":0.0,"sell_floor_units":3.0,"hire_premium":3406.125,"shed_units_now":40.0,"shed_value_now":1557.5,"animals_in_shed_now":6.0,"seeds_unplanted_now":10.375,"seeds_unplanted_cost_now":103.75,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":350.5}},{"agent":"mine, earlier build 147","mine":true,"games":32,"bank_median":12948.0,"games_losing_an_animal":32,"escape_first_day_min":13,"escapes_by_day":{"13":32},"per_game":{"animals_escaped":1.0,"escape_capital":300.0,"escape_yield_units":0.0,"animal_unfed_days":3.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":20.875,"dry_seed_cost":208.75,"plants_decayed":9.625,"decay_units":29.812,"plant_unwatered_days":48.0,"shed_discarded_units":90.688,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":594.875,"silent_loss_coins":508.75,"silent_loss_units":141.375,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":0.0,"fertilizer_forgone":1.0,"care_unfed":0.0,"animal_shed_days":146.0,"idle_tile_days":785.75,"weed_tile_days":275.781,"hand_pass_turns":1373.406,"phantom_hand_actions":0.0,"sell_floor_units":3.0,"hire_premium":3038.812,"shed_units_now":40.0,"shed_value_now":1690.0,"animals_in_shed_now":6.0,"seeds_unplanted_now":8.938,"seeds_unplanted_cost_now":89.375,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":312.125}},{"agent":"mine, earlier build 148","mine":true,"games":32,"bank_median":640.0,"games_losing_an_animal":32,"escape_first_day_min":13,"escapes_by_day":{"13":2,"14":84,"21":81,"22":26,"23":53,"24":40,"25":47,"26":20,"27":56,"28":34},"per_game":{"animals_escaped":13.406,"escape_capital":6096.875,"escape_yield_units":1.219,"animal_unfed_days":42.188,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":41.938,"dry_seed_cost":2355.938,"plants_decayed":14.375,"decay_units":16.594,"plant_unwatered_days":113.625,"shed_discarded_units":1.969,"dead_actions":21.969,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":269.594,"silent_loss_coins":8452.812,"silent_loss_units":51.344,"refused_buy_product":2.125,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":3.719,"fertilizer_forgone":11.562,"care_unfed":0.0,"animal_shed_days":10.938,"idle_tile_days":746.031,"weed_tile_days":200.906,"hand_pass_turns":858.344,"phantom_hand_actions":0.0,"sell_floor_units":12.688,"hire_premium":3423.812,"shed_units_now":23.062,"shed_value_now":1041.625,"animals_in_shed_now":0.188,"seeds_unplanted_now":7.125,"seeds_unplanted_cost_now":516.875,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":21.969,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":82.5}},{"agent":"mine, earlier build 149","mine":true,"games":32,"bank_median":38336.0,"games_losing_an_animal":32,"escape_first_day_min":12,"escapes_by_day":{"12":2,"13":10,"14":8,"15":8,"16":14,"17":48,"18":38,"19":12,"20":16,"21":8,"22":54,"23":16,"24":26,"26":8,"27":6},"per_game":{"animals_escaped":7.312,"escape_capital":3181.25,"escape_yield_units":0.0,"animal_unfed_days":31.562,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":37.25,"plants_dry":48.469,"dry_seed_cost":2943.438,"plants_decayed":5.812,"decay_units":13.625,"plant_unwatered_days":169.062,"shed_discarded_units":0.0,"dead_actions":49.469,"water_dead":9.875,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":215.75,"buy_zero_fill":5.562,"sell_dead_units":3554.0,"sold_units":984.875,"silent_loss_coins":6124.688,"silent_loss_units":37.406,"refused_buy_product":12.062,"refused_buy_seed":7.875,"refused_buy_animal":4.25,"tile_cap_units":23.125,"fertilizer_forgone":0.219,"care_unfed":21.25,"animal_shed_days":38.125,"idle_tile_days":305.562,"weed_tile_days":122.25,"hand_pass_turns":260.281,"phantom_hand_actions":0.0,"sell_floor_units":49.75,"hire_premium":5329.625,"shed_units_now":18.25,"shed_value_now":1606.438,"animals_in_shed_now":0.25,"seeds_unplanted_now":20.969,"seeds_unplanted_cost_now":522.5,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":1.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":1.344,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":3.438}},{"agent":"xuantianfengwu-preempt-selector","mine":false,"games":32,"bank_median":21474.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":2.0,"feed_no_wheat":0.0,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":0.0,"dry_seed_cost":0.0,"plants_decayed":0.0,"decay_units":0.0,"plant_unwatered_days":0.0,"shed_discarded_units":48.531,"dead_actions":0.0,"water_dead":0.0,"harvest_dead":0.0,"plant_dead":0.0,"sell_zero_fill":0.0,"buy_zero_fill":0.0,"sell_dead_units":0.0,"sold_units":433.906,"silent_loss_coins":0.0,"silent_loss_units":48.531,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":16.0,"fertilizer_forgone":0.0,"care_unfed":0.0,"animal_shed_days":2.0,"idle_tile_days":418.0,"weed_tile_days":29.0,"hand_pass_turns":4790.281,"phantom_hand_actions":0.0,"sell_floor_units":49.375,"hire_premium":3642.688,"shed_units_now":27.0,"shed_value_now":1313.719,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.0,"drop_dead":0.0,"place_dead":0.0,"build_dead":0.0,"dig_dead":0.0,"fertilize_dead":0.0,"collect_dead":0.0,"care_dead":0.0,"farmer_pass_turns":535.906}},{"agent":"yhay81-shop-router-0909","mine":false,"games":32,"bank_median":80469.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":28.562,"feed_no_wheat":0.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":18.0,"decay_units":18.0,"plant_unwatered_days":379.0,"shed_discarded_units":8.562,"dead_actions":42.719,"water_dead":11.469,"harvest_dead":6.062,"plant_dead":0.0,"sell_zero_fill":102.688,"buy_zero_fill":0.0,"sell_dead_units":88068.625,"sold_units":1528.375,"silent_loss_coins":110.0,"silent_loss_units":27.562,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":13.938,"fertilizer_forgone":14.688,"care_unfed":24.562,"animal_shed_days":0.938,"idle_tile_days":64.062,"weed_tile_days":2.719,"hand_pass_turns":400.406,"phantom_hand_actions":0.0,"sell_floor_units":194.312,"hire_premium":3609.062,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.0,"seeds_unplanted_cost_now":0.0,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.062,"drop_dead":0.0,"place_dead":0.438,"build_dead":0.0,"dig_dead":0.125,"fertilize_dead":0.0,"collect_dead":3.938,"care_dead":20.5,"farmer_pass_turns":61.312}},{"agent":"yhay81-shop-router-0911-simple","mine":false,"games":32,"bank_median":99544.0,"games_losing_an_animal":0,"escape_first_day_min":-1,"escapes_by_day":{},"per_game":{"animals_escaped":0.0,"escape_capital":0.0,"escape_yield_units":0.0,"animal_unfed_days":31.25,"feed_no_wheat":1.125,"feed_no_animal":0.0,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":375.562,"shed_discarded_units":0.0,"dead_actions":74.25,"water_dead":34.5,"harvest_dead":6.875,"plant_dead":2.688,"sell_zero_fill":110.062,"buy_zero_fill":0.0,"sell_dead_units":84133.125,"sold_units":1584.812,"silent_loss_coins":110.0,"silent_loss_units":18.938,"refused_buy_product":0.0,"refused_buy_seed":1.0,"refused_buy_animal":0.0,"tile_cap_units":22.5,"fertilizer_forgone":15.0,"care_unfed":27.25,"animal_shed_days":1.0,"idle_tile_days":87.312,"weed_tile_days":7.375,"hand_pass_turns":372.875,"phantom_hand_actions":17.5,"sell_floor_units":133.0,"hire_premium":3733.0,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":1.688,"seeds_unplanted_cost_now":93.75,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.125,"drop_dead":0.0,"place_dead":1.0,"build_dead":0.0,"dig_dead":0.875,"fertilize_dead":0.062,"collect_dead":3.0,"care_dead":24.0,"farmer_pass_turns":59.125}},{"agent":"yhay81-shop-router-0913","mine":false,"games":32,"bank_median":81330.0,"games_losing_an_animal":6,"escape_first_day_min":25,"escapes_by_day":{"25":6,"28":6},"per_game":{"animals_escaped":0.375,"escape_capital":162.5,"escape_yield_units":0.062,"animal_unfed_days":31.125,"feed_no_wheat":1.312,"feed_no_animal":0.375,"feed_redundant":0.0,"plants_dry":2.0,"dry_seed_cost":110.0,"plants_decayed":17.938,"decay_units":17.938,"plant_unwatered_days":388.375,"shed_discarded_units":48.75,"dead_actions":45.031,"water_dead":10.375,"harvest_dead":5.5,"plant_dead":0.062,"sell_zero_fill":119.75,"buy_zero_fill":0.0,"sell_dead_units":89146.875,"sold_units":1761.062,"silent_loss_coins":272.5,"silent_loss_units":67.75,"refused_buy_product":0.0,"refused_buy_seed":0.0,"refused_buy_animal":0.0,"tile_cap_units":25.25,"fertilizer_forgone":13.688,"care_unfed":23.938,"animal_shed_days":1.0,"idle_tile_days":65.062,"weed_tile_days":3.688,"hand_pass_turns":388.938,"phantom_hand_actions":0.0,"sell_floor_units":145.219,"hire_premium":3867.375,"shed_units_now":0.0,"shed_value_now":0.0,"animals_in_shed_now":0.0,"seeds_unplanted_now":0.062,"seeds_unplanted_cost_now":6.25,"move_dead":0.0,"locked_dead":0.0,"pickup_dead":0.375,"drop_dead":0.0,"place_dead":3.656,"build_dead":0.0,"dig_dead":1.562,"fertilize_dead":0.625,"collect_dead":10.0,"care_dead":11.188,"farmer_pass_turns":60.562}},{"agent":"yhay81-two-shop-router","mine":false,"games":32,"bank_median":74189.0,"games_losing_an_animal":22,"escape_first_day_min":8,"escapes_by_day":{"8":22},"per_game":{"animals_escaped":0.688,"escape_capital":275.0,"escape_yield_units":0.0,"animal_unfed_days":40.875,"feed_no_wheat":1.5,"feed_no_animal":54.0,"feed_redundant":0.0,"plants_dry":5.688,"dry_seed_cost":452.5,"plants_decayed":16.656,"decay_units":19.094,"plant_unwatered_days":364.312,"shed_discarded_units":21.062,"dead_actions":350.125,"water_dead":81.719,"harvest_dead":55.969,"plant_dead":11.812,"sell_zero_fill":24.75,"buy_zero_fill":5.875,"sell_dead_units":207.156,"sold_units":1219.406,"silent_loss_coins":727.5,"silent_loss_units":43.531,"refused_buy_product":17.25,"refused_buy_seed":3.812,"refused_buy_animal":1.75,"tile_cap_units":3.75,"fertilizer_forgone":7.125,"care_unfed":20.812,"animal_shed_days":14.688,"idle_tile_days":183.719,"weed_tile_days":119.531,"hand_pass_turns":713.25,"phantom_hand_actions":0.0,"sell_floor_units":73.312,"hire_premium":4405.062,"shed_units_now":14.25,"shed_value_now":502.875,"animals_in_shed_now":0.688,"seeds_unplanted_now":17.875,"seeds_unplanted_cost_now":1010.0,"move_dead":0.562,"locked_dead":6.562,"pickup_dead":2.0,"drop_dead":0.0,"place_dead":7.625,"build_dead":0.125,"dig_dead":1.562,"fertilize_dead":10.938,"collect_dead":63.375,"care_dead":52.375,"farmer_pass_turns":83.625}}]}''')
TRACE = json.loads(r'''{"seed":17,"rival":"yhay81-shop-router-0913","defective":{"agent":"v7.76_learned","bank":107957.0,"rival_bank":139950.0,"events":[{"turn":23,"day":0,"what":"animal_unfed_days","n":2},{"turn":27,"day":1,"what":"feed_no_wheat","n":1},{"turn":28,"day":1,"what":"feed_no_wheat","n":1},{"turn":47,"day":1,"what":"animals_escaped","n":2},{"turn":47,"day":1,"what":"animal_unfed_days","n":2},{"turn":49,"day":2,"what":"feed_no_animal","n":1},{"turn":53,"day":2,"what":"feed_no_animal","n":1},{"turn":71,"day":2,"what":"animal_unfed_days","n":1},{"turn":97,"day":4,"what":"feed_no_animal","n":1},{"turn":101,"day":4,"what":"feed_no_animal","n":1},{"turn":113,"day":4,"what":"feed_no_animal","n":1},{"turn":119,"day":4,"what":"animal_unfed_days","n":1},{"turn":122,"day":5,"what":"feed_no_animal","n":1},{"turn":125,"day":5,"what":"feed_no_animal","n":1},{"turn":126,"day":5,"what":"feed_no_animal","n":1},{"turn":146,"day":6,"what":"feed_no_animal","n":1},{"turn":148,"day":6,"what":"feed_no_animal","n":1},{"turn":149,"day":6,"what":"feed_no_animal","n":1},{"turn":171,"day":7,"what":"feed_no_animal","n":2},{"turn":172,"day":7,"what":"feed_no_animal","n":1},{"turn":191,"day":7,"what":"animal_unfed_days","n":1},{"turn":194,"day":8,"what":"feed_no_animal","n":1},{"turn":197,"day":8,"what":"feed_no_animal","n":1},{"turn":199,"day":8,"what":"feed_no_animal","n":1},{"turn":206,"day":8,"what":"feed_no_animal","n":2},{"turn":212,"day":8,"what":"feed_no_animal","n":1},{"turn":215,"day":8,"what":"animals_escaped","n":1},{"turn":215,"day":8,"what":"animal_unfed_days","n":3},{"turn":218,"day":9,"what":"feed_no_animal","n":1},{"turn":222,"day":9,"what":"feed_no_animal","n":1},{"turn":223,"day":9,"what":"feed_no_animal","n":1},{"turn":227,"day":9,"what":"feed_no_animal","n":1},{"turn":228,"day":9,"what":"feed_no_animal","n":1},{"turn":230,"day":9,"what":"feed_no_animal","n":1},{"turn":232,"day":9,"what":"feed_no_animal","n":1},{"turn":234,"day":9,"what":"feed_no_animal","n":1},{"turn":239,"day":9,"what":"feed_no_animal","n":1},{"turn":245,"day":10,"what":"feed_no_animal","n":1},{"turn":247,"day":10,"what":"feed_no_animal","n":1},{"turn":249,"day":10,"what":"feed_no_animal","n":1},{"turn":251,"day":10,"what":"feed_no_animal","n":1},{"turn":255,"day":10,"what":"feed_no_animal","n":1},{"turn":257,"day":10,"what":"feed_no_animal","n":1},{"turn":259,"day":10,"what":"feed_no_animal","n":1},{"turn":261,"day":10,"what":"feed_no_animal","n":2},{"turn":267,"day":11,"what":"feed_no_animal","n":1},{"turn":268,"day":11,"what":"feed_no_animal","n":1},{"turn":271,"day":11,"what":"feed_no_animal","n":1},{"turn":272,"day":11,"what":"feed_no_animal","n":1},{"turn":276,"day":11,"what":"feed_no_animal","n":1},{"turn":278,"day":11,"what":"feed_no_animal","n":1},{"turn":287,"day":11,"what":"animal_unfed_days","n":1},{"turn":290,"day":12,"what":"feed_no_animal","n":1},{"turn":294,"day":12,"what":"feed_no_animal","n":2},{"turn":297,"day":12,"what":"feed_no_animal","n":1},{"turn":298,"day":12,"what":"feed_no_animal","n":1},{"turn":301,"day":12,"what":"feed_no_animal","n":1},{"turn":302,"day":12,"what":"feed_no_animal","n":1},{"turn":303,"day":12,"what":"feed_no_animal","n":1},{"turn":306,"day":12,"what":"feed_no_animal","n":1},{"turn":314,"day":13,"what":"feed_no_animal","n":1},{"turn":318,"day":13,"what":"feed_no_animal","n":2},{"turn":322,"day":13,"what":"feed_no_animal","n":1},{"turn":326,"day":13,"what":"feed_no_animal","n":2},{"turn":327,"day":13,"what":"feed_no_animal","n":1},{"turn":330,"day":13,"what":"feed_no_animal","n":2},{"turn":338,"day":14,"what":"feed_no_animal","n":1},{"turn":341,"day":14,"what":"feed_no_animal","n":1},{"turn":342,"day":14,"what":"feed_no_animal","n":1},{"turn":344,"day":14,"what":"feed_no_animal","n":1},{"turn":345,"day":14,"what":"feed_no_animal","n":1},{"turn":348,"day":14,"what":"feed_no_animal","n":1},{"turn":352,"day":14,"what":"feed_no_animal","n":1},{"turn":353,"day":14,"what":"feed_no_animal","n":1},{"turn":356,"day":14,"what":"feed_no_animal","n":1},{"turn":363,"day":15,"what":"feed_no_animal","n":1},{"turn":365,"day":15,"what":"feed_no_animal","n":1},{"turn":367,"day":15,"what":"feed_no_animal","n":2},{"turn":369,"day":15,"what":"feed_no_animal","n":1},{"turn":371,"day":15,"what":"feed_no_animal","n":1},{"turn":375,"day":15,"what":"feed_no_animal","n":1},{"turn":376,"day":15,"what":"feed_no_animal","n":1},{"turn":379,"day":15,"what":"feed_no_animal","n":1},{"turn":386,"day":16,"what":"feed_no_animal","n":1},{"turn":390,"day":16,"what":"feed_no_animal","n":1},{"turn":391,"day":16,"what":"feed_no_animal","n":1},{"turn":395,"day":16,"what":"feed_no_animal","n":1},{"turn":398,"day":16,"what":"feed_no_animal","n":1},{"turn":399,"day":16,"what":"feed_no_animal","n":1},{"turn":400,"day":16,"what":"feed_no_animal","n":1},{"turn":402,"day":16,"what":"feed_no_animal","n":1},{"turn":403,"day":16,"what":"feed_no_animal","n":1},{"turn":412,"day":17,"what":"feed_no_animal","n":2},{"turn":415,"day":17,"what":"feed_no_animal","n":1},{"turn":416,"day":17,"what":"feed_no_animal","n":1},{"turn":417,"day":17,"what":"feed_no_animal","n":1},{"turn":419,"day":17,"what":"feed_no_animal","n":1},{"turn":423,"day":17,"what":"feed_no_animal","n":1},{"turn":424,"day":17,"what":"feed_no_animal","n":1},{"turn":428,"day":17,"what":"feed_no_animal","n":1},{"turn":435,"day":18,"what":"feed_no_animal","n":1},{"turn":437,"day":18,"what":"feed_no_animal","n":1},{"turn":439,"day":18,"what":"feed_no_animal","n":2},{"turn":441,"day":18,"what":"feed_no_animal","n":1},{"turn":444,"day":18,"what":"feed_no_animal","n":1},{"turn":447,"day":18,"what":"feed_no_animal","n":1},{"turn":448,"day":18,"what":"feed_no_animal","n":1},{"turn":453,"day":18,"what":"feed_no_animal","n":1},{"turn":458,"day":19,"what":"feed_no_animal","n":1},{"turn":460,"day":19,"what":"feed_no_animal","n":1},{"turn":462,"day":19,"what":"feed_no_animal","n":2},{"turn":465,"day":19,"what":"feed_no_animal","n":1},{"turn":466,"day":19,"what":"feed_no_animal","n":1},{"turn":471,"day":19,"what":"feed_no_animal","n":2},{"turn":475,"day":19,"what":"feed_no_animal","n":1},{"turn":481,"day":20,"what":"feed_no_animal","n":1},{"turn":484,"day":20,"what":"feed_no_animal","n":1},{"turn":485,"day":20,"what":"feed_no_animal","n":1},{"turn":486,"day":20,"what":"feed_no_animal","n":1},{"turn":490,"day":20,"what":"feed_no_animal","n":1},{"turn":492,"day":20,"what":"feed_no_animal","n":1},{"turn":496,"day":20,"what":"feed_no_animal","n":1},{"turn":500,"day":20,"what":"feed_no_animal","n":1},{"turn":501,"day":20,"what":"feed_no_animal","n":1},{"turn":506,"day":21,"what":"feed_no_animal","n":1},{"turn":507,"day":21,"what":"feed_no_animal","n":1},{"turn":510,"day":21,"what":"feed_no_animal","n":1},{"turn":511,"day":21,"what":"feed_no_animal","n":1},{"turn":512,"day":21,"what":"feed_no_animal","n":1},{"turn":515,"day":21,"what":"feed_no_animal","n":1},{"turn":519,"day":21,"what":"feed_no_animal","n":1},{"turn":520,"day":21,"what":"feed_no_animal","n":1},{"turn":524,"day":21,"what":"feed_no_animal","n":1},{"turn":530,"day":22,"what":"feed_no_animal","n":1},{"turn":532,"day":22,"what":"feed_no_animal","n":1},{"turn":534,"day":22,"what":"feed_no_animal","n":1},{"turn":535,"day":22,"what":"feed_no_animal","n":1},{"turn":536,"day":22,"what":"feed_no_animal","n":1},{"turn":538,"day":22,"what":"feed_no_animal","n":1},{"turn":542,"day":22,"what":"feed_no_animal","n":1},{"turn":543,"day":22,"what":"feed_no_animal","n":1},{"turn":547,"day":22,"what":"feed_no_animal","n":1},{"turn":555,"day":23,"what":"feed_no_animal","n":1},{"turn":557,"day":23,"what":"feed_no_animal","n":1},{"turn":558,"day":23,"what":"feed_no_animal","n":1},{"turn":559,"day":23,"what":"feed_no_animal","n":1},{"turn":562,"day":23,"what":"feed_no_animal","n":2},{"turn":567,"day":23,"what":"feed_no_animal","n":1},{"turn":568,"day":23,"what":"feed_no_animal","n":1},{"turn":571,"day":23,"what":"feed_no_animal","n":1},{"turn":578,"day":24,"what":"feed_no_animal","n":1},{"turn":579,"day":24,"what":"feed_no_animal","n":1},{"turn":583,"day":24,"what":"feed_no_animal","n":1},{"turn":586,"day":24,"what":"feed_no_animal","n":1},{"turn":590,"day":24,"what":"feed_no_animal","n":1},{"turn":591,"day":24,"what":"feed_no_animal","n":1},{"turn":595,"day":24,"what":"feed_no_animal","n":1},{"turn":599,"day":24,"what":"animal_unfed_days","n":2},{"turn":603,"day":25,"what":"feed_no_animal","n":2},{"turn":606,"day":25,"what":"feed_no_animal","n":2},{"turn":608,"day":25,"what":"feed_no_animal","n":1},{"turn":610,"day":25,"what":"feed_no_animal","n":1},{"turn":614,"day":25,"what":"feed_no_animal","n":1},{"turn":615,"day":25,"what":"feed_no_animal","n":1},{"turn":619,"day":25,"what":"feed_no_animal","n":1},{"turn":623,"day":25,"what":"animal_unfed_days","n":2},{"turn":626,"day":26,"what":"feed_no_animal","n":1},{"turn":628,"day":26,"what":"feed_no_animal","n":1},{"turn":630,"day":26,"what":"feed_no_animal","n":1},{"turn":631,"day":26,"what":"feed_no_animal","n":1},{"turn":632,"day":26,"what":"feed_no_animal","n":1},{"turn":634,"day":26,"what":"feed_no_animal","n":1},{"turn":638,"day":26,"what":"feed_no_animal","n":1},{"turn":639,"day":26,"what":"feed_no_animal","n":1},{"turn":642,"day":26,"what":"feed_no_animal","n":1},{"turn":649,"day":27,"what":"feed_no_animal","n":1},{"turn":652,"day":27,"what":"feed_no_animal","n":5},{"turn":658,"day":27,"what":"feed_no_animal","n":2},{"turn":662,"day":27,"what":"feed_no_animal","n":1},{"turn":679,"day":28,"what":"feed_no_animal","n":2},{"turn":683,"day":28,"what":"feed_no_animal","n":1},{"turn":687,"day":28,"what":"feed_no_animal","n":1},{"turn":689,"day":28,"what":"feed_no_animal","n":1},{"turn":693,"day":28,"what":"feed_no_animal","n":1},{"turn":695,"day":28,"what":"animal_unfed_days","n":5}]},"repaired":{"agent":"v7.77_learned","bank":84337.0,"rival_bank":81844.0,"events":[{"turn":23,"day":0,"what":"animal_unfed_days","n":2},{"turn":71,"day":2,"what":"animal_unfed_days","n":1},{"turn":95,"day":3,"what":"animal_unfed_days","n":3},{"turn":119,"day":4,"what":"animal_unfed_days","n":1},{"turn":191,"day":7,"what":"animal_unfed_days","n":3},{"turn":215,"day":8,"what":"animal_unfed_days","n":1},{"turn":287,"day":11,"what":"animal_unfed_days","n":4},{"turn":499,"day":20,"what":"feed_no_wheat","n":1},{"turn":500,"day":20,"what":"feed_no_wheat","n":1},{"turn":503,"day":20,"what":"animal_unfed_days","n":2},{"turn":599,"day":24,"what":"animal_unfed_days","n":4},{"turn":623,"day":25,"what":"animal_unfed_days","n":2},{"turn":695,"day":28,"what":"animal_unfed_days","n":8}]}}''')
LEAD = json.loads(r'''{"worlds":8,"rows":[{"lead_turns":[20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[],"unfed_lead_turns":[672,672],"games_with_a_loss":2,"games_warned_first":0},{"lead_turns":[343,343,343,343,343,343],"unfed_lead_turns":[456,456,456,456,456,456,456,456,456,456,456,456,456,456,456,456],"games_with_a_loss":16,"games_warned_first":6},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[],"unfed_lead_turns":[672,672],"games_with_a_loss":2,"games_warned_first":0},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[269,366,274,274,274,274,269,366,274,274,274,274],"unfed_lead_turns":[504,504,504,504,504,504,504,504,504,504,504,504,504,504,504,504],"games_with_a_loss":16,"games_warned_first":12},{"lead_turns":[],"unfed_lead_turns":[672,672],"games_with_a_loss":2,"games_warned_first":0},{"lead_turns":[130,130],"unfed_lead_turns":[120,120],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[470,470,470,470,470,470,470,470,470,470,470,470,470,470],"unfed_lead_turns":[672,672,672,672,672,672,672,672,672,672,672,672,672,672],"games_with_a_loss":14,"games_warned_first":14},{"lead_turns":[396,41,396,41],"unfed_lead_turns":[552,288,552,288],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[],"unfed_lead_turns":[672,672],"games_with_a_loss":2,"games_warned_first":0},{"lead_turns":[5,19,5,15,5,5,5,19,5,15,5,5],"unfed_lead_turns":[192,672,672,192,672,192,192,672,192,672,672,192,672,192,192,672],"games_with_a_loss":16,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,672,192,192,672,192,192,672,192,192,672],"games_with_a_loss":12,"games_warned_first":8},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[],"unfed_lead_turns":[],"games_with_a_loss":0,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,240,192,24,24,24,24,168,24,240,192,24,24,24,24,168],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,240,192,24,24,24,24,168,24,240,192,24,24,24,24,168],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[432,384,192,432,408,456,672,432,432,384,192,432,408,456,672,432],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[120,24,168,120,24,24,432,48,120,24,168,120,24,24,432,48],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,96,24,24,24,24,144,24,24,96,24,24,24,24,144,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,336,24,24,216,24,24,24,24,336,24,24,216,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[207,207],"unfed_lead_turns":[360,360],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[],"unfed_lead_turns":[504,432,168,432,504,432,168,432],"games_with_a_loss":8,"games_warned_first":0},{"lead_turns":[396,237,396,237],"unfed_lead_turns":[552,432,552,432],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26],"unfed_lead_turns":[48,48,48,48,48,48,48,48,48,48,48,48,48,48,48,48],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[396,237,396,237],"unfed_lead_turns":[552,432,552,432],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[396,237,396,237],"unfed_lead_turns":[552,432,552,432],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[],"unfed_lead_turns":[672,672],"games_with_a_loss":2,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[672,672],"games_with_a_loss":2,"games_warned_first":0},{"lead_turns":[49,49],"unfed_lead_turns":[168,168],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[207,207],"unfed_lead_turns":[360,360],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[41,41],"unfed_lead_turns":[288,288],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[41,41],"unfed_lead_turns":[288,288],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[41,41],"unfed_lead_turns":[288,288],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[41,41],"unfed_lead_turns":[288,288],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[41,41],"unfed_lead_turns":[288,288],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":10,"games_warned_first":0},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":12,"games_warned_first":12},{"lead_turns":[87,87,87,87,87,87,87,87,87,87,87,87,87,87,87,87],"unfed_lead_turns":[168,168,168,168,168,168,168,168,168,168,168,168,168,168,168,168],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[226,250,58,250,58,250,226,226,373,349,181,397,421,349,349,349],"unfed_lead_turns":[360,384,192,384,192,384,360,360,384,360,192,408,432,360,360,360],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[226,250,58,250,226,250,226,226,373,349,469,397,421,349,349,277],"unfed_lead_turns":[360,384,192,384,360,384,360,360,384,360,480,408,432,360,360,288],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[226,250,58,250,226,250,226,226,373,349,469,397,421,349,349,277],"unfed_lead_turns":[360,384,192,384,360,384,360,360,384,360,480,408,432,360,360,288],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[58,58],"unfed_lead_turns":[192,192],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[5,5,58,5,5,5,58,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192],"games_with_a_loss":8,"games_warned_first":8},{"lead_turns":[12,12],"unfed_lead_turns":[192,192],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[5,5,58,5,5,5,58,5],"unfed_lead_turns":[192,192,192,192,192,192,192,192],"games_with_a_loss":8,"games_warned_first":8},{"lead_turns":[5,19,5,15,5,5,5,19,5,15,5,5],"unfed_lead_turns":[192,672,672,192,672,192,192,672,192,672,672,192,672,192,192,672],"games_with_a_loss":16,"games_warned_first":12},{"lead_turns":[5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,672,192,192,672,192,192,672,192,192,672],"games_with_a_loss":12,"games_warned_first":8},{"lead_turns":[49,49],"unfed_lead_turns":[168,168],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,672,192,192,672,192,192,672,192,192,672],"games_with_a_loss":12,"games_warned_first":8},{"lead_turns":[5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,672,192,192,672,192,192,672,192,192,672],"games_with_a_loss":12,"games_warned_first":8},{"lead_turns":[5,5,5,5,5,5,5,5],"unfed_lead_turns":[192,192,672,192,192,672,192,192,672,192,192,672],"games_with_a_loss":12,"games_warned_first":8},{"lead_turns":[20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[34,34],"unfed_lead_turns":[168,168],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":16},{"lead_turns":[49,49],"unfed_lead_turns":[168,168],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[106,106,34,106,106,106,106,34,106,106],"unfed_lead_turns":[240,240,168,240,240,240,240,168,240,240],"games_with_a_loss":10,"games_warned_first":10},{"lead_turns":[34,34],"unfed_lead_turns":[168,168],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[12,12],"unfed_lead_turns":[192,192],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[12,18,12,18],"unfed_lead_turns":[192,528,192,528],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[12,18,12,18],"unfed_lead_turns":[192,528,192,528],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[12,12],"unfed_lead_turns":[192,192],"games_with_a_loss":2,"games_warned_first":2},{"lead_turns":[],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[24,24,24,24,24,24,24,24,24,24,24,24,24,24,24,24],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[288,240,72,72,72,240,240,240,288,240,72,72,72,240,240,240],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[],"unfed_lead_turns":[384,384,312,408,336,336,312,384,384,384,312,408,336,336,312,384],"games_with_a_loss":16,"games_warned_first":0},{"lead_turns":[61,61,61,61],"unfed_lead_turns":[600,600,600,600],"games_with_a_loss":4,"games_warned_first":4},{"lead_turns":[18,18,18,18,20,18,18,18,18,20],"unfed_lead_turns":[192,192,192,192,192,192,192,192,192,192],"games_with_a_loss":10,"games_warned_first":10}]}''')
EPISODES = json.loads(r'''[{"episode":109028888,"trusted":true,"our_seat":0,"banks":[77710.0,88628.0],"replayed":[77710.0,88628.0],"observed_escapes":[0,0],"ledgers":[{"sell_dead_units":84155,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":105,"buy_zero_fill":0,"shed_discarded_units":6,"sold_units":1733,"feed_no_wheat":2,"feed_no_animal":0,"feed_redundant":0,"water_dead":13,"harvest_dead":4,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":31,"plants_dry":5,"dry_seed_cost":260.0,"plant_unwatered_days":430,"decay_units":18,"plants_decayed":18,"silent_loss_coins":260.0,"silent_loss_units":25,"dead_actions":49,"move_dead":0,"locked_dead":0,"pickup_dead":2,"drop_dead":0,"place_dead":0,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":22,"fertilizer_forgone":15,"care_unfed":27,"animal_shed_days":1,"idle_tile_days":236,"weed_tile_days":32,"hand_pass_turns":396,"farmer_pass_turns":58,"phantom_hand_actions":0,"sell_floor_units":209,"hire_premium":6117,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0},{"sell_dead_units":84091,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":109,"buy_zero_fill":0,"shed_discarded_units":8,"sold_units":1663,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":13,"harvest_dead":5,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":49,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":379,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":27,"dead_actions":46,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":0,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":22,"fertilizer_forgone":15,"care_unfed":45,"animal_shed_days":1,"idle_tile_days":229,"weed_tile_days":3,"hand_pass_turns":439,"farmer_pass_turns":61,"phantom_hand_actions":0,"sell_floor_units":213,"hire_premium":6487,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0}],"events":[[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":266,"what":"sell_zero_fill","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":401,"what":"sell_zero_fill","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":451,"what":"sell_zero_fill","n":1},{"turn":455,"what":"plants_dry","n":1},{"turn":455,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":494,"what":"dead_actions","n":1},{"turn":499,"what":"feed_no_wheat","n":1},{"turn":499,"what":"dead_actions","n":3},{"turn":500,"what":"feed_no_wheat","n":1},{"turn":500,"what":"dead_actions","n":1},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"plants_dry","n":2},{"turn":503,"what":"animal_unfed_days","n":2},{"turn":509,"what":"dead_actions","n":1},{"turn":512,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":579,"what":"sell_zero_fill","n":1},{"turn":595,"what":"sell_zero_fill","n":1},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":609,"what":"dead_actions","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":615,"what":"sell_zero_fill","n":1},{"turn":619,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":662,"what":"sell_zero_fill","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":2},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":1},{"turn":695,"what":"shed_discarded_units","n":6},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":698,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":8},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":8},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":717,"what":"sell_zero_fill","n":6}],[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":359,"what":"animal_unfed_days","n":3},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":367,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":383,"what":"animal_unfed_days","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":407,"what":"animal_unfed_days","n":4},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":431,"what":"animal_unfed_days","n":1},{"turn":455,"what":"animal_unfed_days","n":3},{"turn":455,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":479,"what":"animal_unfed_days","n":2},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":499,"what":"dead_actions","n":1},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"animal_unfed_days","n":3},{"turn":509,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":521,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"animal_unfed_days","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":2},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":551,"what":"shed_discarded_units","n":4},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":573,"what":"sell_zero_fill","n":2},{"turn":575,"what":"shed_discarded_units","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"shed_discarded_units","n":1},{"turn":599,"what":"animal_unfed_days","n":5},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":609,"what":"dead_actions","n":1},{"turn":615,"what":"sell_zero_fill","n":1},{"turn":619,"what":"sell_zero_fill","n":1},{"turn":622,"what":"sell_zero_fill","n":1},{"turn":623,"what":"shed_discarded_units","n":2},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":647,"what":"sell_zero_fill","n":1},{"turn":669,"what":"sell_zero_fill","n":3},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":2},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":2},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"animal_unfed_days","n":9},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":705,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":709,"what":"sell_zero_fill","n":2},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":5},{"turn":716,"what":"sell_zero_fill","n":6},{"turn":717,"what":"sell_zero_fill","n":7}]]},{"episode":109033105,"trusted":true,"our_seat":0,"banks":[78685.0,99442.0],"replayed":[78685.0,99442.0],"observed_escapes":[0,0],"ledgers":[{"sell_dead_units":116352,"refused_buy_product":1734,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":193,"buy_zero_fill":0,"shed_discarded_units":3,"sold_units":1845,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":6,"harvest_dead":17,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":29,"plants_dry":5,"dry_seed_cost":260.0,"plant_unwatered_days":431,"decay_units":18,"plants_decayed":18,"silent_loss_coins":260.0,"silent_loss_units":22,"dead_actions":40,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":2,"build_dead":0,"dig_dead":0,"fertilize_dead":0,"collect_dead":10,"care_dead":5,"tile_cap_units":2,"fertilizer_forgone":13,"care_unfed":25,"animal_shed_days":1,"idle_tile_days":234,"weed_tile_days":29,"hand_pass_turns":536,"farmer_pass_turns":76,"phantom_hand_actions":0,"sell_floor_units":277,"hire_premium":7469,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":4,"seeds_unplanted_cost_now":40.0},{"sell_dead_units":84009,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":97,"buy_zero_fill":0,"shed_discarded_units":2,"sold_units":1672,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":13,"harvest_dead":4,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":29,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":379,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":21,"dead_actions":45,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":0,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":23,"fertilizer_forgone":15,"care_unfed":25,"animal_shed_days":1,"idle_tile_days":229,"weed_tile_days":12,"hand_pass_turns":427,"farmer_pass_turns":59,"phantom_hand_actions":0,"sell_floor_units":172,"hire_premium":6487,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0}],"events":[[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":2},{"turn":190,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":199,"what":"sell_zero_fill","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":321,"what":"sell_zero_fill","n":1},{"turn":323,"what":"sell_zero_fill","n":1},{"turn":333,"what":"sell_zero_fill","n":1},{"turn":338,"what":"sell_zero_fill","n":1},{"turn":339,"what":"sell_zero_fill","n":1},{"turn":340,"what":"sell_zero_fill","n":1},{"turn":359,"what":"dead_actions","n":1},{"turn":360,"what":"sell_zero_fill","n":1},{"turn":362,"what":"sell_zero_fill","n":1},{"turn":373,"what":"sell_zero_fill","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":383,"what":"dead_actions","n":1},{"turn":384,"what":"sell_zero_fill","n":1},{"turn":393,"what":"sell_zero_fill","n":1},{"turn":398,"what":"sell_zero_fill","n":1},{"turn":402,"what":"sell_zero_fill","n":1},{"turn":403,"what":"sell_zero_fill","n":2},{"turn":404,"what":"sell_zero_fill","n":3},{"turn":405,"what":"sell_zero_fill","n":3},{"turn":406,"what":"sell_zero_fill","n":3},{"turn":407,"what":"sell_zero_fill","n":2},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":430,"what":"sell_zero_fill","n":8},{"turn":431,"what":"sell_zero_fill","n":1},{"turn":447,"what":"sell_zero_fill","n":2},{"turn":453,"what":"sell_zero_fill","n":1},{"turn":453,"what":"dead_actions","n":1},{"turn":454,"what":"sell_zero_fill","n":1},{"turn":455,"what":"dead_actions","n":1},{"turn":457,"what":"sell_zero_fill","n":2},{"turn":470,"what":"sell_zero_fill","n":1},{"turn":471,"what":"sell_zero_fill","n":2},{"turn":473,"what":"sell_zero_fill","n":1},{"turn":474,"what":"sell_zero_fill","n":1},{"turn":478,"what":"sell_zero_fill","n":1},{"turn":478,"what":"dead_actions","n":1},{"turn":479,"what":"dead_actions","n":1},{"turn":481,"what":"sell_zero_fill","n":1},{"turn":494,"what":"sell_zero_fill","n":1},{"turn":498,"what":"dead_actions","n":1},{"turn":501,"what":"sell_zero_fill","n":2},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"plants_dry","n":3},{"turn":503,"what":"sell_zero_fill","n":1},{"turn":505,"what":"sell_zero_fill","n":1},{"turn":509,"what":"dead_actions","n":1},{"turn":510,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":518,"what":"dead_actions","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":2},{"turn":535,"what":"sell_zero_fill","n":1},{"turn":537,"what":"sell_zero_fill","n":1},{"turn":545,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":2},{"turn":551,"what":"sell_zero_fill","n":2},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":553,"what":"sell_zero_fill","n":1},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":570,"what":"sell_zero_fill","n":2},{"turn":571,"what":"sell_zero_fill","n":2},{"turn":573,"what":"sell_zero_fill","n":2},{"turn":574,"what":"sell_zero_fill","n":2},{"turn":575,"what":"sell_zero_fill","n":2},{"turn":575,"what":"dead_actions","n":2},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":577,"what":"sell_zero_fill","n":2},{"turn":581,"what":"sell_zero_fill","n":1},{"turn":583,"what":"dead_actions","n":1},{"turn":585,"what":"sell_zero_fill","n":1},{"turn":585,"what":"dead_actions","n":2},{"turn":588,"what":"dead_actions","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":3},{"turn":599,"what":"dead_actions","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":601,"what":"sell_zero_fill","n":1},{"turn":612,"what":"sell_zero_fill","n":2},{"turn":617,"what":"sell_zero_fill","n":1},{"turn":621,"what":"sell_zero_fill","n":1},{"turn":621,"what":"dead_actions","n":1},{"turn":622,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":625,"what":"sell_zero_fill","n":1},{"turn":638,"what":"sell_zero_fill","n":1},{"turn":639,"what":"sell_zero_fill","n":1},{"turn":641,"what":"sell_zero_fill","n":1},{"turn":642,"what":"sell_zero_fill","n":1},{"turn":643,"what":"sell_zero_fill","n":2},{"turn":644,"what":"sell_zero_fill","n":2},{"turn":645,"what":"sell_zero_fill","n":2},{"turn":647,"what":"sell_zero_fill","n":2},{"turn":649,"what":"sell_zero_fill","n":2},{"turn":653,"what":"dead_actions","n":1},{"turn":655,"what":"dead_actions","n":1},{"turn":666,"what":"sell_zero_fill","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":687,"what":"dead_actions","n":1},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":3},{"turn":693,"what":"dead_actions","n":1},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"shed_discarded_units","n":3},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":705,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":2},{"turn":711,"what":"sell_zero_fill","n":3},{"turn":711,"what":"dead_actions","n":2},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":8},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":8},{"turn":715,"what":"sell_zero_fill","n":7},{"turn":716,"what":"sell_zero_fill","n":6},{"turn":717,"what":"sell_zero_fill","n":6}],[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":499,"what":"dead_actions","n":1},{"turn":501,"what":"dead_actions","n":1},{"turn":509,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":573,"what":"sell_zero_fill","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":609,"what":"dead_actions","n":1},{"turn":623,"what":"shed_discarded_units","n":2},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":669,"what":"sell_zero_fill","n":3},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":2},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":705,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":5},{"turn":716,"what":"sell_zero_fill","n":6},{"turn":717,"what":"sell_zero_fill","n":7}]]},{"episode":109033357,"trusted":true,"our_seat":1,"banks":[86664.0,81754.0],"replayed":[86664.0,81754.0],"observed_escapes":[0,0],"ledgers":[{"sell_dead_units":84185,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":134,"buy_zero_fill":0,"shed_discarded_units":14,"sold_units":1621,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":11,"harvest_dead":3,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":107,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":388,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":33,"dead_actions":40,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":1,"build_dead":0,"dig_dead":3,"fertilize_dead":1,"collect_dead":8,"care_dead":13,"tile_cap_units":27,"fertilizer_forgone":15,"care_unfed":103,"animal_shed_days":1,"idle_tile_days":64,"weed_tile_days":3,"hand_pass_turns":434,"farmer_pass_turns":69,"phantom_hand_actions":0,"sell_floor_units":90,"hire_premium":4129,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0},{"sell_dead_units":84183,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":106,"buy_zero_fill":0,"shed_discarded_units":6,"sold_units":1697,"feed_no_wheat":2,"feed_no_animal":0,"feed_redundant":0,"water_dead":13,"harvest_dead":4,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":31,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":380,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":25,"dead_actions":49,"move_dead":0,"locked_dead":0,"pickup_dead":2,"drop_dead":0,"place_dead":0,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":22,"fertilizer_forgone":15,"care_unfed":27,"animal_shed_days":1,"idle_tile_days":64,"weed_tile_days":2,"hand_pass_turns":356,"farmer_pass_turns":59,"phantom_hand_actions":0,"sell_floor_units":151,"hire_premium":3722,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0}],"events":[[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":169,"what":"sell_zero_fill","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":186,"what":"dead_actions","n":1},{"turn":189,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":1},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":258,"what":"sell_zero_fill","n":1},{"turn":258,"what":"dead_actions","n":1},{"turn":264,"what":"sell_zero_fill","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":280,"what":"sell_zero_fill","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":304,"what":"dead_actions","n":1},{"turn":312,"what":"sell_zero_fill","n":1},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":354,"what":"sell_zero_fill","n":1},{"turn":359,"what":"animal_unfed_days","n":3},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":367,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":371,"what":"dead_actions","n":1},{"turn":382,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":383,"what":"animal_unfed_days","n":2},{"turn":388,"what":"dead_actions","n":1},{"turn":389,"what":"sell_zero_fill","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":407,"what":"animal_unfed_days","n":4},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":431,"what":"animal_unfed_days","n":5},{"turn":442,"what":"sell_zero_fill","n":1},{"turn":455,"what":"animal_unfed_days","n":8},{"turn":455,"what":"dead_actions","n":1},{"turn":469,"what":"sell_zero_fill","n":1},{"turn":474,"what":"sell_zero_fill","n":2},{"turn":477,"what":"sell_zero_fill","n":1},{"turn":479,"what":"animal_unfed_days","n":6},{"turn":493,"what":"sell_zero_fill","n":1},{"turn":502,"what":"sell_zero_fill","n":1},{"turn":503,"what":"animal_unfed_days","n":8},{"turn":507,"what":"sell_zero_fill","n":1},{"turn":509,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":521,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"shed_discarded_units","n":5},{"turn":527,"what":"animal_unfed_days","n":6},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":2},{"turn":551,"what":"shed_discarded_units","n":1},{"turn":551,"what":"animal_unfed_days","n":8},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":558,"what":"sell_zero_fill","n":2},{"turn":561,"what":"sell_zero_fill","n":1},{"turn":562,"what":"sell_zero_fill","n":2},{"turn":565,"what":"sell_zero_fill","n":2},{"turn":567,"what":"sell_zero_fill","n":1},{"turn":568,"what":"sell_zero_fill","n":1},{"turn":569,"what":"sell_zero_fill","n":3},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":573,"what":"sell_zero_fill","n":2},{"turn":575,"what":"shed_discarded_units","n":6},{"turn":575,"what":"animal_unfed_days","n":5},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":593,"what":"dead_actions","n":1},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":8},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":602,"what":"sell_zero_fill","n":1},{"turn":609,"what":"dead_actions","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":615,"what":"sell_zero_fill","n":1},{"turn":617,"what":"sell_zero_fill","n":1},{"turn":623,"what":"shed_discarded_units","n":2},{"turn":623,"what":"animal_unfed_days","n":4},{"turn":640,"what":"sell_zero_fill","n":2},{"turn":646,"what":"sell_zero_fill","n":1},{"turn":647,"what":"animal_unfed_days","n":9},{"turn":669,"what":"sell_zero_fill","n":4},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"animal_unfed_days","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":2},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"animal_unfed_days","n":13},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":2},{"turn":708,"what":"dead_actions","n":1},{"turn":709,"what":"sell_zero_fill","n":2},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":717,"what":"sell_zero_fill","n":6}],[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":266,"what":"sell_zero_fill","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":401,"what":"sell_zero_fill","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":451,"what":"sell_zero_fill","n":1},{"turn":455,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":494,"what":"dead_actions","n":1},{"turn":499,"what":"feed_no_wheat","n":1},{"turn":499,"what":"dead_actions","n":3},{"turn":500,"what":"feed_no_wheat","n":1},{"turn":500,"what":"dead_actions","n":1},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"animal_unfed_days","n":2},{"turn":509,"what":"dead_actions","n":1},{"turn":512,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":595,"what":"sell_zero_fill","n":1},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":609,"what":"dead_actions","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":615,"what":"sell_zero_fill","n":1},{"turn":619,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":662,"what":"sell_zero_fill","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":2},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":1},{"turn":695,"what":"shed_discarded_units","n":6},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":698,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":717,"what":"sell_zero_fill","n":6}]]},{"episode":109033849,"trusted":true,"our_seat":1,"banks":[64579.0,65671.0],"replayed":[64579.0,65671.0],"observed_escapes":[0,0],"ledgers":[{"sell_dead_units":84280,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":137,"buy_zero_fill":0,"shed_discarded_units":0,"sold_units":1506,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":13,"harvest_dead":4,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":29,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":379,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":19,"dead_actions":46,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":2,"build_dead":0,"dig_dead":0,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":17,"fertilizer_forgone":15,"care_unfed":25,"animal_shed_days":1,"idle_tile_days":64,"weed_tile_days":2,"hand_pass_turns":363,"farmer_pass_turns":59,"phantom_hand_actions":0,"sell_floor_units":210,"hire_premium":3370,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0},{"sell_dead_units":84183,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":106,"buy_zero_fill":0,"shed_discarded_units":6,"sold_units":1697,"feed_no_wheat":2,"feed_no_animal":0,"feed_redundant":0,"water_dead":13,"harvest_dead":4,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":31,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":380,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":25,"dead_actions":49,"move_dead":0,"locked_dead":0,"pickup_dead":2,"drop_dead":0,"place_dead":0,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":22,"fertilizer_forgone":15,"care_unfed":27,"animal_shed_days":1,"idle_tile_days":64,"weed_tile_days":2,"hand_pass_turns":356,"farmer_pass_turns":59,"phantom_hand_actions":0,"sell_floor_units":208,"hire_premium":3722,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":0,"seeds_unplanted_cost_now":0.0}],"events":[[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":2},{"turn":167,"what":"dead_actions","n":1},{"turn":169,"what":"sell_zero_fill","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":213,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":313,"what":"sell_zero_fill","n":1},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":401,"what":"sell_zero_fill","n":1},{"turn":405,"what":"sell_zero_fill","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":451,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":469,"what":"sell_zero_fill","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":493,"what":"sell_zero_fill","n":1},{"turn":497,"what":"sell_zero_fill","n":1},{"turn":499,"what":"dead_actions","n":1},{"turn":501,"what":"sell_zero_fill","n":2},{"turn":501,"what":"dead_actions","n":1},{"turn":509,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":561,"what":"sell_zero_fill","n":1},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":569,"what":"sell_zero_fill","n":3},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":573,"what":"sell_zero_fill","n":2},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":595,"what":"sell_zero_fill","n":1},{"turn":597,"what":"sell_zero_fill","n":3},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":615,"what":"sell_zero_fill","n":2},{"turn":616,"what":"sell_zero_fill","n":1},{"turn":619,"what":"sell_zero_fill","n":2},{"turn":620,"what":"sell_zero_fill","n":1},{"turn":621,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":639,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":650,"what":"sell_zero_fill","n":1},{"turn":665,"what":"sell_zero_fill","n":1},{"turn":669,"what":"sell_zero_fill","n":4},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":2},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":681,"what":"sell_zero_fill","n":1},{"turn":685,"what":"sell_zero_fill","n":2},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":2},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":705,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":709,"what":"sell_zero_fill","n":3},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":6},{"turn":717,"what":"sell_zero_fill","n":7}],[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":266,"what":"sell_zero_fill","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":401,"what":"sell_zero_fill","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":451,"what":"sell_zero_fill","n":1},{"turn":455,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":494,"what":"dead_actions","n":1},{"turn":499,"what":"feed_no_wheat","n":1},{"turn":499,"what":"dead_actions","n":3},{"turn":500,"what":"feed_no_wheat","n":1},{"turn":500,"what":"dead_actions","n":1},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"animal_unfed_days","n":2},{"turn":509,"what":"dead_actions","n":1},{"turn":512,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":595,"what":"sell_zero_fill","n":1},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":609,"what":"dead_actions","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":615,"what":"sell_zero_fill","n":1},{"turn":619,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":662,"what":"sell_zero_fill","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":2},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":1},{"turn":695,"what":"shed_discarded_units","n":6},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":698,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":717,"what":"sell_zero_fill","n":6}]]},{"episode":109033976,"trusted":true,"our_seat":1,"banks":[99517.0,74409.0],"replayed":[99517.0,74409.0],"observed_escapes":[0,1],"ledgers":[{"sell_dead_units":84195,"refused_buy_product":0,"refused_buy_seed":0,"refused_buy_animal":0,"sell_zero_fill":121,"buy_zero_fill":0,"shed_discarded_units":7,"sold_units":1566,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":12,"harvest_dead":3,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"animal_unfed_days":69,"plants_dry":1,"dry_seed_cost":100.0,"plant_unwatered_days":387,"decay_units":18,"plants_decayed":18,"silent_loss_coins":100.0,"silent_loss_units":25,"dead_actions":40,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":1,"build_dead":0,"dig_dead":3,"fertilize_dead":0,"collect_dead":8,"care_dead":13,"tile_cap_units":29,"fertilizer_forgone":15,"care_unfed":65,"animal_shed_days":1,"idle_tile_days":64,"weed_tile_days":2,"hand_pass_turns":401,"farmer_pass_turns":65,"phantom_hand_actions":0,"sell_floor_units":68,"hire_premium":3953,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":1,"seeds_unplanted_cost_now":10.0},{"sell_dead_units":84424,"refused_buy_product":13,"refused_buy_seed":8,"refused_buy_animal":3,"sell_zero_fill":167,"buy_zero_fill":15,"shed_discarded_units":1,"sold_units":1504,"feed_no_wheat":0,"feed_no_animal":91,"feed_redundant":0,"water_dead":92,"harvest_dead":73,"plant_dead":10,"animals_escaped":1,"escape_capital":400.0,"escape_yield_units":0,"animal_unfed_days":23,"plants_dry":2,"dry_seed_cost":110.0,"plant_unwatered_days":326,"decay_units":11,"plants_decayed":11,"silent_loss_coins":510.0,"silent_loss_units":13,"dead_actions":536,"move_dead":0,"locked_dead":0,"pickup_dead":6,"drop_dead":0,"place_dead":15,"build_dead":0,"dig_dead":8,"fertilize_dead":19,"collect_dead":99,"care_dead":123,"tile_cap_units":22,"fertilizer_forgone":14,"care_unfed":18,"animal_shed_days":5,"idle_tile_days":259,"weed_tile_days":4,"hand_pass_turns":350,"farmer_pass_turns":59,"phantom_hand_actions":0,"sell_floor_units":105,"hire_premium":3721,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":2,"seeds_unplanted_cost_now":110.0}],"events":[[{"turn":1,"what":"sell_zero_fill","n":1},{"turn":23,"what":"animal_unfed_days","n":2},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":258,"what":"sell_zero_fill","n":1},{"turn":258,"what":"dead_actions","n":1},{"turn":264,"what":"sell_zero_fill","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":280,"what":"sell_zero_fill","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":304,"what":"dead_actions","n":1},{"turn":312,"what":"sell_zero_fill","n":1},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":354,"what":"sell_zero_fill","n":1},{"turn":359,"what":"animal_unfed_days","n":3},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":367,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":371,"what":"dead_actions","n":1},{"turn":382,"what":"dead_actions","n":1},{"turn":383,"what":"animal_unfed_days","n":2},{"turn":388,"what":"dead_actions","n":1},{"turn":389,"what":"sell_zero_fill","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":405,"what":"dead_actions","n":1},{"turn":407,"what":"animal_unfed_days","n":4},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":431,"what":"animal_unfed_days","n":2},{"turn":442,"what":"sell_zero_fill","n":1},{"turn":455,"what":"animal_unfed_days","n":4},{"turn":455,"what":"dead_actions","n":1},{"turn":474,"what":"sell_zero_fill","n":2},{"turn":477,"what":"sell_zero_fill","n":1},{"turn":479,"what":"animal_unfed_days","n":2},{"turn":493,"what":"sell_zero_fill","n":1},{"turn":503,"what":"animal_unfed_days","n":4},{"turn":509,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":515,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"animal_unfed_days","n":2},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":551,"what":"shed_discarded_units","n":1},{"turn":551,"what":"animal_unfed_days","n":4},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":558,"what":"sell_zero_fill","n":1},{"turn":561,"what":"sell_zero_fill","n":1},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":2},{"turn":567,"what":"sell_zero_fill","n":1},{"turn":568,"what":"sell_zero_fill","n":1},{"turn":569,"what":"sell_zero_fill","n":3},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":573,"what":"sell_zero_fill","n":2},{"turn":575,"what":"shed_discarded_units","n":5},{"turn":575,"what":"animal_unfed_days","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":599,"what":"animal_unfed_days","n":6},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":602,"what":"sell_zero_fill","n":1},{"turn":609,"what":"dead_actions","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":623,"what":"shed_discarded_units","n":1},{"turn":623,"what":"animal_unfed_days","n":3},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":646,"what":"sell_zero_fill","n":1},{"turn":647,"what":"animal_unfed_days","n":4},{"turn":669,"what":"sell_zero_fill","n":3},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":2},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"animal_unfed_days","n":13},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":709,"what":"sell_zero_fill","n":2},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":7},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":717,"what":"sell_zero_fill","n":6}],[{"turn":21,"what":"dead_actions","n":2},{"turn":22,"what":"dead_actions","n":2},{"turn":23,"what":"animal_unfed_days","n":2},{"turn":30,"what":"sell_zero_fill","n":1},{"turn":30,"what":"buy_zero_fill","n":1},{"turn":31,"what":"buy_zero_fill","n":1},{"turn":47,"what":"animals_escaped","n":1},{"turn":47,"what":"animal_unfed_days","n":2},{"turn":48,"what":"sell_zero_fill","n":1},{"turn":49,"what":"dead_actions","n":1},{"turn":49,"what":"feed_no_animal","n":1},{"turn":50,"what":"dead_actions","n":1},{"turn":51,"what":"dead_actions","n":1},{"turn":65,"what":"buy_zero_fill","n":1},{"turn":66,"what":"dead_actions","n":1},{"turn":67,"what":"dead_actions","n":1},{"turn":69,"what":"dead_actions","n":2},{"turn":70,"what":"dead_actions","n":1},{"turn":78,"what":"dead_actions","n":1},{"turn":79,"what":"dead_actions","n":1},{"turn":83,"what":"dead_actions","n":2},{"turn":83,"what":"feed_no_animal","n":1},{"turn":84,"what":"dead_actions","n":2},{"turn":85,"what":"dead_actions","n":1},{"turn":89,"what":"dead_actions","n":1},{"turn":90,"what":"sell_zero_fill","n":1},{"turn":90,"what":"dead_actions","n":2},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":1},{"turn":95,"what":"dead_actions","n":1},{"turn":97,"what":"dead_actions","n":1},{"turn":97,"what":"feed_no_animal","n":1},{"turn":98,"what":"dead_actions","n":1},{"turn":99,"what":"dead_actions","n":1},{"turn":106,"what":"dead_actions","n":1},{"turn":107,"what":"dead_actions","n":2},{"turn":111,"what":"sell_zero_fill","n":1},{"turn":111,"what":"dead_actions","n":3},{"turn":112,"what":"dead_actions","n":1},{"turn":113,"what":"dead_actions","n":1},{"turn":113,"what":"feed_no_animal","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":120,"what":"sell_zero_fill","n":1},{"turn":122,"what":"dead_actions","n":2},{"turn":122,"what":"feed_no_animal","n":1},{"turn":123,"what":"dead_actions","n":2},{"turn":126,"what":"dead_actions","n":1},{"turn":126,"what":"feed_no_animal","n":1},{"turn":127,"what":"dead_actions","n":1},{"turn":128,"what":"dead_actions","n":2},{"turn":128,"what":"feed_no_animal","n":1},{"turn":129,"what":"dead_actions","n":1},{"turn":130,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":144,"what":"sell_zero_fill","n":1},{"turn":146,"what":"dead_actions","n":1},{"turn":146,"what":"feed_no_animal","n":1},{"turn":147,"what":"dead_actions","n":2},{"turn":149,"what":"dead_actions","n":2},{"turn":149,"what":"feed_no_animal","n":2},{"turn":150,"what":"dead_actions","n":2},{"turn":151,"what":"dead_actions","n":2},{"turn":154,"what":"sell_zero_fill","n":1},{"turn":154,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":169,"what":"sell_zero_fill","n":1},{"turn":169,"what":"buy_zero_fill","n":1},{"turn":170,"what":"dead_actions","n":1},{"turn":170,"what":"feed_no_animal","n":1},{"turn":171,"what":"dead_actions","n":1},{"turn":172,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":3},{"turn":173,"what":"feed_no_animal","n":1},{"turn":174,"what":"dead_actions","n":2},{"turn":176,"what":"dead_actions","n":1},{"turn":176,"what":"feed_no_animal","n":1},{"turn":177,"what":"sell_zero_fill","n":1},{"turn":177,"what":"dead_actions","n":1},{"turn":178,"what":"buy_zero_fill","n":1},{"turn":178,"what":"dead_actions","n":1},{"turn":180,"what":"buy_zero_fill","n":1},{"turn":180,"what":"dead_actions","n":1},{"turn":181,"what":"dead_actions","n":1},{"turn":183,"what":"buy_zero_fill","n":1},{"turn":184,"what":"buy_zero_fill","n":1},{"turn":184,"what":"dead_actions","n":1},{"turn":185,"what":"dead_actions","n":2},{"turn":186,"what":"dead_actions","n":2},{"turn":187,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":194,"what":"dead_actions","n":2},{"turn":194,"what":"feed_no_animal","n":1},{"turn":195,"what":"dead_actions","n":2},{"turn":196,"what":"buy_zero_fill","n":1},{"turn":196,"what":"dead_actions","n":1},{"turn":197,"what":"dead_actions","n":1},{"turn":198,"what":"dead_actions","n":1},{"turn":201,"what":"buy_zero_fill","n":1},{"turn":201,"what":"dead_actions","n":1},{"turn":201,"what":"feed_no_animal","n":1},{"turn":202,"what":"buy_zero_fill","n":1},{"turn":202,"what":"dead_actions","n":1},{"turn":203,"what":"buy_zero_fill","n":1},{"turn":203,"what":"dead_actions","n":1},{"turn":204,"what":"buy_zero_fill","n":1},{"turn":205,"what":"dead_actions","n":1},{"turn":206,"what":"buy_zero_fill","n":1},{"turn":206,"what":"dead_actions","n":1},{"turn":207,"what":"dead_actions","n":1},{"turn":208,"what":"dead_actions","n":1},{"turn":209,"what":"dead_actions","n":1},{"turn":210,"what":"dead_actions","n":2},{"turn":211,"what":"dead_actions","n":3},{"turn":212,"what":"dead_actions","n":2},{"turn":213,"what":"dead_actions","n":1},{"turn":217,"what":"buy_zero_fill","n":1},{"turn":217,"what":"dead_actions","n":1},{"turn":217,"what":"feed_no_animal","n":1},{"turn":218,"what":"dead_actions","n":1},{"turn":219,"what":"dead_actions","n":1},{"turn":220,"what":"sell_zero_fill","n":1},{"turn":220,"what":"dead_actions","n":2},{"turn":222,"what":"dead_actions","n":1},{"turn":223,"what":"dead_actions","n":1},{"turn":223,"what":"feed_no_animal","n":1},{"turn":224,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":2},{"turn":228,"what":"dead_actions","n":1},{"turn":229,"what":"dead_actions","n":2},{"turn":230,"what":"dead_actions","n":3},{"turn":231,"what":"dead_actions","n":1},{"turn":231,"what":"feed_no_animal","n":1},{"turn":233,"what":"sell_zero_fill","n":1},{"turn":233,"what":"dead_actions","n":1},{"turn":234,"what":"dead_actions","n":1},{"turn":235,"what":"dead_actions","n":1},{"turn":236,"what":"dead_actions","n":1},{"turn":236,"what":"feed_no_animal","n":1},{"turn":237,"what":"dead_actions","n":2},{"turn":238,"what":"dead_actions","n":1},{"turn":239,"what":"dead_actions","n":1},{"turn":240,"what":"sell_zero_fill","n":1},{"turn":245,"what":"dead_actions","n":1},{"turn":245,"what":"feed_no_animal","n":1},{"turn":246,"what":"dead_actions","n":1},{"turn":247,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":253,"what":"dead_actions","n":2},{"turn":253,"what":"feed_no_animal","n":1},{"turn":254,"what":"dead_actions","n":1},{"turn":255,"what":"dead_actions","n":1},{"turn":256,"what":"dead_actions","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":261,"what":"dead_actions","n":2},{"turn":261,"what":"feed_no_animal","n":2},{"turn":262,"what":"dead_actions","n":2},{"turn":263,"what":"dead_actions","n":2},{"turn":264,"what":"sell_zero_fill","n":1},{"turn":265,"what":"dead_actions","n":1},{"turn":266,"what":"sell_zero_fill","n":1},{"turn":267,"what":"dead_actions","n":2},{"turn":267,"what":"feed_no_animal","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":269,"what":"dead_actions","n":1},{"turn":270,"what":"sell_zero_fill","n":1},{"turn":271,"what":"dead_actions","n":1},{"turn":271,"what":"feed_no_animal","n":1},{"turn":272,"what":"dead_actions","n":3},{"turn":272,"what":"feed_no_animal","n":1},{"turn":273,"what":"dead_actions","n":2},{"turn":274,"what":"dead_actions","n":3},{"turn":276,"what":"dead_actions","n":1},{"turn":276,"what":"feed_no_animal","n":1},{"turn":277,"what":"dead_actions","n":1},{"turn":278,"what":"dead_actions","n":2},{"turn":280,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":282,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":287,"what":"dead_actions","n":1},{"turn":290,"what":"dead_actions","n":1},{"turn":290,"what":"feed_no_animal","n":1},{"turn":291,"what":"dead_actions","n":1},{"turn":292,"what":"dead_actions","n":1},{"turn":297,"what":"dead_actions","n":1},{"turn":298,"what":"dead_actions","n":1},{"turn":298,"what":"feed_no_animal","n":1},{"turn":299,"what":"dead_actions","n":1},{"turn":300,"what":"dead_actions","n":1},{"turn":302,"what":"dead_actions","n":1},{"turn":302,"what":"feed_no_animal","n":1},{"turn":303,"what":"dead_actions","n":2},{"turn":303,"what":"feed_no_animal","n":1},{"turn":304,"what":"dead_actions","n":2},{"turn":305,"what":"dead_actions","n":1},{"turn":309,"what":"sell_zero_fill","n":1},{"turn":313,"what":"sell_zero_fill","n":2},{"turn":315,"what":"dead_actions","n":1},{"turn":316,"what":"dead_actions","n":1},{"turn":317,"what":"dead_actions","n":1},{"turn":318,"what":"sell_zero_fill","n":1},{"turn":320,"what":"sell_zero_fill","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":322,"what":"dead_actions","n":1},{"turn":322,"what":"feed_no_animal","n":1},{"turn":323,"what":"dead_actions","n":2},{"turn":324,"what":"dead_actions","n":1},{"turn":327,"what":"dead_actions","n":3},{"turn":327,"what":"feed_no_animal","n":2},{"turn":328,"what":"dead_actions","n":3},{"turn":329,"what":"dead_actions","n":3},{"turn":330,"what":"dead_actions","n":2},{"turn":331,"what":"dead_actions","n":1},{"turn":332,"what":"dead_actions","n":1},{"turn":332,"what":"feed_no_animal","n":1},{"turn":333,"what":"dead_actions","n":2},{"turn":334,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":2},{"turn":337,"what":"dead_actions","n":1},{"turn":337,"what":"feed_no_animal","n":1},{"turn":338,"what":"dead_actions","n":1},{"turn":339,"what":"sell_zero_fill","n":1},{"turn":339,"what":"dead_actions","n":1},{"turn":340,"what":"dead_actions","n":1},{"turn":341,"what":"dead_actions","n":1},{"turn":341,"what":"feed_no_animal","n":1},{"turn":342,"what":"dead_actions","n":2},{"turn":343,"what":"dead_actions","n":1},{"turn":344,"what":"dead_actions","n":1},{"turn":349,"what":"dead_actions","n":1},{"turn":349,"what":"feed_no_animal","n":1},{"turn":350,"what":"dead_actions","n":1},{"turn":351,"what":"dead_actions","n":1},{"turn":353,"what":"dead_actions","n":1},{"turn":353,"what":"feed_no_animal","n":1},{"turn":354,"what":"dead_actions","n":1},{"turn":355,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":361,"what":"dead_actions","n":1},{"turn":363,"what":"dead_actions","n":1},{"turn":364,"what":"dead_actions","n":2},{"turn":364,"what":"feed_no_animal","n":2},{"turn":365,"what":"dead_actions","n":2},{"turn":366,"what":"dead_actions","n":3},{"turn":367,"what":"dead_actions","n":1},{"turn":368,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":3},{"turn":369,"what":"feed_no_animal","n":2},{"turn":370,"what":"dead_actions","n":2},{"turn":371,"what":"dead_actions","n":3},{"turn":372,"what":"dead_actions","n":1},{"turn":373,"what":"dead_actions","n":1},{"turn":374,"what":"sell_zero_fill","n":1},{"turn":374,"what":"dead_actions","n":1},{"turn":375,"what":"dead_actions","n":1},{"turn":376,"what":"dead_actions","n":2},{"turn":378,"what":"dead_actions","n":3},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":383,"what":"dead_actions","n":1},{"turn":384,"what":"dead_actions","n":1},{"turn":385,"what":"dead_actions","n":1},{"turn":387,"what":"dead_actions","n":1},{"turn":387,"what":"feed_no_animal","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":389,"what":"dead_actions","n":3},{"turn":389,"what":"feed_no_animal","n":2},{"turn":390,"what":"dead_actions","n":2},{"turn":391,"what":"dead_actions","n":3},{"turn":392,"what":"dead_actions","n":1},{"turn":393,"what":"sell_zero_fill","n":1},{"turn":396,"what":"dead_actions","n":1},{"turn":397,"what":"sell_zero_fill","n":1},{"turn":397,"what":"dead_actions","n":1},{"turn":399,"what":"dead_actions","n":1},{"turn":400,"what":"dead_actions","n":1},{"turn":401,"what":"sell_zero_fill","n":1},{"turn":402,"what":"dead_actions","n":1},{"turn":402,"what":"feed_no_animal","n":1},{"turn":403,"what":"dead_actions","n":2},{"turn":404,"what":"dead_actions","n":2},{"turn":406,"what":"dead_actions","n":1},{"turn":407,"what":"dead_actions","n":1},{"turn":408,"what":"sell_zero_fill","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":412,"what":"dead_actions","n":1},{"turn":412,"what":"feed_no_animal","n":1},{"turn":413,"what":"dead_actions","n":1},{"turn":414,"what":"dead_actions","n":1},{"turn":415,"what":"dead_actions","n":1},{"turn":416,"what":"dead_actions","n":2},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":418,"what":"dead_actions","n":1},{"turn":419,"what":"dead_actions","n":2},{"turn":420,"what":"dead_actions","n":1},{"turn":421,"what":"dead_actions","n":3},{"turn":422,"what":"dead_actions","n":2},{"turn":423,"what":"dead_actions","n":1},{"turn":423,"what":"feed_no_animal","n":1},{"turn":424,"what":"dead_actions","n":3},{"turn":424,"what":"feed_no_animal","n":1},{"turn":425,"what":"dead_actions","n":3},{"turn":426,"what":"dead_actions","n":2},{"turn":427,"what":"sell_zero_fill","n":1},{"turn":427,"what":"dead_actions","n":1},{"turn":437,"what":"dead_actions","n":1},{"turn":437,"what":"feed_no_animal","n":1},{"turn":438,"what":"dead_actions","n":2},{"turn":439,"what":"dead_actions","n":1},{"turn":441,"what":"dead_actions","n":1},{"turn":441,"what":"feed_no_animal","n":1},{"turn":442,"what":"dead_actions","n":1},{"turn":443,"what":"dead_actions","n":2},{"turn":443,"what":"feed_no_animal","n":1},{"turn":444,"what":"dead_actions","n":3},{"turn":444,"what":"feed_no_animal","n":1},{"turn":445,"what":"dead_actions","n":2},{"turn":446,"what":"dead_actions","n":3},{"turn":448,"what":"sell_zero_fill","n":1},{"turn":449,"what":"sell_zero_fill","n":1},{"turn":450,"what":"dead_actions","n":1},{"turn":451,"what":"sell_zero_fill","n":1},{"turn":451,"what":"dead_actions","n":1},{"turn":453,"what":"dead_actions","n":2},{"turn":455,"what":"sell_zero_fill","n":1},{"turn":455,"what":"dead_actions","n":3},{"turn":457,"what":"dead_actions","n":1},{"turn":458,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":2},{"turn":459,"what":"feed_no_animal","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":461,"what":"dead_actions","n":1},{"turn":462,"what":"dead_actions","n":2},{"turn":462,"what":"feed_no_animal","n":2},{"turn":463,"what":"sell_zero_fill","n":1},{"turn":463,"what":"dead_actions","n":2},{"turn":464,"what":"dead_actions","n":2},{"turn":465,"what":"sell_zero_fill","n":1},{"turn":465,"what":"dead_actions","n":3},{"turn":465,"what":"feed_no_animal","n":1},{"turn":466,"what":"dead_actions","n":1},{"turn":467,"what":"dead_actions","n":1},{"turn":469,"what":"sell_zero_fill","n":1},{"turn":469,"what":"dead_actions","n":1},{"turn":472,"what":"dead_actions","n":1},{"turn":473,"what":"sell_zero_fill","n":1},{"turn":473,"what":"dead_actions","n":1},{"turn":474,"what":"dead_actions","n":1},{"turn":475,"what":"dead_actions","n":2},{"turn":477,"what":"sell_zero_fill","n":1},{"turn":477,"what":"dead_actions","n":1},{"turn":479,"what":"dead_actions","n":1},{"turn":482,"what":"sell_zero_fill","n":1},{"turn":482,"what":"dead_actions","n":3},{"turn":482,"what":"feed_no_animal","n":1},{"turn":483,"what":"dead_actions","n":1},{"turn":484,"what":"dead_actions","n":4},{"turn":484,"what":"feed_no_animal","n":1},{"turn":485,"what":"dead_actions","n":4},{"turn":485,"what":"feed_no_animal","n":1},{"turn":486,"what":"dead_actions","n":5},{"turn":487,"what":"dead_actions","n":2},{"turn":489,"what":"sell_zero_fill","n":1},{"turn":490,"what":"dead_actions","n":1},{"turn":492,"what":"dead_actions","n":3},{"turn":492,"what":"feed_no_animal","n":1},{"turn":493,"what":"sell_zero_fill","n":1},{"turn":493,"what":"dead_actions","n":1},{"turn":494,"what":"dead_actions","n":2},{"turn":496,"what":"dead_actions","n":1},{"turn":497,"what":"sell_zero_fill","n":1},{"turn":499,"what":"dead_actions","n":1},{"turn":501,"what":"sell_zero_fill","n":1},{"turn":501,"what":"dead_actions","n":2},{"turn":503,"what":"dead_actions","n":1},{"turn":505,"what":"sell_zero_fill","n":1},{"turn":505,"what":"dead_actions","n":1},{"turn":506,"what":"dead_actions","n":1},{"turn":506,"what":"feed_no_animal","n":1},{"turn":507,"what":"dead_actions","n":1},{"turn":508,"what":"dead_actions","n":1},{"turn":509,"what":"sell_zero_fill","n":1},{"turn":509,"what":"dead_actions","n":1},{"turn":510,"what":"dead_actions","n":1},{"turn":511,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":513,"what":"dead_actions","n":1},{"turn":514,"what":"dead_actions","n":3},{"turn":514,"what":"feed_no_animal","n":1},{"turn":515,"what":"dead_actions","n":1},{"turn":516,"what":"dead_actions","n":1},{"turn":517,"what":"dead_actions","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":518,"what":"dead_actions","n":1},{"turn":519,"what":"dead_actions","n":1},{"turn":519,"what":"feed_no_animal","n":1},{"turn":520,"what":"dead_actions","n":2},{"turn":520,"what":"feed_no_animal","n":1},{"turn":521,"what":"dead_actions","n":3},{"turn":522,"what":"dead_actions","n":3},{"turn":524,"what":"dead_actions","n":1},{"turn":525,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":526,"what":"dead_actions","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":527,"what":"dead_actions","n":1},{"turn":533,"what":"dead_actions","n":2},{"turn":533,"what":"feed_no_animal","n":2},{"turn":534,"what":"dead_actions","n":2},{"turn":535,"what":"dead_actions","n":4},{"turn":536,"what":"dead_actions","n":2},{"turn":537,"what":"dead_actions","n":1},{"turn":539,"what":"dead_actions","n":1},{"turn":539,"what":"feed_no_animal","n":1},{"turn":540,"what":"dead_actions","n":1},{"turn":541,"what":"dead_actions","n":1},{"turn":543,"what":"dead_actions","n":1},{"turn":544,"what":"dead_actions","n":1},{"turn":545,"what":"dead_actions","n":2},{"turn":548,"what":"dead_actions","n":1},{"turn":548,"what":"feed_no_animal","n":1},{"turn":549,"what":"sell_zero_fill","n":2},{"turn":549,"what":"dead_actions","n":1},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":550,"what":"dead_actions","n":1},{"turn":551,"what":"dead_actions","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":554,"what":"dead_actions","n":1},{"turn":554,"what":"feed_no_animal","n":1},{"turn":555,"what":"dead_actions","n":1},{"turn":556,"what":"dead_actions","n":1},{"turn":558,"what":"sell_zero_fill","n":1},{"turn":558,"what":"dead_actions","n":1},{"turn":558,"what":"feed_no_animal","n":1},{"turn":559,"what":"dead_actions","n":1},{"turn":560,"what":"dead_actions","n":2},{"turn":560,"what":"feed_no_animal","n":1},{"turn":561,"what":"sell_zero_fill","n":1},{"turn":561,"what":"dead_actions","n":1},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":562,"what":"dead_actions","n":1},{"turn":563,"what":"dead_actions","n":1},{"turn":565,"what":"sell_zero_fill","n":2},{"turn":566,"what":"dead_actions","n":1},{"turn":568,"what":"dead_actions","n":1},{"turn":569,"what":"sell_zero_fill","n":2},{"turn":569,"what":"dead_actions","n":1},{"turn":569,"what":"feed_no_animal","n":1},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":570,"what":"dead_actions","n":1},{"turn":571,"what":"dead_actions","n":3},{"turn":573,"what":"sell_zero_fill","n":1},{"turn":573,"what":"dead_actions","n":1},{"turn":575,"what":"dead_actions","n":2},{"turn":578,"what":"dead_actions","n":2},{"turn":578,"what":"feed_no_animal","n":1},{"turn":579,"what":"dead_actions","n":1},{"turn":580,"what":"dead_actions","n":1},{"turn":581,"what":"dead_actions","n":3},{"turn":581,"what":"feed_no_animal","n":1},{"turn":582,"what":"dead_actions","n":1},{"turn":583,"what":"dead_actions","n":1},{"turn":584,"what":"dead_actions","n":2},{"turn":584,"what":"feed_no_animal","n":1},{"turn":585,"what":"dead_actions","n":1},{"turn":586,"what":"dead_actions","n":1},{"turn":587,"what":"dead_actions","n":1},{"turn":588,"what":"dead_actions","n":1},{"turn":588,"what":"feed_no_animal","n":1},{"turn":589,"what":"dead_actions","n":2},{"turn":590,"what":"dead_actions","n":2},{"turn":591,"what":"dead_actions","n":2},{"turn":593,"what":"dead_actions","n":3},{"turn":595,"what":"sell_zero_fill","n":1},{"turn":595,"what":"dead_actions","n":1},{"turn":597,"what":"sell_zero_fill","n":3},{"turn":597,"what":"dead_actions","n":2},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":1},{"turn":601,"what":"sell_zero_fill","n":1},{"turn":602,"what":"sell_zero_fill","n":1},{"turn":603,"what":"dead_actions","n":1},{"turn":603,"what":"feed_no_animal","n":1},{"turn":604,"what":"dead_actions","n":1},{"turn":605,"what":"dead_actions","n":1},{"turn":606,"what":"dead_actions","n":1},{"turn":608,"what":"dead_actions","n":1},{"turn":608,"what":"feed_no_animal","n":1},{"turn":609,"what":"sell_zero_fill","n":1},{"turn":609,"what":"dead_actions","n":2},{"turn":610,"what":"dead_actions","n":1},{"turn":612,"what":"dead_actions","n":1},{"turn":612,"what":"feed_no_animal","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":613,"what":"dead_actions","n":2},{"turn":614,"what":"dead_actions","n":1},{"turn":615,"what":"sell_zero_fill","n":2},{"turn":616,"what":"dead_actions","n":2},{"turn":616,"what":"feed_no_animal","n":1},{"turn":617,"what":"sell_zero_fill","n":1},{"turn":617,"what":"dead_actions","n":1},{"turn":618,"what":"dead_actions","n":1},{"turn":619,"what":"sell_zero_fill","n":1},{"turn":620,"what":"dead_actions","n":1},{"turn":621,"what":"sell_zero_fill","n":1},{"turn":622,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":626,"what":"dead_actions","n":1},{"turn":626,"what":"feed_no_animal","n":1},{"turn":627,"what":"dead_actions","n":1},{"turn":628,"what":"dead_actions","n":1},{"turn":629,"what":"sell_zero_fill","n":1},{"turn":629,"what":"dead_actions","n":1},{"turn":630,"what":"dead_actions","n":1},{"turn":630,"what":"feed_no_animal","n":1},{"turn":631,"what":"dead_actions","n":1},{"turn":632,"what":"dead_actions","n":1},{"turn":633,"what":"dead_actions","n":1},{"turn":634,"what":"dead_actions","n":1},{"turn":634,"what":"feed_no_animal","n":1},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":635,"what":"dead_actions","n":2},{"turn":635,"what":"feed_no_animal","n":1},{"turn":636,"what":"dead_actions","n":2},{"turn":637,"what":"dead_actions","n":1},{"turn":639,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":647,"what":"sell_zero_fill","n":1},{"turn":649,"what":"dead_actions","n":1},{"turn":649,"what":"feed_no_animal","n":1},{"turn":650,"what":"dead_actions","n":1},{"turn":651,"what":"dead_actions","n":1},{"turn":652,"what":"dead_actions","n":1},{"turn":652,"what":"feed_no_animal","n":1},{"turn":653,"what":"dead_actions","n":1},{"turn":654,"what":"dead_actions","n":1},{"turn":655,"what":"dead_actions","n":1},{"turn":658,"what":"dead_actions","n":1},{"turn":658,"what":"feed_no_animal","n":1},{"turn":659,"what":"dead_actions","n":1},{"turn":660,"what":"dead_actions","n":1},{"turn":662,"what":"sell_zero_fill","n":1},{"turn":665,"what":"sell_zero_fill","n":1},{"turn":666,"what":"dead_actions","n":1},{"turn":666,"what":"feed_no_animal","n":1},{"turn":667,"what":"dead_actions","n":1},{"turn":668,"what":"dead_actions","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":2},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":2},{"turn":675,"what":"dead_actions","n":2},{"turn":676,"what":"dead_actions","n":2},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":677,"what":"dead_actions","n":1},{"turn":679,"what":"dead_actions","n":1},{"turn":679,"what":"feed_no_animal","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":2},{"turn":682,"what":"dead_actions","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":686,"what":"dead_actions","n":1},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":687,"what":"dead_actions","n":1},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":1},{"turn":695,"what":"shed_discarded_units","n":1},{"turn":695,"what":"animal_unfed_days","n":5},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":698,"what":"sell_zero_fill","n":1},{"turn":699,"what":"dead_actions","n":1},{"turn":705,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":716,"what":"dead_actions","n":1},{"turn":717,"what":"sell_zero_fill","n":6},{"turn":717,"what":"dead_actions","n":1}]]}]''')
COW = json.loads(r'''{"base":{"0":{"farmer":["PASS"],"hands":[],"market":[["BUY_ANIMAL","COW",1],["BUY_PRODUCT","WHEAT",3]]},"1":{"farmer":["PICKUP","COW",1],"hands":[],"market":[]},"2":{"farmer":["BUILD_PASTURE"],"hands":[],"market":[]},"3":{"farmer":["PLACE","COW"],"hands":[],"market":[]},"5":{"farmer":["PICKUP","WHEAT",1],"hands":[],"market":[]},"6":{"farmer":["FEED"],"hands":[],"market":[]},"30":{"farmer":["FEED"],"hands":[],"market":[]},"54":{"farmer":["FEED"],"hands":[],"market":[]}},"fix":{"0":{"farmer":["PASS"],"hands":[],"market":[["BUY_ANIMAL","COW",1],["BUY_PRODUCT","WHEAT",3]]},"1":{"farmer":["PICKUP","COW",1],"hands":[],"market":[]},"2":{"farmer":["BUILD_PASTURE"],"hands":[],"market":[]},"3":{"farmer":["PLACE","COW"],"hands":[],"market":[]},"5":{"farmer":["PICKUP","WHEAT",1],"hands":[],"market":[]},"6":{"farmer":["FEED"],"hands":[],"market":[]},"29":{"farmer":["PICKUP","WHEAT",1],"hands":[],"market":[]},"30":{"farmer":["FEED"],"hands":[],"market":[]},"53":{"farmer":["PICKUP","WHEAT",1],"hands":[],"market":[]},"54":{"farmer":["FEED"],"hands":[],"market":[]}},"base_events":[[30,"feed_no_wheat",1],[47,"animal_unfed_days",1],[54,"feed_no_wheat",1],[71,"animal_unfed_days",1],[71,"animals_escaped",1],[71,"escape_capital",400.0]],"fix_events":[]}''')
CASE = json.loads(r'''{"seed":17,"rival":"yhay81-shop-router-0913","rows":[{"bank":123809.0,"rival_bank":100760.0,"events":[{"turn":1,"what":"sell_zero_fill","n":1},{"turn":23,"what":"animal_unfed_days","n":2},{"turn":31,"what":"buy_zero_fill","n":1},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":2},{"turn":190,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":196,"what":"sell_zero_fill","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":2},{"turn":226,"what":"sell_zero_fill","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":321,"what":"sell_zero_fill","n":1},{"turn":323,"what":"sell_zero_fill","n":1},{"turn":333,"what":"sell_zero_fill","n":1},{"turn":338,"what":"sell_zero_fill","n":1},{"turn":339,"what":"sell_zero_fill","n":1},{"turn":340,"what":"sell_zero_fill","n":1},{"turn":359,"what":"dead_actions","n":1},{"turn":360,"what":"sell_zero_fill","n":1},{"turn":362,"what":"sell_zero_fill","n":1},{"turn":367,"what":"sell_zero_fill","n":1},{"turn":373,"what":"sell_zero_fill","n":1},{"turn":383,"what":"dead_actions","n":1},{"turn":384,"what":"sell_zero_fill","n":1},{"turn":393,"what":"sell_zero_fill","n":1},{"turn":398,"what":"sell_zero_fill","n":1},{"turn":402,"what":"sell_zero_fill","n":1},{"turn":403,"what":"sell_zero_fill","n":2},{"turn":404,"what":"sell_zero_fill","n":3},{"turn":404,"what":"dead_actions","n":1},{"turn":405,"what":"sell_zero_fill","n":3},{"turn":406,"what":"sell_zero_fill","n":3},{"turn":407,"what":"sell_zero_fill","n":3},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":430,"what":"sell_zero_fill","n":8},{"turn":431,"what":"sell_zero_fill","n":1},{"turn":447,"what":"sell_zero_fill","n":2},{"turn":453,"what":"sell_zero_fill","n":1},{"turn":453,"what":"dead_actions","n":1},{"turn":454,"what":"sell_zero_fill","n":1},{"turn":455,"what":"dead_actions","n":1},{"turn":457,"what":"sell_zero_fill","n":2},{"turn":470,"what":"sell_zero_fill","n":1},{"turn":471,"what":"sell_zero_fill","n":2},{"turn":473,"what":"sell_zero_fill","n":1},{"turn":474,"what":"sell_zero_fill","n":1},{"turn":478,"what":"sell_zero_fill","n":1},{"turn":478,"what":"dead_actions","n":1},{"turn":479,"what":"dead_actions","n":1},{"turn":481,"what":"sell_zero_fill","n":1},{"turn":494,"what":"sell_zero_fill","n":1},{"turn":498,"what":"dead_actions","n":1},{"turn":501,"what":"sell_zero_fill","n":2},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"sell_zero_fill","n":1},{"turn":505,"what":"sell_zero_fill","n":1},{"turn":509,"what":"dead_actions","n":1},{"turn":510,"what":"dead_actions","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":518,"what":"dead_actions","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":2},{"turn":529,"what":"sell_zero_fill","n":1},{"turn":535,"what":"sell_zero_fill","n":1},{"turn":537,"what":"sell_zero_fill","n":1},{"turn":545,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":2},{"turn":551,"what":"sell_zero_fill","n":2},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":553,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":569,"what":"sell_zero_fill","n":1},{"turn":570,"what":"sell_zero_fill","n":2},{"turn":571,"what":"sell_zero_fill","n":3},{"turn":573,"what":"sell_zero_fill","n":3},{"turn":574,"what":"sell_zero_fill","n":2},{"turn":575,"what":"sell_zero_fill","n":2},{"turn":575,"what":"dead_actions","n":2},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":577,"what":"sell_zero_fill","n":1},{"turn":583,"what":"dead_actions","n":1},{"turn":585,"what":"dead_actions","n":2},{"turn":588,"what":"dead_actions","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"shed_discarded_units","n":1},{"turn":599,"what":"animal_unfed_days","n":8},{"turn":599,"what":"sell_zero_fill","n":3},{"turn":599,"what":"dead_actions","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":601,"what":"sell_zero_fill","n":1},{"turn":612,"what":"sell_zero_fill","n":1},{"turn":617,"what":"sell_zero_fill","n":1},{"turn":621,"what":"sell_zero_fill","n":1},{"turn":621,"what":"dead_actions","n":1},{"turn":622,"what":"sell_zero_fill","n":1},{"turn":623,"what":"shed_discarded_units","n":2},{"turn":623,"what":"animal_unfed_days","n":4},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":625,"what":"sell_zero_fill","n":1},{"turn":638,"what":"sell_zero_fill","n":1},{"turn":639,"what":"sell_zero_fill","n":1},{"turn":641,"what":"sell_zero_fill","n":1},{"turn":642,"what":"sell_zero_fill","n":4},{"turn":643,"what":"sell_zero_fill","n":2},{"turn":644,"what":"sell_zero_fill","n":2},{"turn":645,"what":"sell_zero_fill","n":4},{"turn":647,"what":"animal_unfed_days","n":3},{"turn":647,"what":"sell_zero_fill","n":2},{"turn":649,"what":"sell_zero_fill","n":2},{"turn":653,"what":"dead_actions","n":1},{"turn":655,"what":"dead_actions","n":1},{"turn":669,"what":"sell_zero_fill","n":4},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":671,"what":"animal_unfed_days","n":3},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":687,"what":"dead_actions","n":1},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":2},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":3},{"turn":693,"what":"dead_actions","n":1},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":695,"what":"animal_unfed_days","n":11},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":2},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":709,"what":"sell_zero_fill","n":1},{"turn":710,"what":"sell_zero_fill","n":2},{"turn":711,"what":"sell_zero_fill","n":3},{"turn":711,"what":"dead_actions","n":2},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":712,"what":"dead_actions","n":1},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":7},{"turn":716,"what":"sell_zero_fill","n":6},{"turn":717,"what":"sell_zero_fill","n":6}],"ledger":{"sell_dead_units":116671,"refused_buy_product":1,"refused_buy_seed":0,"refused_buy_animal":0,"refused_hire":0,"refused_buy_land":0,"hire_paid":5892,"sell_zero_fill":207,"buy_zero_fill":1,"sell_revenue":149669.0,"total_spend":28860.0,"shed_discarded_units":3,"sold_units":1560,"feed_no_wheat":0,"feed_no_animal":0,"feed_redundant":0,"water_dead":6,"harvest_dead":18,"plant_dead":0,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"escape_first_day":-1,"animal_unfed_days":44,"plants_dry":1,"dry_seed_cost":100.0,"dry_yield_units":0,"dry_first_day":21,"plant_unwatered_days":379,"decay_units":18,"plants_decayed":18,"silent_loss_coins":100.0,"silent_loss_units":21,"dead_actions":44,"move_dead":0,"locked_dead":0,"pickup_dead":0,"drop_dead":0,"place_dead":2,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":11,"care_dead":6,"tile_cap_units":10,"fertilizer_forgone":13,"care_unfed":40,"animal_shed_days":1,"idle_tile_days":64,"weed_tile_days":1,"hand_pass_turns":521,"farmer_pass_turns":69,"phantom_hand_actions":0,"sell_floor_units":81,"hire_premium":5610,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":1,"seeds_unplanted_cost_now":10.0},"agent":"V42, Ahmed Berat \u00d6zer, verbatim"},{"bank":107957.0,"rival_bank":139950.0,"events":[{"turn":23,"what":"animal_unfed_days","n":2},{"turn":26,"what":"dead_actions","n":2},{"turn":27,"what":"feed_no_wheat","n":1},{"turn":27,"what":"dead_actions","n":1},{"turn":28,"what":"feed_no_wheat","n":1},{"turn":28,"what":"dead_actions","n":1},{"turn":47,"what":"animals_escaped","n":2},{"turn":47,"what":"animal_unfed_days","n":2},{"turn":49,"what":"dead_actions","n":1},{"turn":49,"what":"feed_no_animal","n":1},{"turn":50,"what":"dead_actions","n":1},{"turn":51,"what":"dead_actions","n":1},{"turn":53,"what":"dead_actions","n":1},{"turn":53,"what":"feed_no_animal","n":1},{"turn":54,"what":"dead_actions","n":1},{"turn":55,"what":"dead_actions","n":1},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":88,"what":"buy_zero_fill","n":1},{"turn":89,"what":"dead_actions","n":1},{"turn":90,"what":"sell_zero_fill","n":1},{"turn":90,"what":"dead_actions","n":2},{"turn":91,"what":"dead_actions","n":1},{"turn":92,"what":"dead_actions","n":1},{"turn":93,"what":"dead_actions","n":1},{"turn":94,"what":"dead_actions","n":1},{"turn":95,"what":"dead_actions","n":1},{"turn":96,"what":"sell_zero_fill","n":1},{"turn":97,"what":"dead_actions","n":1},{"turn":97,"what":"feed_no_animal","n":1},{"turn":98,"what":"dead_actions","n":1},{"turn":99,"what":"dead_actions","n":1},{"turn":101,"what":"dead_actions","n":1},{"turn":101,"what":"feed_no_animal","n":1},{"turn":102,"what":"dead_actions","n":1},{"turn":103,"what":"dead_actions","n":1},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":2},{"turn":112,"what":"dead_actions","n":1},{"turn":113,"what":"dead_actions","n":1},{"turn":113,"what":"feed_no_animal","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":120,"what":"sell_zero_fill","n":1},{"turn":122,"what":"dead_actions","n":2},{"turn":122,"what":"feed_no_animal","n":1},{"turn":123,"what":"dead_actions","n":2},{"turn":125,"what":"dead_actions","n":1},{"turn":125,"what":"feed_no_animal","n":1},{"turn":126,"what":"dead_actions","n":2},{"turn":126,"what":"feed_no_animal","n":1},{"turn":127,"what":"dead_actions","n":1},{"turn":128,"what":"dead_actions","n":1},{"turn":131,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":134,"what":"sell_zero_fill","n":1},{"turn":146,"what":"dead_actions","n":1},{"turn":146,"what":"feed_no_animal","n":1},{"turn":147,"what":"dead_actions","n":2},{"turn":148,"what":"dead_actions","n":1},{"turn":148,"what":"feed_no_animal","n":1},{"turn":149,"what":"dead_actions","n":3},{"turn":149,"what":"feed_no_animal","n":1},{"turn":150,"what":"sell_zero_fill","n":1},{"turn":150,"what":"dead_actions","n":1},{"turn":151,"what":"dead_actions","n":1},{"turn":153,"what":"dead_actions","n":1},{"turn":154,"what":"dead_actions","n":1},{"turn":166,"what":"sell_zero_fill","n":1},{"turn":167,"what":"sell_zero_fill","n":1},{"turn":169,"what":"buy_zero_fill","n":1},{"turn":171,"what":"dead_actions","n":2},{"turn":171,"what":"feed_no_animal","n":2},{"turn":172,"what":"dead_actions","n":3},{"turn":172,"what":"feed_no_animal","n":1},{"turn":173,"what":"dead_actions","n":3},{"turn":174,"what":"dead_actions","n":2},{"turn":175,"what":"sell_zero_fill","n":1},{"turn":175,"what":"dead_actions","n":1},{"turn":176,"what":"buy_zero_fill","n":1},{"turn":176,"what":"dead_actions","n":2},{"turn":177,"what":"dead_actions","n":1},{"turn":178,"what":"dead_actions","n":1},{"turn":180,"what":"dead_actions","n":1},{"turn":183,"what":"dead_actions","n":1},{"turn":184,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":2},{"turn":190,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":1},{"turn":193,"what":"dead_actions","n":1},{"turn":194,"what":"dead_actions","n":2},{"turn":194,"what":"feed_no_animal","n":1},{"turn":195,"what":"sell_zero_fill","n":1},{"turn":195,"what":"buy_zero_fill","n":2},{"turn":195,"what":"dead_actions","n":3},{"turn":196,"what":"dead_actions","n":1},{"turn":197,"what":"dead_actions","n":1},{"turn":197,"what":"feed_no_animal","n":1},{"turn":198,"what":"dead_actions","n":2},{"turn":199,"what":"sell_zero_fill","n":1},{"turn":199,"what":"dead_actions","n":2},{"turn":199,"what":"feed_no_animal","n":1},{"turn":200,"what":"dead_actions","n":1},{"turn":201,"what":"dead_actions","n":2},{"turn":202,"what":"sell_zero_fill","n":1},{"turn":202,"what":"dead_actions","n":2},{"turn":203,"what":"dead_actions","n":2},{"turn":205,"what":"dead_actions","n":1},{"turn":206,"what":"dead_actions","n":4},{"turn":206,"what":"feed_no_animal","n":2},{"turn":207,"what":"dead_actions","n":3},{"turn":208,"what":"dead_actions","n":2},{"turn":210,"what":"sell_zero_fill","n":1},{"turn":212,"what":"dead_actions","n":2},{"turn":212,"what":"feed_no_animal","n":1},{"turn":213,"what":"sell_zero_fill","n":1},{"turn":214,"what":"sell_zero_fill","n":1},{"turn":215,"what":"animals_escaped","n":1},{"turn":215,"what":"animal_unfed_days","n":3},{"turn":217,"what":"sell_zero_fill","n":1},{"turn":217,"what":"buy_zero_fill","n":1},{"turn":218,"what":"dead_actions","n":1},{"turn":218,"what":"feed_no_animal","n":1},{"turn":219,"what":"dead_actions","n":2},{"turn":220,"what":"dead_actions","n":1},{"turn":222,"what":"dead_actions","n":1},{"turn":222,"what":"feed_no_animal","n":1},{"turn":223,"what":"dead_actions","n":3},{"turn":223,"what":"feed_no_animal","n":1},{"turn":224,"what":"dead_actions","n":3},{"turn":225,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":227,"what":"feed_no_animal","n":1},{"turn":228,"what":"dead_actions","n":2},{"turn":228,"what":"feed_no_animal","n":1},{"turn":229,"what":"dead_actions","n":2},{"turn":230,"what":"dead_actions","n":2},{"turn":230,"what":"feed_no_animal","n":1},{"turn":231,"what":"dead_actions","n":1},{"turn":232,"what":"dead_actions","n":2},{"turn":232,"what":"feed_no_animal","n":1},{"turn":233,"what":"dead_actions","n":1},{"turn":234,"what":"dead_actions","n":3},{"turn":234,"what":"feed_no_animal","n":1},{"turn":235,"what":"dead_actions","n":1},{"turn":236,"what":"dead_actions","n":2},{"turn":237,"what":"sell_zero_fill","n":1},{"turn":237,"what":"dead_actions","n":1},{"turn":238,"what":"dead_actions","n":1},{"turn":239,"what":"dead_actions","n":1},{"turn":239,"what":"feed_no_animal","n":1},{"turn":245,"what":"dead_actions","n":1},{"turn":245,"what":"feed_no_animal","n":1},{"turn":246,"what":"dead_actions","n":1},{"turn":247,"what":"dead_actions","n":2},{"turn":247,"what":"feed_no_animal","n":1},{"turn":248,"what":"dead_actions","n":1},{"turn":249,"what":"dead_actions","n":2},{"turn":249,"what":"feed_no_animal","n":1},{"turn":250,"what":"dead_actions","n":1},{"turn":251,"what":"dead_actions","n":2},{"turn":251,"what":"feed_no_animal","n":1},{"turn":252,"what":"dead_actions","n":1},{"turn":253,"what":"dead_actions","n":1},{"turn":255,"what":"dead_actions","n":2},{"turn":255,"what":"feed_no_animal","n":1},{"turn":256,"what":"dead_actions","n":1},{"turn":257,"what":"dead_actions","n":3},{"turn":257,"what":"feed_no_animal","n":1},{"turn":258,"what":"dead_actions","n":1},{"turn":259,"what":"dead_actions","n":2},{"turn":259,"what":"feed_no_animal","n":1},{"turn":260,"what":"dead_actions","n":1},{"turn":261,"what":"dead_actions","n":3},{"turn":261,"what":"feed_no_animal","n":2},{"turn":262,"what":"dead_actions","n":2},{"turn":263,"what":"dead_actions","n":2},{"turn":266,"what":"dead_actions","n":1},{"turn":267,"what":"dead_actions","n":1},{"turn":267,"what":"feed_no_animal","n":1},{"turn":268,"what":"dead_actions","n":2},{"turn":268,"what":"feed_no_animal","n":1},{"turn":269,"what":"sell_zero_fill","n":1},{"turn":269,"what":"dead_actions","n":3},{"turn":270,"what":"dead_actions","n":1},{"turn":271,"what":"dead_actions","n":1},{"turn":271,"what":"feed_no_animal","n":1},{"turn":272,"what":"dead_actions","n":2},{"turn":272,"what":"feed_no_animal","n":1},{"turn":273,"what":"dead_actions","n":3},{"turn":274,"what":"dead_actions","n":1},{"turn":275,"what":"dead_actions","n":1},{"turn":276,"what":"dead_actions","n":2},{"turn":276,"what":"feed_no_animal","n":1},{"turn":277,"what":"dead_actions","n":3},{"turn":278,"what":"dead_actions","n":2},{"turn":278,"what":"feed_no_animal","n":1},{"turn":279,"what":"dead_actions","n":2},{"turn":280,"what":"dead_actions","n":2},{"turn":285,"what":"dead_actions","n":1},{"turn":286,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":1},{"turn":290,"what":"dead_actions","n":1},{"turn":290,"what":"feed_no_animal","n":1},{"turn":291,"what":"dead_actions","n":1},{"turn":292,"what":"dead_actions","n":1},{"turn":294,"what":"dead_actions","n":2},{"turn":294,"what":"feed_no_animal","n":2},{"turn":295,"what":"dead_actions","n":3},{"turn":296,"what":"dead_actions","n":2},{"turn":297,"what":"dead_actions","n":2},{"turn":297,"what":"feed_no_animal","n":1},{"turn":298,"what":"dead_actions","n":3},{"turn":298,"what":"feed_no_animal","n":1},{"turn":299,"what":"sell_zero_fill","n":1},{"turn":299,"what":"dead_actions","n":2},{"turn":300,"what":"dead_actions","n":1},{"turn":301,"what":"dead_actions","n":1},{"turn":301,"what":"feed_no_animal","n":1},{"turn":302,"what":"dead_actions","n":2},{"turn":302,"what":"feed_no_animal","n":1},{"turn":303,"what":"dead_actions","n":3},{"turn":303,"what":"feed_no_animal","n":1},{"turn":304,"what":"dead_actions","n":2},{"turn":305,"what":"dead_actions","n":1},{"turn":306,"what":"dead_actions","n":1},{"turn":306,"what":"feed_no_animal","n":1},{"turn":307,"what":"dead_actions","n":1},{"turn":308,"what":"dead_actions","n":1},{"turn":309,"what":"sell_zero_fill","n":1},{"turn":314,"what":"dead_actions","n":1},{"turn":314,"what":"feed_no_animal","n":1},{"turn":315,"what":"dead_actions","n":2},{"turn":316,"what":"dead_actions","n":1},{"turn":318,"what":"dead_actions","n":2},{"turn":318,"what":"feed_no_animal","n":2},{"turn":319,"what":"dead_actions","n":3},{"turn":320,"what":"dead_actions","n":2},{"turn":321,"what":"sell_zero_fill","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":322,"what":"dead_actions","n":1},{"turn":322,"what":"feed_no_animal","n":1},{"turn":323,"what":"sell_zero_fill","n":1},{"turn":323,"what":"dead_actions","n":2},{"turn":324,"what":"dead_actions","n":1},{"turn":325,"what":"sell_zero_fill","n":1},{"turn":326,"what":"dead_actions","n":2},{"turn":326,"what":"feed_no_animal","n":2},{"turn":327,"what":"dead_actions","n":3},{"turn":327,"what":"feed_no_animal","n":1},{"turn":328,"what":"dead_actions","n":3},{"turn":329,"what":"dead_actions","n":1},{"turn":330,"what":"dead_actions","n":2},{"turn":330,"what":"feed_no_animal","n":2},{"turn":331,"what":"dead_actions","n":2},{"turn":332,"what":"dead_actions","n":2},{"turn":333,"what":"sell_zero_fill","n":1},{"turn":337,"what":"sell_zero_fill","n":1},{"turn":338,"what":"sell_zero_fill","n":1},{"turn":338,"what":"dead_actions","n":3},{"turn":338,"what":"feed_no_animal","n":1},{"turn":339,"what":"sell_zero_fill","n":2},{"turn":339,"what":"dead_actions","n":4},{"turn":340,"what":"sell_zero_fill","n":2},{"turn":340,"what":"dead_actions","n":3},{"turn":341,"what":"dead_actions","n":2},{"turn":341,"what":"feed_no_animal","n":1},{"turn":342,"what":"sell_zero_fill","n":1},{"turn":342,"what":"dead_actions","n":3},{"turn":342,"what":"feed_no_animal","n":1},{"turn":343,"what":"sell_zero_fill","n":1},{"turn":343,"what":"dead_actions","n":3},{"turn":344,"what":"dead_actions","n":2},{"turn":344,"what":"feed_no_animal","n":1},{"turn":345,"what":"dead_actions","n":2},{"turn":345,"what":"feed_no_animal","n":1},{"turn":346,"what":"dead_actions","n":2},{"turn":347,"what":"dead_actions","n":1},{"turn":348,"what":"dead_actions","n":1},{"turn":348,"what":"feed_no_animal","n":1},{"turn":349,"what":"sell_zero_fill","n":1},{"turn":349,"what":"dead_actions","n":1},{"turn":350,"what":"dead_actions","n":1},{"turn":352,"what":"dead_actions","n":1},{"turn":352,"what":"feed_no_animal","n":1},{"turn":353,"what":"dead_actions","n":2},{"turn":353,"what":"feed_no_animal","n":1},{"turn":354,"what":"dead_actions","n":2},{"turn":355,"what":"dead_actions","n":1},{"turn":356,"what":"dead_actions","n":1},{"turn":356,"what":"feed_no_animal","n":1},{"turn":357,"what":"dead_actions","n":1},{"turn":358,"what":"dead_actions","n":1},{"turn":359,"what":"dead_actions","n":1},{"turn":360,"what":"sell_zero_fill","n":1},{"turn":362,"what":"sell_zero_fill","n":1},{"turn":363,"what":"dead_actions","n":2},{"turn":363,"what":"feed_no_animal","n":1},{"turn":364,"what":"dead_actions","n":1},{"turn":365,"what":"dead_actions","n":2},{"turn":365,"what":"feed_no_animal","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":367,"what":"dead_actions","n":3},{"turn":367,"what":"feed_no_animal","n":2},{"turn":368,"what":"dead_actions","n":3},{"turn":369,"what":"dead_actions","n":4},{"turn":369,"what":"feed_no_animal","n":1},{"turn":370,"what":"dead_actions","n":2},{"turn":371,"what":"dead_actions","n":3},{"turn":371,"what":"feed_no_animal","n":1},{"turn":372,"what":"dead_actions","n":1},{"turn":373,"what":"sell_zero_fill","n":1},{"turn":373,"what":"dead_actions","n":2},{"turn":375,"what":"sell_zero_fill","n":1},{"turn":375,"what":"dead_actions","n":1},{"turn":375,"what":"feed_no_animal","n":1},{"turn":376,"what":"dead_actions","n":2},{"turn":376,"what":"feed_no_animal","n":1},{"turn":377,"what":"dead_actions","n":3},{"turn":378,"what":"dead_actions","n":1},{"turn":379,"what":"dead_actions","n":1},{"turn":379,"what":"feed_no_animal","n":1},{"turn":380,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":383,"what":"dead_actions","n":1},{"turn":384,"what":"sell_zero_fill","n":1},{"turn":385,"what":"dead_actions","n":1},{"turn":386,"what":"dead_actions","n":2},{"turn":386,"what":"feed_no_animal","n":1},{"turn":387,"what":"dead_actions","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":390,"what":"dead_actions","n":1},{"turn":390,"what":"feed_no_animal","n":1},{"turn":391,"what":"dead_actions","n":2},{"turn":391,"what":"feed_no_animal","n":1},{"turn":392,"what":"dead_actions","n":2},{"turn":393,"what":"sell_zero_fill","n":2},{"turn":393,"what":"dead_actions","n":1},{"turn":395,"what":"sell_zero_fill","n":1},{"turn":395,"what":"dead_actions","n":1},{"turn":395,"what":"feed_no_animal","n":1},{"turn":396,"what":"dead_actions","n":1},{"turn":397,"what":"dead_actions","n":1},{"turn":398,"what":"sell_zero_fill","n":1},{"turn":398,"what":"dead_actions","n":1},{"turn":398,"what":"feed_no_animal","n":1},{"turn":399,"what":"dead_actions","n":2},{"turn":399,"what":"feed_no_animal","n":1},{"turn":400,"what":"dead_actions","n":3},{"turn":400,"what":"feed_no_animal","n":1},{"turn":401,"what":"dead_actions","n":2},{"turn":402,"what":"sell_zero_fill","n":2},{"turn":402,"what":"dead_actions","n":3},{"turn":402,"what":"feed_no_animal","n":1},{"turn":403,"what":"sell_zero_fill","n":2},{"turn":403,"what":"dead_actions","n":2},{"turn":403,"what":"feed_no_animal","n":1},{"turn":404,"what":"sell_zero_fill","n":3},{"turn":404,"what":"dead_actions","n":3},{"turn":405,"what":"sell_zero_fill","n":3},{"turn":405,"what":"dead_actions","n":1},{"turn":406,"what":"sell_zero_fill","n":3},{"turn":407,"what":"sell_zero_fill","n":2},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":412,"what":"dead_actions","n":3},{"turn":412,"what":"feed_no_animal","n":2},{"turn":413,"what":"dead_actions","n":2},{"turn":414,"what":"dead_actions","n":2},{"turn":415,"what":"dead_actions","n":4},{"turn":415,"what":"feed_no_animal","n":1},{"turn":416,"what":"dead_actions","n":3},{"turn":416,"what":"feed_no_animal","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":417,"what":"dead_actions","n":3},{"turn":417,"what":"feed_no_animal","n":1},{"turn":418,"what":"sell_zero_fill","n":1},{"turn":418,"what":"dead_actions","n":4},{"turn":419,"what":"dead_actions","n":3},{"turn":419,"what":"feed_no_animal","n":1},{"turn":420,"what":"dead_actions","n":1},{"turn":421,"what":"dead_actions","n":2},{"turn":422,"what":"dead_actions","n":2},{"turn":423,"what":"dead_actions","n":1},{"turn":423,"what":"feed_no_animal","n":1},{"turn":424,"what":"dead_actions","n":2},{"turn":424,"what":"feed_no_animal","n":1},{"turn":425,"what":"dead_actions","n":2},{"turn":426,"what":"dead_actions","n":2},{"turn":428,"what":"dead_actions","n":1},{"turn":428,"what":"feed_no_animal","n":1},{"turn":429,"what":"dead_actions","n":1},{"turn":430,"what":"sell_zero_fill","n":8},{"turn":430,"what":"dead_actions","n":1},{"turn":431,"what":"sell_zero_fill","n":1},{"turn":432,"what":"sell_zero_fill","n":1},{"turn":434,"what":"sell_zero_fill","n":1},{"turn":435,"what":"dead_actions","n":1},{"turn":435,"what":"feed_no_animal","n":1},{"turn":436,"what":"dead_actions","n":1},{"turn":437,"what":"dead_actions","n":4},{"turn":437,"what":"feed_no_animal","n":1},{"turn":438,"what":"dead_actions","n":1},{"turn":439,"what":"dead_actions","n":5},{"turn":439,"what":"feed_no_animal","n":2},{"turn":440,"what":"dead_actions","n":2},{"turn":441,"what":"dead_actions","n":4},{"turn":441,"what":"feed_no_animal","n":1},{"turn":442,"what":"dead_actions","n":2},{"turn":443,"what":"dead_actions","n":1},{"turn":444,"what":"dead_actions","n":1},{"turn":444,"what":"feed_no_animal","n":1},{"turn":445,"what":"dead_actions","n":1},{"turn":446,"what":"dead_actions","n":1},{"turn":447,"what":"sell_zero_fill","n":2},{"turn":447,"what":"dead_actions","n":1},{"turn":447,"what":"feed_no_animal","n":1},{"turn":448,"what":"dead_actions","n":2},{"turn":448,"what":"feed_no_animal","n":1},{"turn":449,"what":"dead_actions","n":2},{"turn":450,"what":"dead_actions","n":1},{"turn":451,"what":"dead_actions","n":1},{"turn":453,"what":"sell_zero_fill","n":1},{"turn":453,"what":"dead_actions","n":2},{"turn":453,"what":"feed_no_animal","n":1},{"turn":454,"what":"dead_actions","n":1},{"turn":455,"what":"dead_actions","n":2},{"turn":457,"what":"sell_zero_fill","n":2},{"turn":458,"what":"sell_zero_fill","n":1},{"turn":458,"what":"dead_actions","n":2},{"turn":458,"what":"feed_no_animal","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":2},{"turn":460,"what":"feed_no_animal","n":1},{"turn":461,"what":"dead_actions","n":1},{"turn":462,"what":"dead_actions","n":5},{"turn":462,"what":"feed_no_animal","n":2},{"turn":463,"what":"dead_actions","n":4},{"turn":464,"what":"dead_actions","n":3},{"turn":465,"what":"dead_actions","n":1},{"turn":465,"what":"feed_no_animal","n":1},{"turn":466,"what":"dead_actions","n":3},{"turn":466,"what":"feed_no_animal","n":1},{"turn":467,"what":"dead_actions","n":2},{"turn":468,"what":"dead_actions","n":1},{"turn":469,"what":"dead_actions","n":2},{"turn":470,"what":"sell_zero_fill","n":2},{"turn":471,"what":"sell_zero_fill","n":2},{"turn":471,"what":"dead_actions","n":2},{"turn":471,"what":"feed_no_animal","n":2},{"turn":472,"what":"dead_actions","n":2},{"turn":473,"what":"sell_zero_fill","n":1},{"turn":473,"what":"dead_actions","n":2},{"turn":474,"what":"sell_zero_fill","n":1},{"turn":474,"what":"dead_actions","n":1},{"turn":475,"what":"dead_actions","n":1},{"turn":475,"what":"feed_no_animal","n":1},{"turn":476,"what":"dead_actions","n":1},{"turn":477,"what":"dead_actions","n":1},{"turn":478,"what":"sell_zero_fill","n":1},{"turn":478,"what":"dead_actions","n":1},{"turn":479,"what":"dead_actions","n":1},{"turn":480,"what":"sell_zero_fill","n":1},{"turn":481,"what":"sell_zero_fill","n":1},{"turn":481,"what":"dead_actions","n":1},{"turn":481,"what":"feed_no_animal","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":484,"what":"feed_no_animal","n":1},{"turn":485,"what":"dead_actions","n":4},{"turn":485,"what":"feed_no_animal","n":1},{"turn":486,"what":"dead_actions","n":3},{"turn":486,"what":"feed_no_animal","n":1},{"turn":487,"what":"dead_actions","n":2},{"turn":488,"what":"dead_actions","n":2},{"turn":489,"what":"dead_actions","n":1},{"turn":490,"what":"dead_actions","n":1},{"turn":490,"what":"feed_no_animal","n":1},{"turn":491,"what":"dead_actions","n":2},{"turn":492,"what":"dead_actions","n":3},{"turn":492,"what":"feed_no_animal","n":1},{"turn":493,"what":"sell_zero_fill","n":1},{"turn":493,"what":"dead_actions","n":2},{"turn":494,"what":"sell_zero_fill","n":1},{"turn":494,"what":"dead_actions","n":1},{"turn":496,"what":"dead_actions","n":2},{"turn":496,"what":"feed_no_animal","n":1},{"turn":497,"what":"dead_actions","n":1},{"turn":498,"what":"dead_actions","n":2},{"turn":499,"what":"dead_actions","n":1},{"turn":500,"what":"dead_actions","n":2},{"turn":500,"what":"feed_no_animal","n":1},{"turn":501,"what":"sell_zero_fill","n":2},{"turn":501,"what":"dead_actions","n":4},{"turn":501,"what":"feed_no_animal","n":1},{"turn":502,"what":"dead_actions","n":3},{"turn":503,"what":"sell_zero_fill","n":2},{"turn":503,"what":"dead_actions","n":2},{"turn":505,"what":"sell_zero_fill","n":1},{"turn":505,"what":"dead_actions","n":1},{"turn":506,"what":"dead_actions","n":1},{"turn":506,"what":"feed_no_animal","n":1},{"turn":507,"what":"dead_actions","n":2},{"turn":507,"what":"feed_no_animal","n":1},{"turn":508,"what":"dead_actions","n":2},{"turn":509,"what":"dead_actions","n":2},{"turn":510,"what":"dead_actions","n":3},{"turn":510,"what":"feed_no_animal","n":1},{"turn":511,"what":"dead_actions","n":3},{"turn":511,"what":"feed_no_animal","n":1},{"turn":512,"what":"dead_actions","n":3},{"turn":512,"what":"feed_no_animal","n":1},{"turn":513,"what":"sell_zero_fill","n":1},{"turn":513,"what":"dead_actions","n":3},{"turn":514,"what":"dead_actions","n":3},{"turn":515,"what":"dead_actions","n":1},{"turn":515,"what":"feed_no_animal","n":1},{"turn":516,"what":"dead_actions","n":1},{"turn":517,"what":"dead_actions","n":2},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":518,"what":"dead_actions","n":1},{"turn":519,"what":"dead_actions","n":1},{"turn":519,"what":"feed_no_animal","n":1},{"turn":520,"what":"dead_actions","n":2},{"turn":520,"what":"feed_no_animal","n":1},{"turn":521,"what":"sell_zero_fill","n":2},{"turn":521,"what":"dead_actions","n":2},{"turn":522,"what":"dead_actions","n":2},{"turn":524,"what":"dead_actions","n":1},{"turn":524,"what":"feed_no_animal","n":1},{"turn":525,"what":"dead_actions","n":1},{"turn":526,"what":"dead_actions","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":2},{"turn":527,"what":"dead_actions","n":1},{"turn":529,"what":"sell_zero_fill","n":1},{"turn":530,"what":"dead_actions","n":1},{"turn":530,"what":"feed_no_animal","n":1},{"turn":531,"what":"dead_actions","n":2},{"turn":532,"what":"dead_actions","n":2},{"turn":532,"what":"feed_no_animal","n":1},{"turn":533,"what":"dead_actions","n":3},{"turn":534,"what":"dead_actions","n":2},{"turn":534,"what":"feed_no_animal","n":1},{"turn":535,"what":"sell_zero_fill","n":1},{"turn":535,"what":"dead_actions","n":2},{"turn":535,"what":"feed_no_animal","n":1},{"turn":536,"what":"dead_actions","n":3},{"turn":536,"what":"feed_no_animal","n":1},{"turn":537,"what":"sell_zero_fill","n":1},{"turn":537,"what":"dead_actions","n":2},{"turn":538,"what":"sell_zero_fill","n":1},{"turn":538,"what":"dead_actions","n":2},{"turn":538,"what":"feed_no_animal","n":1},{"turn":539,"what":"dead_actions","n":2},{"turn":540,"what":"dead_actions","n":1},{"turn":541,"what":"dead_actions","n":1},{"turn":542,"what":"dead_actions","n":1},{"turn":542,"what":"feed_no_animal","n":1},{"turn":543,"what":"dead_actions","n":2},{"turn":543,"what":"feed_no_animal","n":1},{"turn":544,"what":"dead_actions","n":2},{"turn":545,"what":"sell_zero_fill","n":1},{"turn":545,"what":"dead_actions","n":2},{"turn":547,"what":"dead_actions","n":1},{"turn":547,"what":"feed_no_animal","n":1},{"turn":548,"what":"dead_actions","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":549,"what":"dead_actions","n":1},{"turn":550,"what":"sell_zero_fill","n":2},{"turn":551,"what":"sell_zero_fill","n":2},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":553,"what":"sell_zero_fill","n":1},{"turn":555,"what":"sell_zero_fill","n":1},{"turn":555,"what":"dead_actions","n":1},{"turn":555,"what":"feed_no_animal","n":1},{"turn":556,"what":"dead_actions","n":1},{"turn":557,"what":"dead_actions","n":2},{"turn":557,"what":"feed_no_animal","n":1},{"turn":558,"what":"sell_zero_fill","n":1},{"turn":558,"what":"dead_actions","n":2},{"turn":558,"what":"feed_no_animal","n":1},{"turn":559,"what":"dead_actions","n":3},{"turn":559,"what":"feed_no_animal","n":1},{"turn":560,"what":"dead_actions","n":3},{"turn":561,"what":"sell_zero_fill","n":1},{"turn":561,"what":"dead_actions","n":1},{"turn":562,"what":"sell_zero_fill","n":2},{"turn":562,"what":"dead_actions","n":3},{"turn":562,"what":"feed_no_animal","n":2},{"turn":563,"what":"dead_actions","n":3},{"turn":564,"what":"dead_actions","n":2},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":565,"what":"dead_actions","n":3},{"turn":567,"what":"dead_actions","n":1},{"turn":567,"what":"feed_no_animal","n":1},{"turn":568,"what":"dead_actions","n":2},{"turn":568,"what":"feed_no_animal","n":1},{"turn":569,"what":"dead_actions","n":2},{"turn":570,"what":"sell_zero_fill","n":2},{"turn":570,"what":"dead_actions","n":1},{"turn":571,"what":"sell_zero_fill","n":2},{"turn":571,"what":"dead_actions","n":2},{"turn":571,"what":"feed_no_animal","n":1},{"turn":572,"what":"dead_actions","n":1},{"turn":573,"what":"sell_zero_fill","n":2},{"turn":573,"what":"dead_actions","n":1},{"turn":574,"what":"sell_zero_fill","n":2},{"turn":574,"what":"dead_actions","n":1},{"turn":575,"what":"sell_zero_fill","n":2},{"turn":575,"what":"dead_actions","n":2},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":577,"what":"sell_zero_fill","n":2},{"turn":578,"what":"dead_actions","n":1},{"turn":578,"what":"feed_no_animal","n":1},{"turn":579,"what":"dead_actions","n":2},{"turn":579,"what":"feed_no_animal","n":1},{"turn":580,"what":"dead_actions","n":2},{"turn":581,"what":"sell_zero_fill","n":1},{"turn":581,"what":"dead_actions","n":3},{"turn":582,"what":"dead_actions","n":1},{"turn":583,"what":"dead_actions","n":4},{"turn":583,"what":"feed_no_animal","n":1},{"turn":584,"what":"dead_actions","n":3},{"turn":585,"what":"sell_zero_fill","n":1},{"turn":585,"what":"dead_actions","n":4},{"turn":586,"what":"dead_actions","n":1},{"turn":586,"what":"feed_no_animal","n":1},{"turn":587,"what":"dead_actions","n":1},{"turn":588,"what":"dead_actions","n":3},{"turn":590,"what":"dead_actions","n":1},{"turn":590,"what":"feed_no_animal","n":1},{"turn":591,"what":"dead_actions","n":3},{"turn":591,"what":"feed_no_animal","n":1},{"turn":592,"what":"dead_actions","n":2},{"turn":593,"what":"dead_actions","n":3},{"turn":595,"what":"dead_actions","n":2},{"turn":595,"what":"feed_no_animal","n":1},{"turn":596,"what":"dead_actions","n":1},{"turn":597,"what":"sell_zero_fill","n":2},{"turn":597,"what":"dead_actions","n":2},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":2},{"turn":599,"what":"sell_zero_fill","n":3},{"turn":599,"what":"dead_actions","n":1},{"turn":600,"what":"plants_decayed","n":1},{"turn":600,"what":"decay_units","n":1},{"turn":601,"what":"sell_zero_fill","n":2},{"turn":603,"what":"sell_zero_fill","n":1},{"turn":603,"what":"dead_actions","n":2},{"turn":603,"what":"feed_no_animal","n":2},{"turn":604,"what":"dead_actions","n":3},{"turn":605,"what":"dead_actions","n":1},{"turn":606,"what":"dead_actions","n":3},{"turn":606,"what":"feed_no_animal","n":2},{"turn":607,"what":"dead_actions","n":2},{"turn":608,"what":"sell_zero_fill","n":1},{"turn":608,"what":"dead_actions","n":4},{"turn":608,"what":"feed_no_animal","n":1},{"turn":609,"what":"dead_actions","n":1},{"turn":610,"what":"sell_zero_fill","n":1},{"turn":610,"what":"dead_actions","n":3},{"turn":610,"what":"feed_no_animal","n":1},{"turn":611,"what":"sell_zero_fill","n":2},{"turn":611,"what":"dead_actions","n":1},{"turn":612,"what":"sell_zero_fill","n":2},{"turn":612,"what":"dead_actions","n":2},{"turn":613,"what":"dead_actions","n":1},{"turn":614,"what":"dead_actions","n":1},{"turn":614,"what":"feed_no_animal","n":1},{"turn":615,"what":"dead_actions","n":2},{"turn":615,"what":"feed_no_animal","n":1},{"turn":616,"what":"dead_actions","n":3},{"turn":617,"what":"sell_zero_fill","n":1},{"turn":617,"what":"dead_actions","n":2},{"turn":619,"what":"dead_actions","n":2},{"turn":619,"what":"feed_no_animal","n":1},{"turn":620,"what":"dead_actions","n":1},{"turn":621,"what":"sell_zero_fill","n":1},{"turn":621,"what":"dead_actions","n":2},{"turn":622,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":625,"what":"sell_zero_fill","n":2},{"turn":626,"what":"dead_actions","n":1},{"turn":626,"what":"feed_no_animal","n":1},{"turn":627,"what":"sell_zero_fill","n":1},{"turn":627,"what":"dead_actions","n":1},{"turn":628,"what":"sell_zero_fill","n":1},{"turn":628,"what":"dead_actions","n":2},{"turn":628,"what":"feed_no_animal","n":1},{"turn":629,"what":"sell_zero_fill","n":1},{"turn":629,"what":"dead_actions","n":2},{"turn":630,"what":"dead_actions","n":2},{"turn":630,"what":"feed_no_animal","n":1},{"turn":631,"what":"dead_actions","n":2},{"turn":631,"what":"feed_no_animal","n":1},{"turn":632,"what":"dead_actions","n":3},{"turn":632,"what":"feed_no_animal","n":1},{"turn":633,"what":"dead_actions","n":1},{"turn":634,"what":"dead_actions","n":2},{"turn":634,"what":"feed_no_animal","n":1},{"turn":635,"what":"dead_actions","n":1},{"turn":636,"what":"dead_actions","n":2},{"turn":637,"what":"sell_zero_fill","n":2},{"turn":638,"what":"sell_zero_fill","n":1},{"turn":638,"what":"dead_actions","n":1},{"turn":638,"what":"feed_no_animal","n":1},{"turn":639,"what":"sell_zero_fill","n":1},{"turn":639,"what":"dead_actions","n":3},{"turn":639,"what":"feed_no_animal","n":1},{"turn":640,"what":"dead_actions","n":2},{"turn":641,"what":"sell_zero_fill","n":1},{"turn":641,"what":"dead_actions","n":1},{"turn":642,"what":"sell_zero_fill","n":1},{"turn":642,"what":"dead_actions","n":1},{"turn":642,"what":"feed_no_animal","n":1},{"turn":643,"what":"sell_zero_fill","n":2},{"turn":643,"what":"dead_actions","n":1},{"turn":644,"what":"sell_zero_fill","n":2},{"turn":644,"what":"dead_actions","n":1},{"turn":645,"what":"sell_zero_fill","n":2},{"turn":645,"what":"dead_actions","n":1},{"turn":647,"what":"sell_zero_fill","n":2},{"turn":647,"what":"dead_actions","n":1},{"turn":649,"what":"sell_zero_fill","n":2},{"turn":649,"what":"dead_actions","n":1},{"turn":649,"what":"feed_no_animal","n":1},{"turn":650,"what":"dead_actions","n":1},{"turn":651,"what":"dead_actions","n":1},{"turn":652,"what":"dead_actions","n":5},{"turn":652,"what":"feed_no_animal","n":5},{"turn":653,"what":"dead_actions","n":5},{"turn":654,"what":"dead_actions","n":3},{"turn":655,"what":"dead_actions","n":3},{"turn":658,"what":"dead_actions","n":2},{"turn":658,"what":"feed_no_animal","n":2},{"turn":659,"what":"dead_actions","n":2},{"turn":660,"what":"dead_actions","n":2},{"turn":662,"what":"dead_actions","n":1},{"turn":662,"what":"feed_no_animal","n":1},{"turn":663,"what":"dead_actions","n":1},{"turn":664,"what":"dead_actions","n":1},{"turn":666,"what":"sell_zero_fill","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":3},{"turn":670,"what":"dead_actions","n":1},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":671,"what":"dead_actions","n":1},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":673,"what":"sell_zero_fill","n":1},{"turn":674,"what":"sell_zero_fill","n":2},{"turn":675,"what":"dead_actions","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":677,"what":"dead_actions","n":1},{"turn":679,"what":"dead_actions","n":3},{"turn":679,"what":"feed_no_animal","n":2},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":4},{"turn":681,"what":"dead_actions","n":2},{"turn":682,"what":"dead_actions","n":1},{"turn":683,"what":"dead_actions","n":1},{"turn":683,"what":"feed_no_animal","n":1},{"turn":684,"what":"dead_actions","n":1},{"turn":685,"what":"sell_zero_fill","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":686,"what":"dead_actions","n":1},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":687,"what":"dead_actions","n":3},{"turn":687,"what":"feed_no_animal","n":1},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":688,"what":"dead_actions","n":1},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":689,"what":"dead_actions","n":1},{"turn":689,"what":"feed_no_animal","n":1},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":690,"what":"dead_actions","n":1},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":691,"what":"dead_actions","n":1},{"turn":692,"what":"sell_zero_fill","n":3},{"turn":692,"what":"dead_actions","n":1},{"turn":693,"what":"sell_zero_fill","n":3},{"turn":693,"what":"dead_actions","n":2},{"turn":693,"what":"feed_no_animal","n":1},{"turn":694,"what":"sell_zero_fill","n":2},{"turn":694,"what":"dead_actions","n":1},{"turn":695,"what":"animal_unfed_days","n":5},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":695,"what":"dead_actions","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":699,"what":"dead_actions","n":1},{"turn":705,"what":"sell_zero_fill","n":1},{"turn":705,"what":"dead_actions","n":2},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":2},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":3},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":712,"what":"dead_actions","n":1},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":2},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":714,"what":"dead_actions","n":2},{"turn":715,"what":"sell_zero_fill","n":7},{"turn":715,"what":"dead_actions","n":3},{"turn":716,"what":"sell_zero_fill","n":6},{"turn":717,"what":"sell_zero_fill","n":6},{"turn":717,"what":"dead_actions","n":2}],"ledger":{"sell_dead_units":116751,"refused_buy_product":1740,"refused_buy_seed":3,"refused_buy_animal":6,"refused_hire":1,"refused_buy_land":0,"hire_paid":4939,"sell_zero_fill":258,"buy_zero_fill":6,"sell_revenue":140827.0,"total_spend":35870.0,"shed_discarded_units":0,"sold_units":1518,"feed_no_wheat":2,"feed_no_animal":192,"feed_redundant":0,"water_dead":33,"harvest_dead":97,"plant_dead":3,"animals_escaped":3,"escape_capital":1200.0,"escape_yield_units":0,"escape_first_day":1,"animal_unfed_days":20,"plants_dry":2,"dry_seed_cost":110.0,"dry_yield_units":1,"dry_first_day":15,"plant_unwatered_days":356,"decay_units":15,"plants_decayed":15,"silent_loss_coins":1310.0,"silent_loss_units":16,"dead_actions":805,"move_dead":0,"locked_dead":0,"pickup_dead":10,"drop_dead":0,"place_dead":29,"build_dead":0,"dig_dead":3,"fertilize_dead":17,"collect_dead":211,"care_dead":208,"tile_cap_units":0,"fertilizer_forgone":12,"care_unfed":12,"animal_shed_days":1,"idle_tile_days":115,"weed_tile_days":2,"hand_pass_turns":498,"farmer_pass_turns":76,"phantom_hand_actions":0,"sell_floor_units":0,"hire_premium":4664,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":4,"seeds_unplanted_cost_now":40.0},"agent":"v7.76, mine, the defective build"},{"bank":84337.0,"rival_bank":81844.0,"events":[{"turn":18,"what":"dead_actions","n":2},{"turn":19,"what":"dead_actions","n":2},{"turn":23,"what":"animal_unfed_days","n":2},{"turn":54,"what":"dead_actions","n":1},{"turn":60,"what":"dead_actions","n":1},{"turn":71,"what":"animal_unfed_days","n":1},{"turn":83,"what":"dead_actions","n":1},{"turn":89,"what":"dead_actions","n":1},{"turn":91,"what":"dead_actions","n":1},{"turn":95,"what":"animal_unfed_days","n":3},{"turn":107,"what":"dead_actions","n":1},{"turn":111,"what":"dead_actions","n":1},{"turn":115,"what":"dead_actions","n":1},{"turn":119,"what":"animal_unfed_days","n":1},{"turn":119,"what":"sell_zero_fill","n":1},{"turn":123,"what":"dead_actions","n":1},{"turn":129,"what":"dead_actions","n":1},{"turn":132,"what":"dead_actions","n":1},{"turn":138,"what":"dead_actions","n":1},{"turn":155,"what":"dead_actions","n":1},{"turn":162,"what":"dead_actions","n":1},{"turn":166,"what":"dead_actions","n":1},{"turn":167,"what":"dead_actions","n":1},{"turn":173,"what":"dead_actions","n":1},{"turn":178,"what":"dead_actions","n":1},{"turn":185,"what":"dead_actions","n":1},{"turn":188,"what":"dead_actions","n":1},{"turn":191,"what":"animal_unfed_days","n":3},{"turn":191,"what":"dead_actions","n":2},{"turn":195,"what":"dead_actions","n":1},{"turn":205,"what":"dead_actions","n":1},{"turn":211,"what":"dead_actions","n":1},{"turn":212,"what":"dead_actions","n":1},{"turn":215,"what":"animal_unfed_days","n":1},{"turn":220,"what":"dead_actions","n":1},{"turn":227,"what":"dead_actions","n":1},{"turn":228,"what":"dead_actions","n":1},{"turn":233,"what":"dead_actions","n":1},{"turn":245,"what":"dead_actions","n":1},{"turn":246,"what":"dead_actions","n":1},{"turn":251,"what":"dead_actions","n":1},{"turn":253,"what":"sell_zero_fill","n":1},{"turn":257,"what":"dead_actions","n":1},{"turn":259,"what":"dead_actions","n":1},{"turn":260,"what":"dead_actions","n":1},{"turn":266,"what":"sell_zero_fill","n":1},{"turn":268,"what":"dead_actions","n":1},{"turn":281,"what":"dead_actions","n":1},{"turn":287,"what":"animal_unfed_days","n":4},{"turn":317,"what":"dead_actions","n":1},{"turn":321,"what":"dead_actions","n":1},{"turn":323,"what":"dead_actions","n":1},{"turn":335,"what":"dead_actions","n":1},{"turn":361,"what":"sell_zero_fill","n":1},{"turn":365,"what":"dead_actions","n":1},{"turn":366,"what":"dead_actions","n":1},{"turn":369,"what":"dead_actions","n":1},{"turn":370,"what":"dead_actions","n":1},{"turn":381,"what":"dead_actions","n":1},{"turn":383,"what":"plants_dry","n":1},{"turn":388,"what":"dead_actions","n":1},{"turn":391,"what":"dead_actions","n":1},{"turn":401,"what":"sell_zero_fill","n":1},{"turn":409,"what":"sell_zero_fill","n":1},{"turn":417,"what":"sell_zero_fill","n":1},{"turn":451,"what":"sell_zero_fill","n":1},{"turn":455,"what":"sell_zero_fill","n":1},{"turn":457,"what":"dead_actions","n":1},{"turn":459,"what":"dead_actions","n":1},{"turn":460,"what":"dead_actions","n":1},{"turn":484,"what":"dead_actions","n":1},{"turn":486,"what":"dead_actions","n":1},{"turn":494,"what":"dead_actions","n":1},{"turn":499,"what":"feed_no_wheat","n":1},{"turn":499,"what":"dead_actions","n":3},{"turn":500,"what":"feed_no_wheat","n":1},{"turn":500,"what":"dead_actions","n":1},{"turn":501,"what":"dead_actions","n":1},{"turn":503,"what":"animal_unfed_days","n":2},{"turn":509,"what":"dead_actions","n":1},{"turn":512,"what":"sell_zero_fill","n":1},{"turn":518,"what":"sell_zero_fill","n":1},{"turn":526,"what":"sell_zero_fill","n":1},{"turn":527,"what":"plants_dry","n":1},{"turn":527,"what":"sell_zero_fill","n":1},{"turn":549,"what":"sell_zero_fill","n":1},{"turn":550,"what":"sell_zero_fill","n":1},{"turn":552,"what":"plants_decayed","n":7},{"turn":552,"what":"decay_units","n":7},{"turn":562,"what":"sell_zero_fill","n":1},{"turn":565,"what":"sell_zero_fill","n":1},{"turn":570,"what":"sell_zero_fill","n":1},{"turn":576,"what":"plants_decayed","n":3},{"turn":576,"what":"decay_units","n":3},{"turn":595,"what":"sell_zero_fill","n":1},{"turn":597,"what":"sell_zero_fill","n":1},{"turn":598,"what":"sell_zero_fill","n":1},{"turn":599,"what":"animal_unfed_days","n":4},{"turn":599,"what":"sell_zero_fill","n":1},{"turn":600,"what":"plants_decayed","n":4},{"turn":600,"what":"decay_units","n":4},{"turn":609,"what":"dead_actions","n":1},{"turn":613,"what":"sell_zero_fill","n":1},{"turn":615,"what":"sell_zero_fill","n":1},{"turn":619,"what":"sell_zero_fill","n":1},{"turn":623,"what":"animal_unfed_days","n":2},{"turn":623,"what":"sell_zero_fill","n":1},{"turn":635,"what":"sell_zero_fill","n":1},{"turn":640,"what":"sell_zero_fill","n":1},{"turn":662,"what":"sell_zero_fill","n":1},{"turn":669,"what":"dead_actions","n":1},{"turn":670,"what":"sell_zero_fill","n":2},{"turn":671,"what":"sell_zero_fill","n":3},{"turn":672,"what":"plants_decayed","n":4},{"turn":672,"what":"decay_units","n":4},{"turn":674,"what":"sell_zero_fill","n":1},{"turn":676,"what":"dead_actions","n":1},{"turn":677,"what":"sell_zero_fill","n":1},{"turn":680,"what":"sell_zero_fill","n":1},{"turn":680,"what":"dead_actions","n":1},{"turn":686,"what":"sell_zero_fill","n":2},{"turn":687,"what":"sell_zero_fill","n":2},{"turn":688,"what":"sell_zero_fill","n":2},{"turn":689,"what":"sell_zero_fill","n":3},{"turn":690,"what":"sell_zero_fill","n":2},{"turn":691,"what":"sell_zero_fill","n":1},{"turn":692,"what":"sell_zero_fill","n":2},{"turn":693,"what":"sell_zero_fill","n":2},{"turn":694,"what":"sell_zero_fill","n":1},{"turn":695,"what":"shed_discarded_units","n":6},{"turn":695,"what":"animal_unfed_days","n":8},{"turn":695,"what":"sell_zero_fill","n":1},{"turn":697,"what":"sell_zero_fill","n":1},{"turn":698,"what":"sell_zero_fill","n":1},{"turn":708,"what":"sell_zero_fill","n":1},{"turn":708,"what":"dead_actions","n":1},{"turn":710,"what":"sell_zero_fill","n":1},{"turn":711,"what":"sell_zero_fill","n":4},{"turn":711,"what":"dead_actions","n":1},{"turn":712,"what":"sell_zero_fill","n":8},{"turn":713,"what":"sell_zero_fill","n":9},{"turn":713,"what":"dead_actions","n":1},{"turn":714,"what":"sell_zero_fill","n":9},{"turn":715,"what":"sell_zero_fill","n":6},{"turn":716,"what":"sell_zero_fill","n":5},{"turn":717,"what":"sell_zero_fill","n":6}],"ledger":{"sell_dead_units":84195,"refused_buy_product":0,"refused_buy_seed":1,"refused_buy_animal":0,"refused_hire":0,"refused_buy_land":0,"hire_paid":3986,"sell_zero_fill":106,"buy_zero_fill":0,"sell_revenue":112693.0,"total_spend":31356.0,"shed_discarded_units":6,"sold_units":1685,"feed_no_wheat":2,"feed_no_animal":0,"feed_redundant":0,"water_dead":31,"harvest_dead":6,"plant_dead":2,"animals_escaped":0,"escape_capital":0.0,"escape_yield_units":0,"escape_first_day":-1,"animal_unfed_days":31,"plants_dry":2,"dry_seed_cost":110.0,"dry_yield_units":1,"dry_first_day":15,"plant_unwatered_days":376,"decay_units":18,"plants_decayed":18,"silent_loss_coins":110.0,"silent_loss_units":25,"dead_actions":72,"move_dead":0,"locked_dead":0,"pickup_dead":2,"drop_dead":0,"place_dead":1,"build_dead":0,"dig_dead":1,"fertilize_dead":0,"collect_dead":3,"care_dead":24,"tile_cap_units":22,"fertilizer_forgone":15,"care_unfed":27,"animal_shed_days":1,"idle_tile_days":84,"weed_tile_days":2,"hand_pass_turns":356,"farmer_pass_turns":59,"phantom_hand_actions":0,"sell_floor_units":178,"hire_premium":3722,"shed_units_now":0,"shed_value_now":0.0,"animals_in_shed_now":0,"seeds_unplanted_now":1,"seeds_unplanted_cost_now":80.0},"agent":"v7.77, mine, the repair"}]}''')
ANIM = json.loads(r'''{"cow_base":[{"t":0,"action":["PASS"],"market":[["BUY_ANIMAL","COW",1],["BUY_PRODUCT","WHEAT",3]],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{},"money":3000.0,"day":0,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3,"COW":1},"money":2521.0,"day":0,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":1,"action":["PICKUP","COW",1],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3,"COW":1},"money":2521.0,"day":0,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":2,"action":["BUILD_PASTURE"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE"}]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":3,"action":["PLACE","COW"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE"}]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":4,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":5,"action":["PICKUP","WHEAT",1],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":6,"action":["FEED"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":7,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":8,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":9,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":10,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":11,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":12,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":13,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":14,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":15,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":16,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":17,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":18,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":19,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":20,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":21,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":22,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":23,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":24,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":25,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":26,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":27,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":28,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":29,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":30,"action":["FEED"],"market":[],"events":[["feed_no_wheat",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":7},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":31,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":8},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":32,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":9},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":33,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":10},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":34,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":11},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":35,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":12},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":36,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":13},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":37,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":14},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":38,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":15},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":39,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":16},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":40,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":17},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":41,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":18},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":42,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":19},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":43,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":20},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":44,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":21},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":45,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":22},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":46,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":23},"ledger":{"feed_no_wheat":1,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":47,"action":["PASS"],"market":[],"events":[["animal_unfed_days",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":0},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":48,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":1},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":49,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":2},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":50,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":3},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":51,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":4},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":52,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":5},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":53,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":6},"ledger":{"feed_no_wheat":1,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":54,"action":["FEED"],"market":[],"events":[["feed_no_wheat",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":7},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":55,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":8},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":56,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":9},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":57,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":10},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":58,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":11},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":59,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":12},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":60,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":13},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":61,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":14},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":62,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":15},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":63,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":16},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":64,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":17},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":65,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":18},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":66,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":19},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":67,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":20},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":68,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":21},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":69,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":22},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":70,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":23},"ledger":{"feed_no_wheat":2,"animal_unfed_days":1,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":71,"action":["PASS"],"market":[],"events":[["animal_unfed_days",1],["animals_escaped",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":1}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":2,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE"}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":3,"hour":0},"ledger":{"feed_no_wheat":2,"animal_unfed_days":2,"animals_escaped":1,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}}],"cow_fix":[{"t":0,"action":["PASS"],"market":[["BUY_ANIMAL","COW",1],["BUY_PRODUCT","WHEAT",3]],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{},"money":3000.0,"day":0,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3,"COW":1},"money":2521.0,"day":0,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":1,"action":["PICKUP","COW",1],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3,"COW":1},"money":2521.0,"day":0,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":2,"action":["BUILD_PASTURE"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE"}]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":3,"action":["PLACE","COW"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE"}]],"farmer":[4,4],"inventory":{"COW":1},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":4,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":5,"action":["PICKUP","WHEAT",1],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":3},"money":2521.0,"day":0,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":6,"action":["FEED"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":7,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":8,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":9,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":10,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":11,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":12,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":13,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":14,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":15,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":16,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":17,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":18,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":19,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":20,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":21,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":22,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":23,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":0,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":24,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":25,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":26,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":27,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":28,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":29,"action":["PICKUP","WHEAT",1],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":2},"money":2521.0,"day":1,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":30,"action":["FEED"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":31,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":32,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":33,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":34,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":35,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":36,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":37,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":38,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":39,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":40,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":41,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":42,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":43,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":44,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":45,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":46,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":47,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":1,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":48,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":49,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":50,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":51,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":52,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":53,"action":["PICKUP","WHEAT",1],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":1},"money":2521.0,"day":2,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{},"money":2521.0,"day":2,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":54,"action":["FEED"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{"WHEAT":1},"shed":{},"money":2521.0,"day":2,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":55,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":56,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":57,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":58,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":59,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":60,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":61,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":62,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":63,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":64,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":65,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":66,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":67,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":68,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":69,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":70,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":71,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":2,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2521.0,"day":3,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}}],"rot":[{"t":0,"action":["PASS"],"market":[["BUY_SEED","WHEAT",1]],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{},"money":3000.0,"day":0,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":1,"action":["PLANT","WHEAT"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":2,"action":["WATER"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":3,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":4,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":5,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":6,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":7,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":8,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":9,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":10,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":11,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":12,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":13,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":14,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":15,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":16,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":17,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":18,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":19,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":20,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":21,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":22,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":23,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":1}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":0,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":24,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":25,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":26,"action":["WATER"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":27,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":28,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":29,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":30,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":31,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":32,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":33,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":34,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":35,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":36,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":37,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":38,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":39,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":40,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":41,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":42,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":43,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":44,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":45,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":46,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":47,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":1,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":48,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":49,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":50,"action":["WATER"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":51,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":52,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":53,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":54,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":55,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":56,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":57,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":58,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":59,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":60,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":61,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":62,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":63,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":64,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":65,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":66,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":67,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":68,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":69,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":70,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":71,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":2,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":72,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":73,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":74,"action":["WATER"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":75,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":76,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":77,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":78,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":79,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":80,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":81,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":82,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":83,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":84,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":85,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":86,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":87,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":88,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":89,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":90,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":91,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":92,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":93,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":94,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":95,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":3,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":96,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":97,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":98,"action":["WATER"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":99,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":100,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":101,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":102,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":103,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":104,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":105,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":106,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":107,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":108,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":109,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":110,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":111,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":112,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":113,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":114,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":115,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":116,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":117,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":118,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":119,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":true,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":4,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":120,"action":["PASS"],"market":[],"events":[["decay_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":4,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":1,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":121,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":1,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":122,"action":["PASS"],"market":[],"events":[["decay_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":3,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":2,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":123,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":2,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":124,"action":["PASS"],"market":[],"events":[["decay_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":2,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":3,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":125,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":3,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":126,"action":["PASS"],"market":[],"events":[["decay_units",1],["plants_decayed",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PLANT","crop":"WHEAT","yield_units":1,"watered_today":false,"consecutive_unwatered":0}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":127,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":128,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":129,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":130,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":131,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":132,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":133,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":134,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":135,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":136,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":137,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":138,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":139,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":140,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":141,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":142,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":143,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":5,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":144,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":0},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":1},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":145,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":1},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":2},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":146,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":2},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":3},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":147,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":3},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":4},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":148,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":4},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":5},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":149,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":5},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":6},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":150,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":6},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":7},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":151,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":7},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":8},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":152,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":8},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":9},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":153,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":9},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":10},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":154,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":10},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":11},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":155,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":11},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":12},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":156,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":12},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":13},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":157,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":13},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":14},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":158,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":14},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":15},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":159,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":15},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":16},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":160,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":16},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":17},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":161,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":17},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":18},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":162,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":18},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":19},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":163,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":19},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":20},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":164,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":20},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":21},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":165,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":21},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":22},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":166,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":22},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":167,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":6,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"WEED"}]],"farmer":[4,4],"inventory":{},"shed":{},"money":2990.0,"day":7,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":4,"plants_decayed":1,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}}],"cap":[{"t":23,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":39},"money":1430.0,"day":0,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":39},"money":1430.0,"day":1,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":47,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":38},"money":1430.0,"day":1,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":38},"money":1430.0,"day":2,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":71,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":37},"money":1430.0,"day":2,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":37},"money":1430.0,"day":3,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":95,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":36},"money":1430.0,"day":3,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":36},"money":1430.0,"day":4,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":119,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":35},"money":1430.0,"day":4,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":35},"money":1430.0,"day":5,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":143,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":34},"money":1430.0,"day":5,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":34},"money":1430.0,"day":6,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":167,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":33},"money":1430.0,"day":6,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":33},"money":1430.0,"day":7,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":191,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,null,null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":0,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":32},"money":1430.0,"day":7,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":1,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":32},"money":1430.0,"day":8,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":215,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":1,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":31},"money":1430.0,"day":8,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":1,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":31},"money":1430.0,"day":9,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":239,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":1,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":30},"money":1430.0,"day":9,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":2,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":30},"money":1430.0,"day":10,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":263,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[null,null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":2,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":29},"money":1430.0,"day":10,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":2,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":29},"money":1430.0,"day":11,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":287,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":2,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":28},"money":1430.0,"day":11,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":3,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":28},"money":1430.0,"day":12,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":311,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":3,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":27},"money":1430.0,"day":12,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":3,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":27},"money":1430.0,"day":13,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":335,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":3,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":26},"money":1430.0,"day":13,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":4,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":26},"money":1430.0,"day":14,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":359,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":4,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":25},"money":1430.0,"day":14,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":4,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":25},"money":1430.0,"day":15,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":383,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":4,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":24},"money":1430.0,"day":15,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":5,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":24},"money":1430.0,"day":16,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":407,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":5,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":23},"money":1430.0,"day":16,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":5,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":23},"money":1430.0,"day":17,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":431,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":5,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":22},"money":1430.0,"day":17,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":22},"money":1430.0,"day":18,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":455,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":21},"money":1430.0,"day":18,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":21},"money":1430.0,"day":19,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":0}},{"t":479,"action":["PASS"],"market":[],"events":[["tile_cap_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":20},"money":1430.0,"day":19,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":20},"money":1430.0,"day":20,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":1}},{"t":503,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":19},"money":1430.0,"day":20,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":19},"money":1430.0,"day":21,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":1}},{"t":527,"action":["PASS"],"market":[],"events":[["tile_cap_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":18},"money":1430.0,"day":21,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":18},"money":1430.0,"day":22,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":2}},{"t":551,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":17},"money":1430.0,"day":22,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":17},"money":1430.0,"day":23,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":2}},{"t":575,"action":["PASS"],"market":[],"events":[["tile_cap_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":16},"money":1430.0,"day":23,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":16},"money":1430.0,"day":24,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":3}},{"t":599,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":15},"money":1430.0,"day":24,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":15},"money":1430.0,"day":25,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":3}},{"t":623,"action":["PASS"],"market":[],"events":[["tile_cap_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":14},"money":1430.0,"day":25,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":14},"money":1430.0,"day":26,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":4}},{"t":647,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":13},"money":1430.0,"day":26,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":13},"money":1430.0,"day":27,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":4}},{"t":671,"action":["PASS"],"market":[],"events":[["tile_cap_units",1]],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":12},"money":1430.0,"day":27,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":12},"money":1430.0,"day":28,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":5}},{"t":695,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":11},"money":1430.0,"day":28,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":false,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":11},"money":1430.0,"day":29,"hour":0},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":5}},{"t":719,"action":["PASS"],"market":[],"events":[],"before":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":10},"money":1430.0,"day":29,"hour":23},"after":{"tiles":[[null,null,null,null,null],[null,null,{"kind":"WEED"},null,null],[null,{"kind":"WEED"},null,null,null],[null,null,null,null,null],[{"kind":"WEED"},null,null,null,{"kind":"PASTURE","animal":"COW","yield_units":6,"fed_today":true,"consecutive_unfed":0}]],"farmer":[4,4],"inventory":{},"shed":{"WHEAT":10},"money":1430.0,"day":29,"hour":23},"ledger":{"feed_no_wheat":0,"animal_unfed_days":0,"animals_escaped":0,"decay_units":0,"plants_decayed":0,"shed_discarded_units":0,"harvest_dead":0,"tile_cap_units":5}}]}''')
import pandas as pd
pd.set_option('display.max_colwidth', 90)
ROWS = CENSUS['rows']
MINE_SET = lambda r: r['mine']
from matplotlib.colors import ListedColormap
# the per-turn strip's classes: a loss wins over anything else on the same turn
CLASS = {'dead_actions': 1, 'feed_no_animal': 1,
         'decay_units': 2,
         'sell_zero_fill': 3, 'buy_zero_fill': 3,
         'feed_no_wheat': 4, 'animal_unfed_days': 4,
         'animals_escaped': 5, 'plants_dry': 5, 'plants_decayed': 5, 'shed_discarded_units': 5}
NAMES = ['nothing', 'action ignored', 'produce rotting', 'market slot burned', 'precursor', 'loss']
PAL = ListedColormap(['#f7f9fa', '#c9d1d6', '#f1c27d', '#8e7cc3', '#e67e22', '#a93226'])
g = lambda r, k: r['per_game'].get(k, 0.0)
print(f"{len(ROWS)} agents, {sum(r['games'] for r in ROWS):,} games, "
      f"rival {CENSUS['rival']}, {CENSUS['worlds']} worlds x 2 seats each")


<a id="s1"></a>
## 1. What the engine does in silence

Four rules, all reasonable, all silent.

* **Animals.** At the nightly refresh an animal that was not fed has its unfed counter raised, and at two the animal is gone. The structure stays.
* **Crops.** A plant that ends two days unwatered becomes a weed, seed and growth included.
* **Ripe produce.** A plant left standing past its last yield day loses one unit every two steps until it too is a weed.
* **The shed.** Production arriving at a shed that already holds a hundred items is discarded.

And a fifth that is not a rule but a consequence: **an action the engine cannot execute is accepted and ignored.** A FEED with nothing to feed, a WATER on bare soil, a HARVEST on an unripe plant, a PLANT with no seed, all return normally.

Then there is what the engine never destroys and the farm still never gets.
An animal offers one unit of fertilizer every morning, and if nobody collects it the offer is simply replaced by tomorrow's.
A cow holds six units of milk and then holds six, whatever else it produces.
A hired hand that is told nothing does nothing and is paid the same.
And whatever is still in the shed when the season ends is worth exactly nothing, because the score is money.
**None of that is a loss the engine inflicts. All of it is value the farm had and did not use**, and it is the larger ledger of the two.

Watch two of them happen.
The boards below are the engine's own state, drawn turn by turn from the north-west quadrant of a farm; the caption under each is what the engine recorded on that turn, and the counters at the right edge are the ledger as it fills.
On the left, the cow from the opening paragraph; on the right, the same tape with one PICKUP added each morning.


In [ ]:
# The board-as-a-GIF idea is Georgy Mamarin's, from his guide 'Kaggriculture, Visualized:
# what every crop pays' (https://www.kaggle.com/code/georgymamarin/kaggriculture-visualized-what-every-crop-pays),
# where a whole season plays on the board with the money race beside it. The drawing below
# is my own and puts the ledger's timeline beside the board; the technique, a FuncAnimation
# written to a GIF and embedded inline, is his, and it renders on Kaggle with nothing else.
import base64, pathlib, tempfile
from matplotlib.animation import FuncAnimation, PillowWriter
from PIL import Image, ImageSequence
from IPython.display import HTML, display

TILE = {'PASTURE': '#c9b48a', 'COOP': '#d9c9a3', 'WEED': '#8a9a6a', 'PLANT': '#7fb069', None: '#efe6d3'}

def draw_board(ax, snap, title, event):
    ax.clear(); ax.set_xlim(-.3, 6.1); ax.set_ylim(-2.2, 5.9); ax.set_aspect('equal'); ax.axis('off'); ax.grid(False)
    ax.add_patch(mpatch.Rectangle((-.15, -.15), 5.3, 5.3, fc='#e3d6bd', ec='none', zorder=0))
    for y in range(5):
        for x in range(5):
            t = snap['tiles'][y][x]
            kind = t.get('kind') if isinstance(t, dict) else None
            ax.add_patch(mpatch.FancyBboxPatch((x + .06, 4 - y + .06), .88, .88, boxstyle='round,pad=0,rounding_size=.14',
                                               fc=TILE.get(kind, '#efe6d3'), ec='white', lw=1, zorder=1))
            if isinstance(t, dict) and t.get('animal'):
                fed = t.get('fed_today'); unfed = t.get('consecutive_unfed', 0)
                c = '#5b3a1e' if fed else '#b03a2e'
                ax.text(x + .5, 4 - y + .72, 'COW', ha='center', va='center', fontsize=10, weight='bold', color=c, zorder=3)
                ax.text(x + .5, 4 - y + .44, 'fed' if fed else f'unfed {unfed}', ha='center', va='center', fontsize=7, color=c, zorder=3)
                if t.get('yield_units'):
                    ax.text(x + .5, 4 - y + .18, f"milk {t['yield_units']}", ha='center', va='center', fontsize=7.5, color='#1b262e', zorder=3)
            elif isinstance(t, dict) and kind == 'PLANT':
                ax.text(x + .5, 4 - y + .62, t.get('crop', '').lower(), ha='center', va='center', fontsize=9, weight='bold', color='#1e4d2b', zorder=3)
                ax.text(x + .5, 4 - y + .3, f"yield {t.get('yield_units', 0)}", ha='center', va='center', fontsize=7.5, color='#1e4d2b', zorder=3)
            elif kind == 'WEED':
                ax.text(x + .5, 4 - y + .5, 'weed', ha='center', va='center', fontsize=8, color='#3d4a2e', zorder=3)
    ax.add_patch(mpatch.FancyBboxPatch((5.08, -1.0), .8, .8, boxstyle='round,pad=0,rounding_size=.14', fc='#6e4b2a', ec='none', zorder=2))
    ax.text(5.48, -.6, 'shed', ha='center', va='center', fontsize=7.5, color='white', zorder=3)
    fx, fy = snap['farmer']
    if fx < 5 and fy < 5:
        ax.add_patch(mpatch.FancyBboxPatch((fx + .02, 4 - fy + .02), .96, .96, boxstyle='round,pad=0,rounding_size=.14',
                                           fc='none', ec='#1a5fb4', lw=3, zorder=4))
    inv = ', '.join(f'{k.lower()} {v}' for k, v in snap['inventory'].items()) or 'nothing'
    shed = ', '.join(f'{k.lower()} {v}' for k, v in snap['shed'].items()) or 'empty'
    ax.set_title(title, fontsize=11, color=INK, loc='left', pad=8)
    ax.text(-.15, 5.55, f"day {snap['day']}, hour {snap['hour']}     in hand: {inv}     shed: {shed}", fontsize=8.5, color='#4b5a63', va='top')
    loud = any(w in event for w in ('nothing', 'GONE', 'rot', 'clipped', 'WEED'))
    ax.text(-.15, -1.25, event, fontsize=10.5, color=WARN if loud else '#4b5a63', va='top', weight='bold' if loud else 'normal')
    ax.text(-.15, -2.1, 'blue frame: the farmer', fontsize=7.5, color='#1a5fb4', va='bottom')

def caption(f):
    ev = {k: n for k, n in f['events']}
    if 'animals_escaped' in ev: return 'night: the cow is GONE, 400 coins with it'
    if 'feed_no_wheat' in ev: return 'FEED with an empty hand: nothing happens'
    if 'plants_decayed' in ev: return 'the last unit rots: the tile is a WEED'
    if 'decay_units' in ev: return 'a unit rots on the plant'
    if 'tile_cap_units' in ev: return 'night: milk clipped at the cap, a unit that will never exist'
    if 'animal_unfed_days' in ev: return 'night: an unfed day is counted'
    if f['market']: return 'market: ' + ', '.join(' '.join(map(str, o)) for o in f['market'])
    a = f['action']
    return 'farmer: ' + ' '.join(map(str, a)) if a != ['PASS'] else ''

def draw_timeline(ax, series, t_now, xlabel):
    """series: list of (label, turns, values, color, style). A step chart with a cursor."""
    ax.clear(); ax.grid(True, alpha=.25)
    top = 1
    for label, ts, vs, color, ls in series:
        ax.step(ts, vs, where='post', color=color, ls=ls, lw=2, label=label)
        top = max(top, max(vs) if vs else 1)
    ax.axvline(t_now, color=INK, lw=1.2)
    ax.set_ylim(-.15, top + .8); ax.set_xlim(min(series[0][1]), max(series[0][1]))
    for d in range(0, 721, 24):
        if min(series[0][1]) <= d <= max(series[0][1]): ax.axvline(d, color='#95a5a6', lw=.6, ls=':')
    ax.set_xlabel(xlabel); ax.set_ylabel('count'); ax.legend(frameon=False, fontsize=8, loc='upper left')
    ax.set_title('the ledger as the game plays', fontsize=10, loc='left', pad=8)

def pick(frames, every, must=()):
    must = set(must) | {f['t'] for f in frames if f['events']}
    return [f for f in frames if f['t'] % every == 0 or f['t'] in must]

def ledger_series(frames, keys, colors, ls='-'):
    ts = [f['t'] for f in frames]
    return [(k.replace('_', ' '), ts, [f['ledger'][k] for f in frames], c, ls) for k, c in zip(keys, colors)]

def gif(boards, series_fn, xlabel, fps=2, figsize=(14, 5), hold_ms=3000):
    """boards: list of (title, sampled frames); series_fn(t) gives the timeline series. One GIF."""
    fig, axes = plt.subplots(1, len(boards) + 1, figsize=figsize,
                             gridspec_kw={'width_ratios': [1] * len(boards) + [1.35], 'wspace': .32})
    n = max(len(fr) for _, fr in boards)
    def frame(i):
        t_now = boards[0][1][min(i, len(boards[0][1]) - 1)]['t']
        for ax, (title, fr) in zip(axes[:-1], boards):
            f = fr[min(i, len(fr) - 1)]
            draw_board(ax, f['after'], title, caption(f))
        draw_timeline(axes[-1], series_fn(), t_now, xlabel)
    anim = FuncAnimation(fig, frame, frames=n)
    path = pathlib.Path(tempfile.mkdtemp()) / 'board.gif'
    anim.save(str(path), writer=PillowWriter(fps=fps), dpi=85)
    plt.close(fig)
    im = Image.open(path); frames_ = [f.copy() for f in ImageSequence.Iterator(im)]
    dur = [int(1000 / fps)] * len(frames_); dur[-1] = hold_ms
    frames_[0].save(str(path), save_all=True, append_images=frames_[1:], duration=dur, loop=0)
    return HTML(f'<img src="data:image/gif;base64,{base64.b64encode(path.read_bytes()).decode()}" style="max-width:100%">')

KEYS = ['feed_no_wheat', 'animal_unfed_days', 'animals_escaped']
must = (0, 1, 2, 3, 5, 6, 29, 30, 53, 54)
cow_series = lambda: (ledger_series(ANIM['cow_base'], KEYS, [WARN, '#95a5a6', '#7b241c'])
                      + [('animals escaped, repaired tape', [f['t'] for f in ANIM['cow_fix']],
                          [f['ledger']['animals_escaped'] for f in ANIM['cow_fix']], GOOD, '--')])
display(gif([('as written', pick(ANIM['cow_base'], 4, must)),
             ('with a PICKUP each morning', pick(ANIM['cow_fix'], 4, must))],
            cow_series, 'turn (24 a day)', fps=2, figsize=(15, 5.6)))


**How to read it.** Two boards, one tape each, and the ledger beside them as the game plays.
The frame advances two turns a second and stops on the turns where something happens, so the whole three days take about twenty seconds; the line under each board is the engine's own event on that turn, red when it is a loss or a warning.

On the left, the cow is fed once at hour 6 of day 0.
At hour 6 of day 1 the hand comes back empty, because inventories are emptied into the shed at midnight, and the red line in the chart steps up: **the warning, at turn 30**.
The grey line counts the unfed nights at hour 23 of days 1 and 2, and at the second one the dark red line steps: **the loss, at turn 71**, forty-one turns after the warning.
The right board runs the same tape with one PICKUP added each morning, and the dashed green line never leaves zero.

The second animation is the rule nobody plans for: **ripe produce does not wait.** A wheat plant, watered every day, ripe from day 2 and never harvested.


In [ ]:
rot = ANIM['rot']
rot_series = lambda: (ledger_series(rot, ['decay_units', 'plants_decayed'], [WARN, '#7b241c'])
                      + [('yield on the plant', [f['t'] for f in rot],
                          [f['after']['tiles'][4][4].get('yield_units', 0) if isinstance(f['after']['tiles'][4][4], dict) else 0 for f in rot],
                          GOOD, '-')])
shown = [f for f in rot if 44 <= f['t'] <= 130]         # from the second watering to the weed
display(gif([('wheat: watered daily, never harvested', pick(shown, 6))],
            rot_series, 'turn (24 a day)', fps=2, figsize=(12, 5.6)))
cap = ANIM['cap']
cap_series = lambda: ([('milk held on the cow', [f['t'] for f in cap],
                        [f['after']['tiles'][4][4].get('yield_units', 0) if isinstance(f['after']['tiles'][4][4], dict) else 0 for f in cap],
                        GOOD, '-')] + ledger_series(cap, ['tile_cap_units'], [WARN]))
display(gif([('a cow fed daily, never milked: one frame a night', cap)],
            cap_series, 'turn (24 a day)', fps=2, figsize=(12, 5.6)))


**How to read the two.** In the first, the green line is the yield sitting on the plant: it climbs to four as the plant is watered on days 2, 3 and 4, holds through day 4 while nobody harvests, and from turn 120 the red line takes it down a unit every two turns until the tile is a weed.
The yield counter on the tile is the only sign the engine gives.
In the second, one frame per night for a whole season, the green line is the milk held on the cow: it climbs to six by day 18 and then holds, and from then on every second night the red line steps, a unit that would have accrued and never will.

The cow from the first animation, as code, in eight actions.
The tape is the scenario; the ledger underneath is what the engine reports about it, turn by turn, and then the same scenario with one line added.


In [ ]:
def show(title, actions, events):
    print(title)
    for t, a in sorted(actions.items(), key=lambda kv: int(kv[0])):
        ops = [a['farmer']] + a['hands']
        mk = ', '.join(' '.join(map(str, o)) for o in a['market'])
        print(f"  turn {int(t):>3}  farmer {' '.join(map(str, a['farmer'])):<16} market {mk}")
    print('  ledger events:')
    if not events:
        print('    none: the counters stay at zero for 72 turns')
    for t, what, n in events:
        print(f"    turn {t:>3} (day {t // 24})  {what} +{n:g}")
    print()

show('AS WRITTEN: buy a cow and three wheat, place it, feed it, FEED again each morning',
     COW['base'], COW['base_events'])
show('WITH ONE LINE ADDED: PICKUP WHEAT before each morning FEED', COW['fix'], COW['fix_events'])


The first version fails on day 1 and the ledger says so at turn 30, forty-one turns before the cow leaves at turn 71.
`feed_no_wheat` is the hand arriving with nothing; `animal_unfed_days` is the engine's own countdown; `animals_escaped` with `escape_capital` is the loss and its price.

The second version is the same tape with a PICKUP each morning, and every counter stays at zero.
**The bug is not that the agent forgot to feed.** It fed, every day. The bug is that inventories are emptied into the shed at midnight, so wheat has to be picked up again each morning, and nothing in the observation tells a tape that its hand is empty.


<a id="s2"></a>
## 2. Where the counters live

The engine is a C++ port of the competition environment at 1.32.7, bit-exact against it, playing an episode in well under a millisecond.
How it earns the word bit-exact, and the 24,000 episodes a second it reaches, is a separate page: [From 1 to 24k episodes a second](https://www.kaggle.com/code/destbreso/from-1-to-24k-episodes-a-second).

It is driven two ways.
**L0** takes fixed action streams and runs thousands of episodes a second with the interpreter lock released, which is what makes a twelve-seed panel cheap.
**L1** hands a live Python agent the same observation dictionary the real interpreter builds, turn by turn, so an agent runs unmodified.

The telemetry is orthogonal to both.
**Each stage of a turn writes the losses it causes into a block of per-farm counters, and `Game.telemetry(player)` reads the block at any moment.**
The counters are write-only from the engine's point of view: nothing in the simulation ever reads one, which is why adding them changed no result and why the golden traces still reproduce banks to the coin.

The diagram is drawn from the source layout.
The four boxes in the middle are the four functions a turn calls, in order, and each is connected to the counters it writes.


In [ ]:
STAGES = [
    ('apply_unit_actions', 'farmer and hands: move, plant, water,\nharvest, feed, build, pick up, drop',
     'every op that did nothing: feed_no_wheat, water_dead,\nharvest_dead, plant_dead, pickup_dead, place_dead...\nhand_pass_turns, phantom_hand_actions, tile_cap_units'),
    ('process_market', 'ten orders a turn, settled unit by unit,\nslot by slot, player 0 first',
     'refused_buy_*, refused_hire, refused_buy_land,\nsell_zero_fill, buy_zero_fill,\nsell_floor_units, hire_premium'),
    ('town_consume + decay_plants', 'shops draw down stock, prices refresh,\na standing crop loses a unit every two steps',
     'decay_units, plants_decayed'),
    ('end_of_day', 'refresh plants and animals, spawn weeds,\ndrop inventories into the shed',
     'animals_escaped, escape_capital, plants_dry, dry_seed_cost,\nshed_discarded_units, fertilizer_forgone, tile_cap_units,\ncare_unfed, animal_shed_days, idle_tile_days')]

fig, ax = plt.subplots(figsize=(14, 8.4))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off'); ax.grid(False)

def box(x, y, w, h, title, body='', fc='#ffffff', ec=INK, lw=1.2, ls='-', fs=11, bs=9.2, tc=None):
    ax.add_patch(mpatch.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=1.4',
                                       fc=fc, ec=ec, lw=lw, ls=ls, zorder=2))
    ax.text(x + w / 2, y + h - 2.4, title, ha='center', va='top', fontsize=fs,
            weight='bold', color=tc or ec, zorder=3)
    if body:
        ax.text(x + w / 2, y + (h - 5) / 2, body, ha='center', va='center', fontsize=bs,
                color='#4b5a63', zorder=3, linespacing=1.4)

def arrow(x1, y1, x2, y2, color=INK, lw=1.3, rad=0):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1), zorder=4,
                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw,
                                connectionstyle=f'arc3,rad={rad}', shrinkA=1, shrinkB=1))

ax.text(12, 98, 'HOW YOU DRIVE IT', fontsize=10.5, weight='bold', color='#6b7b85', ha='center')
box(1, 78, 22, 15, 'L0  batch', 'run_episode / run_many\nfixed action streams\n24,442 episodes a second', fc='#f4f7f9')
box(1, 60, 22, 15, 'L1  step', 'Game(seed)\nobserve(seat) / step(a, b)\nlive agents, unmodified', fc='#f4f7f9')
box(1, 43, 22, 14, 'L2  tensors', 'not built: the schema\nbelongs to its first consumer',
    fc='#ffffff', ec='#aab7bf', ls='--', tc='#8b99a1')

ax.add_patch(mpatch.FancyBboxPatch((27, 28), 41, 66, boxstyle='round,pad=0,rounding_size=1.8',
                                   fc='#fbfcfd', ec=INK, lw=1.8, zorder=1))
ax.text(47.5, 98.4, 'Sim::step', ha='center', va='top', fontsize=12.5, weight='bold', color=INK)
ax.text(47.5, 95.6, 'one turn, zero heap allocation', ha='center', va='top', fontsize=9, color='#6b7b85')
for i, (name, body, _) in enumerate(STAGES):
    y = 77 - i * 15.6
    box(29, y, 37, 13.6, name, body, fc='#eef3f6', fs=10.5, bs=8.8)
    if i:
        arrow(47.5, y + 15.6, y_ := y + 13.9, y_, color='#7f8c8d', lw=1.5) if False else arrow(47.5, y + 15.6, 47.5, y + 13.9, color='#7f8c8d', lw=1.5)
arrow(23, 85, 28.6, 83, rad=-.1)
arrow(23, 67, 28.6, 77, rad=.1)

box(72, 75, 27, 18, 'State', 'farms[2]: tiles, shed, seeds, money,\nhands, quadrants, hires today\nmarket, shops, step, day', fc='#f4f7f9')
arrow(68, 86, 71.6, 86)
arrow(71.6, 81, 68, 81, rad=.35)
ax.text(85.5, 72.4, 'trivially copyable: a search can fork an episode', ha='center', fontsize=8.6,
        color='#6b7b85', style='italic')

ax.add_patch(mpatch.FancyBboxPatch((72, 13), 27, 56, boxstyle='round,pad=0,rounding_size=1.8',
                                   fc='#fdf6ec', ec=MINE, lw=1.8, zorder=1))
ax.text(85.5, 67, 'TELEMETRY', ha='center', va='top', fontsize=12, weight='bold', color=MINE)
ax.text(85.5, 63.3, 'per-farm counters, written where the\nengine acts and never read back',
        ha='center', va='top', fontsize=8.8, color='#8a6d30', linespacing=1.35)
for i, (_, _, counters) in enumerate(STAGES):
    ys, yc = 77 - i * 15.6, 54.5 - i * 11.6
    ax.text(85.5, yc, counters, ha='center', va='center', fontsize=7.6, color='#6b5a31', linespacing=1.45)
    arrow(66.2, ys + 6.8, 71.6, yc, color=MINE, lw=1.1, rad=-.15)
box(72, 2, 27, 8.6, 'Game.telemetry(player)', 'one dict, read at any turn', fc='#fdf6ec', ec=MINE, fs=10, bs=8.8)
arrow(85.5, 12.6, 85.5, 10.9, color=MINE)

box(1, 2, 67, 23, 'Bit-exact against kaggle_environments 1.32.7',
    "CPython's Mersenne Twister reproduced exactly, dict insertion order where the 100-item shed cap\n"
    'decides which goods survive, slot-by-slot market resolution, and the 1.32.7 hinge repricing of\n'
    'carrot, tomato and egg. Golden traces reproduce banks to the coin and 6 of 6 are observation-exact\n'
    'over 10,066 field blocks each. Five recorded ladder episodes reproduce to the coin in section 11.\n'
    'Adding the counters changed none of it, which is what write-only buys.',
    fc='#f1f7f2', ec=GOOD, fs=11, bs=9)
fig.tight_layout(); plt.show()


**How to read it.** Follow one turn from the left: an L0 stream or an L1 agent hands `Sim::step` an action, the four stages run top to bottom against the state on the right, and each stage drops its counters into the amber block as a side effect.
Nothing flows back from the amber block, which is the whole guarantee: the ledger can be read at every turn and it cannot alter the game.

The four books, with what every line means.
The split between a **loss** and its **precursor** is the part of the first book that makes it worth reading.

One decision in the listing deserves its reason: **a PASS is not a dead action.** The engine did exactly what was asked, which was nothing, and the agent knows it asked for nothing, so there is nothing silent about it.
It is in the efficiency book instead, because an idle unit is a turn of labour not used, and it is split in two: a hired hand costs a day's wage and its idle turns are wage paid for nothing, while the farmer is free and its idle turns are only opportunity.
A hand with no order at all counts the same as one told to PASS, since the engine treats them alike.
`animals_escaped` tells you an animal died, which is a post mortem.
`feed_no_wheat` tells you a hand reached an animal and had nothing to give it, and that fires while the animal is still alive.


In [ ]:
LEDGER = [
    ('1 losses', 'loss', 'animals_escaped', 'animals gone at two consecutive unfed days'),
    ('1 losses', 'loss', 'escape_capital', 'their purchase price, in coins'),
    ('1 losses', 'loss', 'escape_yield_units', 'product held on the tile and lost with it'),
    ('1 losses', 'loss', 'plants_dry', 'plants turned to weed by two dry days'),
    ('1 losses', 'loss', 'dry_seed_cost', 'their seed price, in coins'),
    ('1 losses', 'loss', 'plants_decayed', 'plants that rotted all the way to weed'),
    ('1 losses', 'loss', 'decay_units', 'yield units rotted off a plant left standing'),
    ('1 losses', 'loss', 'shed_discarded_units', 'production the hundred-item cap destroyed'),
    ('1 losses', 'precursor', 'feed_no_wheat', 'FEED on a hungry animal, carrier held no wheat'),
    ('1 losses', 'precursor', 'animal_unfed_days', 'animal-days ending unfed: the runway to a loss'),
    ('1 losses', 'precursor', 'plant_unwatered_days', 'plant-days ending unwatered'),
    ('2 ignored', 'farm action', 'feed_no_animal', 'FEED at a tile whose animal already left'),
    ('2 ignored', 'farm action', 'feed_redundant', 'FEED on an animal already fed today'),
    ('2 ignored', 'farm action', 'water_dead', 'WATER on a non-plant or an already-watered tile'),
    ('2 ignored', 'farm action', 'harvest_dead', 'HARVEST that took nothing'),
    ('2 ignored', 'farm action', 'plant_dead', 'PLANT dropped for want of seed, or a busy tile'),
    ('2 ignored', 'farm action', 'collect_dead, care_dead, fertilize_dead', 'COLLECT, CARE or FERTILIZE with nothing there'),
    ('2 ignored', 'farm action', 'pickup_dead, drop_dead, place_dead', 'shed operations away from the shed, or of nothing'),
    ('2 ignored', 'farm action', 'build_dead, dig_dead, move_dead, locked_dead', 'BUILD on a busy tile, DIG on an empty one, a step off the board or onto locked land'),
    ('2 ignored', 'farm action', 'phantom_hand_actions', 'orders to hands the farm does not have'),
    ('2 ignored', 'market order', 'sell_zero_fill', 'a SELL order that delivered nothing: one of ten slots burned'),
    ('2 ignored', 'market order', 'buy_zero_fill', 'a BUY order that bought nothing: usually a broken financing chain'),
    ('2 ignored', 'market order', 'refused_buy_*, refused_hire, refused_buy_land', 'units and orders refused on funds or capacity'),
    ('3 efficiency', 'unused', 'fertilizer_forgone', 'animal-days whose fertilizer was on offer and never collected'),
    ('3 efficiency', 'unused', 'tile_cap_units', 'yield clipped at a tile holding cap: not harvested in time'),
    ('3 efficiency', 'unused', 'sell_floor_units', 'units sold at the price floor of one coin'),
    ('3 efficiency', 'unused', 'hand_pass_turns', 'hired hands that did nothing this turn'),
    ('3 efficiency', 'unused', 'farmer_pass_turns', 'the farmer standing still, kept apart: the farmer is free'),
    ('3 efficiency', 'unused', 'idle_tile_days', 'unlocked tiles empty or weed at the end of a day'),
    ('3 efficiency', 'unused', 'weed_tile_days', 'of those, weeds nobody dug'),
    ('3 efficiency', 'unused', 'animal_shed_days', 'animals still in the shed at the end of a day'),
    ('3 efficiency', 'unused', 'care_unfed', 'CARE without FEED the same day: the bonus never accrues'),
    ('3 efficiency', 'cost structure', 'hire_premium', 'the same-day hire ladder above one coin a hand: the price of labour, since hands leave at midnight'),
    ('4 stranded', 'at the end', 'shed_units_now', 'units in the shed when the game ended: worth nothing'),
    ('4 stranded', 'at the end', 'shed_value_now', 'what they would have fetched at the final prices'),
    ('4 stranded', 'at the end', 'animals_in_shed_now', 'animals bought and never placed'),
    ('4 stranded', 'at the end', 'seeds_unplanted_cost_now', 'coins of seed never planted'),
]
pd.DataFrame(LEDGER, columns=['book', 'kind', 'counter', 'what it counts']).set_index(['book', 'kind', 'counter'])


<a id="s3"></a>
## 3. What the report looks like

One command, one agent, one rival, sixteen seeds from both seats.
The output below is verbatim, and the agent under test is **my own submission, live on the ladder at the moment it ran**.

```
python tools/silent_leaks.py --agent my_agent.py --rival rival.py \
       --seeds 1-16 --both-seats --require-zero-escapes
```


In [ ]:
REPORT = "v7.76_learned against yhay81-shop-router-0913: 32 games, playing...\n\nv7.76_learned against yhay81-shop-router-0913: 32 games\n\nescapes: 74 (engine), 74 (observation control) AGREE\ngames losing an animal: 32 of 32, first on day 1\n  seed     3 seat 0  4 animal(s) from day 1, 1,800 coins, feed_no_wheat 8, bank 83,572 margin -52,602\n  seed     3 seat 1  4 animal(s) from day 1, 1,800 coins, feed_no_wheat 8, bank 83,572 margin -52,602\n  seed     1 seat 0  4 animal(s) from day 1, 1,700 coins, feed_no_wheat 2, bank 60,312 margin -44,024\n  seed     1 seat 1  4 animal(s) from day 1, 1,700 coins, feed_no_wheat 2, bank 60,312 margin -44,024\n  seed     2 seat 0  3 animal(s) from day 1, 1,200 coins, feed_no_wheat 2, bank 96,166 margin -9,523\n  seed     2 seat 1  3 animal(s) from day 1, 1,200 coins, feed_no_wheat 2, bank 96,166 margin -9,523\n  seed     4 seat 0  2 animal(s) from day 1, 800 coins, feed_no_wheat 5, bank 88,265 margin -29,971\n  seed     4 seat 1  2 animal(s) from day 1, 800 coins, feed_no_wheat 5, bank 88,265 margin -29,971\n\nPER GAME, mean:\n\n  LOSS\n    animals_escaped              2.31   animals gone at two consecutive unfed days\n    escape_capital             943.75   their purchase price, in coins\n    escape_yield_units           0.00   product held on the tile and lost with it\n    plants_dry                   2.06   plants turned to weed by two dry days\n    dry_seed_cost              118.12   their seed price, in coins\n    plants_decayed              14.62   plants that rotted all the way to weed\n    decay_units                 14.62   yield units rotted off a plant left standing\n    shed_discarded_units         1.50   production the hundred-item cap destroyed\n\n  PRECURSOR\n    feed_no_wheat                4.81   FEED on a hungry animal, carrier held no wheat\n    animal_unfed_days           25.81   animal-days ending unfed: the runway to a loss\n    plant_unwatered_days       350.56   plant-days ending unwatered\n\n  DEAD ACTION\n    feed_no_animal             125.19   FEED at a tile whose animal already left\n    feed_redundant               0.00   FEED on an animal already fed today\n    water_dead                  50.19   WATER on a non-plant or an already-watered tile\n    harvest_dead                69.62   HARVEST that took nothing\n    plant_dead                   5.00   PLANT dropped for want of seed, or a busy tile\n    pickup_dead                 12.00   PICKUP away from the shed, or of what it lacks\n    place_dead                  19.53   PLACE with nothing to place, or nowhere to put it\n    build_dead                   0.00   BUILD on a tile that is not empty\n    dig_dead                     4.31   DIG on an empty tile, or on an animal\n    fertilize_dead              18.72   FERTILIZE on a non-plant, or with none carried\n    collect_dead               135.69   COLLECT_FERTILIZER with none available\n    care_dead                  152.62   CARE on no animal, or twice in a day\n    move_dead                    0.00   a move off the board\n    locked_dead                  0.00   a tile action on locked land\n    phantom_hand_actions         0.00   orders to hands the farm does not have\n\n  EFFICIENCY\n    fertilizer_forgone          12.75   animal-days whose fertilizer was on offer and never collected\n    tile_cap_units              17.06   yield clipped at a tile's holding cap: not harvested in time\n    hand_pass_turns            388.34   hired hands that did nothing this turn\n    farmer_pass_turns           61.19   the farmer's PASS turns, apart: the farmer is free, a hand costs a day\n    idle_tile_days             144.62   unlocked tiles empty or weed at the end of a day\n    weed_tile_days               4.53   of those, weeds nobody dug\n    animal_shed_days             0.88   animals still in the shed at the end of a day\n    care_unfed                  20.19   CARE without FEED the same day: the bonus never accrues\n    sell_floor_units            97.44   units sold at the price floor of 1\n\n  MARKET\n    hire_premium             4,027.62   the same-day hire ladder above one coin a hand: the price of labour, hands leave at midnight\n\n  STRANDED AT THE END\n    shed_units_now               0.00   units still in the shed when the game ended: worth nothing\n    shed_value_now               0.00   what they would have fetched at the final prices\n    animals_in_shed_now          0.00   animals bought and never placed\n    seeds_unplanted_cost_now      86.25   coins of seed never planted\n\n  MARKET\n    sell_zero_fill             183.88   a SELL order that delivered nothing, one slot burned\n    buy_zero_fill               10.81   a BUY order that bought nothing: usually a broken financing chain\n    refused_buy_product        124.88   BUY_PRODUCT units refused on funds or capacity\n    refused_buy_seed             4.88   BUY_SEED units refused on funds\n    refused_buy_animal           3.38   BUY_ANIMAL units refused on funds or capacity\n    refused_hire                 0.19   HIREs refused on funds or the unit cap\n    refused_buy_land             0.00   BUY_LANDs refused on funds, or none left\n\n  SILENT LOSS, coins          1,062   units        17.0\n\nFAIL: 74 animal(s) escaped, 30,200 coins\n"
print(REPORT)


**How to read it, block by block.**

The first line after the header is the instrument checking itself: `escapes: 74 (engine), 74 (observation control) AGREE`.
The same quantity is computed twice, once by the engine at the site where it removes the animal, and once from outside by reading the observation at hour 23 for an animal that already has one unfed day and was not fed today.
The two routes share no code.
When they disagree the tool says so and reports neither, because one of them is wrong and nothing in the disagreement says which.

`games losing an animal: 32 of 32, first on day 1` is the finding, and the eight rows under it are the worst games with their price: **four animals gone on day 1 for 1,800 coins, and the game lost by 52,602.**

`PER GAME, mean` is the ledger itself, the four books of section 2 as the tool groups them: LOSS and PRECURSOR are the first book, DEAD ACTION and MARKET the second, EFFICIENCY the third, STRANDED the fourth.
Read LOSS for what it cost, PRECURSOR for whether it could have been seen coming, DEAD ACTION for turns thrown away, MARKET for slots and financing, EFFICIENCY for what the farm had and did not use.
Here `feed_no_animal` at 125 a game is the tape feeding a pasture its cow left on day 1, for the rest of the season.

Read EFFICIENCY last and read it against the bank: hands paid for 388 turns of standing still is not a defect, it is a tape that hires more labour than it schedules, and the number says how much labour was available for whatever the tape left undone.

The exit code is 1 because of `--require-zero-escapes`, which is how the tool sits in a pre-submission check: it is a gate, not a report, when you ask it to be.

The control is eight lines and it is the whole reason the number is admissible.


In [ ]:
CONTROL_CODE = '''
def escapes_from_the_observation(game, seat, step):
    """An animal with one unfed day that was not fed today leaves tonight.
    Read at hour 23, before the nightly refresh runs."""
    if step % 24 != 23:
        return 0
    farm = game.observe(seat)["farms"][seat]
    return sum(1 for row in farm["tiles"] for tile in row
               if isinstance(tile, dict) and "animal" in tile
               and tile["consecutive_unfed"] >= 1 and not tile["fed_today"])
'''
print(CONTROL_CODE)


<a id="s4"></a>
## 4. The warning arrives before the loss

A count per game says an agent is bleeding.
It does not say whether a fix could have acted in time, and that is what decides whether an instrument is useful or merely damning.

One episode first, turn by turn, with every counter that moved.
The defective agent is on the left and its repair on the right, same seed, same rival.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6), sharey=True)
MARKS = {'animal_unfed_days': ('o', '#95a5a6', 'animal-day ending unfed'),
         'feed_no_wheat': ('v', WARN, 'FEED with no wheat carried'),
         'animals_escaped': ('X', '#7b241c', 'animal gone'),
         'feed_no_animal': ('.', '#bdc3c7', 'FEED at an empty tile')}
ORDER = list(MARKS)
for ax, key, ttl in zip(axes, ('defective', 'repaired'), ('before: 3 animals lost', 'after: none lost')):
    side = TRACE[key]
    for ev in side['events']:
        m, c, _ = MARKS[ev['what']]
        ax.scatter([ev['turn']], [ORDER.index(ev['what'])], marker=m, s=48 if m != '.' else 12,
                   color=c, zorder=3, linewidths=0)
    ax.set_ylim(-.5, 4.15); ax.set_yticks(range(len(ORDER)))
    ax.set_yticklabels([MARKS[k][2] for k in ORDER])
    ax.set_xlim(0, 720); ax.set_xlabel('turn (24 per day)')
    ax.set_title(f"{side['agent']}  {ttl}\nmargin {side['bank'] - side['rival_bank']:+,.0f}", fontsize=9.5)
    for d in (1, 8):
        ax.axvline(d * 24, color=INK, lw=.6, ls=':')
esc = min(e['turn'] for e in TRACE['defective']['events'] if e['what'] == 'animals_escaped')
warn = min(e['turn'] for e in TRACE['defective']['events'] if e['what'] == 'feed_no_wheat')
axes[0].axvspan(warn, esc, color=WARN, alpha=.18, zorder=0)
axes[0].annotate(f'warned at turn {warn}, lost at turn {esc}:\n{esc - warn} turns to buy one unit of wheat',
                 xy=(esc, 2.15), xytext=(215, 3.75), fontsize=8.5, color=WARN, va='center',
                 arrowprops=dict(arrowstyle='->', color=WARN, lw=.8, connectionstyle='arc3,rad=-.2'))
axes[1].annotate('the repair: the funding guard no longer\nsells the wheat the herd eats',
                 xy=(360, 3.75), ha='center', va='center', fontsize=8.5, color=GOOD)
fig.suptitle(f"one episode, seed {TRACE['seed']}, against {TRACE['rival']}", fontsize=10)
fig.tight_layout(); plt.show()
print(f'first warning at turn {warn}, first loss at turn {esc}: {esc - warn} turns of notice')


**How to read it.** Time runs left to right across the thirty days; each row is one counter and each mark is one turn on which it moved.
The red band on the left panel is the notice: the first FEED that found no wheat at turn 27, the first animals gone at turn 47.

The grey dust along the top row of the left panel is the detail I find most useful.
Those are FEED actions issued at a tile whose animal left on day 1, and they continue for the remaining twenty-eight days.
**The agent spends the rest of the season feeding a ghost**, and every one of those turns is a hand that could have carried something.

One episode is an anecdote.
The question it raises is general: across every agent that loses animals, how much notice does the precursor give?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7), gridspec_kw={'width_ratios': [1.35, 1]})
leads = np.array(sorted(t for r in LEAD['rows'] for t in r['lead_turns']))
unfed = np.array(sorted(t for r in LEAD['rows'] for t in r['unfed_lead_turns']))
ax = axes[0]
for v, color, label in ((unfed, '#95a5a6', 'an animal-day ended unfed'),
                        (leads, WARN, 'a FEED found no wheat to give')):
    y = 100 * (1 - np.arange(len(v)) / len(v))
    ax.step(v, y, where='post', color=color, lw=1.7, label=label)
ax.axvline(24, color=INK, lw=.9, ls=':')
ax.text(26, 92, 'one game day', fontsize=7.5, color=INK)
med = np.median(leads)
ax.plot([med], [50], marker='o', ms=5, color=WARN, zorder=5)
ax.annotate(f'half the games give\n{med:.0f} turns or more', xy=(med, 50), xytext=(62, 84),
            fontsize=8, color=WARN, arrowprops=dict(arrowstyle='->', color=WARN, lw=.8))
ax.set_xscale('symlog', linthresh=10); ax.set_xlim(0, 520); ax.set_ylim(0, 100)
ax.set_xlabel('turns of notice before the first animal is lost (log scale past 10)')
ax.set_ylabel('share of games with at least this much notice, percent')
ax.legend(frameon=False, fontsize=8, loc='lower left')
ax.set_title(f'the notice you get, {len(leads):,} games that warned first', fontsize=9.5)
ax = axes[1]
frac = [r['games_warned_first'] / r['games_with_a_loss'] for r in LEAD['rows'] if r['games_with_a_loss']]
ax.hist(frac, bins=np.linspace(0, 1, 21), color=INK, alpha=.85)
ax.set_xlabel("share of the agent's losing games that warned first")
ax.set_ylabel('agents')
ax.set_title(f'{sum(1 for f in frac if f > .99)} of {len(frac)} agents are warned\nin every game they lose an animal', fontsize=9.5)
fig.tight_layout(); plt.show()
for th in (12, 24, 48):
    print(f'{100 * (leads >= th).mean():5.1f} percent of those games give at least {th} turns of notice')


**How to read it.** The left panel is a survival curve of the notice: at any point on the x-axis, the height is the share of games that got at least that many turns between the first warning and the first animal lost.
The x-axis is logarithmic past ten turns so that the five-turn and the four-hundred-turn cases fit on one line.

The two precursors have different characters, and the curves show it.
The grey one, an animal-day ending unfed, is **the engine's own countdown**: an animal needs two unfed nights to leave, so the counter fires exactly one day early, every time, which is the vertical step at 24.
It also fires for every deliberate skip, including the last week of the season, so as an alarm it is loud.
The red one, a FEED that found nothing, is **specific**: the agent intended to feed and could not, which is the signature of a supply defect and not of a strategy.
Its notice varies with where in the day the hand arrives, from five turns to most of a week, with a median of twenty.

The right panel is the stronger claim.
For **fifty of the eighty-one agents** that lose animals at all, the specific warning fires first in every single losing game.
For those agents the loss is not a surprise, it is a signal nothing was listening for.

What twenty turns buys: the fix needs the wheat in the hand's inventory, so the notice has to cover a purchase, a pickup at the shed and the walk to the animal.
Twenty turns does. Five may not.


<a id="s5"></a>
## 5. Units and orders are different questions

Two market counters measure a habit rather than a loss, and it is worth separating them before anyone acts on a number.

`sell_dead_units` is the SELL volume the shed could not cover.
The ordinary way to say "sell everything I have" in this environment is to ask for a thousand units and let the settle fill whatever exists, so a healthy agent books tens of thousands of undelivered units by design.
**The quantity measures the idiom.**
The refused-purchase units carry the same ambiguity: "buy as much wheat as I can afford" asks for a large quantity on purpose.

What measures something is an order that delivered **nothing at all**, because that one burned a slot, and an agent may place ten market orders in a turn and no more.
For purchases the empty order usually means the financing chain broke: the purchase this turn depended on a sale that did not happen.
The figure is both sell counters across the field, four orders of magnitude apart.


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.3))
dead = np.array([g(r, 'sell_dead_units') for r in ROWS])
zero = np.array([g(r, 'sell_zero_fill') for r in ROWS])
o = np.argsort(dead)
ax.fill_between(range(len(ROWS)), dead[o], color='#d5dbdb', label='undelivered SELL units (the idiom)')
ax.plot(range(len(ROWS)), zero[o], color=WARN, lw=1.5, label='SELL orders that filled nothing (the measurement)')
ax.set_yscale('symlog'); ax.set_xlabel('agents, ordered by undelivered units')
ax.set_ylabel('per game (log scale)'); ax.legend(frameon=False, fontsize=8)
ax.set_title('a counter can be true and still measure the wrong thing', fontsize=10)
fig.tight_layout(); plt.show()
print(f'median undelivered units {np.median(dead):,.0f} a game; median orders that filled nothing {np.median(zero):,.0f}')


**How to read it.** Agents are sorted left to right by their undelivered sell volume, and the y-axis is logarithmic.
The grey area climbs to a million units a game at the right edge, which is an agent asking for a thousand of everything every turn; the red line, the orders that actually filled nothing, stays between ten and two hundred for almost everybody.
A reader who took the grey number as a loss would rank the field upside down.

The two real ladder games in section 11 make the same point for purchases: one game shows **1,734 refused units and not one empty order**, which is the idiom, and another shows **24 refused units and fifteen empty orders**, which is a financing chain that broke fifteen times.


<a id="s6"></a>
## 6. The field

Every agent below plays the same sixteen shop worlds from both seats against the same fixed public rival, so what varies across the rows is the agent and not the conditions.

The community agents are held byte-identical to their published notebooks, with author and licence recorded, in a dataset I keep updated: [Kaggriculture donor agents](https://www.kaggle.com/datasets/destbreso/kaggriculture-donor-agents-20260902).
Attach it to a notebook and every row here is one command away from being re-run.

The first figure is **when animals leave**, summed over the whole field.


In [ ]:
days = {}
for r in ROWS:
    for d, k in r['escapes_by_day'].items():
        days[int(d)] = days.get(int(d), 0) + k
fig, ax = plt.subplots(figsize=(8, 3.1))
xs = sorted(days)
ax.bar(xs, [days[d] for d in xs], color=[WARN if d in (1, 8) else INK for d in xs], width=.72)
for d in (1, 8):
    if d in days:
        ax.annotate(f'day {d}\n{days[d]}', xy=(d, days[d]), xytext=(d, days[d] * 1.06), ha='center', fontsize=8, color=WARN)
ax.axvspan(24.5, 29, color='#95a5a6', alpha=.16, zorder=0)
ax.annotate('the last week: an animal fed here\nreturns almost nothing, so this is a trade',
            xy=(26.8, max(days.values()) * .72), ha='center', fontsize=7.5, color='#5d6d7e')
ax.set_xlabel('game day the animal left'); ax.set_ylabel('animals, whole field')
ax.set_title(f'{sum(days.values()):,} animals lost across {len(ROWS)} agents', fontsize=10)
fig.tight_layout(); plt.show()


**How to read it.** Three clusters, and they are three different things.

The tallest bar is the **last day of the season** and it is not a defect.
An animal fed on day 28 returns almost nothing, so an agent that stops feeding at the end is making a trade.
A nonzero counter is a question, never a verdict.

The two red bars are the defects.
**Day 1** is the opening paragraph of this page: animals bought on day 0 whose feed never reached a hand on day 1, because the inventory was emptied overnight or because the cash for the wheat went elsewhere.
**Day 8** follows the first wheat stock-out: the opening purchase has been eaten and the first harvest has not yet replaced it.
An animal lost on either day takes a whole season of production with it and was bought with a fifth of the opening cash.
That is the loss worth an instrument, and from outside the farm there is no way to see it happen.

The next figure asks the same question of everything the farm grows rather than of the herd.


In [ ]:
CH = [('decay_units', 'rotted where it grew', '#c0392b'),
      ('shed_discarded_units', 'destroyed by the shed cap', '#e08e45'),
      ('escape_yield_units', 'left with a starved animal', '#7d3c98'),
      ('dry_yield_units', 'lost with a dried-out crop', '#b7791f')]
V = np.array([[g(r, k) for k, _, _ in CH] for r in ROWS], dtype=float)
order = np.argsort(-V.sum(1))
fig, ax = plt.subplots(figsize=(10.4, 3.9))
bottom = np.zeros(len(ROWS))
for i, (_, label, color) in enumerate(CH):
    ax.bar(range(len(ROWS)), V[order, i], bottom=bottom, width=1.0, color=color, label=label, linewidth=0)
    bottom += V[order, i]
ax.set_xlim(-.5, len(ROWS) - .5)
ax.set_xlabel('agents, ordered by total unsold production'); ax.set_ylabel('units a game')
ax.legend(frameon=False, fontsize=8, ncol=4, loc='upper right')
ax.set_title('produce that was grown, paid for, and never reached the market', fontsize=10)
ax.grid(axis='x', alpha=0)
fig.tight_layout(); plt.show()
lost, sold = V.sum(1), np.array([g(r, 'sold_units') for r in ROWS])
share = 100 * lost / np.maximum(sold + lost, 1)
print(f'median agent: {np.median(lost):.1f} units a game never sold against {np.median(sold):.0f} sold, '
      f'so {np.median(share):.1f} percent of what it accounted for; the worst agent leaves {lost.max():.0f} units a game')
print(f'rot is {100 * V[:, 0].sum() / V.sum():.0f} percent of all of it, the shed cap {100 * V[:, 1].sum() / V.sum():.0f} percent')


**How to read it.** Each bar is one agent, stacked by the channel through which its produce was lost, in absolute units a game.
Two channels account for all of it: **rot on the tile, two thirds, and the shed cap, one third.**
The other two, produce that left with a starved animal or died with a dried crop, are too thin to see, which is itself information: those losses are capital and seed, not yield.

Against a median of about 1,500 units sold a game the rot looks small, under two percent.
It is also universal, already grown and already paid for, and it is the one channel where a fix costs only a hand-turn.

Dead actions are the other universal.
The next figure breaks the median forty-one ignored actions a game into their kinds.


In [ ]:
KINDS = [(('feed_no_animal',), 'FEED, no animal there', '#bdc3c7'),
         (('feed_redundant',), 'FEED, already fed', '#95a5a6'),
         (('feed_no_wheat',), 'FEED, no wheat carried', '#b7791f'),
         (('water_dead',), 'WATER, nothing to water', '#5dade2'),
         (('harvest_dead',), 'HARVEST, nothing to take', '#c0392b'),
         (('plant_dead',), 'PLANT, no seed or busy tile', '#1e8449'),
         (('pickup_dead', 'drop_dead', 'place_dead'), 'shed ops away from the shed, or of nothing', '#7d3c98'),
         (('build_dead', 'dig_dead'), 'BUILD on a busy tile, DIG on an empty one', '#e08e45'),
         (('collect_dead',), 'COLLECT_FERTILIZER, none on offer', '#2e86c1'),
         (('care_dead',), 'CARE, no animal or already cared for', '#85c1e9'),
         (('fertilize_dead',), 'FERTILIZE, no plant or none carried', '#1a5276'),
         (('move_dead', 'locked_dead'), 'a step off the board, or onto locked land', '#34495e')]
D = np.array([[sum(g(r, k) for k in ks) for ks, _, _ in KINDS] for r in ROWS], dtype=float)
order = np.argsort(-D.sum(1))
fig, ax = plt.subplots(figsize=(10.4, 3.9))
bottom = np.zeros(len(ROWS))
for i, (_, label, color) in enumerate(KINDS):
    ax.bar(range(len(ROWS)), D[order, i], bottom=bottom, width=1.0, color=color, label=label, linewidth=0)
    bottom += D[order, i]
ax.set_yscale('symlog', linthresh=20); ax.set_xlim(-.5, len(ROWS) - .5)
ax.set_xlabel('agents, ordered by actions ignored'); ax.set_ylabel('actions a game (log scale past 20)')
ax.legend(frameon=False, fontsize=7.2, ncol=2, loc='upper right')
ax.set_title('what the ignored actions were trying to do', fontsize=10)
ax.grid(axis='x', alpha=0)
fig.tight_layout(); plt.show()
tot = D.sum(0)
for (k, label, _), v in sorted(zip(KINDS, tot), key=lambda kv: -kv[1]):
    print(f'  {label:<30} {100 * v / tot.sum():5.1f} percent of all ignored actions')


**How to read it.** Same layout, one bar per agent, and the y-axis is logarithmic past twenty so the two agents above a thousand do not flatten everyone else.

The largest kind is not a mistake so much as a habit, and it is the sell sentinel of section 5 wearing work clothes.
A **CARE on an animal already cared for today** is a tape repeating, turn after turn, an action that counts once a day, because repeating is free and remembering is not; it is close to half of everything the engine ignores, and it costs nothing unless the hand had somewhere else to be.
A **COLLECT_FERTILIZER with none on offer** is the same habit on the other daily action.
The next three are mistakes.
A **WATER on nothing** is a tape watering a tile that is a weed or bare, which is a tape that outlived its plan.
A **HARVEST that takes nothing** is a tape arriving before the plant is ripe or after it has been picked, which is a calendar error.
A **FEED at an empty tile** is the ghost from section 4, and the agents that have it have it in bulk.
A step off the board or onto locked land, and a shed operation issued away from the shed, are the smallest kinds and the most mechanical: a hand that is not where its tape believes it is.

None of them costs money directly.
Each costs a hand-turn, and labour is the one input on this board that is nearly free to buy and impossible to get back.


<a id="s7"></a>
## 7. Four leaks, measured across everything I hold

The four panels are the four ways the farm loses value without saying so.
Each dot is one agent, sorted low to high within its panel, and the dashed line is the field median.


In [ ]:
PANELS = [('animals_escaped', 'animals lost to starvation', 'per game'),
          ('decay_units', 'yield units rotted in the field', 'per game'),
          ('shed_discarded_units', 'units destroyed by the shed cap', 'per game'),
          ('dead_actions', 'actions the engine ignored', 'per game')]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6))
for ax, (key, ttl, unit) in zip(axes.ravel(), PANELS):
    v = np.array([g(r, key) for r in ROWS])
    o = np.argsort(v)
    colors = [MINE if MINE_SET(ROWS[i]) else INK for i in o]
    ax.scatter(range(len(v)), v[o], s=14, c=colors, linewidths=0, alpha=.75)
    ax.axhline(np.median(v), color=WARN, lw=.9, ls='--')
    ax.set_title(f'{ttl}\nmedian {np.median(v):,.1f}, worst {v.max():,.1f}', fontsize=9)
    ax.set_ylabel(unit); ax.set_xticks([])
    if v.max() > 20 * max(np.median(v), .5):
        ax.set_yscale('symlog', linthresh=1)
    ax.set_ylim(bottom=0)
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], marker='o', ls='', color=INK, label='community agents'),
                    Line2D([], [], marker='o', ls='', color=MINE, label='mine'),
                    Line2D([], [], ls='--', color=WARN, label='field median')],
           loc='upper right', frameon=False, fontsize=8, ncol=3, bbox_to_anchor=(.99, 1.005))
fig.suptitle(f'{len(ROWS)} agents, {sum(r["games"] for r in ROWS):,} games, each dot one agent sorted low to high', fontsize=10, x=.28)
fig.tight_layout(); plt.show()


**How to read it.** A flat stretch is many agents at the same value, a steep tail is a few agents far from the rest.
The starvation panel is flat at zero for half the field and then climbs three decades; the rotting panel has a long shelf at exactly eighteen; the shed-cap and dead-action panels rise smoothly with a handful of outliers.
Amber dots are mine and are spread through every panel, which is what a hundred abandoned builds look like.

The shelf at eighteen is worth a second look, and the next figure is built to explain it.

If agents are clustered by their **ledger** rather than by their bank, the structure that appears is the field's lineage.
Two agents replaying tapes descended from the same published notebook walk past the same tiles on the same days, so they rot the same units, ignore the same actions and burn the same market slots.
An instrument that counts a mechanism shows family resemblance for free, where an instrument that scores an outcome cannot.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

KEYS = ['animals_escaped', 'escape_capital', 'animal_unfed_days', 'feed_no_wheat', 'feed_no_animal',
        'feed_redundant', 'plants_dry', 'dry_yield_units', 'plants_decayed', 'decay_units',
        'plant_unwatered_days', 'shed_discarded_units', 'water_dead', 'harvest_dead', 'plant_dead',
        'sell_zero_fill', 'buy_zero_fill', 'sold_units', 'dead_actions']
M = np.array([[g(r, k) for k in KEYS] for r in ROWS], dtype=float)
live = M.std(0) > 0                      # a counter every agent reads the same on carries nothing
KEYS = [k for k, ok in zip(KEYS, live) if ok]; M = M[:, live]
M = np.log10(M + 1)                      # the counters span four orders of magnitude
M = (M - M.mean(0)) / (M.std(0) + 1e-9)  # so no single column dominates the distance
Z = linkage(pdist(M, 'euclidean'), method='ward')
order = dendrogram(Z, no_plot=True)['leaves']
labels = fcluster(Z, t=6, criterion='maxclust')

fig, (axd, axh) = plt.subplots(1, 2, figsize=(11.6, 6.4), gridspec_kw={'width_ratios': [1, 3.1]})
dendrogram(Z, ax=axd, orientation='left', no_labels=True, color_threshold=Z[-5, 2], link_color_func=lambda k: '#5d6d7e')
axd.set_xticks([]); axd.grid(False)
for s in axd.spines.values():
    s.set_visible(False)
axd.set_title('agents, clustered by their\nledger alone', fontsize=9)
im = axh.imshow(M[order], aspect='auto', cmap='RdBu_r', vmin=-2.2, vmax=2.2, interpolation='nearest')
axh.set_xticks(range(len(KEYS)))
axh.set_xticklabels([k.replace('_', ' ') for k in KEYS], rotation=55, ha='right', fontsize=7.5)
axh.set_yticks([]); axh.grid(False)
axh.set_title('each row an agent, each column a counter, standardised', fontsize=9)
fig.colorbar(im, ax=axh, fraction=.02, pad=.01).set_label('standard deviations from the field', fontsize=7)
fig.tight_layout(); plt.show()

import collections
big = collections.Counter(labels).most_common(1)[0][0]
members = [ROWS[i] for i in range(len(ROWS)) if labels[i] == big]
comm = [r['agent'] for r in members if not r['mine']]
authors = collections.Counter(a.split('-')[0] for a in comm)
print(f'the largest group holds {len(members)} of {len(ROWS)} agents; its rows sit {np.mean(M[labels == big].std(0)):.2f} '
      f'standard deviations apart on average against 1.00 for the field')
print('its community members, by author:', ', '.join(f'{a} x{n}' for a, n in authors.most_common()))
print(f'plus {len(members) - len(comm)} of my own builds, which sit on the same chassis')


**How to read it.** The heatmap is one row per agent and one column per counter, red above the field's average and blue below, in standard deviations; the tree on the left groups rows that look alike.
A block of near-identical rows is a family.

The largest group is exactly that, and the line printed under the figure names its members by author: **all ten versions of one author's notebook, the routers it descends from, the agents another author grafted onto it, two dozen of my own builds on the same chassis**, and a handful of others that share its habits.
Its rows sit well under half a standard deviation apart where the field sits a whole one.
The ledger did not know any of those names; it recovered the lineage from what the agents lose and leave.

The same table also says who is clean.


In [ ]:
clean = [r for r in ROWS if not r['mine'] and g(r, 'animals_escaped') == 0 and r['bank_median'] > 60000]
clean.sort(key=lambda r: -r['bank_median'])
comm = sum(1 for r in ROWS if not r['mine'])
print(f'{len(clean)} of {comm} community agents lose no animal in any of the {ROWS[0]["games"]} games '
      f'while banking above 60,000. The strongest of them:')
pd.DataFrame([(r['agent'], int(r['bank_median']), round(g(r, 'decay_units'), 1), round(g(r, 'dead_actions'), 1))
              for r in clean[:12]], columns=['agent', 'median bank', 'rot, units a game', 'actions ignored a game']).set_index('agent')


<a id="s7b"></a>
## 8. The larger book: what the farm had and did not use

Everything so far was the engine taking something away.
The second book is the engine offering something and the agent not taking it, and across the field it is the bigger number by a wide margin.
Six channels, one panel each, the same layout as before: each dot an agent, sorted, the dashed line the median.


In [ ]:
EFF = [('fertilizer_forgone', 'fertilizer offered and never collected', 'animal-days a game'),
       ('tile_cap_units', 'milk and fruit held at the cap, nobody came', 'units a game'),
       ('sell_floor_units', 'units sold for one coin', 'units a game'),
       ('hand_pass_turns', 'hired hands standing still', 'hand-turns a game'),
       ('idle_tile_days', 'unlocked land left empty or weed', 'tile-nights a game'),
       ('stranded_coins', 'seed and stock stranded at the end', 'coins')]
for r in ROWS:
    r['per_game']['stranded_coins'] = g(r, 'shed_value_now') + g(r, 'seeds_unplanted_cost_now')
fig, axes = plt.subplots(2, 3, figsize=(12, 5.8))
for ax, (key, ttl, unit) in zip(axes.ravel(), EFF):
    v = np.array([g(r, key) for r in ROWS])
    o = np.argsort(v)
    colors = [MINE if MINE_SET(ROWS[i]) else INK for i in o]
    ax.scatter(range(len(v)), v[o], s=14, c=colors, linewidths=0, alpha=.75)
    ax.axhline(np.median(v), color=WARN, lw=.9, ls='--')
    ax.set_title(f'{ttl}\nmedian {np.median(v):,.0f}, worst {v.max():,.0f}', fontsize=9)
    ax.set_ylabel(unit); ax.set_xticks([])
    if v.max() > 20 * max(np.median(v), .5):
        ax.set_yscale('symlog', linthresh=1)
    ax.set_ylim(bottom=0)
fig.suptitle(f'{len(ROWS)} agents, each dot one agent sorted low to high; amber is mine', fontsize=10, x=.3)
fig.tight_layout(); plt.show()
fert = np.array([g(r, 'fertilizer_forgone') for r in ROWS]); cap = np.array([g(r, 'tile_cap_units') for r in ROWS])
print(f'median agent: {np.median(fert):.0f} fertilizer units a game never collected, worth about '
      f'{100 * np.median(fert):,.0f} coins at the base price of 100; {np.median(cap):.0f} units of produce held at a cap')
floor = np.array([g(r, 'sell_floor_units') for r in ROWS])
print(f'median agent sells {np.median(floor):.0f} units a game for one coin each')
strand = np.array([g(r, 'stranded_coins') for r in ROWS])
print(f'median agent ends the season with {np.median(strand):,.0f} coins of seed and stock stranded; '
      f'the worst quarter end with {np.percentile(strand, 75):,.0f} or more, the worst agent with {strand.max():,.0f}')


**How to read it.** Six things the farm was offered and did not take, and none of them appears in any other instrument I know of, because none of them is an event: each is the absence of one.

**Fertilizer** is the largest by value.
An animal puts one unit on offer every morning, worth a hundred coins at base price, and the offer does not accumulate: an uncollected unit is simply replaced by tomorrow's.
Most agents do collect most of it, and the median still forgoes about eleven units a game, which at base price is more coins than any loss in the first book costs the median agent.

**The tile cap** is the same mechanism on the produce side: a cow holds six units of milk and then holds six, a tomato plant holds four, and every unit that would have accrued above the cap is a unit that was never harvested in time.

**Units sold for one coin** is the one I did not expect.
The price floor is one coin, a sale at the floor does not even count as supply, and the median agent makes about a hundred and sixty of them a game: produce grown, carried and sold for less than the seed it came from, almost always in a last-day dump into a market that everybody is dumping into.
Whether holding it would have fetched more is a question for the pricing model; that it fetched nothing is a fact.

**Hands standing still** is labour bought and not scheduled.
Hands leave at midnight and are hired again each morning, so a hand's cost is a day and its idle turns cost nothing extra; the counter measures opportunity, and it matters where the same tape is short of hands elsewhere, which the dead-action census in section 6 says it usually is.

**Idle land** is bought once and then forgotten, at a thousand to four thousand coins a quadrant.

**Stranded at the end** turned out to be rare for stock and common for seed.
Almost every agent empties its shed before the last step, because the score is money and the strongest public chassis carry a terminal sale; what stays behind is seed, bought and never planted, a median of under a hundred coins and up to four thousand.


<a id="s8"></a>
## 9. What the whole census says

Every line of this table is knowable only from inside the engine.


In [ ]:
import statistics as st
n = len(ROWS)
esc = [r for r in ROWS if g(r, 'animals_escaped') > 0]
ours = [r for r in ROWS if MINE_SET(r)]
q = lambda key: st.median([g(r, key) for r in ROWS])
p90 = lambda key: sorted(g(r, key) for r in ROWS)[int(.9 * n)]
leads = [t for r in LEAD['rows'] for t in r['lead_turns']]
LINES = [
    ('agents that lose at least one animal', f'{len(esc)} of {n}'),
    ('of those, mine', f"{sum(1 for r in ours if g(r,'animals_escaped') > 0)} of {len(ours)} I keep"),
    ('worst agent, animals per game', f"{max(g(r,'animals_escaped') for r in ROWS):.2f}"),
    ('days the losses cluster on', 'day 1, day 8, and the last week'),
    ('notice before the first loss, median', f'{np.median(leads):.0f} turns'),
    ('yield units rotting in the field, median agent', f"{q('decay_units'):.1f} per game"),
    ('the same at the ninetieth percentile', f"{p90('decay_units'):.1f} per game"),
    ('units destroyed by the shed cap, median', f"{q('shed_discarded_units'):.1f} per game"),
    ('actions the engine ignored, median', f"{q('dead_actions'):.0f} per game"),
    ('the same, worst agent', f"{max(g(r,'dead_actions') for r in ROWS):.0f} per game"),
    ('SELL orders that filled nothing, median', f"{q('sell_zero_fill'):.0f} per game"),
    ('BUY orders that bought nothing, median', f"{q('buy_zero_fill'):.1f} per game"),
    ('coins lost in silence, median agent', f"{q('silent_loss_coins'):,.0f} per game"),
    ('coins lost in silence, worst agent', f"{max(g(r,'silent_loss_coins') for r in ROWS):,.0f} per game"),
    ('fertilizer offered and never collected, median', f"{q('fertilizer_forgone'):.0f} units per game"),
    ('produce held at a tile cap, median', f"{q('tile_cap_units'):.0f} units per game"),
    ('hired hands standing still, median', f"{q('hand_pass_turns'):.0f} hand-turns per game"),
    ('land left idle, median', f"{q('idle_tile_days'):.0f} tile-nights per game"),
    ('units sold for one coin, median', f"{q('sell_floor_units'):.0f} per game"),
    ('seed and stock stranded at the end, median', f"{q('stranded_coins'):,.0f} coins per game"),
    ('the same, worst agent', f"{max(g(r,'stranded_coins') for r in ROWS):,.0f} coins per game"),
]
pd.DataFrame(LINES, columns=['what the census says', 'value']).set_index('what the census says')


My own agents are over-represented in the second row and the reason is not what it looks like.
The roster is every build I ever kept, a hundred of them, most abandoned the day they were measured and only a handful ever fielded.
The community rows are published notebooks, which is a filter my scratch builds never had to pass.

**The line I did not expect is the rotting.**
Losing an animal is a defect, it happens to a minority of agents, and a guard fixes it.
Leaving yield to rot happens to almost everybody, including the strongest public agents I hold, and at the median it is about eighteen units a game walked past.
Whether harvesting them pays depends on what the hand would otherwise have done with the turn, and on whether the shed has room, since the same census shows the cap already destroying five units a game.
That is a question about the agent rather than the engine, and nobody can ask it without a counter for it.


<a id="s10"></a>
## 10. Case study: the agent most of the field is running

The largest group in section 7 is one lineage, and its root is public: [Kaggriculture V42, Production That Fits the Market](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v42-production-that-fits-the-marke) by Ahmed Berat Özer, Apache-2.0, the most recent of a line of notebooks that a large part of the ladder runs verbatim or lightly modified.
I run it myself: it is the base my current candidate is built on.
So here is what the agent most people are fielding does with its hands, read from the engine, exactly as published.

First the report, the same command as section 3, sixteen seeds from both seats.


In [ ]:
REPORT_V42 = "ahmedberatozer-v42-production-that-fits-the-market against yhay81-shop-router-0913: 32 games, playing...\n\nahmedberatozer-v42-production-that-fits-the-market against yhay81-shop-router-0913: 32 games\n\nescapes: 0 (engine), 0 (observation control) AGREE\ngames losing an animal: 0 of 32\n\nPER GAME, mean:\n\n  LOSS\n    animals_escaped              0.00   animals gone at two consecutive unfed days\n    escape_capital               0.00   their purchase price, in coins\n    escape_yield_units           0.00   product held on the tile and lost with it\n    plants_dry                   1.00   plants turned to weed by two dry days\n    dry_seed_cost              100.00   their seed price, in coins\n    plants_decayed              17.94   plants that rotted all the way to weed\n    decay_units                 17.94   yield units rotted off a plant left standing\n    shed_discarded_units        13.22   production the hundred-item cap destroyed\n\n  PRECURSOR\n    feed_no_wheat                0.00   FEED on a hungry animal, carrier held no wheat\n    animal_unfed_days           80.59   animal-days ending unfed: the runway to a loss\n    plant_unwatered_days       384.75   plant-days ending unwatered\n\n  DEAD ACTION\n    feed_no_animal               0.00   FEED at a tile whose animal already left\n    feed_redundant               0.00   FEED on an animal already fed today\n    water_dead                  10.94   WATER on a non-plant or an already-watered tile\n    harvest_dead                 4.59   HARVEST that took nothing\n    plant_dead                   0.00   PLANT dropped for want of seed, or a busy tile\n    pickup_dead                  0.00   PICKUP away from the shed, or of what it lacks\n    place_dead                   1.28   PLACE with nothing to place, or nowhere to put it\n    build_dead                   0.00   BUILD on a tile that is not empty\n    dig_dead                     3.50   DIG on an empty tile, or on an animal\n    fertilize_dead               0.81   FERTILIZE on a non-plant, or with none carried\n    collect_dead                 7.50   COLLECT_FERTILIZER with none available\n    care_dead                   12.62   CARE on no animal, or twice in a day\n    move_dead                    0.00   a move off the board\n    locked_dead                  0.00   a tile action on locked land\n    phantom_hand_actions         0.00   orders to hands the farm does not have\n\n  EFFICIENCY\n    fertilizer_forgone          15.00   animal-days whose fertilizer was on offer and never collected\n    tile_cap_units              23.78   yield clipped at a tile's holding cap: not harvested in time\n    hand_pass_turns            457.25   hired hands that did nothing this turn\n    farmer_pass_turns           67.94   the farmer's PASS turns, apart: the farmer is free, a hand costs a day\n    idle_tile_days             100.66   unlocked tiles empty or weed at the end of a day\n    weed_tile_days               2.78   of those, weeds nobody dug\n    animal_shed_days             0.94   animals still in the shed at the end of a day\n    care_unfed                  76.16   CARE without FEED the same day: the bonus never accrues\n    sell_floor_units            95.59   units sold at the price floor of 1\n\n  MARKET\n    hire_premium             5,004.31   the same-day hire ladder above one coin a hand: the price of labour, hands leave at midnight\n\n  STRANDED AT THE END\n    shed_units_now               0.00   units still in the shed when the game ended: worth nothing\n    shed_value_now               0.00   what they would have fetched at the final prices\n    animals_in_shed_now          0.00   animals bought and never placed\n    seeds_unplanted_cost_now      15.62   coins of seed never planted\n\n  MARKET\n    sell_zero_fill             135.97   a SELL order that delivered nothing, one slot burned\n    buy_zero_fill                1.00   a BUY order that bought nothing: usually a broken financing chain\n    refused_buy_product          5.12   BUY_PRODUCT units refused on funds or capacity\n    refused_buy_seed             0.00   BUY_SEED units refused on funds\n    refused_buy_animal           0.00   BUY_ANIMAL units refused on funds or capacity\n    refused_hire                 0.00   HIREs refused on funds or the unit cap\n    refused_buy_land             0.00   BUY_LANDs refused on funds, or none left\n\n  SILENT LOSS, coins            100   units        31.2\n"
print(REPORT_V42)


**The first book is clean.** No animal lost in 32 games, no FEED that found nothing, no ghost fed, no order to a hand that does not exist.
The dead actions are well under the field's median and they are the daily-action habit of section 6, CARE and COLLECT repeated where they only count once.

**The second book says where its headroom is.** Set it against the field median, counter by counter.


In [ ]:
v42 = next(r for r in ROWS if r['agent'].startswith('ahmedberatozer-v42'))
FIELD = [('animals_escaped', 'animals lost', 'a game'),
         ('dead_actions', 'actions ignored', 'a game'),
         ('feed_no_wheat', 'FEED with no wheat', 'a game'),
         ('animal_unfed_days', 'animal-days unfed', 'a game'),
         ('care_unfed', 'CARE on an unfed day, not credited', 'a game'),
         ('decay_units', 'produce rotted on the tile', 'units'),
         ('tile_cap_units', 'produce held at a tile cap', 'units'),
         ('shed_discarded_units', 'destroyed by the shed cap', 'units'),
         ('fertilizer_forgone', 'fertilizer never collected', 'units'),
         ('sell_floor_units', 'sold for one coin', 'units'),
         ('hand_pass_turns', 'hired hands standing still', 'hand-turns'),
         ('idle_tile_days', 'land left idle', 'tile-nights'),
         ('stranded_coins', 'seed and stock stranded at the end', 'coins'),
         ('sell_zero_fill', 'SELL orders that filled nothing', 'a game')]
rows = []
for k, name, unit in FIELD:
    a = g(v42, k); m = np.median([g(r, k) for r in ROWS])
    verdict = 'cleaner' if a < m * .8 else ('worse' if a > m * 1.25 else 'as the field')
    rows.append((name, round(a, 1), round(m, 1), verdict))
pd.DataFrame(rows, columns=['per game', 'V42', 'field median', 'V42 against the field']).set_index('per game')


**How to read it.** Three things stand out, and none of them is visible from its bank or its rating.

**It feeds every other day.** Eighty animal-days end unfed in a season with zero escapes, which is only possible by riding the engine's rule exactly: an animal leaves on the second consecutive unfed night, never the first, so alternate days are safe and halve the wheat bill.
That is a deliberate economy and the ledger shows it as one, because the precursor that marks a defect, a FEED that finds nothing, reads zero.

**It cares on the days it does not feed, and the engine does not credit it.** The care bonus accrues only on a day the animal was also fed, so about seventy-six CARE actions a game earn nothing.
That is a hand-turn a day per animal, and it is the kind of rule an author cannot see from outside: the action returns normally, the tile looks cared for, and the bonus simply never arrives.

**Its headroom is in collection, not production.** It leaves more fertilizer uncollected than the median agent, holds more produce at the tile cap, and loses more to the shed cap, all while banking near the top of the field.
The reading is that it produces more than its hands carry, so the next coin for this lineage is a hand-turn spent collecting rather than a better plan; and since a large part of the ladder runs this file, the same coin is on offer to all of them.

Where it stands beside my own two builds, in one world, turn by turn.


In [ ]:
strips, labels = [], []
for r in CASE['rows']:
    row = np.zeros(720, dtype=int)
    for ev in r['events']:
        c = CLASS.get(ev['what'], 0)
        if c > row[ev['turn']]:
            row[ev['turn']] = c
    strips.append(row)
    labels.append(f"{r['agent']:<34} {r['bank']:>8,.0f} vs {r['rival_bank']:>8,.0f}")
fig, ax = plt.subplots(figsize=(12, 2.6))
ax.imshow(np.array(strips), aspect='auto', cmap=PAL, vmin=0, vmax=5, interpolation='nearest')
ax.set_yticks(range(len(strips))); ax.set_yticklabels(labels, fontsize=8, family='monospace')
ax.set_xticks(range(0, 721, 96)); ax.set_xticklabels([f'day {t // 24}' for t in range(0, 721, 96)])
ax.grid(False)
ax.legend(handles=[mpatch.Patch(color=PAL(i), label=NAMES[i]) for i in range(1, 6)],
          loc='upper center', bbox_to_anchor=(.5, -.3), ncol=5, frameon=False, fontsize=8)
ax.set_title(f"seed {CASE['seed']}, seat 0, against {CASE['rival']}: three agents, one world", fontsize=10)
fig.tight_layout(); plt.show()
for r in CASE['rows']:
    L = r['ledger']
    print(f"{r['agent']:<34} escapes {L['animals_escaped']}  ignored {L['dead_actions']:>4}  "
          f"fertilizer forgone {L['fertilizer_forgone']:>3}  at cap {L['tile_cap_units']:>3}  floor sales {L['sell_floor_units']:>4}  "
          f"idle hand-turns {L['hand_pass_turns']:>4}")


**How to read it.** The top row is the published base; the middle row is my build on the same chassis with the funding defect, grey with ghost feeds from the second week and dark red where the animals left; the bottom row is the repair, which looks like the base again.
The orange ticks in the top row are the alternate-day feeding, one every other night, and they are the reason the base's row is otherwise so quiet: what it does with its herd is on purpose.

None of this is a criticism of a notebook I run myself.
It is the point of the instrument: **an agent that a large part of the field is fielding has a ledger, the ledger has a shape, and the shape says where the next coin is** for everyone running it.

**The lineage, version by version.** Ten versions of this notebook are public, from v27 to v42, and in September a report on the forum said one of the middle ones, the version most people had copied at the time, was losing cows on day 1 in some worlds.
The ledger can settle that, because every version is in the census on the same worlds against the same rival.


In [ ]:
ORDER = ['v27', 'v31', 'v34', 'v35', 'v36', 'v38', 'v39', 'v40', 'v41', 'v42']
line = sorted([r for r in ROWS if r['agent'].startswith('ahmedberatozer-')],
              key=lambda r: next(i for i, v in enumerate(ORDER) if f'-{v}' in r['agent']))
COLS2 = [('animals_escaped', 'animals lost'), ('feed_no_wheat', 'FEED, no wheat'),
         ('animal_unfed_days', 'animal-days unfed'), ('care_unfed', 'CARE on unfed day'),
         ('dead_actions', 'actions ignored'), ('tile_cap_units', 'held at tile cap'),
         ('shed_discarded_units', 'shed cap'), ('sell_floor_units', 'sold for one coin'),
         ('fertilizer_forgone', 'fertilizer forgone')]
tbl = pd.DataFrame([[next(v for v in ORDER if f'-{v}' in r['agent']), int(r['bank_median'])] + [round(g(r, k), 1) for k, _ in COLS2]
                    for r in line], columns=['version', 'median bank'] + [c for _, c in COLS2]).set_index('version')
display(tbl)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
x = range(len(line))
SPECS = [([('feed_no_wheat', 'FEED with no wheat'), ('animals_escaped', 'animals lost')], 'the reported defect: never there on these worlds'),
         ([('animal_unfed_days', 'animal-days unfed'), ('care_unfed', 'CARE on an unfed day')], 'what changed at v38: alternate-day feeding'),
         ([('tile_cap_units', 'held at the tile cap'), ('sell_floor_units', 'sold for one coin')], 'the second book moved the other way')]
for ax, (keys, ttl) in zip(axes, SPECS):
    for (k, label), color in zip(keys, (WARN, INK)):
        ax.plot(x, [g(r, k) for r in line], marker='o', ms=4, lw=1.4, color=color, label=label)
    ax.set_xticks(list(x)); ax.set_xticklabels([v for v in ORDER], fontsize=8)
    ax.set_title(ttl, fontsize=9); ax.legend(frameon=False, fontsize=7.5); ax.set_ylim(bottom=0)
fig.suptitle('one public lineage, ten versions, the same 32 games each', fontsize=10)
fig.tight_layout(); plt.show()


**How to read it.** The reported defect is the left panel, and on these worlds it was never there: no version loses an animal, and the only precursor, a FEED that found no wheat, reads one in eight games up to v38 and zero from v39 against this rival.

**Does V42 fix it? Yes, and the ledger says exactly what was fixed.** The mechanism the report described is a hand arriving to feed with nothing in it, which is `feed_no_wheat`, and it is a defect whether or not an animal dies of it.
Pushed harder than the census, over 64 worlds from both seats against four rivals, **V39** still shows that precursor once in five games against an idle rival and once in sixteen against V34, with no animal lost in 512 games; **V41 and V42 read zero on every world and rival tried**, 256 and 384 games, with no animal lost either.
Its author's own note for v41, the feed shortfalls taken to zero at the source, is what the ledger sees.
So the loss the report described did not reproduce in any version here, and the precursor it needs was real, small, and gone by V41.

What the versions actually changed is in the other two panels.
At v38 the herd went to alternate-day feeding: unfed animal-days more than doubled, with zero escapes, and the uncredited CARE actions doubled with them, because the tape kept caring every day.
The same versions doubled the produce held at the tile cap and cut the units sold for one coin by more than half, which is the sell side getting better while the collect side fell behind.
**A defect somebody reported was not in the ledger; two trades nobody reported were.** That is the difference between reading an agent from its games and reading it from the engine.


<a id="s9"></a>
## 11. What it is for

Six uses, ordered from the one I run every day to the ones I would want somebody to build.

### 11.1 A gate before a submission, and a regression test between versions

The report in section 3 exits nonzero on any escape.
That is the whole of the first use: **run it on the file you are about to submit, against two or three public rivals, and do not submit on a red exit.**
The submission it caught had cleared a parity check, a robustness sweep and a head-to-head against its predecessor, because none of those instruments count what the engine destroys.

The second use is the same tool run twice.
Section 4's pair is a regression test: **the ledger of the new version must be no worse than the old on every line**, and a line that got worse names the change that did it.
A bank comparison cannot do this, because a change can lose a cow and gain a better sell timing in the same game and read as an improvement.

### 11.2 Auditing a real ladder episode, both sides

A replay file carries the resolved seed and both action streams.
Fed back into the engine at that seed they reproduce the episode exactly, and **the ledger then says what each side lost in silence during a game that counted for rating, including the opponent's side**, which no observation of yours ever showed you.

I did this for the five most recent games of my live submission.
The recorded final banks are the check on every one of them.


In [ ]:
REPLAY_CODE = '''
import json, kagsim

PASS = {"farmer": ["PASS"], "hands": [], "market": []}
replay = json.load(open("episode-109033976-replay.json"))     # kaggle competitions replay <id>
seed = replay["info"]["seed"]
steps = replay["steps"]

# the action stored at step t is the reply to the observation at step t-1
tape = [[(steps[t + 1][p].get("action") or PASS) if t + 1 < len(steps) else PASS
         for t in range(720)] for p in (0, 1)]

game = kagsim.Game(seed)
for t in range(720):
    game.step(tape[0][t], tape[1][t])

recorded = [steps[-1][p]["reward"] for p in (0, 1)]
assert [game.reward(0), game.reward(1)] == recorded, "not reproduced: trust nothing below"
for p in (0, 1):
    print(replay["info"]["TeamNames"][p], game.telemetry(p))
'''
print(REPLAY_CODE)


In [ ]:
# The ledger of five real games, both sides. Columns are the counters that
# separate a defect from a habit; the banks are the recorded ones, which the
# replay reproduced to the coin before any ledger was read.
COLS = [('animals_escaped', 'animals lost'), ('feed_no_wheat', 'FEED, no wheat'),
        ('feed_no_animal', 'FEED, no animal'), ('decay_units', 'rotted'),
        ('shed_discarded_units', 'shed cap'), ('dead_actions', 'ignored'),
        ('sell_zero_fill', 'empty SELLs'), ('buy_zero_fill', 'empty BUYs'),
        ('refused_buy_product', 'refused wheat units')]
rows = []
for e in EPISODES:
    if not e['trusted']:
        rows.append([e['episode'], 'REFUSED', None, None] + [None] * len(COLS)); continue
    me = e['our_seat']
    for side, p in (('me', me), ('rival', 1 - me)):
        L = e['ledgers'][p]
        rows.append([e['episode'], side, int(e['banks'][p]), 'won' if e['banks'][p] > e['banks'][1 - p] else 'lost']
                    + [int(L[k]) for k, _ in COLS])
table = pd.DataFrame(rows, columns=['episode', 'side', 'bank', 'result'] + [c for _, c in COLS]).set_index(['episode', 'side'])
display(table)
ok = sum(1 for e in EPISODES if e['trusted'])
print(f'{ok} of {len(EPISODES)} episodes reproduced to the coin; the control agreed with the engine on every one '
      f'(engine {[e["ledgers"][e["our_seat"]]["animals_escaped"] for e in EPISODES if e["trusted"]]}, '
      f'observation {[e["observed_escapes"][e["our_seat"]] for e in EPISODES if e["trusted"]]})')


**How to read it.** Each pair of rows is one rated game, my side and the opponent's, with the counters that tell a defect from a habit.

The fifth game is the one to look at.
**My live agent lost a cow, issued 266 actions the engine ignored, and placed fifteen purchase orders that bought nothing**, in a game it lost by 25,000.
The census in section 6 had read zero escapes for that same agent against its fixed rival, because the defect needs a particular opponent and world to trigger, which is exactly why auditing the games that actually happened is a different instrument from playing a fixed panel.

The third game shows the purchase ambiguity of section 5 in one row: 1,734 refused wheat units and not a single empty order.
That agent asks for more wheat than it can afford on purpose and takes what it gets.

Every rival row is a farm I never saw from the inside until now.
The rivals rot the same eighteen units and burn a hundred empty sell orders a game, like everybody, and none of them lost an animal.

The same five games as a timeline, which is where the totals stop hiding the when.


In [ ]:
# One strip per side per episode: 720 turns, coloured by the kind of event the
# engine recorded on that turn. Losses win over everything else when two kinds
# land on the same turn.
trusted = [e for e in EPISODES if e['trusted']]
strips, labels = [], []
for e in trusted:
    me = e['our_seat']
    for side, p in (('me', me), ('rival', 1 - me)):
        row = np.zeros(720, dtype=int)
        for ev in e['events'][p]:
            c = CLASS.get(ev['what'], 0)
            if c > row[ev['turn']]:
                row[ev['turn']] = c
        strips.append(row)
        won = e['banks'][p] > e['banks'][1 - p]
        labels.append(f"{e['episode']}  {side:<6} {e['banks'][p]:>8,.0f}{' won' if won else '    '}")
fig, ax = plt.subplots(figsize=(12, .42 * len(strips) + 1.4))
ax.imshow(np.array(strips), aspect='auto', cmap=PAL, vmin=0, vmax=5, interpolation='nearest')
ax.set_yticks(range(len(strips))); ax.set_yticklabels(labels, fontsize=7.5, family='monospace')
for i in range(0, len(strips), 2):
    ax.axhline(i - .5, color='white', lw=2)
ax.set_xticks(range(0, 721, 96)); ax.set_xticklabels([f'day {t // 24}' for t in range(0, 721, 96)])
ax.grid(False)
handles = [mpatch.Patch(color=PAL(i), label=NAMES[i]) for i in range(1, 6)]
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5, -.18), ncol=5, frameon=False, fontsize=8)
ax.set_title('five rated games, turn by turn: what the engine recorded against each side', fontsize=10)
fig.tight_layout(); plt.show()


**How to read it.** Every row is one side of one game, left to right across the thirty days; a coloured tick is a turn on which the engine recorded something against that side, and the palette runs from a wasted turn in grey to a loss in dark red.

Read the last pair.
My side is grey from the first week onward, which is the ghost being fed, and it carries a violet run of empty purchases where the financing chain broke; the rival's row is almost blank.
In the other four pairs the two sides look alike: the same amber ticks of rot in the last week, the same scattered violet of sell orders that found an empty shed, and the orange precursors at the season's end that are the deliberate stop.

The strip is the instrument I would put in front of an agent author first, because it answers the question the totals cannot: **not how much, but when, and against whom.**

### 11.3 An objective for search and learning, with the one way it goes wrong

The bank is a sparse reward: one number, at the end, moved by the shop draw more than by most decisions.
The ledger is dense and causal: it moves on the turn of the mistake and it names the mistake.
That is the shape a learning signal wants, and there are three ways to use it.

As a **constraint in a search**.
The market half of this ledger already has a consumer: a schedule search one repository over uses the refused-purchase counters to reject candidates, and it caught a best genome that was refusing 24 of its own 104 animal purchases while scoring well on the bank.
The farm half extends that to escapes and rot: a candidate with a loss in its ledger is not a candidate.

As a **penalty term in a learned policy's reward**, dense where the bank is sparse: the coins of `escape_capital` and `dry_seed_cost` are already in the bank's units, and a small charge per ignored action prices the labour the bank cannot see.

As a **filter on demonstrations** for behavioural cloning: of two recorded episodes of the same agent, prefer the one with the cleaner ledger, because the other contains a mistake the clone would learn.

**The way it goes wrong is Goodhart's, and the census shows it.** An agent that does nothing has a perfect ledger and a bank of 3,000, and there is one row in section 7 that sits at zero on every counter while banking a fifth of the field.
Zero silent loss is trivially achievable by not farming.
So the ledger is a sub-objective or a constraint, never the objective, and the right statement is the one a search actually needs: **maximise the bank subject to no losses the engine can name.**

The efficiency book plays a different role in the same search: not a constraint but a **gradient**.
Of the six channels in section 8, the largest one for a given agent names the feature to build next, in the bank's own units: a hundred coins per uncollected fertilizer unit, the seed price per unplanted seed, the difference between the floor and the day's price per unit dumped, a hand-turn per idle hand.
That is a ranking of improvements before any of them is built, which a bank comparison can only give after.

### 11.4 Does it rank agents?

No, and it is worth being plain about why.
The figure below is every agent's silent loss against its bank, and the correlation is small and negative.


In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 4.2))
x = np.array([g(r, 'silent_loss_coins') for r in ROWS])
y = np.array([r['bank_median'] for r in ROWS])
mine = np.array([MINE_SET(r) for r in ROWS])
ax.scatter(x[~mine], y[~mine], s=22, color=INK, alpha=.55, linewidths=0, label='community agents')
ax.scatter(x[mine], y[mine], s=26, color=MINE, alpha=.8, linewidths=0, label='mine')
for r in ROWS:
    if 'live 56220106' in r['agent']:
        ax.annotate('my live submission', (g(r, 'silent_loss_coins'), r['bank_median']), fontsize=7.5, color=WARN,
                    ha='right', xytext=(-8, 6), textcoords='offset points', arrowprops=dict(arrowstyle='->', color=WARN, lw=.8))
ax.set_xscale('symlog'); ax.set_xlabel('coins lost in silence per game (escaped capital + dead seed, log scale)')
ax.set_ylabel('median bank'); ax.legend(frameon=False, fontsize=8)
rc = np.corrcoef(np.log10(x + 1), y)[0, 1]
ax.set_title(f'the leak explains almost none of the bank (r = {rc:+.2f} on log coins)', fontsize=10)
fig.tight_layout(); plt.show()


**How to read it.** Each dot is an agent; right means more coins lost in silence, up means a higher bank.
The cloud has no slope to speak of, and the agents banking the most sit at the same leak as agents banking half of it.

That is not a weakness of the instrument, it is the reason the losses survive: **a component that does not move the total is a component nobody optimises.**
What the ledger gives a ranking is headroom rather than order.
The coins in the x-axis are recoverable by a code change and not by a strategic bet, and on a ladder where the top hundred are separated by a few points of rating, the cheapest improvement available is the one that removes a defect instead of adding an idea.
As a tiebreak between agents of equal bank I would expect the cleaner ledger to have the higher ceiling; that is a hypothesis this page does not test.

### 11.5 A canary for engine changes

When the environment is rebalanced, a fixed tape replayed on the new engine shows the change in its ledger before it shows in its bank, and shows which mechanism moved.
The 1.32.7 repricing of carrot, tomato and egg touched no counter here; a change to decay or to the unfed threshold would light one up on every agent at once.

### 11.6 Reading somebody else's agent

This page is that use, applied 154 times.
A published agent's ledger is a description of what it does with its hands that its author may not have, and a notebook that ships with its own ledger has told its readers something a leaderboard position cannot.


<a id="s10"></a>
## 12. Running it on your own agent

The engine installs from its repository, which is the distribution channel on purpose: a competition-scoped tool tied to a moving environment is worse than useless if a stale package hands somebody a confidently wrong simulator.

```bash
pip install git+https://github.com/destbreso/kaggriculture-cppsim
python -c "import kagsim; print(kagsim.run_episode(kagsim.Stream([]), kagsim.Stream([]), seed=11))"
```

**Validate before trusting it, always**, and the fixtures ship with it: `python tests/test_golden.py` replays recorded episodes and demands the banks match to the coin.
Kaggle notebook kernels run without internet, so for a notebook either attach the repository as a dataset or install it locally.

Your agent is an ordinary callable taking the observation dictionary, exactly as the competition calls it, so nothing about it has to change.
**The census in this page is the loop below and nothing else.**


In [ ]:
CENSUS_CODE = '''
import kagsim

def ledger(agent, rival, seeds, seats=(0, 1)):
    """Play the worlds and average the engine's per-farm counters."""
    total, games = {}, 0
    for seed in seeds:
        for seat in seats:
            game = kagsim.Game(seed)
            while not game.done:
                mine = dict(game.observe(seat));      mine["player"] = seat
                theirs = dict(game.observe(1 - seat)); theirs["player"] = 1 - seat
                a, b = agent(mine), rival(theirs)
                game.step(a if seat == 0 else b, b if seat == 0 else a)
            for k, v in game.telemetry(seat).items():
                if isinstance(v, (int, float)):
                    total[k] = total.get(k, 0) + v
            games += 1
    return {k: v / games for k, v in total.items()}, games

per_game, n = ledger(my_agent, rival_agent, seeds=range(1, 17))   # one row of the table
print(per_game["animals_escaped"], "animals a game over", n, "games")
'''
print(CENSUS_CODE)


The command-line version, which produced the report in section 3, ships with the engine as `tools/silent_leaks.py`: one file, standard library plus the engine, with the observation-side control built in.
The cell below writes it next to this notebook, so it is also in this page's output files.


In [ ]:
TOOL = "#!/usr/bin/env python3\n\"\"\"Read the engine's silent-loss ledger for any two agents. Stdlib plus kagsim.\n\n    python tools/silent_leaks.py --agent my_agent.py --rival theirs.py\n    python tools/silent_leaks.py --agent my_agent.py --rival theirs.py --seeds 1-32 --both-seats\n    python tools/silent_leaks.py --agent my_agent.py --rival theirs.py --require-zero-escapes\n\nAn agent file is whatever Kaggle runs: a Python file whose last module-level\ncallable of one or two arguments takes the observation dict and returns the\naction dict. Nothing about it has to change to be measured here.\n\nWHY THIS EXISTS. The environment refuses, starves and rots in silence. An\nanimal two days unfed is gone in the morning and the pasture looks exactly like\none that never held a cow; a crop nobody watered becomes a weed; a FEED issued\nby a hand carrying no wheat changes nothing at all. None of it appears in the\nobservation, so the author sees a bank that came out lower and no reason why.\n\nTHREE LEDGERS, NOT ONE. Losses the engine inflicts (an animal gone, a crop dried,\nproduce rotted, the shed cap); actions it accepted and ignored; and EFFICIENCY,\nthe value the farm had and did not use: fertilizer on offer and never collected,\nmilk held at the animal's cap because nobody came, hands told to PASS, land left\nidle, and everything still in the shed when the season ended, which the bank\ncounts as nothing. An instrument that measures one channel invites tuning to\nthat channel, so this one measures them all.\n\nTHE USEFUL HALF IS THE PRECURSOR. `animals_escaped` is a post mortem.\n`feed_no_wheat` fires while the animal is still alive, a median of about one\ngame day earlier, which is time enough to buy one unit of wheat.\n\nTHE CONTROL IS NOT CEREMONY. The escape count is computed twice: once by the\nengine at the site where it removes the animal, and once from the outside by\nreading the observation at hour 23 for an animal with one unfed day that was\nnot fed today. The two share no code. When they disagree this prints DISAGREE\nand you should trust neither until you know why.\n\nONE COUNTER IS DELIBERATELY NOT IN THE LEDGER. `sell_dead_units`, the SELL\nvolume the shed could not cover, reads in the tens of thousands for a perfectly\nhealthy agent, because the ordinary way to say \"sell everything I have\" is to\nask for a thousand units and let the settle fill what exists. It measures the\nidiom. `sell_zero_fill`, an order that delivered nothing at all, measures a\nmarket slot burned, and there are only ten a turn.\n\nApache-2.0, same as the engine. Part of github.com/destbreso/kaggriculture-cppsim.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport pathlib\nimport sys\n\nimport kagsim\n\nPASS = {\"farmer\": [\"PASS\"], \"hands\": [], \"market\": []}\n\nLEDGER = (\n    (\"LOSS\", \"animals_escaped\", \"animals gone at two consecutive unfed days\"),\n    (\"LOSS\", \"escape_capital\", \"their purchase price, in coins\"),\n    (\"LOSS\", \"escape_yield_units\", \"product held on the tile and lost with it\"),\n    (\"LOSS\", \"plants_dry\", \"plants turned to weed by two dry days\"),\n    (\"LOSS\", \"dry_seed_cost\", \"their seed price, in coins\"),\n    (\"LOSS\", \"plants_decayed\", \"plants that rotted all the way to weed\"),\n    (\"LOSS\", \"decay_units\", \"yield units rotted off a plant left standing\"),\n    (\"LOSS\", \"shed_discarded_units\", \"production the hundred-item cap destroyed\"),\n    (\"PRECURSOR\", \"feed_no_wheat\", \"FEED on a hungry animal, carrier held no wheat\"),\n    (\"PRECURSOR\", \"animal_unfed_days\", \"animal-days ending unfed: the runway to a loss\"),\n    (\"PRECURSOR\", \"plant_unwatered_days\", \"plant-days ending unwatered\"),\n    (\"DEAD ACTION\", \"feed_no_animal\", \"FEED at a tile whose animal already left\"),\n    (\"DEAD ACTION\", \"feed_redundant\", \"FEED on an animal already fed today\"),\n    (\"DEAD ACTION\", \"water_dead\", \"WATER on a non-plant or an already-watered tile\"),\n    (\"DEAD ACTION\", \"harvest_dead\", \"HARVEST that took nothing\"),\n    (\"DEAD ACTION\", \"plant_dead\", \"PLANT dropped for want of seed, or a busy tile\"),\n    (\"DEAD ACTION\", \"pickup_dead\", \"PICKUP away from the shed, or of what it lacks\"),\n    (\"DEAD ACTION\", \"place_dead\", \"PLACE with nothing to place, or nowhere to put it\"),\n    (\"DEAD ACTION\", \"build_dead\", \"BUILD on a tile that is not empty\"),\n    (\"DEAD ACTION\", \"dig_dead\", \"DIG on an empty tile, or on an animal\"),\n    (\"DEAD ACTION\", \"fertilize_dead\", \"FERTILIZE on a non-plant, or with none carried\"),\n    (\"DEAD ACTION\", \"collect_dead\", \"COLLECT_FERTILIZER with none available\"),\n    (\"DEAD ACTION\", \"care_dead\", \"CARE on no animal, or twice in a day\"),\n    (\"DEAD ACTION\", \"move_dead\", \"a move off the board\"),\n    (\"DEAD ACTION\", \"locked_dead\", \"a tile action on locked land\"),\n    (\"DEAD ACTION\", \"phantom_hand_actions\", \"orders to hands the farm does not have\"),\n    (\"EFFICIENCY\", \"fertilizer_forgone\", \"animal-days whose fertilizer was on offer and never collected\"),\n    (\"EFFICIENCY\", \"tile_cap_units\", \"yield clipped at a tile's holding cap: not harvested in time\"),\n    (\"EFFICIENCY\", \"hand_pass_turns\", \"hired hands that did nothing this turn\"),\n    (\"EFFICIENCY\", \"farmer_pass_turns\", \"the farmer's PASS turns, apart: the farmer is free, a hand costs a day\"),\n    (\"EFFICIENCY\", \"idle_tile_days\", \"unlocked tiles empty or weed at the end of a day\"),\n    (\"EFFICIENCY\", \"weed_tile_days\", \"of those, weeds nobody dug\"),\n    (\"EFFICIENCY\", \"animal_shed_days\", \"animals still in the shed at the end of a day\"),\n    (\"EFFICIENCY\", \"care_unfed\", \"CARE without FEED the same day: the bonus never accrues\"),\n    (\"EFFICIENCY\", \"sell_floor_units\", \"units sold at the price floor of 1\"),\n    (\"MARKET\", \"hire_premium\", \"the same-day hire ladder above one coin a hand: the price of labour, hands leave at midnight\"),\n    (\"STRANDED AT THE END\", \"shed_units_now\", \"units still in the shed when the game ended: worth nothing\"),\n    (\"STRANDED AT THE END\", \"shed_value_now\", \"what they would have fetched at the final prices\"),\n    (\"STRANDED AT THE END\", \"animals_in_shed_now\", \"animals bought and never placed\"),\n    (\"STRANDED AT THE END\", \"seeds_unplanted_cost_now\", \"coins of seed never planted\"),\n    (\"MARKET\", \"sell_zero_fill\", \"a SELL order that delivered nothing, one slot burned\"),\n    (\"MARKET\", \"buy_zero_fill\", \"a BUY order that bought nothing: usually a broken financing chain\"),\n    (\"MARKET\", \"refused_buy_product\", \"BUY_PRODUCT units refused on funds or capacity\"),\n    (\"MARKET\", \"refused_buy_seed\", \"BUY_SEED units refused on funds\"),\n    (\"MARKET\", \"refused_buy_animal\", \"BUY_ANIMAL units refused on funds or capacity\"),\n    (\"MARKET\", \"refused_hire\", \"HIREs refused on funds or the unit cap\"),\n    (\"MARKET\", \"refused_buy_land\", \"BUY_LANDs refused on funds, or none left\"),\n)\n\n\ndef load_agent(path):\n    \"\"\"Import a Kaggle agent file and return its entry point.\n\n    Kaggle runs the LAST module-level callable taking one or two arguments,\n    which is why several public agents end with `agent = globals().pop('agent')`:\n    re-binding a name does not move it to the end of the module dict, so without\n    that line an inner helper can win.\n    \"\"\"\n    path = pathlib.Path(path)\n    tag = f\"_agent_{abs(hash(str(path)))}\"\n    spec = importlib.util.spec_from_file_location(tag, str(path))\n    if spec is None or spec.loader is None:\n        raise SystemExit(f\"cannot import {path}\")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[tag] = module\n    d = str(path.parent)\n    added = d not in sys.path\n    if added:\n        sys.path.insert(0, d)\n    try:\n        spec.loader.exec_module(module)\n    finally:\n        if added and sys.path and sys.path[0] == d:\n            sys.path.pop(0)\n        sys.modules.pop(tag, None)\n    fns = [v for v in vars(module).values()\n           if callable(v) and getattr(v, \"__module__\", \"\") == module.__name__\n           and getattr(v, \"__code__\", None) and v.__code__.co_argcount in (1, 2)]\n    if not fns:\n        raise SystemExit(f\"{path} declares no entry point\")\n    return fns[-1]\n\n\ndef observed_escapes(observation, seat, step):\n    \"\"\"The CONTROL, computed from the observation and nothing else.\n\n    An animal whose unfed counter already stands at one and that was not fed\n    today reaches two at tonight's refresh and is gone. Read at hour 23.\n    \"\"\"\n    if step % 24 != 23:\n        return 0\n    farms = observation.get(\"farms\") or []\n    if len(farms) <= seat:\n        return 0\n    n = 0\n    for row in (farms[seat].get(\"tiles\") or []):\n        for tile in row or []:\n            if (isinstance(tile, dict) and \"animal\" in tile\n                    and int(tile.get(\"consecutive_unfed\", 0)) >= 1\n                    and not tile.get(\"fed_today\")):\n                n += 1\n    return n\n\n\ndef play(agent, rival, seed, seat):\n    \"\"\"One episode. Returns the engine ledger and the control's escape count.\"\"\"\n    game = kagsim.Game(seed)\n    control = 0\n    for step in range(720):\n        if game.done:\n            break\n        mine = dict(game.observe(seat) or {})\n        mine[\"player\"] = seat\n        theirs = dict(game.observe(1 - seat) or {})\n        theirs[\"player\"] = 1 - seat\n        control += observed_escapes(mine, seat, step)\n        a, b = agent(mine), rival(theirs) if rival else PASS\n        game.step(a if seat == 0 else b, b if seat == 0 else a)\n    return (dict(game.telemetry(seat)), control,\n            game.reward(seat), game.reward(seat) - game.reward(1 - seat))\n\n\ndef _name(path):\n    \"\"\"Kaggle agents are all called main.py, so the directory is the name.\"\"\"\n    p = pathlib.Path(path)\n    return p.parent.name if p.stem == \"main\" else p.stem\n\n\ndef parse_seeds(text):\n    out = []\n    for part in text.split(\",\"):\n        part = part.strip()\n        if \"-\" in part:\n            lo, hi = part.split(\"-\")\n            out.extend(range(int(lo), int(hi) + 1))\n        elif part:\n            out.append(int(part))\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__.split(\"\\n\")[0])\n    ap.add_argument(\"--agent\", required=True, help=\"the agent under test\")\n    ap.add_argument(\"--rival\", default=None, help=\"a second agent file, or omit for an idle rival\")\n    ap.add_argument(\"--seeds\", default=\"1-16\", help=\"e.g. 1-16 or 3,7,11\")\n    ap.add_argument(\"--both-seats\", action=\"store_true\",\n                    help=\"play each seed from both seats; the market settles player 0 first\")\n    ap.add_argument(\"--require-zero-escapes\", action=\"store_true\", help=\"exit 1 on any loss\")\n    ap.add_argument(\"--json\", default=None, help=\"write the per-game rows here\")\n    a = ap.parse_args()\n\n    agent = load_agent(a.agent)\n    rival = load_agent(a.rival) if a.rival else None\n    seeds = parse_seeds(a.seeds)\n    seats = (0, 1) if a.both_seats else (0,)\n    label = f\"{_name(a.agent)} against {_name(a.rival) if a.rival else 'idle'}\"\n    print(f\"{label}: {len(seeds) * len(seats)} games, playing...\", flush=True)\n\n    rows, total, control = [], {}, 0\n    for seed in seeds:\n        for seat in seats:\n            tel, ctl, bank, margin = play(agent, rival, seed, seat)\n            control += ctl\n            rows.append({\"seed\": seed, \"seat\": seat, \"bank\": bank, \"margin\": margin,\n                         \"observed_escapes\": ctl,\n                         \"telemetry\": {k: (float(v) if isinstance(v, float) else int(v))\n                                       for k, v in tel.items()}})\n            for k, v in tel.items():\n                if isinstance(v, (int, float)) and k not in (\"escape_first_day\", \"dry_first_day\"):\n                    total[k] = total.get(k, 0) + v\n\n    n = len(rows)\n    engine = total.get(\"animals_escaped\", 0)\n    print(f\"\\n{label}: {n} games\\n\")\n    print(f\"escapes: {engine} (engine), {control} (observation control) \"\n          f\"{'AGREE' if engine == control else 'DISAGREE, trust neither until you know why'}\")\n    hit = [r for r in rows if r[\"telemetry\"][\"animals_escaped\"]]\n    first = [r[\"telemetry\"][\"escape_first_day\"] for r in hit\n             if r[\"telemetry\"][\"escape_first_day\"] >= 0]\n    print(f\"games losing an animal: {len(hit)} of {n}\"\n          + (f\", first on day {min(first)}\" if first else \"\"))\n    for r in sorted(hit, key=lambda r: -r[\"telemetry\"][\"escape_capital\"])[:8]:\n        t = r[\"telemetry\"]\n        print(f\"  seed {r['seed']:>5} seat {r['seat']}  {t['animals_escaped']} animal(s) \"\n              f\"from day {t['escape_first_day']}, {t['escape_capital']:,.0f} coins, \"\n              f\"feed_no_wheat {t['feed_no_wheat']}, bank {r['bank']:,.0f} \"\n              f\"margin {r['margin']:+,.0f}\")\n\n    print(\"\\nPER GAME, mean:\")\n    kind = None\n    for k, key, what in LEDGER:\n        if k != kind:\n            print(f\"\\n  {k}\")\n            kind = k\n        print(f\"    {key:<22} {total.get(key, 0) / n:>10,.2f}   {what}\")\n    print(f\"\\n  {'SILENT LOSS, coins':<22} {total.get('silent_loss_coins', 0) / n:>10,.0f}\"\n          f\"   {'units':<8} {total.get('silent_loss_units', 0) / n:>8,.1f}\")\n\n    if a.json:\n        pathlib.Path(a.json).write_text(json.dumps(rows, indent=1))\n        print(f\"\\nwrote {a.json}\")\n    if a.require_zero_escapes and engine:\n        print(f\"\\nFAIL: {engine:,} animal(s) escaped, \"\n              f\"{total.get('escape_capital', 0):,.0f} coins\")\n        return 1\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
import hashlib, pathlib
path = pathlib.Path('silent_leaks.py'); path.write_text(TOOL)
print(f'wrote {path} ({len(TOOL.splitlines())} lines, sha256 {hashlib.sha256(TOOL.encode()).hexdigest()[:12]})')
print()
print('\n'.join(TOOL.splitlines()[:36]))
print('    ...')


Three habits separate a counter from a measurement.

**Read the precursor and not only the loss**, because the loss is a post mortem and the precursor is a warning with twenty turns on it.
**Carry a second route to any number you intend to act on**, which is why the escapes are computed twice and why a disagreement reports neither.
**Read a nonzero count as a question**: the last-week animals in section 6 are a deliberate trade, and the undelivered units in section 5 are a habit.


<a id="s11"></a>
## 13. Links and credit

* the engine, the telemetry and the tool: [github.com/destbreso/kaggriculture-cppsim](https://github.com/destbreso/kaggriculture-cppsim)
* the agents measured here, kept updated: [Kaggriculture donor agents](https://www.kaggle.com/datasets/destbreso/kaggriculture-donor-agents-20260902)
* how the engine reaches 24,000 episodes a second and stays exact: [From 1 to 24k episodes a second](https://www.kaggle.com/code/destbreso/from-1-to-24k-episodes-a-second)
* the two pages this one continues: [X-ray your agent](https://www.kaggle.com/code/destbreso/x-ray-your-agent), one agent read from its replays, and [A macroeconomic x-ray of the meta](https://www.kaggle.com/code/destbreso/a-macroeconomic-x-ray-of-the-kaggriculture-meta), the field's economy read from the ladder

The board animations in section 1 follow the form of [Kaggriculture, Visualized: what every crop pays](https://www.kaggle.com/code/georgymamarin/kaggriculture-visualized-what-every-crop-pays) by Georgy Mamarin, whose guide plays a whole season on the board in six seconds; the drawing here is mine, the idea is his.

The telemetry layer, the tool, the audit and this census are mine.
The engine they sit in began as a port of the environment to C++ and carries its upstream credit in the repository, where it belongs and where it is exact.

Among the community agents measured here, the ones by Ahmed Berat Özer read zero escapes on every game, which matches what his notebooks set out to fix, and thirty-two of fifty-two authors are clean on the herd.
The instrument was pointed at my own live submission first, and it is the one that failed.
